# ML-03 · V5.00 FINAL FULL RESEARCH
## Экономические и инфраструктурные характеристики муниципалитетов и динамика населения

**Полный исследовательский Colab-ноутбук: четыре этапа, один сквозной индикатор выполнения.**

**Что проверяется.** Насколько наблюдаемые муниципальные социально-экономические и инфраструктурные показатели улучшают прогноз миграции и изменения численности населения относительно исторического метода; устойчивость по годам, регионам, размеру муниципалитетов, группам факторов и семействам моделей. Горизонты: **1 и 3 года**.

**Что исключено из научной программы.** ATT, DR-DID, event-study, причинные интерпретации, обучение пятилетних моделей, искусственная реконструкция муниципальных границ и площади. Невыполнимые пятилетние тесты включены только в журнал исключённого дизайна. Временная доступность и история ОКТМО проверяются как ограничения достоверности, а не подменяются неподтверждёнными сведениями.

**Источник.** Приватный GitHub `AntonDozhdikov/Demograf_sovetnik`, замороженный handoff стадии I (52 актива), + документируемые дополнительные разделы БДМО. Внимание: не все 52 массива имеют муниципальный статистический уровень. Вход: секрет Colab `TRIAD-RU-Colab` либо `GITHUB_TOKEN`; не записывается в результаты.

**Результаты.** Книга Excel, таблицы CSV, графики, полный исследовательский ZIP с ноутбуком и индексом, технический Evidence ZIP с Parquet/моделями, отдельные QA и манифесты. Значения метрик появляются **только после фактического запуска**.
 

# Этап I / IV. Источники, подготовка исследовательского стенда

#### Параметры запуска, замороженные горизонты и запрет причинной реконструкции.

In [1]:
# 01 · ML03 V5.00 · FULL по умолчанию, строгие межъячеечные контракты
CFG = {
    'MODE':'FULL',                    # 'SMOKE' для быстрой проверки / 'FULL' для исследования
    'PUBLISH_FULL_PASS':False,       # строгая научная публикация запрещена без прохождения FULL QA
    'REPO':'AntonDozhdikov/Demograf_sovetnik',
    'SOURCE_REF':'58473f995440daac295ca32532959f101bfa14ea',
    'HANDOFF':'handoff/07_v2',
    'SEGMENT':'stage2/ml03_factors',
    'VERSION':'stage2-ml3-v5.0.0-final',
    'WORKDIR':'/content/ml03_factors',
    'SEED':1729,
    'DOWNLOAD_ALL_STAGE1':True,     # все 52 SHA-verified Stage I в локальный cache; JOIN только подходящих единиц
    'MAX_SINGLE_ASSET_GB':8.0,
    'MIN_DISK_FREE_GB':6.0,
    'MAX_EVIDENCE_GB':1.9,
    'HORIZONS':[1,3],
    'TARGETS':['migration_per_1000','population_change_pct'],
    'FEATURE_PUBLICATION_LAG':1,
    'DEV_YEARS':2,
    'TEST_YEARS':2,
    'MIN_TRAIN_YEARS':3,
    'MIN_TRAIN_ROWS':120,
    'MIN_TEST_ROWS':40,
    'MIN_BOOTSTRAP_REGIONS':8,
    'MAX_FEATURES_PER_GROUP':12,
    'MAX_FEATURES_FROM_ONE_ASSET':5,
    'MAX_CANDIDATE_CODES_PER_ASSET':20,
    'MAX_CATALOG_CODES_PER_ASSET':None,  # не скрывать нерассмотренные коды
    'MAX_CODE_SLICES_PER_ASSET_DOMAIN':None,
    'MIN_ECON_DOMAINS':2,
    'MIN_SOCIAL_DOMAINS':2,
    'MIN_CANONICAL_TEST_YEARS':2,
    'MIN_SPATIAL_MATCHES':30,
    'SPATIAL_PUBLICATION_YEAR':None,
    'SPATIAL_PUBLICATION_EVIDENCE':'',
    'SPATIAL_BOUNDARY_CROSSWALK_VERIFIED':False,
    'SPATIAL_BOUNDARY_CROSSWALK_EVIDENCE':'',
    'MIN_SPATIAL_CANONICAL_TEST_ROWS':40,
    'MAX_FUTURE_MUTATION_ROWS':30,
    'MIN_BOUNDARY_SENSITIVITY_REGIONS':8,
    'MIN_FE_ENTITY_OBSERVATIONS':3,
    'MIN_INDICATOR_MUNICIPALITIES':25,
    'MIN_INDICATOR_OBS_TRAIN':100,
    'MIN_INDICATOR_YEARS_TRAIN':2,
    'MIN_RECENT_MUNICIPALITY_YEARS':70,
    'CATALOG_RECENT_WINDOW':5,
    'MAX_DIMENSIONAL_SCANS':1500,  # если превышено — NOT_EXHAUSTIVE, не научный PASS
    'MAX_DIMENSIONAL_SLICES_PER_CODE':2000,  # размерные комбинации проверяются SQL-фильтром до лимита,
    'MIN_MUNICIPAL_YEAR_SUPPORT':60,
    'TARGET_LABEL_AVAILABILITY_LAG':1,
    'MAX_NDFL_SLICE_VALUES':400,  # ограничение числа тестируемых допустимых разрезов, не отказ от всего кода
    'FAIL_FAST_SQL':True,
    'SQL_PREFLIGHT_REAL_CODES':True,
    'EXPLORE_SECONDARY_MIGRATION_WINDOW':True,
    'PRIMARY_TEST_WINDOWS':{'migration_per_1000':[2021,2022],'population_change_pct':[2022,2023]},
    'MAX_FEATURE_STALENESS_YEARS':3,
    'ENABLE_REGIONAL_ECONOMIC_CONTEXT':True,
    'REGIONAL_ECONOMIC_MAX_FEATURES':5,
    'REGIONAL_ECONOMIC_MIN_REGIONS':10,
    'REGIONAL_CONTEXT_IS_NOT_MUNICIPAL_G1':True,
    'MIN_DISTINCT_EVALUATION_YEARS':2,
    'MAX_CONFLICT_SHARE':0.002,
    'CORRELATED_PROXY_CUTOFF':0.95,
    'MODEL_FAMILIES':['RIDGE','HGB','CATBOOST','LIGHTGBM','ELASTICNET'],
    'SEEDS_FINAL':[1729,1733,1747,1753,1777,1783,1799,1801,1811,1823],
    'SEED_REPETITION_REPORT':True,
    'ENABLE_CHECKPOINTS':True,
    'BOOTSTRAPS':499,
    'MAX_IMPORTANCE_ROWS':1200,
    'MAX_IMPORTANCE_FEATURES':22,
    'MAX_SHAP_ROWS':260,
    'SMOKE_MUNICIPALITIES':500,
    'DUCKDB_MEMORY':'6GB',
    'DUCKDB_THREADS':4,
    'EXCEL_MAX_ROWS_PER_SHEET':110000,
    # Source enrichment: additive official BDMO sections outside the immutable Stage I 52.
    'GITHUB_COLAB_SECRET':'TRIAD-RU-Colab',  # имя секрета, не значение; резерв — GITHUB_TOKEN
    'RUN_CAUSAL_EXPERIMENT':False, # В V5 причинные оценки полностью исключены
    'VERIFIED_BOUNDARY_HISTORY_INPUT':'', # optional path, source/years must be independently certified
    'VERIFIED_PUBLICATION_VINTAGES_INPUT':'', # optional indicator/year public-release receipts
    'VERIFIED_SPATIAL_VINTAGE_INPUT':'', # optional historical geocodes and geometry crosswalk  # исторический источник, без реконструкции
    'G5_RETROSPECTIVE_EXPLORATORY':True,  # без подмены прогноза прошлым знанием
    'H5_CALENDAR_AUDIT':True,  # исключительно архив исключённых вариантов; моделирование H5 запрещено
    'SOURCE_PUBLICATION_AUDIT':True,  # даты публикации из документированных источников, без домыслов
    'ENABLE_EXTERNAL_BDMO':True,
    'BDMO_EXTERNAL_MAX_SOURCE_GB':12.0,
    'BDMO_EXTERNAL_MAX_UNPACKED_GB':20.0,
    'BDMO_EXTERNAL_MAX_MEMBERS_PER_SECTION':150,
    # Research-grade explainable ensemble: selected solely by chronological DEV.
    'ADVANCED_ENABLE':True,
    'ADVANCED_REQUIRE_ALL_FAMILIES':True,
    'ADVANCED_N_JOBS':4,
    'ADVANCED_MAX_CANDIDATE_FAMILIES':6,
    'ADVANCED_ENSEMBLE_K':3,
    'ADVANCED_EXPORT_MEMBERS':True,
    'ADVANCED_WEIGHT_RIDGE':0.02,
    'ADVANCED_EXPLAIN_ENABLE':True,
    'ADVANCED_SHAP_ROWS':180,
    'ADVANCED_BACKGROUND_ROWS':120,
    'ADVANCED_SHAP_TOLERANCE':3e-4,
    'ADVANCED_EXPLAIN_BOOTSTRAPS':299,
    'ADVANCED_PERM_ROWS':700,
    'ADVANCED_PERM_REPEATS':3,
    'ADVANCED_PDP_TRAIN_ROWS':450,
    'ADVANCED_PDP_TOP':4,
    'ADVANCED_PDP_BINS':13,
            'ARCHIVES_INCLUDE_RAW_SOURCE_CACHE':False, # raw Stage I cache can be very large
    'AUTO_COLAB_DOWNLOAD':True,  # только компактный научный архив; тяжёлый не скачивается
    'GITHUB_TECHNICAL_PUBLISH':True, # тяжелый Evidence + модели только в приватный Release репозитория
    'RESEARCH_ARCHIVE_MAX_MB':45,   # лимит научного ZIP для быстрой загрузки
    'RESEARCH_LOCAL_SHAP_SAMPLE':1500,
    'GITHUB_VERIFY_ASSET_SHA_IF_AVAILABLE':True,
    'REFERENCE_V4_ARCHIVE_LOCAL':'',  # необязательный путь к локальному техническому Evidence V4.02/V4.03 для парных строк
    'V403_REFERENCE_GITHUB':True, # 302 MB технического архива V4.03, только для сравнения фактических ключей
    'V403_REFERENCE_TAG':'ml03-v403-results-20261122T103412Z',
    'V403_REFERENCE_SHA256':'64b2e83d3d7a371141f67adcca0dae3caa2b2416ee8981c5c1f159f69bc8f041',
    'MAX_REFERENCE_ARCHIVE_MB':420,
    'ENABLE_STAGE1_BUDGET_BALANCE':True,
    'ENABLE_STAGE1_ROAD_DISCOVERY':True,
    'V5_H5_EXCLUDED_NO_FIT':True,
    'STRICT_CROSS_VERSION_ROW_QA':False,  # истинное сравнение ключей выполняется, если доступен архив предсказаний
    'SCIENTIFIC_RELEASE_ALLOWED':True,  # только подтверждённое прогнозное исследование; без причинных выводов
}
assert CFG['MODE'] in ('SMOKE','FULL')
assert all(h>0 and isinstance(h,int) for h in CFG['HORIZONS'])
assert CFG['FEATURE_PUBLICATION_LAG']>=1
# Версия и режим отображаются одним общим индикатором в следующей ячейке.


#### Среда Colab, зависимости, единый интерактивный прогресс.

In [2]:
# 02 · ENVIRONMENT, MINIMAL COLAB DEPENDENCIES, AUDIT LOG
import sys, os, shutil, platform, subprocess, importlib.util, json, time, hashlib, re, math, itertools, warnings, zipfile, io, base64, traceback, statistics, datetime, collections, urllib.parse, ast, zlib, pprint
from pathlib import Path
WORK=Path(CFG['WORKDIR']);WORK.mkdir(parents=True,exist_ok=True)
for d in ('cache','evidence','charts','models'): (WORK/d).mkdir(exist_ok=True)
usage=shutil.disk_usage(str(WORK))
# Среда и свободное место пишутся в технический журнал, а не в вывод ноутбука.
ensure_disk=usage.free/2**30>=CFG['MIN_DISK_FREE_GB']
if not ensure_disk:raise RuntimeError('INSUFFICIENT_COLAB_DISK_CAPACITY')
NEEDED={'duckdb':'duckdb>=1.1,<2','pyarrow':'pyarrow>=17,<24','pandas':'pandas>=2.1,<3',
        'numpy':'numpy>=1.26,<3','scipy':'scipy>=1.11,<2','sklearn':'scikit-learn>=1.4,<2',
        'matplotlib':'matplotlib>=3.8,<4','openpyxl':'openpyxl>=3.1,<4',
        'nbformat':'nbformat>=5.9,<6','requests':'requests>=2.31,<3',
        'ipywidgets':'ipywidgets>=8,<9','joblib':'joblib>=1.3,<2'}
OPTIONAL={'catboost':'catboost>=1.2,<2','lightgbm':'lightgbm>=4,<5',
          'linearmodels':'linearmodels>=6,<9','shap':'shap>=0.46,<1','xgboost':'xgboost>=2.0,<4'}
missing=[v for k,v in NEEDED.items() if importlib.util.find_spec(k) is None]
if missing:
    result=subprocess.run([sys.executable,'-m','pip','-q','install',*missing],capture_output=True,text=True)
    if result.returncode:raise RuntimeError('MANDATORY_PIP_DEPENDENCY_FAILURE')
optional_missing=[v for k,v in OPTIONAL.items() if importlib.util.find_spec(k) is None]
if optional_missing:
    result=subprocess.run([sys.executable,'-m','pip','-q','install',*optional_missing],capture_output=True,text=True)
    if result.returncode:
        if CFG['MODE']=='FULL' and CFG['ADVANCED_ENABLE'] and CFG['ADVANCED_REQUIRE_ALL_FAMILIES']:
            raise RuntimeError('ADVANCED_FULL_DEPENDENCY_INSTALL_FAILED: '+result.stderr[-1000:])
        pass  # Уточнение о недоступности — в статусе экспериментов, без засорения интерфейса.
import numpy as np, pandas as pd, duckdb, requests, pyarrow as pa, pyarrow.parquet as pq
import matplotlib;matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge,ElasticNet
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error
from openpyxl import Workbook,load_workbook
from openpyxl.styles import Font,PatternFill,Alignment
from openpyxl.utils import get_column_letter
import nbformat,joblib
pd.options.mode.copy_on_write=True
STATUS={};WARNINGS=[];TRACE=WORK/'ml03_technical.log';START=time.monotonic()
if TRACE.exists():TRACE.unlink()
def log(msg,severity='INFO'):
    text=str(msg)
    text=re.sub(r'(?i)(?:gh'+'p_|github'+'_pat_)[a-z0-9_]+','[REDACTED]',text)
    with open(TRACE,'a',encoding='utf-8') as f:
        f.write(datetime.datetime.now(datetime.timezone.utc).isoformat()+f' [{severity}] '+text+'\n')
def mark(name,status,reason=''):
    STATUS[name]={'status':status,'reason':str(reason)[:480]}
    log(name+': '+status+': '+str(reason)[:480])
    if status not in ('PASS','NOT_FEASIBLE','CAUSAL_NOT_IDENTIFIED'):
        WARNINGS.append({'step':name,'status':status,'reason':str(reason)[:480]})
def ensure(ok,msg):
    if not ok:raise AssertionError(msg)
def digest(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
log('Environment OK: Python='+sys.version.split()[0]+' CPU='+str(os.cpu_count())+' free_disk_gib='+str(round(usage.free/2**30,1)))
# Fail before expensive work if the required runtime is not available.
ensure(pd.__version__.split('.')[0] in ('2',), 'PANDAS_MAJOR_VERSION_NOT_TESTED') if 'ensure' in globals() else None



# ML-03 V5.00: РОВНО ОДНО визуальное окно прогресса (ipywidgets.HTML).
# Виджет отображается один раз в cell 02. Межъячеечные обновления меняют его value,
# не создавая новых FloatProgress, tqdm, display() или многократных выводов Colab.
from IPython.display import display as _single_display
try:
    import ipywidgets as _widgets
except ImportError:
    _widgets=None
_PHASE_BOUNDS={'ENV':(0,26),'AUTH':(26,47),'INPUT':(47,181),'BDMO':(181,208),'INVENTORY':(208,239),'SCAN':(239,307),'FACTORS':(307,333),'PANEL':(333,375),'TASKS':(375,398),'DEV':(398,483),'TRAIN':(483,594),'ADVANCED':(594,698),'STATS':(698,745),'SHAP':(745,802),'SENSITIVITY':(802,848),'EXCEL':(848,866),'EVIDENCE':(866,907),'FINAL':(907,925),'ARCHIVE_RESEARCH':(925,953),'ARCHIVE_TECH':(953,992),'DELIVERY':(992,1000)}
_PHASE_TITLES={'ENV':'I/IV · Среда','AUTH':'I/IV · GitHub и каталог','INPUT':'I/IV · 52 источника','BDMO':'I/IV · Дополнительные БДМО','INVENTORY':'I/IV · Инвентаризация','SCAN':'I/IV · Предметный отбор','FACTORS':'I/IV · Бюджет, дороги и классификация','PANEL':'I/IV · Панель','TASKS':'II/IV · Дизайн и исключённый H5','DEV':'II/IV · Подбор моделей DEV','TRAIN':'II/IV · Обучение и тест','ADVANCED':'II/IV · Ансамбли','STATS':'III/IV · Парная статистика','SHAP':'III/IV · SHAP и устойчивость','SENSITIVITY':'III/IV · Утечки, лаги, ретроспективы','EXCEL':'III/IV · Книга Excel','EVIDENCE':'IV/IV · Evidence','FINAL':'IV/IV · Код и воспроизводимость','ARCHIVE_RESEARCH':'IV/IV · Исследовательский ZIP','ARCHIVE_TECH':'IV/IV · GitHub Evidence','DELIVERY':'IV/IV · Контроль итогов'}
class _OneProgress:
    def __init__(self):
        self.n=0;self.total=1000;self.desc='Инициализация';self.postfix='';self._last=0.;self._last_percent=-1
        self.widget=_widgets.HTML(value=self._html()) if _widgets else None
        if self.widget is not None:_single_display(self.widget)
    def _html(self):
        import html
        pct=int(round(100*self.n/self.total));name=html.escape(self.desc)
        sub=html.escape(self.postfix)
        return ('<div style="max-width:920px;padding:15px 17px;border:1px solid #d4dce5;'
          'border-radius:13px;background:#0b1c31;color:#f5f9ff;font:14px Arial,sans-serif">'
          '<div style="display:flex;justify-content:space-between;gap:14px;margin-bottom:10px">'
          '<b>ML-03 V5.00 · Объединённый запуск · '+name+'</b><b>'+str(pct)+'%</b></div>'
          '<div role="progressbar" aria-valuenow="'+str(pct)+'" aria-valuemin="0" aria-valuemax="100"'
          ' style="height:15px;border-radius:9px;background:#35445b;overflow:hidden">'
          '<div style="width:'+str(pct)+'%;height:100%;background:linear-gradient(90deg,#21a5af,#61d8b0);'
          'border-radius:9px"></div></div><div style="margin-top:8px;color:#a8c5df;font-size:12px">'
          +sub+'</div></div>')
    def update(self,increment):
        self.n=min(self.total,max(0,self.n+int(increment)));self.refresh()
    def set_description_str(self,s,refresh=False):self.desc=str(s);self.refresh(force=refresh)
    def set_postfix_str(self,s,refresh=False):self.postfix=str(s);self.refresh(force=refresh)
    def refresh(self,force=False):
        now=time.monotonic();pct=int(100*self.n/self.total)
        if not force and now-self._last<.3 and pct==self._last_percent:return
        self._last=now;self._last_percent=pct
        if self.widget is not None:self.widget.value=self._html()
        elif force or pct%10==0:log(f'PROGRESS {pct}% {self.desc} {self.postfix}')
    def close(self):self.refresh(force=True)
_PROGRESS_BAR=_OneProgress();_PROGRESS_LAST=None
PROGRESS_LEDGER=[]
def progress(phase,done=1,total=1,detail=''):
    global _PROGRESS_LAST
    if phase not in _PHASE_BOUNDS:raise KeyError('PROGRESS_UNKNOWN_PHASE_'+str(phase))
    start,end=_PHASE_BOUNDS[phase];total=max(1,int(total));done=max(0,min(int(done),total))
    goal=start+int((end-start)*done/total)
    goal=max(goal,_PROGRESS_BAR.n)  # монотонная шкала, определяется фактическим выполнением этапов
    detail=str(detail).replace('\n',' ')[:85]
    state=(_PHASE_TITLES[phase],done,total,detail)
    if state!=_PROGRESS_LAST:
        _PROGRESS_BAR.desc=state[0];_PROGRESS_BAR.postfix=f'{done}/{total} · {detail}'
        _PROGRESS_LAST=state
    if goal>_PROGRESS_BAR.n:_PROGRESS_BAR.update(goal-_PROGRESS_BAR.n)
    else:_PROGRESS_BAR.refresh()
    if done==total:PROGRESS_LEDGER.append({'phase':phase,'completed':done,'total':total,'bar_value':_PROGRESS_BAR.n})
    return None

def quiet_print(*args,**kwargs):
    # Отладочные сообщения доступны в ml03_technical.log, но не создают сотни строк в Colab.
    try:log(' '.join(str(a) for a in args))
    except Exception:pass

# Every warning goes to the technical log, not the live notebook output. Exceptions remain visible.
def _ml03_warning_to_log(message, category, filename, lineno, file=None, line=None):
    log(f'{category.__name__}: {message} [{Path(filename).name}:{lineno}]','WARN')
warnings.showwarning=_ml03_warning_to_log

progress('ENV',1,1,'Среда и зависимости подготовлены')

# Не маскируем научные ошибки: подавляется только многословный pandas PerformanceWarning.
warnings.filterwarnings('ignore', message='DataFrame is highly fragmented.*', category=pd.errors.PerformanceWarning)


#### Доступ к приватному GitHub (секрет не экспортируется).

In [3]:
# 03 · SECURE AUTH / READ-ONLY PREFLIGHT
from getpass import getpass
# Colab secret name is TRIAD-RU-Colab; never write its VALUE to logs, ZIP or notebook.
_TOKEN_KEY=str(CFG.get('GITHUB_COLAB_SECRET','TRIAD-RU-Colab'))
TOKEN=''
try:
    from google.colab import userdata
    for _secret in (_TOKEN_KEY,'GITHUB_TOKEN'):
        try:TOKEN=str(userdata.get(_secret) or '').strip()
        except (KeyError,ValueError,Exception):TOKEN=''
        if TOKEN:break
except ImportError:pass
if not TOKEN:
    TOKEN=(os.environ.get('TRIAD_RU_COLAB') or os.environ.get('GITHUB_TOKEN') or '').strip()
if not TOKEN:TOKEN=getpass("GitHub token (скрытый ввод, если секрет Colab не найден): ").strip()
if not TOKEN: raise RuntimeError("BLOCKED_AUTH: требуется токен приватного GitHub")
API=f"https://api.github.com/repos/{CFG['REPO']}"
SESSION=requests.Session()
SESSION.headers.update({"Authorization":"Bearer "+TOKEN,"Accept":"application/vnd.github+json",
                        "X-GitHub-Api-Version":"2022-11-28","User-Agent":"Demograf-sovetnik-ML03"})
# В трассировки не передаются объекты HTTP-ошибок с заголовками/URL авторизации.
def api(method,endpoint,*,expected=(200,),**kwargs):
    url=endpoint if endpoint.startswith('https://') else API+endpoint
    r=SESSION.request(method,url,timeout=kwargs.pop('timeout',(20,90)),**kwargs)
    if r.status_code not in expected:
        code="ACCESS_DENIED" if r.status_code in (401,403) else "NOT_FOUND_OR_NO_RIGHTS" if r.status_code==404 else "HTTP_FAILURE"
        raise RuntimeError(f"{code}: GitHub HTTP {r.status_code}; resource={endpoint.split('?')[0][:130]}")
    return r
repo_obj=api('GET','').json()
ensure(repo_obj.get('private') is True,"Ожидался приватный репозиторий; настройки не меняем")
source_sha=api('GET',f"/commits/{CFG['SOURCE_REF']}").json()['sha']
SOURCE_COMMIT_SHA=source_sha  # CFG не изменяется после ячейки 01: воспроизводимый источник

def repo_bytes(path):
    obj=api('GET',f"/contents/{urllib.parse.quote(path,safe='/')}?ref={source_sha}").json()
    if obj.get('content'):
        return base64.b64decode(obj['content'])
    # Для крупных git blobs получаем Git blob, не доверяя download_url с временными параметрами.
    blob=api('GET','/git/blobs/'+obj['sha']).json()
    if blob.get('encoding') != 'base64': raise RuntimeError("UNKNOWN_GIT_BLOB_ENCODING")
    return base64.b64decode(blob['content'])
RECEIPT=json.loads(repo_bytes(CFG['HANDOFF']+'/publication_receipt.json'))
raw_index=repo_bytes(CFG['HANDOFF']+'/dataset_index.json')
receipt_index=next((x for x in RECEIPT['files'] if x['path'].endswith('/dataset_index.json')),None)
ensure(receipt_index is not None and hashlib.sha256(raw_index).hexdigest()==receipt_index['sha256'],
       'HANDOFF_INDEX_SHA_MISMATCH')
INDEX=json.loads(raw_index); ASSETS=INDEX['assets']
ensure(len(ASSETS)==52 and len({a['asset_id'] for a in ASSETS})==len(ASSETS),
       'Нарушен контракт индекса: ожидаются 52 уникальных массива')
ensure(all(a['sha256_verified'] and a['locator']['type'] in ('git','release') for a in ASSETS),
       'Индекс содержит неверифицированные/неизвестные источники')
mark('PREFLIGHT_CATALOG','PASS',f"52 актива; immutable source SHA {source_sha[:12]}")
quiet_print(f'GitHub READ: OK; handoff={INDEX["version"]}; sources={len(ASSETS)}; SHA={source_sha[:12]}')
progress('AUTH',1,1,'Каталог и commit подтверждены')

#### 52 Stage I: контроль SHA-256, статистических единиц и муниципальной панели.

In [4]:
# 04 · V3.00 · VERIFIED 52 SOURCE INVENTORY, MUNICIPAL/REGIONAL/ORGANIZATION GRAIN
import tempfile
source_receipts=[]; DOWNLOAD_QA=[]; LOCAL_ASSETS={}
def safe_remote_download(asset):
    """SHA-verified cache; GitHub git/release; no untrusted URL in logs."""
    name=hashlib.sha256(asset['asset_id'].encode()).hexdigest()[:22]+'.parquet'
    dst=WORK/'cache'/name
    if dst.exists() and digest(dst)==asset['sha256']:
        if asset['asset_id'] not in {x['asset_id'] for x in source_receipts}:
            source_receipts.append({'asset_id':asset['asset_id'],'sha256':asset['sha256'],'local_bytes':dst.stat().st_size,'source_ref':source_sha,'cache_hit':True})
        LOCAL_ASSETS[asset['asset_id']]=str(dst)
        return dst
    dst.unlink(missing_ok=True); tmp=dst.with_suffix('.part');tmp.unlink(missing_ok=True)
    loc=asset['locator']; total=0
    try:
        if loc['type']=='git':
            path=urllib.parse.quote(loc['path'],safe='/')
            obj=api('GET',f'/contents/{path}?ref={source_sha}').json()
            blob=api('GET','/git/blobs/'+obj['sha'],timeout=(30,240)).json()
            ensure(blob.get('encoding')=='base64','GIT_BLOB_ENCODING_UNSUPPORTED')
            buf=base64.b64decode(blob['content'])
            with open(tmp,'wb') as f:f.write(buf)
            total=len(buf)
        else:
            u=API+'/releases/assets/'+str(int(loc['asset_id']))
            r=SESSION.get(u,headers={'Accept':'application/octet-stream'},stream=True,timeout=(30,1200))
            if r.status_code!=200:
                r.close();raise RuntimeError('RELEASE_ACCESS_HTTP_'+str(r.status_code))
            if 'json' in r.headers.get('Content-Type','').lower():
                j=r.json();r.close();download=j.get('browser_download_url','')
                host=urllib.parse.urlparse(download)
                ensure(host.scheme=='https' and host.hostname in ('github.com','api.github.com'), 'INVALID_RELEASE_HOST')
                r=SESSION.get(download,stream=True,timeout=(30,1200))
                if r.status_code!=200 or 'json' in r.headers.get('Content-Type','').lower():
                    code=r.status_code;r.close();raise RuntimeError('RELEASE_BINARY_HTTP_'+str(code))
            try:
                with open(tmp,'wb') as f:
                    for block in r.iter_content(4*1024*1024):
                        if block:
                            total+=len(block)
                            ensure(total<=int(CFG['MAX_SINGLE_ASSET_GB']*2**30),'ASSET_LIMIT_EXCEEDED')
                            f.write(block)
            finally:r.close()
        ensure(digest(tmp)==asset['sha256'],'INPUT_SHA_MISMATCH_'+asset['asset_id'])
        tmp.replace(dst);LOCAL_ASSETS[asset['asset_id']]=str(dst)
        source_receipts.append({'asset_id':asset['asset_id'],'sha256':asset['sha256'],'local_bytes':dst.stat().st_size,'source_ref':source_sha,'cache_hit':False})
        return dst
    except Exception:tmp.unlink(missing_ok=True);raise

def get_asset(aid):
    a=next((v for v in ASSETS if v['asset_id']==aid),None)
    if a is None:raise KeyError('UNKNOWN_HANDOFF_ASSET '+aid)
    return a

SOURCE_CLASSES={
 'G1':['02::labor','02::socio1','02::socio2','02::socio3','02::5ndfl','02::7ndfl'],
 'G2':['02::housing','02::construction','06::BDMO6::BDMO6_member00::full'],
 'G3':['02::education','02::social'],
 'G4':['02::budget'],
 'G5':['06::TRANSPORT','06::D04::D04_member00::spatial_reference::derived']}
def source_group(a):
    return next((g for g,pp in SOURCE_CLASSES.items() if any(a['asset_id'].startswith(p) for p in pp)),'CONTEXT_ONLY')
BASE_ASSET='01::0'
# Дополнительный обход всего реестра Stage I: длинные муниципальные наборы независимо от
# папки/источника; запрещено превращать региональные и организационные записи в ОКТМО.
SOURCE_ADAPTER_AUDIT=[]
ECONOMY_MANIFEST=json.loads(repo_bytes('manifests/02_economy_full_v1.json'))
ECONOMY_SOURCE_MANIFEST_QA=pd.DataFrame([{'source_key':k,'source':v.get('source'),
 'rows_in_manifest':v.get('rows'),'column_count':len(v.get('columns') or []),
 'has_object_level':('object_level' in (v.get('columns') or [])),
 'has_municipal_key':bool({'oktmo','object_oktmo'}&set(v.get('columns') or [])),
 'report_grain':'REQUIRES_CONTENT_VALIDATION' if 'object_level' in (v.get('columns') or []) else 'MUNICIPAL_LONG_CANDIDATE' if 'oktmo' in (v.get('columns') or []) else 'UNKNOWN'}
 for k,v in ECONOMY_MANIFEST.get('files',{}).items()])

def eligible_municipal_long_schema(a):
    cc=set(a.get('columns_manifest') or [])
    ok=('oktmo' in cc or 'object_oktmo' in cc) and {'indicator_code','indicator_name','indicator_value','year'}.issubset(cc)
    return bool(ok)

def classify_asset_for_scan(a):
    declared=source_group(a)
    if declared!='CONTEXT_ONLY':return declared
    if eligible_municipal_long_schema(a):return 'DISCOVERY_MULTIDOMAIN'
    return 'CONTEXT_ONLY'

FACTOR_ASSETS=[a for a in ASSETS if a['asset_id']!='01::0' and
               classify_asset_for_scan(a)!='CONTEXT_ONLY']
# An education organization's OKTMO is a LOCATION OF ORGANIZATION, not proof of
# municipal total or municipal population coverage. Keep a full exclusion receipt.
ORGANIZATION_GRAIN_PREFIXES=('04::E02::','05::E04::','05::E05::')
ORGANIZATION_GRAIN_EXCLUSIONS=[{'asset_id':a['asset_id'],
    'reason':'ORGANIZATION_LEVEL_NOT_MUNICIPAL_AGGREGATE; NO_VALIDATED_ORGANIZATION_TO_MUNICIPAL_CONCORDANCE',
    'status':'EXPLICITLY_EXCLUDED_FROM_MUNICIPAL_MODEL'} for a in ASSETS
    if a['asset_id'].startswith(ORGANIZATION_GRAIN_PREFIXES)]
UNADAPTED_MUNICIPAL_WIDE=[a['asset_id'] for a in ASSETS
 if {'oktmo','year'}.issubset(set(a.get('columns_manifest') or [])) and
 not eligible_municipal_long_schema(a) and a['asset_id']!='01::0'
 and not a['asset_id'].startswith(ORGANIZATION_GRAIN_PREFIXES)]
SOURCE_ADAPTER_AUDIT=[{'asset_id':a['asset_id'],'declared_group':source_group(a),
 'scan_group':classify_asset_for_scan(a),'schema_type':('LONG_MUNICIPAL' if eligible_municipal_long_schema(a)
 else 'NON_MUNICIPAL_OR_WIDE_UNVERIFIED'),
 'has_oktmo':int(bool({'oktmo','object_oktmo'}&set(a.get('columns_manifest') or []))),
 'decision':('SCAN' if a in FACTOR_ASSETS else 'INDEX_ONLY_NO_VALIDATED_MUNICIPAL_GRAIN')}
 for a in ASSETS]

DOWNLOAD_ASSETS=(ASSETS if (CFG['MODE']=='FULL' and CFG['DOWNLOAD_ALL_STAGE1']) else
    [get_asset(BASE_ASSET)]+FACTOR_ASSETS)
if CFG['MODE']=='SMOKE':
    allow={'01::0','02::5ndfl::00::00','02::7ndfl::00::00',
           '02::housing::00::00','02::education::00::00','02::budget::00::00',
           '02::labor::00::00','02::social::00::00','02::construction::00::00',
           '02::socio1::00::00','02::socio2::00::00','02::socio3::00::00',
           '06::D04::D04_member00::spatial_reference::derived'}
    DOWNLOAD_ASSETS=[a for a in DOWNLOAD_ASSETS if a['asset_id'] in allow]
progress('INPUT',0,len(DOWNLOAD_ASSETS),'Сверка файлов с SHA-256')
input_map=pd.DataFrame([{'asset_id':a['asset_id'],'source':a['source_id'],
 'rows_manifest':a['rows_actual'],'group':source_group(a),
 'storage':a['locator']['type'],'status':'INDEXED','sha256':a['sha256']} for a in ASSETS])

# All 52 assets are indexed, including 03-05 (not misinterpreted as municipal observations).
# FULL downloads every asset by default, while only validated relevant sources enter ML-03 JOINs.
for ai,a in enumerate(DOWNLOAD_ASSETS,1):
    try:
        p=safe_remote_download(a)
        DOWNLOAD_QA.append({'asset_id':a['asset_id'],'source_group':source_group(a),'status':'SHA_PASS',
                             'bytes':p.stat().st_size,'error':''})
    except Exception as ex:
        DOWNLOAD_QA.append({'asset_id':a['asset_id'],'source_group':source_group(a),'status':'ERROR',
                             'bytes':0,'error':type(ex).__name__+': '+str(ex)[:120]})
        log('download '+a['asset_id']+': '+type(ex).__name__,'WARN')
    progress('INPUT',ai,len(DOWNLOAD_ASSETS),f'SHA: {sum(v["status"]=="SHA_PASS" for v in DOWNLOAD_QA)} из {len(DOWNLOAD_ASSETS)}')
DOWNLOAD_QA=pd.DataFrame(DOWNLOAD_QA)
required=['01::0']+[a['asset_id'] for a in FACTOR_ASSETS if a in DOWNLOAD_ASSETS]
missing=[aid for aid in required if aid not in LOCAL_ASSETS]
ensure('01::0' in LOCAL_ASSETS,'BLOCKED_INPUT_D01_NOT_AVAILABLE')
mark('PASS_INPUT','PASS' if not missing else 'FAIL',
     f'stage1 available={len(LOCAL_ASSETS)}/{len(DOWNLOAD_ASSETS)}; required missing={len(missing)}')
mark('ALL_STAGE1_DOWNLOAD','PASS' if CFG['MODE']=='FULL' and CFG['DOWNLOAD_ALL_STAGE1'] and len(LOCAL_ASSETS)==len(ASSETS) else 'PARTIAL',
     f'SHA downloaded={len(LOCAL_ASSETS)}/{len(ASSETS)}; includes independent education sources without ML05 JOIN')
if CFG['MODE']=='FULL' and CFG['DOWNLOAD_ALL_STAGE1']:
    ensure(len(LOCAL_ASSETS)==len(ASSETS),
           'FULL_STAGE1_INCOMPLETE_DOWNLOADS; review DOWNLOAD_QA and auth/release assets')

input_map['status']=input_map['asset_id'].map({r['asset_id']:'SHA_VERIFIED' for r in source_receipts}).fillna('INDEX_ONLY_OR_UNAVAILABLE')
STAGE1_PHYSICAL_AUDIT=[]
for a in ASSETS:
    aid=a['asset_id'];p=LOCAL_ASSETS.get(aid)
    if not p:
        STAGE1_PHYSICAL_AUDIT.append({'asset_id':aid,'status':'INDEXED_NOT_DOWNLOADED',
          'expected_rows':a.get('rows_actual'),'actual_rows':None,'metadata_columns':None,'sha_verified':False})
        continue
    try:
        pf=pq.ParquetFile(p);nr=pf.metadata.num_rows;nc=len(pf.schema.names)
        schema_columns=set(pf.schema.names);manifest_columns=set(a.get('columns_manifest') or [])
        ok=nr==a.get('rows_actual') and (not manifest_columns or manifest_columns==schema_columns)
        STAGE1_PHYSICAL_AUDIT.append({'asset_id':aid,'status':'PASS' if ok else 'SCHEMA_OR_ROWS_MISMATCH',
          'expected_rows':a.get('rows_actual'),'actual_rows':nr,'metadata_columns':nc,'sha_verified':True,
          'schema_columns_missing':','.join(sorted(manifest_columns-schema_columns))})
    except Exception as exc:
        STAGE1_PHYSICAL_AUDIT.append({'asset_id':aid,'status':'PARQUET_METADATA_ERROR',
          'expected_rows':a.get('rows_actual'),'actual_rows':None,'metadata_columns':None,
          'sha_verified':True,'exception_type':type(exc).__name__})
STAGE1_PHYSICAL_AUDIT=pd.DataFrame(STAGE1_PHYSICAL_AUDIT)
physical_errors=STAGE1_PHYSICAL_AUDIT.status.isin(['SCHEMA_OR_ROWS_MISMATCH','PARQUET_METADATA_ERROR']).sum()
mark('STAGE1_PHYSICAL_AUDIT','PASS' if physical_errors==0 else 'FAIL',
  f'{len(STAGE1_PHYSICAL_AUDIT)} registered assets; {len(LOCAL_ASSETS)} physical reads; metadata mismatches={int(physical_errors)}')
if physical_errors and CFG['MODE']=='FULL':raise RuntimeError('STAGE1_PHYSICAL_AUDIT_FAILED')

con=duckdb.connect(str(WORK/'ml03.duckdb'))
con.execute('SET enable_progress_bar = false')  # единый индикатор — только _PROGRESS_BAR
con.execute(f"SET memory_limit='{CFG['DUCKDB_MEMORY']}'");con.execute(f"SET threads={int(CFG['DUCKDB_THREADS'])}")
base=pd.read_parquet(LOCAL_ASSETS['01::0'],columns=['oktmo','region','mun_type','municipality','year',
    'population','average_population','migration','migration_rate','births','deaths'])
for c in ('oktmo','region','mun_type','municipality'):
    base[c]=base[c].astype('string').str.strip()
base=base.rename(columns={'oktmo':'entity_id','region':'region_name','year':'observation_year'})
for c in ('population','average_population','migration','migration_rate','births','deaths'):
    base[c]=pd.to_numeric(base[c],errors='coerce')
base['observation_year']=pd.to_numeric(base['observation_year'],errors='coerce')
base=base[base.entity_id.notna()&base.entity_id.ne('')&base.observation_year.between(1990,2026)].copy()
base['observation_year']=base.observation_year.astype(int)
base['entity_id']=base.entity_id.astype(str)
repeats=base[base.duplicated(['entity_id','observation_year'],keep=False)].copy()
if len(repeats):
    valuecols=['population','average_population','migration','migration_rate','births','deaths','region_name','mun_type']
    diversity=repeats.groupby(['entity_id','observation_year'])[valuecols].nunique(dropna=False)
    ensure(diversity.le(1).all().all(),'D01_CONFLICTING_ENTITY_YEAR')
    base=base.drop_duplicates(['entity_id','observation_year'])
base.loc[~base.population.gt(0),'population']=np.nan
base=base.sort_values(['entity_id','observation_year']).reset_index(drop=True)
YEARS=sorted(base.observation_year.unique().tolist())

# Independent target audit: no forced equivalence between reported migration_rate and net-migration counts.
MIGRATION_AUDIT=[];record=base.migration_rate
for denom in ['average_population','population']:
    for numerator_scale in [1.,10.,100.,1000.]:
        candidate=base.migration/base[denom]*numerator_scale
        good=record.notna() & candidate.notna() & base[denom].gt(0)
        if good.sum()<100:continue
        delta=(record[good]-candidate[good]).abs()
        consistent=(delta<=np.maximum(1e-5,0.05*record[good].abs())).mean()
        MIGRATION_AUDIT.append({'denominator':denom,'numerator_scale':numerator_scale,
           'n_pairs':int(good.sum()),'agreement_5pct':float(consistent),'median_abs_error':float(delta.median()),
           'median_observed_rate':float(record[good].median())})
MIGRATION_QA=pd.DataFrame(MIGRATION_AUDIT).sort_values(['agreement_5pct','n_pairs','median_abs_error'],ascending=[False,False,True])
if len(MIGRATION_QA):
    best=MIGRATION_QA.iloc[0];valid_rate=bool(best.agreement_5pct>=0.90 and best.n_pairs>=300)
else:valid_rate=False
if valid_rate:
    denom=best['denominator'];scale=best['numerator_scale']
    base['migration_per_1000']=1000.0*base.migration/base[denom]  # persons per 1000 residents; source migration_rate is a fraction when scale=1
    mark('MIGRATION_TARGET_SEMANTICS','PASS',f'{denom}, source_ratio_scale={scale}, converted_to_per_1000=True, n={best.n_pairs}, agreement={best.agreement_5pct:.3f}')
else:
    base['migration_per_1000']=np.nan
    mark('MIGRATION_TARGET_SEMANTICS','NOT_FEASIBLE','Reported migration rate inconsistent with all inspected denominators/scales; source-definition audit required')
base['boundary_uncertainty']='UNVERIFIED_CROSS_YEAR_BOUNDARIES'
base['source_vintage']=source_sha
base_keys=base[['entity_id','observation_year','region_name']].drop_duplicates()
con.register('base_keys',base_keys)
quiet_print('Демография D01:',len(base),'наблюдений;',base.entity_id.nunique(),'ОКТМО;',min(YEARS),'-',max(YEARS))

# Test windows are declared in CONFIG; D01-only presence checks prevent selecting
# a future year with no migration labels. Selection never examines outcome magnitudes.
TARGET_TEST_WINDOWS={t:list(map(int,CFG['PRIMARY_TEST_WINDOWS'][t])) for t in CFG['TARGETS']}
TARGET_CALENDAR_PREFLIGHT=[]
# D01 does not contain population_change_pct: construct availability from real
# population at target year and at origin=target-h, without evaluating magnitude.
_POP_SOURCE=base.loc[base.population.gt(0),['entity_id','observation_year','population']].copy()
_POP_SOURCE= _POP_SOURCE.drop_duplicates(['entity_id','observation_year'])
for target,years in TARGET_TEST_WINDOWS.items():
    ensure(target in ('migration_per_1000','population_change_pct'), 'UNKNOWN_D01_TARGET_'+str(target))
    for year in years:
        by_h={}
        for h in CFG['HORIZONS']:
            if target=='migration_per_1000':
                observed=base.loc[base.observation_year.eq(year),'migration_per_1000'].notna()
                n=int(observed.sum())
            else:
                current=_POP_SOURCE.loc[_POP_SOURCE.observation_year.eq(year),['entity_id']]
                origin=_POP_SOURCE.loc[_POP_SOURCE.observation_year.eq(year-int(h)),['entity_id']]
                n=int(current.merge(origin,on='entity_id',how='inner',validate='one_to_one').shape[0])
            by_h[int(h)]=n
        # Record each horizon separately; no silent reuse of h1 counts for h3/h5.
        for h,n in by_h.items():
            TARGET_CALENDAR_PREFLIGHT.append({'target':target,'target_year':int(year),'horizon':int(h),
              'available_labels':int(n),'role':'PREREGISTERED_TWO_YEAR_PRIMARY',
              'status':'OBSERVED' if n>0 else 'MISSING',
              'availability_derived_from':'D01_MIGRATION_PER_1000' if target=='migration_per_1000' else 'D01_POPULATION_ORIGIN_AND_TARGET'})
TARGET_CALENDAR_PREFLIGHT=pd.DataFrame(TARGET_CALENDAR_PREFLIGHT)
if TARGET_CALENDAR_PREFLIGHT.loc[TARGET_CALENDAR_PREFLIGHT.horizon.eq(1),'status'].eq('MISSING').any():
    mark('PRIMARY_TARGET_D01_GATE','FAIL','Preregistered target years are unobserved; revise protocol explicitly, never silently shift')
    if CFG['MODE']=='FULL':raise RuntimeError('PRIMARY_TARGET_D01_GATE_FAILURE: '+TARGET_CALENDAR_PREFLIGHT.to_json(orient='records'))
else:mark('PRIMARY_TARGET_D01_GATE','PASS',f'Observed outcome labels in primary h=1 windows: {TARGET_TEST_WINDOWS}')



#### Каталог БДМО: проверка единиц, размерностей, реального муниципального охвата.

In [5]:
# 05 · V5.00 · SUBJECT-VALIDATED FACTOR INVENTORY · SQL/LEVEL CHECKED
# Дополнительные муниципальные источники БДМО: только проверяемые фактические данные.
# Разделы основного каталога Stage I (52) остаются неизменными и отдельно аудируются.
import html as _html
from html.parser import HTMLParser as _HTMLParser
BDMO_EXTERNAL_QA_ROWS=[];BDMO_EXTERNAL_ASSETS=[];BDMO_EXTERNAL_RECEIPTS=[]
BDMO_OFFICIAL_CATALOGUE='https://tochno.st/datasets/bdmo'
BDMO_EXTRA_SECTIONS={1:'G1',2:'G1',3:'G3',6:'G5',8:'G2',9:'G1',19:'G3',32:'G1',33:'G3',39:'G1',42:'G4',70:'G5'}
BDMO_EXTERNAL_DESIGN=pd.DataFrame([{'section':s,'assigned_group':g,'source':'BDMO/Rosstat via tochno.st',
    'data_grain':'municipality_year_long, validated at query time',
    'selection':'all codes catalogued, only preregistered semantic domains retained'}
    for s,g in BDMO_EXTRA_SECTIONS.items()])

class _LinkCollector(_HTMLParser):
    def __init__(self):super().__init__();self.hrefs=[]
    def handle_starttag(self,tag,attrs):
        if tag=='a':
            d=dict(attrs)
            if d.get('href'):self.hrefs.append(_html.unescape(d['href']))

def _supplementary_urls():
    r=SESSION.get(BDMO_OFFICIAL_CATALOGUE,timeout=(15,60));r.raise_for_status()
    parser=_LinkCollector();parser.feed(r.text)
    u=set(parser.hrefs)|set(re.findall(r'https?://[^\s\"<>]+?(?:\.zip)',_html.unescape(r.text)))
    result={}
    for sec in BDMO_EXTRA_SECTIONS:
        pattern=rf'/by_(?:section|indicator)/data_section{sec}_.*?\.zip'
        matches=[v for v in u if re.search(pattern,v) and urllib.parse.urlparse(v).hostname=='storage.yandexcloud.net']
        # Fallback is the same dated catalogue release already audited in Stage I,
        # not a claim that the remote file exists: HTTP response must validate it.
        if not matches:
            base_url=next((x.get('source_url') for x in ASSETS if x.get('source_id')=='BDMO' and '/by_section/' in str(x.get('source_url'))),None)
            if base_url and sec!=32:
                mm=re.search(r'data_section\d+_',base_url)
                if mm:matches=[base_url[:mm.start()]+'data_section'+str(sec)+'_'+base_url[mm.end():].replace('data_section13_',f'data_section{sec}_')]
        if matches:result[sec]=max(matches,key=lambda s:re.search(r'v(20\d{6})',s).group(1) if re.search(r'v(20\d{6})',s) else '')
    return result

def _supplement_one_section(sec,url,group):
    pu=urllib.parse.urlparse(url)
    ensure(pu.scheme=='https' and pu.hostname=='storage.yandexcloud.net' and '/tochno-st-catalog/Rosstat/' in pu.path,'BDMO_EXTERNAL_URL_UNTRUSTED')
    zpath=WORK/'cache'/f'bdmo_external_{sec}.zip';part=zpath.with_suffix('.part')
    side=zpath.with_suffix('.source_url.txt')
    if zpath.exists() and (not side.exists() or side.read_text(encoding='utf-8')!=url):
        zpath.unlink()
    if not zpath.exists():
        part.unlink(missing_ok=True);nr=0;h=hashlib.sha256()
        try:
            rr=SESSION.get(url,timeout=(30,180),stream=True);rr.raise_for_status()
            with open(part,'wb') as fd:
                for buf in rr.iter_content(4*1024*1024):
                    if not buf:continue
                    nr+=len(buf)
                    ensure(nr<=CFG['BDMO_EXTERNAL_MAX_SOURCE_GB']*2**30,'BDMO_EXTERNAL_ZIP_SIZE_LIMIT')
                    if nr%(512*1024*1024)<len(buf):
                        ensure(shutil.disk_usage(WORK).free>CFG['MIN_DISK_FREE_GB']*2**30,
                               'BDMO_EXTERNAL_DISK_SAFETY_STOP')
                    h.update(buf);fd.write(buf)
            rr.close();ensure(zipfile.is_zipfile(part),'BDMO_EXTERNAL_NOT_ZIP')
            part.replace(zpath);side.write_text(url,encoding='utf-8')
        except Exception:part.unlink(missing_ok=True);raise
    ensure(zipfile.is_zipfile(zpath),'BDMO_EXTERNAL_CACHE_NOT_ZIP')
    source_sha=digest(zpath);indexed=0
    with zipfile.ZipFile(zpath) as z:
        ensure(z.testzip() is None,'BDMO_EXTERNAL_ZIP_CRC')
        infos=[i for i in z.infolist() if not i.is_dir() and i.filename.lower().endswith('.parquet')]
        infos.sort(key=lambda x:x.filename)
        if not infos:
            BDMO_EXTERNAL_QA_ROWS.append({'section':sec,'status':'SOURCE_FORMAT_NOT_PARQUET',
                'reason':'The validated archive contains no direct Parquet member; ' +
                         'nested/CSV sources require separate schema and grain validation',
                'members_sample':json.dumps([i.filename for i in z.infolist()[:20]],ensure_ascii=False)})
            return 0
        ensure(len(infos)<=CFG['BDMO_EXTERNAL_MAX_MEMBERS_PER_SECTION'],'BDMO_EXTERNAL_TOO_MANY_MEMBERS')
        for idx,member in enumerate(infos):
            ensure(member.file_size<CFG['BDMO_EXTERNAL_MAX_UNPACKED_GB']*2**30,'BDMO_EXTERNAL_UNPACKED_MEMBER_LIMIT')
            dest=WORK/'cache'/f'bdmo_external_{sec}_{idx:03d}.parquet'
            if not dest.exists() or dest.stat().st_size!=member.file_size:
                tmp=dest.with_suffix('.part')
                try:
                    with z.open(member) as src,open(tmp,'wb') as dst:shutil.copyfileobj(src,dst,length=4*1024*1024)
                    ensure(tmp.stat().st_size==member.file_size,'BDMO_EXTERNAL_MEMBER_BYTES')
                    tmp.replace(dest)
                except Exception:tmp.unlink(missing_ok=True);raise
            info=pq.ParquetFile(dest);cols=info.schema.names
            needed={'year','indicator_code','indicator_name','indicator_value','oktmo'}
            if not needed.issubset(cols):
                BDMO_EXTERNAL_QA_ROWS.append({'section':sec,'member':member.filename,'status':'SCHEMA_UNSUPPORTED','columns':','.join(cols[:35])});continue
            aid=f'SUPP::BDMO::{sec}::{idx:03d}'
            entry={'asset_id':aid,'source_id':'BDMO','role':'municipal_long_supplement',
                'segment':'BDMO_EXTERNAL','columns_manifest':cols,'rows_actual':info.metadata.num_rows,
                'source_url':url,'source_sha256':source_sha,'sha256':digest(dest),
                'locator':{'type':'verified_external_cache','path':str(dest)}}
            BDMO_EXTERNAL_ASSETS.append(entry);LOCAL_ASSETS[aid]=str(dest)
            SOURCE_CLASSES[group].append(aid)
            if entry not in FACTOR_ASSETS:FACTOR_ASSETS.append(entry)
            BDMO_EXTERNAL_RECEIPTS.append({'asset_id':aid,'group':group,'source_url':url,'source_sha256':source_sha,
                'parquet_sha256':entry['sha256'],'source_member':member.filename,'rows':entry['rows_actual'],
                'columns':len(cols),'status':'SHA_VERIFIED_AND_SCHEMA_CHECKED'})
            BDMO_EXTERNAL_QA_ROWS.append({'section':sec,'member':member.filename,'status':'ADAPTED',
                'asset_id':aid,'rows':entry['rows_actual'],'group':group})
            indexed+=1
    return indexed

if CFG['ENABLE_EXTERNAL_BDMO'] and CFG['MODE']=='FULL':
    try:
        urls=_supplementary_urls();BDMO_EXTERNAL_QA_ROWS.append({'section':0,'status':'CATALOGUE_OPENED','n_urls_found':len(urls)})
        for _bi,(section,group) in enumerate(BDMO_EXTRA_SECTIONS.items(),1):
            progress('BDMO',_bi-1,len(BDMO_EXTRA_SECTIONS),f'Раздел {section}')
            if section not in urls:
                BDMO_EXTERNAL_QA_ROWS.append({'section':section,'group':group,'status':'URL_NOT_DISCOVERED'});continue
            try:
                n=_supplement_one_section(section,urls[section],group)
                if n==0:log(f'BDMO section {section}: no verified Parquet member; review format audit','WARN')
                quiet_print(f'Дополнительный БДМО раздел {section}: {n} Parquet; SHA проверен')
            except Exception as ex:
                BDMO_EXTERNAL_QA_ROWS.append({'section':section,'group':group,'status':'SOURCE_LOAD_FAILED',
                    'reason':type(ex).__name__+': '+str(ex)[:240]})
                log(f'External BDMO section {section} rejected: {type(ex).__name__}: {str(ex)[:180]}','WARN')
    except Exception as ex:
        BDMO_EXTERNAL_QA_ROWS.append({'section':0,'status':'SOURCE_CATALOGUE_UNAVAILABLE',
            'reason':type(ex).__name__+': '+str(ex)[:240]})
        log('BDMO external catalogue unavailable: '+type(ex).__name__,'WARN')
progress('BDMO',1,1,f'Проверено разделов: {len(BDMO_EXTRA_SECTIONS)}')
BDMO_EXTERNAL_RECOVERY=pd.DataFrame([{
    'section':int(s),'group':g,'status':next((x.get('status') for x in reversed(BDMO_EXTERNAL_QA_ROWS)
        if x.get('section')==s),'NOT_ATTEMPTED'),
    'source_url':urls.get(s,'') if 'urls' in locals() else '',
    'must_not_claim_loaded_without_sha':True}
    for s,g in BDMO_EXTRA_SECTIONS.items()])
BDMO_EXTERNAL_QA=pd.DataFrame(BDMO_EXTERNAL_QA_ROWS)
mark('EXTERNAL_BDMO_SOURCE_AUDIT','PASS' if len(BDMO_EXTERNAL_ASSETS)>=3 else 'PARTIAL',
     f'External verified Parquet={len(BDMO_EXTERNAL_ASSETS)}, sections={sorted(set(int(s.split("::")[2]) for s in LOCAL_ASSETS if s.startswith("SUPP::BDMO::")))}; not part of original 52')
_ORIGINAL_GET_ASSET=get_asset
def get_asset(aid):
    for _a in BDMO_EXTERNAL_ASSETS:
        if _a['asset_id']==aid:return _a
    return _ORIGINAL_GET_ASSET(aid)

PATTERNS = {
 'G1':'зарплат|заработ|оплат.{0,12}труда|доход|инвестиц|занятост|занятых|безработ|предпринимател|предприяти|оборот|производств|налог|уровен.{0,8}жизни|средн.{0,15}душ',
 'G2':'жил|жилищ|квартир|строительств|коммунал|водоснабж|водоотвед|теплоснабж|газификац|электроснабж|канализац|благоустрой',
 'G3':'образован|школ|дошколь|детск.{0,8}сад|медицин|врач|фельдшер|больниц|поликлиник|амбулатор|социал|койко|здравоохран|спорт|культур',
 'G4':'бюджет|расход|дотаци|субсид|субвенци|межбюджет|налогов.{0,10}доход|казн',
 'G5':'расстояни|транспорт|доступност|пространств|территори|площад.{0,18}(?:земл|муницип)|автомобил.{0,12}дорог|покрыти.{0,12}дорог|связь|телефон|интернет'}
SEMANTIC_DOMAINS = {
 'G1':[('wages','заработн.{0,10}плат|средн.{0,12}зарплат|оплат.{0,7}труда|фонд.{0,10}оплат'),('employment','численност.{0,10}(?:занят|работающ|работник)|число.{0,10}(?:занят|работающ|работник)|уровень.{0,10}безработ|число.{0,12}безработ|рабоч.{0,8}мест|занят.{0,10}насел'),('business','мал.{0,12}предприят|средн.{0,12}предприят|предпринимател|субъект.{0,12}мсп|оборот.{0,12}организац|зарегистрированн.{0,15}организац'),('investment','инвестиц|капитальн.{0,10}вложен'),('retail','розничн.{0,14}торгов|оборот.{0,15}торгов|общественн.{0,15}питан'),('paid_services','платн.{0,12}услуг|услуг.{0,12}населен'),('income','денежн.{0,12}доход|сумм.{0,12}доход|совокупн.{0,10}доход|налогооблагаем.{0,10}доход|среднедушев.{0,12}доход|доход.{0,12}населени')],
 'G2':[('housing_stock','жил.{0,8}фонд|жил.{0,9}площад|обеспеченн.{0,12}жиль|площад.{0,12}жил'),('housing_construction','ввод.{0,30}(?:жил|жиль|квартир)|введен.{0,20}(?:жил|квартир)|строительств.{0,14}жил|построенн.{0,12}квартир'),('housing_access','улучшивш.{0,12}жилищ|нуждающ.{0,12}жилищ|семе.{0,12}жилищ'),('utilities','водоснабж|водоотвед|теплоснабж|газификац|канализац|коммунал|благоустроенн.{0,12}жил|электроснабж')],
 'G3':[('health','врач|фельдшер|поликлиник|больниц|больнич|здравоохран|медицин|амбулатор|койко|медработ'),('education','образован|школ|дошколь|детск.{0,8}сад|обучающ|учащ|педагог'),('social','социальн.{0,14}(учрежден|обслужив|защит|услуг)|дом.{0,12}культур|спортивн.{0,10}сооруж')],
 'G5':[('territorial_area','площад.{0,18}(?:земл|муницип)|площад.{0,8}территори'),('roads','автомобил.{0,12}дорог|протяж[её]нност.{0,12}дорог|дорог.{0,12}покрыти'),('connectivity','телефонн|интернет|связ[ьи]|доступност.{0,12}сети')],
 'G4':[('municipal_spending','расход|содержание.{0,12}работник|затрат'),('municipal_revenue','доход.{0,12}бюджет|налогов.{0,12}доход|собственн.{0,10}доход'),('intergovernmental','дотаци|субвенци|субсид|межбюджет|трансферт'),('budget_balance','дефицит|профицит|долг|сбалансированност|бюджет')]}
EXCLUDED_INCOME='ценн.{0,8}бумаг|дивиденд|вычет|ставк.{0,12}налог|исчисленн.{0,12}налог|сумм.{0,12}налог|полученн.{0,8}сведен|количеств.{0,12}сведен|числ.{0,9}сведен|операц.{0,14}бумаг|код.{0,8}доход'
REJECT_NONMEASUREMENT='методик|справочн|код.{0,7}показател|инструкция|наименование|описание|классификатор'

def classify_indicator(group,asset_id,name,unit):
    title=re.sub(r'\s+',' ',str(name).casefold()).strip();un=str(unit).casefold()
    if not title or re.search(REJECT_NONMEASUREMENT,title):return None,'NON_MEASUREMENT'
    if group not in SEMANTIC_DOMAINS:return None,'NO_REGISTERED_DOMAIN'
    if group=='G1' and re.search(EXCLUDED_INCOME,title):return None,'NARROW_TAX_OR_DEDUCTION_PROXY'
    if group=='G1' and 'ndfl' in asset_id.lower():
        if not re.search(r'денежн.{0,12}доход|сумм.{0,12}доход|совокупн.{0,10}доход|налогооблагаем.{0,10}доход',title):return None,'NDFL_NOT_INCOME_AMOUNT'
        if re.search(r'штук|единиц|количеств|сведен|запис',un):return None,'NDFL_COUNT_NOT_INCOME'
    # V5: предметная категория определяется самим показателем, а не источником.
    if group=='G1' and re.search(r'инвестиц.{0,50}(?:основн|капитал)|инвестиц.{0,10}в|капитальн.{0,10}вложен',title):
        return 'investment','V5_INVESTMENT_OVERRIDES_BUSINESS'
    if group=='G3' and re.search(r'социал.{0,20}обслужив|обслуживаем.{0,40}(?:отделени|социал)|специализированн.{0,30}обслужив',title):
        return 'social','V5_HOME_SOCIAL_CARE_NOT_HOSPITAL'
    # 'Продано товаров (предприятия со штатом >15)' measures SALES, not workers.
    if group=='G1' and re.search(r'продано товар|продаж.{0,25}товар|отгружено товар|отгрузк.{0,20}товар',title):
        return 'business','SALES_VOLUME_NOT_EMPLOYMENT'
    # A count of medical social workers is not automatically hospital infrastructure.
    for dom,pattern in SEMANTIC_DOMAINS[group]:
        if re.search(pattern,title):return dom,''
    return None,'NO_SUBSTANTIVE_MATCH'

TOTAL_DIM_VALUES={'всего','итого','все','все категории','total','all','общий','общая','общие'}
NDFL_RISK_DIMENSIONS=('individual_type','income','deduction','tax_rate','report_type','index_form','tax_authority')

def eligible_slice(asset_id,dims):
    if 'ndfl' not in asset_id.lower():return True,'CATEGORY_SPECIFIED_NO_COLLAPSE'
    for k in NDFL_RISK_DIMENSIONS:
        if k in dims and str(dims[k]).casefold().strip() not in TOTAL_DIM_VALUES:
            return False,'NON_VERIFIED_TOTAL_NDFL_DIMENSION_'+k
    return True,'NDFL_ALL_RISK_DIMENSIONS_EXPLICIT_TOTAL_OR_ABSENT'

ensure(classify_indicator('G1','02::5ndfl::00::00','Количество сведений о доходах','Штука')[0] is None,'SEMANTIC_CANARY_TAX_COUNTS')
ensure(classify_indicator('G1','02::labor','Средняя начисленная заработная плата','рублей')[0]=='wages','SEMANTIC_CANARY_WAGES')
ensure(classify_indicator('G3','02::social','Количество врачей','человек')[0]=='health','SEMANTIC_CANARY_HEALTH')
ensure(classify_indicator('G1','SUPP::BDMO','Продано товаров несобственного производства (без субъектов малого предпринимательства), средняя численность работников которых превышает 15 человек','рублей')[0]=='business','SEMANTIC_CANARY_SALES_NOT_EMPLOYMENT')
ensure(not eligible_slice('02::5ndfl::00::00',{'income':'2204','tax_rate':'9'})[0],'SEMANTIC_CANARY_NDFL_SLICES')
mark('SEMANTIC_CANARY','PASS','Tax counts, wage concept, medicine and narrow NDFL slice guards')

PRIMARY_TARGET_YEARS=sorted(set(sum(TARGET_TEST_WINDOWS.values(),[])))
FACTOR_CUTOFF_BY_H={int(h):int(min(min(yy) for yy in TARGET_TEST_WINDOWS.values())-h-CFG['FEATURE_PUBLICATION_LAG']) for h in CFG['HORIZONS']}
TRAIN_CATALOG_MAX_YEAR=max(FACTOR_CUTOFF_BY_H.values());TRAIN_CATALOG_MIN_YEAR=TRAIN_CATALOG_MAX_YEAR-CFG['CATALOG_RECENT_WINDOW']
CATALOG_ROWS=[];FEATURE_PARTS=[];conflict_rows=[];SOURCE_QA=[];FEATURE_CANDIDATES=[];SELECTED=[];CANDIDATE_AUDIT=[];G5_DIAGNOSTICS=[];SCAN_AUDIT=[];FACTOR_ELIGIBILITY=[];DIMENSION_AUDIT=[]
LAST_SLICE_REASON='NOT_STARTED'

def qstr(v):return "'"+str(v).replace("'","''")+"'"
def qident(s):
    v=str(s);ensure(bool(v) and '\x00' not in v,'INVALID_SOURCE_COLUMN')
    return '"'+v.replace('"','""')+'"'

SQL_FAILURES=[];SQL_EXECUTION_AUDIT=[];REAL_SQL_PREFLIGHT=[]

def sql_run(sql,params=None,*,asset_id='',code='',stage='',preview=False):
    args=list(params or []);t0=time.monotonic()
    try:
        result=con.execute(sql,args);data=result.fetchone() if preview else result.df()
        SQL_EXECUTION_AUDIT.append({'asset_id':asset_id,'code':str(code),'stage':stage,'status':'PASS','elapsed_ms':int((time.monotonic()-t0)*1000),'sql_sha256':hashlib.sha256(sql.encode()).hexdigest()})
        return data
    except Exception as exc:
        safe_sql=re.sub(r'(?i)(ghp_|github_pat_)[a-z0-9_]+','[REDACTED]',sql)
        detail={'asset_id':asset_id,'code':str(code),'stage':stage,'status':'SCAN_ERROR','exception_type':type(exc).__name__,'message':str(exc)[:2000],'sql':safe_sql[:12000],'sql_sha256':hashlib.sha256(sql.encode()).hexdigest(),'parameters_types':[type(v).__name__ for v in args],'elapsed_ms':int((time.monotonic()-t0)*1000)}
        SQL_FAILURES.append(detail);log('SQL_SCAN_ERROR '+json.dumps({k:v for k,v in detail.items() if k!='sql'},ensure_ascii=False),'WARN')
        raise

def column_key(a):
    cc=a.get('columns_manifest') or []
    return 'oktmo' if 'oktmo' in cc else 'object_oktmo' if 'object_oktmo' in cc else None

def dimensions_for(a):
    cc=a.get('columns_manifest') or [];aid=a['asset_id']
    if 'ndfl' in aid.lower():return [k for k in NDFL_RISK_DIMENSIONS if k in cc]
    omit={'indicator_section_code','indicator_section','indicator_code','indicator_name','indicator_unit','indicator_period','indicator_value','year','region_id','region_name','region','oktmo','object_oktmo','oktmo_stable','oktmo_history','oktmo_year_from','oktmo_year_to','mun_level','mun_district','municipality','mun_type','mun_type_oktmo','object_name','object_level','object_okato','source','comment','version_date','report_date','report_region_id','report_region_name','null_value_reason'}
    return [c for c in cc if c not in omit]

def val_sql(alias='a'):
    return f"TRY_CAST(REPLACE(REPLACE(REPLACE(TRIM(CAST({alias}.indicator_value AS VARCHAR)), ' ', ''), chr(160), ''), ',', '.') AS DOUBLE)"

def _grp_eligible_from_source(a):
    g=classify_asset_for_scan(a)
    return ['G1','G2','G3','G4'] if g=='DISCOVERY_MULTIDOMAIN' else [g] if g in SEMANTIC_DOMAINS else []

def inventory_asset(a,path,group):
    key=column_key(a);cc=set(a.get('columns_manifest') or [])
    if key is None or not {'indicator_code','indicator_name','indicator_value','year'}.issubset(cc):return pd.DataFrame()
    unit=f"COALESCE(TRIM(CAST(a.{qident('indicator_unit')} AS VARCHAR)), '')" if 'indicator_unit' in cc else "''"
    k=qident(key);yr='TRY_CAST(a.'+qident('year')+' AS INTEGER)'
    level_predicate=("regexp_matches(lower(coalesce(cast(a."+qident('object_level')+" AS VARCHAR),'')), 'муниципал|городск.{0,8}округ')" if 'object_level' in cc else 'TRUE')
    sql=f"""SELECT TRIM(CAST(a.{qident('indicator_code')} AS VARCHAR)) AS indicator_code,
      TRIM(CAST(a.{qident('indicator_name')} AS VARCHAR)) AS indicator_name,{unit} AS indicator_unit,
      COUNT(DISTINCT TRIM(CAST(a.{k} AS VARCHAR))) AS n_municipalities,
      COUNT(DISTINCT {yr}) AS n_years,
      COUNT(DISTINCT CASE WHEN {yr} BETWEEN {TRAIN_CATALOG_MIN_YEAR} AND {TRAIN_CATALOG_MAX_YEAR}
      THEN TRIM(CAST(a.{k} AS VARCHAR))||':'||CAST({yr} AS VARCHAR) END) AS recent_keys,
      MIN({yr}) AS min_year,MAX({yr}) AS max_year
      FROM read_parquet(?) a JOIN base_keys b ON TRIM(CAST(a.{k} AS VARCHAR))=b.entity_id
         AND {yr}=b.observation_year
      WHERE {yr}<={TRAIN_CATALOG_MAX_YEAR} AND {level_predicate}
       AND regexp_matches(lower(coalesce(cast(a.{qident('indicator_name')} AS VARCHAR),'')),?)
       AND a.{qident('indicator_code')} IS NOT NULL
      GROUP BY 1,2,3 HAVING COUNT(DISTINCT a.{k})>={CFG['MIN_INDICATOR_MUNICIPALITIES']}
      ORDER BY indicator_code,indicator_name,indicator_unit"""
    return sql_run(sql,[str(path),PATTERNS[group]],asset_id=a['asset_id'],stage='INVENTORY')

def _slice_dimensions_supported(a,dims):
    if 'ndfl' not in a['asset_id'].lower():return True,'NON_NDFL_DIMENSIONAL_SLICE'
    ok,reason=eligible_slice(a['asset_id'],dims)
    if not ok:return False,reason
    if any(k in dims for k in ('income','deduction','tax_rate','individual_type','report_type','index_form')):
        return True,'NDFL_EXPLICIT_TOTAL_VALUES_NO_COLLAPSE'
    return True,'NDFL_UNSTRATIFIED_NO_AGGREGATION'

def factor_raw_sql(a):
    key=column_key(a);cc=set(a.get('columns_manifest') or []);dims=dimensions_for(a)
    ensure(key is not None and {'indicator_code','indicator_name','indicator_value','year'}.issubset(cc),'INVALID_FACTOR_SCHEMA')
    d_expr=','.join(f"COALESCE(NULLIF(TRIM(CAST(a.{qident(d)} AS VARCHAR)),''),'∅') AS {qident('dim_'+str(i).zfill(2))}" for i,d in enumerate(dims))
    d_expr=','+d_expr if d_expr else ''
    unitfilter=f" AND COALESCE(TRIM(CAST(a.{qident('indicator_unit')} AS VARCHAR)),'')=?" if 'indicator_unit' in cc else ''
    level_filter=(" AND regexp_matches(lower(coalesce(cast(a."+qident('object_level')+" AS VARCHAR),'')), 'муниципал|городск.{0,8}округ')" if 'object_level' in cc else '')
    raw=f"""SELECT TRIM(CAST(a.{qident(key)} AS VARCHAR)) AS entity_id,
       TRY_CAST(a.{qident('year')} AS INTEGER) AS observation_year,{val_sql()} AS val{d_expr}
       FROM read_parquet(?) a JOIN base_keys b
         ON TRIM(CAST(a.{qident(key)} AS VARCHAR))=b.entity_id
         AND TRY_CAST(a.{qident('year')} AS INTEGER)=b.observation_year
       WHERE TRIM(CAST(a.{qident('indicator_code')} AS VARCHAR))=?
         AND TRIM(CAST(a.{qident('indicator_name')} AS VARCHAR))=? {unitfilter} {level_filter}"""
    return raw,dims,bool(unitfilter)

def select_dimensional_slice(a,path,code,name,unit,group):
    global LAST_SLICE_REASON
    LAST_SLICE_REASON='NO_VALID_CANDIDATE'
    raw,dims,with_unit=factor_raw_sql(a)
    baseparams=[str(path),str(code),str(name)]+([str(unit)] if with_unit else [])
    d_alias=[qident('dim_'+str(i).zfill(2)) for i in range(len(dims))]
    d_select=','.join(d_alias)+',' if dims else ''
    pos=' GROUP BY '+','.join(map(str,range(1,len(dims)+1))) if dims else ''
    summary_sql=f"""WITH v AS ({raw}) SELECT {d_select}
       COUNT(DISTINCT CASE WHEN observation_year<={TRAIN_CATALOG_MAX_YEAR} AND val IS NOT NULL
           THEN entity_id||':'||CAST(observation_year AS VARCHAR) END) AS n_train_keys,
       COUNT(DISTINCT CASE WHEN observation_year BETWEEN {TRAIN_CATALOG_MIN_YEAR} AND {TRAIN_CATALOG_MAX_YEAR}
         AND val IS NOT NULL THEN entity_id||':'||CAST(observation_year AS VARCHAR) END) AS n_recent_keys,
       COUNT(DISTINCT CASE WHEN observation_year<={TRAIN_CATALOG_MAX_YEAR} AND val IS NOT NULL
         THEN observation_year END) AS n_train_years FROM v {pos}"""
    if 'ndfl' in a['asset_id'].lower():
        filter_terms=[f'lower(trim({d_alias[i]})) IN ('+','.join(qstr(s) for s in sorted(TOTAL_DIM_VALUES))+')' for i,d in enumerate(dims) if d in NDFL_RISK_DIMENSIONS]
        predicate=' AND '.join(filter_terms) if filter_terms else 'TRUE'
        budget=max(1,int(CFG['MAX_NDFL_SLICE_VALUES']))
    else:
        predicate='TRUE';budget=max(1,int(CFG['MAX_DIMENSIONAL_SLICES_PER_CODE']))
    total_sql='SELECT COUNT(*) FROM ('+summary_sql+') total_slices'
    total=int(sql_run(total_sql,baseparams,asset_id=a['asset_id'],code=code,stage='SLICE_COUNT',preview=True)[0])
    valid_sql='SELECT COUNT(*) FROM ('+summary_sql+') grouped WHERE '+predicate
    eligible=int(sql_run(valid_sql,baseparams,asset_id=a['asset_id'],code=code,stage='SLICE_ELIGIBLE_COUNT',preview=True)[0])
    take=min(eligible,budget)
    DIMENSION_AUDIT.append({'asset_id':a['asset_id'],'indicator_code':code,'slices_found':total,'semantically_eligible':eligible,'inspected':take,'remaining':max(eligible-take,0),'rejected_semantically':total-eligible,'status':'NO_VALID_TOTAL_SLICE' if 'ndfl' in a['asset_id'].lower() and eligible==0 else 'CANDIDATE_BUDGET_PARTIAL' if eligible>budget else 'EXHAUSTIVE_SQL_FILTERED'})
    if not eligible:
        LAST_SLICE_REASON='NO_VALID_TOTAL_SLICE' if 'ndfl' in a['asset_id'].lower() else 'NO_DIMENSION_SLICES'
        return None,None,None
    top_sql='SELECT * FROM ('+summary_sql+') grouped WHERE '+predicate+' ORDER BY n_recent_keys DESC,n_train_keys DESC LIMIT ?'
    top=sql_run(top_sql,baseparams+[take],asset_id=a['asset_id'],code=code,stage='SLICE_RANKING')
    if top.empty:return None,None,None
    for row in top.to_dict('records'):
        sl={d:str(row['dim_'+str(i).zfill(2)]) for i,d in enumerate(dims)}
        ok,why=_slice_dimensions_supported(a,sl)
        if not ok:
            CANDIDATE_AUDIT.append({'asset_id':a['asset_id'],'code':code,'name':name,'decision':why,'slice':json.dumps(sl,ensure_ascii=False)})
            continue
        if int(row['n_train_keys'])<CFG['MIN_INDICATOR_OBS_TRAIN'] or int(row['n_train_years'])<CFG['MIN_INDICATOR_YEARS_TRAIN']:continue
        pred=' AND '.join(f'{d_alias[i]}=?' for i in range(len(dims))) if dims else 'TRUE'

        # FIX: DuckDB requires the output alias "value" to be quoted.
        audit_sql=f"""WITH v AS ({raw}), sliced AS
          (SELECT entity_id,observation_year,val FROM v WHERE {pred})
          SELECT entity_id,observation_year,
            MIN(val) AS "value",
            COUNT(DISTINCT val) AS n_different,
            COUNT(*) AS n_raw
          FROM sliced GROUP BY entity_id,observation_year"""

        audit=sql_run(audit_sql,baseparams+[sl[d] for d in dims],asset_id=a['asset_id'],code=code,stage='EXTRACT_SLICE')
        if audit.empty:continue
        conflict=audit.n_different.gt(1);train_mask=audit.observation_year.le(TRAIN_CATALOG_MAX_YEAR)
        train_conf=conflict&train_mask;share=float(train_conf.sum()/max(1,train_mask.sum()))
        if conflict.any():
            conflict_rows.append({'asset':a['asset_id'],'code':code,'n_collisions':int(conflict.sum()),'share':share,'slice':json.dumps(sl,ensure_ascii=False),'decision':'REJECT_NONUNIQUE_SLICED_KEY' if share>CFG['MAX_CONFLICT_SHARE'] else 'EXCLUDE_CONFLICTED_ENTITY_YEARS'})
        if share>CFG['MAX_CONFLICT_SHARE']:continue
        audit=audit.loc[~conflict,['entity_id','observation_year','value']].copy()
        train_values=audit.observation_year.le(TRAIN_CATALOG_MAX_YEAR)&audit['value'].notna()
        if int(train_values.sum())<CFG['MIN_INDICATOR_OBS_TRAIN']:continue
        ensure(not audit.duplicated(['entity_id','observation_year']).any(),'NON_UNIQUE_FEATURE_OUTPUT')
        LAST_SLICE_REASON='ELIGIBLE'
        return audit,sl,row
    LAST_SLICE_REASON='PARTIAL_CANDIDATES_INSPECTED' if eligible>budget else 'NO_VALID_SLICE_AFTER_SUPPORT_CHECK'
    return None,None,None

def run_duckdb_sql_contracts():
    tmp=WORK/'cache'/'ml03_stage3_sql_contract.parquet';tmp_ndfl=WORK/'cache'/'ml03_stage3_ndfl_contract.parquet'
    audit0=len(SQL_EXECUTION_AUDIT);dim0=len(DIMENSION_AUDIT);conf0=len(conflict_rows)
    selected=base_keys.loc[base_keys.observation_year.between(TRAIN_CATALOG_MIN_YEAR,TRAIN_CATALOG_MAX_YEAR),['entity_id','observation_year']].drop_duplicates()
    eligible_ids=selected.groupby('entity_id').observation_year.nunique()
    eligible_ids=eligible_ids[eligible_ids>=2].index[:100]
    selected=selected[selected.entity_id.isin(eligible_ids)].groupby('entity_id').head(2)
    ensure(len(selected)>=100 and selected.observation_year.nunique()>=2 and selected.entity_id.nunique()>=CFG['MIN_INDICATOR_MUNICIPALITIES'],'SQL_UNIT_FIXTURE_INSUFFICIENT_BASE_KEYS')
    fixture=selected.rename(columns={'entity_id':'oktmo','observation_year':'year'}).copy()
    fixture['indicator_code']='Y48010001';fixture['indicator_name']='Площадь жилищного фонда';fixture['indicator_unit']='кв. м'
    fixture['indicator_value']='1 234,5';fixture['group']='Все';fixture['order']='Вся территория';fixture['index"quoted']='Данные'
    con.register('ml03_fixture_df',fixture)
    try:
        con.execute('COPY (SELECT * FROM ml03_fixture_df) TO '+qstr(str(tmp))+' (FORMAT PARQUET)')
        toy={'asset_id':'02::housing::SYNTHETIC_SQL_CONTRACT','columns_manifest':list(fixture.columns)}
        inv=inventory_asset(toy,tmp,'G2')
        ensure(not inv.empty and inv.indicator_code.eq('Y48010001').any(),'SQL_UNIT_INVENTORY_EMPTY')
        dat,sl,support=select_dimensional_slice(toy,tmp,'Y48010001','Площадь жилищного фонда','кв. м','G2')
        ensure(dat is not None and len(dat)>=100,'SQL_UNIT_EXTRACT_NO_ROWS')
        ensure(dat['value'].notna().all() and np.allclose(dat['value'].to_numpy(),1234.5),'SQL_UNIT_LOCALE_DECIMAL')
        ensure(not dat.duplicated(['entity_id','observation_year']).any(),'SQL_UNIT_DUPLICATE_KEYS')
        ensure(sl['group']=='Все' and sl['order']=='Вся территория' and sl['index"quoted']=='Данные','SQL_UNIT_DIMENSION_ROUNDTRIP')

        # Municipal-level contract: a D05-like region row with a matching OKTMO
        # must still be excluded, even if its identifier happens to coincide.
        lev_fixture=fixture.iloc[:2].copy().drop(columns=['group','order','index"quoted'])
        lev_fixture['object_oktmo']=lev_fixture.pop('oktmo')
        lev_fixture['object_level']=['Регион','Муниципальный район']
        con.register('ml03_level_fixture_df',lev_fixture)
        level_path=WORK/'cache'/'ml03_level_contract.parquet'
        con.execute('COPY (SELECT * FROM ml03_level_fixture_df) TO '+qstr(str(level_path))+' (FORMAT PARQUET)')
        try:
            lev_asset={'asset_id':'02::labor::SYNTHETIC_LEVEL','columns_manifest':list(lev_fixture.columns)}
            lvi=inventory_asset(lev_asset,level_path,'G2')
            ensure(len(lvi)<=1,'LEVEL_INVENTORY_UNEXPECTED')
            raw_level,_,hasunit=factor_raw_sql(lev_asset)
            pp=[str(level_path),'Y48010001','Площадь жилищного фонда']+(['кв. м'] if hasunit else [])
            lv=sql_run('SELECT COUNT(*) AS n FROM ('+raw_level+') filtered',pp,asset_id=lev_asset['asset_id'],stage='OBJECT_LEVEL_UNIT',preview=True)
            ensure(int(lv[0])==1,'REGIONAL_OBJECT_ROW_LEAKED_IN_MUNICIPAL_JOIN')
        finally:
            con.unregister('ml03_level_fixture_df');level_path.unlink(missing_ok=True)
        ndfl=fixture.iloc[:1].copy().drop(columns=['group','order','index"quoted'])
        ndfl['indicator_code']='SYNTHETIC_NDFL_TOTAL';ndfl['indicator_name']='Общая сумма денежных доходов'
        ndfl['indicator_unit']='рублей';ndfl['income']='9999';ndfl['tax_rate']='13'
        sparse=pd.concat([ndfl.assign(income=str(10000+i)) for i in range(2101)],ignore_index=True)
        total=fixture.drop(columns=['group','order','index"quoted']).copy()
        total['indicator_code']='SYNTHETIC_NDFL_TOTAL';total['indicator_name']='Общая сумма денежных доходов'
        total['indicator_unit']='рублей';total['income']='всего';total['tax_rate']='всего'
        ndfl=pd.concat([sparse,total],ignore_index=True)
        con.register('ml03_ndfl_fixture_df',ndfl)
        con.execute('COPY (SELECT * FROM ml03_ndfl_fixture_df) TO '+qstr(str(tmp_ndfl))+' (FORMAT PARQUET)')
        toy_ndfl={'asset_id':'02::5ndfl::SYNTHETIC_SQL_CONTRACT','columns_manifest':list(ndfl.columns)}
        a,sl2,z=select_dimensional_slice(toy_ndfl,tmp_ndfl,'SYNTHETIC_NDFL_TOTAL','Общая сумма денежных доходов','рублей','G1')
        ensure(a is not None and len(a)>=100,'SQL_UNIT_NDFL_NO_TOTAL_SLICE')
        ensure(sl2=={'income':'всего','tax_rate':'всего'},'SQL_UNIT_NDFL_NARROW_SLICE_INCLUDED')
        record=DIMENSION_AUDIT[-1]
        ensure(record['slices_found']>2000 and record['semantically_eligible']==1 and record['inspected']==1 and record['remaining']==0,'SQL_UNIT_NDFL_LIMIT_ORDER')
        mark('SQL_DUCKDB_UNIT_CONTRACT','PASS',f'real DuckDB queries on Parquet; quoted columns; numeric parsing; >{record["slices_found"]} NDFL slices -> 1 eligible total')
        return {'status':'PASS','housing_rows':len(dat),'ndfl_total_rows':len(a),'ndfl_slices_found':record['slices_found'],'ndfl_semantically_eligible':1,'real_duckdb':True,'synthetic_only':True}
    finally:
        try:con.unregister('ml03_fixture_df');con.unregister('ml03_ndfl_fixture_df')
        except Exception:pass
        tmp.unlink(missing_ok=True);tmp_ndfl.unlink(missing_ok=True)
        del SQL_EXECUTION_AUDIT[audit0:];del DIMENSION_AUDIT[dim0:];del conflict_rows[conf0:]

SQL_DUCKDB_CONTRACT_REPORT=run_duckdb_sql_contracts()
REQUIRED_SQL_PREFLIGHT={'02::housing::00::00':['Y48010001','Y48011010'],'02::education::00::00':['Y48014001','Y48014003'],'02::budget::00::00':['Y48013001','Y48013002']}

def execute_real_sql_preflight():
    for aid,codes in REQUIRED_SQL_PREFLIGHT.items():
        if aid not in LOCAL_ASSETS:
            for code in codes:REAL_SQL_PREFLIGHT.append({'asset_id':aid,'code':code,'status':'NOT_DOWNLOADED','n_rows':0})
            continue
        a=get_asset(aid)
        for code in codes:
            try:
                row=sql_run('SELECT COUNT(*) FROM read_parquet(?) WHERE TRIM(CAST('+qident('indicator_code')+' AS VARCHAR))=?',[str(LOCAL_ASSETS[aid]),code],asset_id=aid,code=code,stage='REAL_PREFLIGHT_COUNT',preview=True)
                n=int(row[0]);REAL_SQL_PREFLIGHT.append({'asset_id':aid,'code':code,'status':'PASS' if n>0 else 'NO_CODE_ROWS','n_rows':n})
                raw,_,with_unit=factor_raw_sql(a)
                header_cols='indicator_name, indicator_unit' if 'indicator_unit' in a['columns_manifest'] else 'indicator_name'
                hdr=sql_run('SELECT '+header_cols+' FROM read_parquet(?) WHERE TRIM(CAST('+qident('indicator_code')+' AS VARCHAR))=? LIMIT 1',[str(LOCAL_ASSETS[aid]),code],asset_id=aid,code=code,stage='REAL_PREFLIGHT_HEADER')
                if len(hdr):
                    name=str(hdr.iloc[0]['indicator_name']).strip()
                    unit=str(hdr.iloc[0]['indicator_unit']).strip() if with_unit and pd.notna(hdr.iloc[0]['indicator_unit']) else ''
                    p=[str(LOCAL_ASSETS[aid]),code,name]+([unit] if with_unit else [])
                    observed=sql_run('SELECT COUNT(*) FROM ('+raw+') source_example',p,asset_id=aid,code=code,stage='REAL_PREFLIGHT_RAW_JOINED',preview=True)
                    joined=int(observed[0]);REAL_SQL_PREFLIGHT[-1]['n_joined']=joined
                    if joined==0:REAL_SQL_PREFLIGHT[-1]['status']='NO_MATCHED_MUNICIPAL_YEARS'
                    else:
                        dat,chosen_dims,validation=select_dimensional_slice(a,LOCAL_ASSETS[aid],code,name,unit,source_group(a))
                        if dat is None or dat.empty:
                            REAL_SQL_PREFLIGHT[-1]['status']='NO_VALIDATED_DIMENSIONAL_SLICE'
                            REAL_SQL_PREFLIGHT[-1]['slice_rejection_reason']=LAST_SLICE_REASON
                        elif dat.duplicated(['entity_id','observation_year']).any():
                            REAL_SQL_PREFLIGHT[-1]['status']='DUPLICATE_ENTITY_YEAR'
                        else:
                            REAL_SQL_PREFLIGHT[-1]['selected_keys']=int(len(dat))
                            REAL_SQL_PREFLIGHT[-1]['selected_nonmissing_values']=int(dat['value'].notna().sum())
                            REAL_SQL_PREFLIGHT[-1]['selected_dimensions_json']=json.dumps(chosen_dims,ensure_ascii=False)
                            if not dat['value'].notna().any():REAL_SQL_PREFLIGHT[-1]['status']='NO_NUMERIC_VALUES'
                else:REAL_SQL_PREFLIGHT[-1]['status']='NO_HEADER'
            except Exception as ex:
                REAL_SQL_PREFLIGHT.append({'asset_id':aid,'code':code,'status':'SCAN_ERROR','n_rows':0,'exception_type':type(ex).__name__,'message':str(ex)[:300]})
    return pd.DataFrame(REAL_SQL_PREFLIGHT)

REAL_SQL_PREFLIGHT_DF=execute_real_sql_preflight()
if CFG['MODE']=='FULL' and CFG['SQL_PREFLIGHT_REAL_CODES']:
    failed=REAL_SQL_PREFLIGHT_DF[~REAL_SQL_PREFLIGHT_DF.status.eq('PASS')]
    if len(failed):
        out=WORK/'ml03_sql_preflight_failures.json';out.write_text(json.dumps(REAL_SQL_PREFLIGHT,ensure_ascii=False,indent=2),encoding='utf-8')
        mark('REAL_STAGE1_SQL_PREFLIGHT','FAIL',f'Failed real SQL tests: {len(failed)}/6. Details: {out.name}')
        raise RuntimeError('FAIL_PIPELINE_SQL_PREFLIGHT: '+str(failed[['asset_id','code','status']].to_dict('records')))
mark('REAL_STAGE1_SQL_PREFLIGHT','PASS' if REAL_SQL_PREFLIGHT_DF.status.eq('PASS').all() else 'PARTIAL',f"real source code checks={len(REAL_SQL_PREFLIGHT_DF)}, passed={int(REAL_SQL_PREFLIGHT_DF.status.eq('PASS').sum())}; DuckDB fixture={SQL_DUCKDB_CONTRACT_REPORT['status']}")

progress('INVENTORY',0,len(FACTOR_ASSETS),'Аудит схем и индикаторов')
for _fi,a in enumerate(FACTOR_ASSETS,1):
    aid=a['asset_id'];groups=_grp_eligible_from_source(a)
    progress('INVENTORY',_fi,len(FACTOR_ASSETS),f'{_fi} из {len(FACTOR_ASSETS)} массивов')
    if aid not in LOCAL_ASSETS:
        SOURCE_QA.append({'asset_id':aid,'group':source_group(a),'status':'NOT_DOWNLOADED'});continue
    if not eligible_municipal_long_schema(a):
        SOURCE_QA.append({'asset_id':aid,'group':source_group(a),'status':'SCHEMA_NOT_ADAPTED','reason':'WIDE_OR_NON_MUNICIPAL_UNVALIDATED'});continue
    for g in groups:
        try:
            inv=inventory_asset(a,LOCAL_ASSETS[aid],g)
            SOURCE_QA.append({'asset_id':aid,'group':g,'status':'INVENTORIED_EXHAUSTIVE','n_candidates':len(inv),'cutoff_year':TRAIN_CATALOG_MAX_YEAR})
            if len(inv):
                inv['asset_id']=aid;inv['group']=g;CATALOG_ROWS.append(inv)
        except Exception as ex:
            SOURCE_QA.append({'asset_id':aid,'group':g,'status':'INVENTORY_ERROR','error_type':type(ex).__name__,'error_message':str(ex)[:1000]})
            log('INVENTORY_ERROR '+aid+': '+type(ex).__name__+' '+str(ex)[:400],'WARN')

progress('INVENTORY',1,1,'Инвентаризация завершена')
CATALOG=pd.concat(CATALOG_ROWS,ignore_index=True) if CATALOG_ROWS else pd.DataFrame()
if len(CATALOG):
    result=[classify_indicator(x['group'],x['asset_id'],x['indicator_name'],x['indicator_unit']) for x in CATALOG.to_dict('records')]
    CATALOG['domain']=[v[0] for v in result];CATALOG['semantic_rejection_reason']=[v[1] for v in result]
    for x in CATALOG.loc[CATALOG.domain.isna()].to_dict('records'):
        CANDIDATE_AUDIT.append({'asset_id':x['asset_id'],'code':x['indicator_code'],'name':x['indicator_name'],'decision':x['semantic_rejection_reason'],'slice':''})
    inspection_by_group=collections.Counter()
    _scan_total=int(CATALOG.loc[CATALOG.group.isin(('G1','G2','G3','G4'))&CATALOG.domain.notna()].shape[0])
    _scan_done=0
    progress('SCAN',0,_scan_total,'Размерные разрезы показателей')
    for g in ('G1','G2','G3','G4'):
        ranked=CATALOG.loc[CATALOG.group.eq(g)&CATALOG.domain.notna()].sort_values(['recent_keys','n_municipalities','n_years'],ascending=False)
        for idx,row in enumerate(ranked.to_dict('records')):
            _scan_done+=1
            if _scan_done%5==0 or _scan_done==_scan_total:
                progress('SCAN',_scan_done,_scan_total,f'{g}: {_scan_done}/{_scan_total}')
            if inspection_by_group[g]>=CFG['MAX_DIMENSIONAL_SCANS']:
                SCAN_AUDIT.append({'group':g,'asset_id':row['asset_id'],'code':row['indicator_code'],'status':'NOT_INSPECTED_CAP','reason':'RESOURCE_BUDGET','rank':idx})
                continue
            inspection_by_group[g]+=1;a=get_asset(row['asset_id'])
            try:
                dat,dims,support=select_dimensional_slice(a,LOCAL_ASSETS[a['asset_id']],row['indicator_code'],row['indicator_name'],row['indicator_unit'],g)
                reason='ELIGIBLE' if dat is not None else LAST_SLICE_REASON
                SCAN_AUDIT.append({'group':g,'asset_id':a['asset_id'],'code':row['indicator_code'],'status':reason,'rank':idx})
                if dat is None:continue
                feature_score=(int(support['n_recent_keys']),int(support['n_train_keys']))
                FEATURE_CANDIDATES.append({'group':g,'domain':row['domain'],'asset_id':a['asset_id'],'code':row['indicator_code'],'name':row['indicator_name'],'unit':row['indicator_unit'],'dims':dims,'support':support,'data':dat,'score':feature_score})
            except Exception as ex:
                SCAN_AUDIT.append({'group':g,'asset_id':a['asset_id'],'code':row['indicator_code'],'status':'SCAN_ERROR','reason':type(ex).__name__+': '+str(ex)[:400],'rank':idx})
                log('DIMENSION_SCAN '+a['asset_id']+': '+type(ex).__name__+' '+str(ex)[:400],'WARN')
    incomplete=any(r['status'] in ('NOT_INSPECTED_CAP','SCAN_ERROR') for r in SCAN_AUDIT)
else:incomplete=False
progress('SCAN',1,1,'Предметный отбор завершён')

SCAN_AUDIT=pd.DataFrame(SCAN_AUDIT);DIMENSION_AUDIT=pd.DataFrame(DIMENSION_AUDIT)
if len(CATALOG):
    g1=CATALOG.loc[CATALOG.group.eq('G1')].copy()
    if g1.empty:
        G1_DISCOVERY=pd.DataFrame([{'asset_id':a['asset_id'],'domain':'UNRESOLVED','discovered_codes':0,'discovered_rows':0,'reason_code':'UNADAPTED_MUNICIPAL_WIDE' if a['asset_id'] in UNADAPTED_MUNICIPAL_WIDE else 'NO_MATCHING_G1_INDICATORS_AT_VALIDATED_MUNICIPAL_GRAIN'} for a in ASSETS if a['asset_id']!='01::0'])
    g1['scan_key']=g1.asset_id.astype(str)+'|'+g1.indicator_code.astype(str)+'|'+g1.indicator_name.astype(str)
    insp=SCAN_AUDIT.loc[SCAN_AUDIT.group.eq('G1')].copy() if len(SCAN_AUDIT) else pd.DataFrame()
    if not g1.empty:
        G1_DISCOVERY=g1.groupby(['asset_id','domain'],dropna=False).agg(discovered_codes=('indicator_code','nunique'),discovered_rows=('indicator_name','size'),earliest_year=('min_year','min'),latest_year=('max_year','max'),max_municipalities=('n_municipalities','max')).reset_index()
    checked=insp.groupby('asset_id',dropna=False).status.value_counts().unstack(fill_value=0) if len(insp) else pd.DataFrame()
    if len(checked):
        for label in ('ELIGIBLE','NO_VALID_TOTAL_SLICE','NO_VALID_SLICE_AFTER_SUPPORT_CHECK','SCAN_ERROR','NOT_INSPECTED_CAP'):
            G1_DISCOVERY['n_'+label.lower()]=G1_DISCOVERY.asset_id.map(checked.get(label,pd.Series(dtype=int))).fillna(0).astype(int)
    G1_DISCOVERY['data_grain']='MUNICIPALITY_YEAR_EXACT_OKTMO_OR_NOT_VALIDATED'
    if not g1.empty:G1_DISCOVERY['reason_code']=np.where(G1_DISCOVERY.domain.isna(),'NO_SEMANTIC_MATCH','AUDIT_SLICES_IN_SCAN_AUDIT')
else:
    G1_DISCOVERY=pd.DataFrame([{'asset_id':a['asset_id'],'domain':None,'discovered_codes':0,'discovered_rows':0,'reason_code':'NO_MATCHED_MUNICIPAL_INDICATORS'} for a in ASSETS])

G1_EMPTY_CAUSE='NOT_FULLY_INSPECTED' if incomplete else 'UNADAPTED_MUNICIPAL_WIDE' if UNADAPTED_MUNICIPAL_WIDE else 'NO_VALIDATED_ECONOMIC_FEATURE_ON_STAGE_I'
DISCOVERY_EXHAUSTIVE=not incomplete and not UNADAPTED_MUNICIPAL_WIDE and (DIMENSION_AUDIT.empty or not DIMENSION_AUDIT.status.eq('CANDIDATE_BUDGET_PARTIAL').any()) and not any(r.get('status')=='INVENTORY_ERROR' for r in SOURCE_QA) and not SQL_FAILURES
mark('DISCOVERY_EXHAUSTIVENESS','PASS' if DISCOVERY_EXHAUSTIVE else 'PARTIAL',f'inspected={int(len(SCAN_AUDIT))}; unadapted_municipal_wide={len(UNADAPTED_MUNICIPAL_WIDE)}; incomplete={not DISCOVERY_EXHAUSTIVE}')

SOURCE_GRAIN_AUDIT=[]
for a in ASSETS:
    aid=a['asset_id'];records=[r for r in SOURCE_QA if r.get('asset_id')==aid]
    rawstage=next((r.get('status') for r in reversed(records) if r.get('status') in ('INVENTORY_ERROR','SCHEMA_NOT_ADAPTED','INVENTORIED_EXHAUSTIVE','NOT_DOWNLOADED')),None)
    stage=('NOT_DOWNLOADED' if aid not in LOCAL_ASSETS else 'TARGET_D01' if aid=='01::0' else 'SCANNED' if any(r.get('asset_id')==aid and r.get('status')=='ELIGIBLE' for r in SCAN_AUDIT.to_dict('records')) else 'SCAN_ERROR' if any(r.get('asset_id')==aid and r.get('status')=='SCAN_ERROR' for r in SCAN_AUDIT.to_dict('records')) else 'INVENTORIED_NO_MODEL_SLICE' if rawstage=='INVENTORIED_EXHAUSTIVE' else 'SOURCE_SCHEMA_ERROR' if rawstage=='INVENTORY_ERROR' else 'NOT_APPLICABLE_GRAIN' if not eligible_municipal_long_schema(a) else 'NOT_SEMANTICALLY_REVIEWED')
    SOURCE_GRAIN_AUDIT.append({'asset_id':aid,'downloaded_sha':aid in LOCAL_ASSETS,'columns_manifest_n':len(a.get('columns_manifest') or []),'source_grain':'MUNICIPAL_LONG_SCHEMA_AND_LEVEL_FILTER' if eligible_municipal_long_schema(a) and 'object_level' in (a.get('columns_manifest') or []) else 'MUNICIPAL_LONG_SCHEMA_NO_EXPLICIT_LEVEL' if eligible_municipal_long_schema(a) else 'NOT_VALIDATED_MUNICIPAL_LONG','stage':stage,'reason':rawstage or source_group(a)})

SOURCE_GRAIN_AUDIT=pd.DataFrame(SOURCE_GRAIN_AUDIT);SCAN_SQL_ERRORS=pd.DataFrame(SQL_FAILURES)
# The 52-source inventory is exhaustive at file/metadata level, not necessarily at validated content grain.
SOURCE_GRAIN_AUDIT['scientific_join_role']=SOURCE_GRAIN_AUDIT.stage.map(lambda s:'TARGET' if s=='TARGET_D01' else 'MUNICIPAL_LONG_CANDIDATE' if s in ('SCANNED','INVENTORIED_NO_MODEL_SLICE') else 'INDEX_ONLY_NOT_JOINED')
SCAN_SQL_AUDIT=pd.DataFrame(SQL_EXECUTION_AUDIT);REAL_SQL_PREFLIGHT_QA=pd.DataFrame(REAL_SQL_PREFLIGHT)
DATA_SMOKE_REPORT={'version':CFG['VERSION'],'asset_count':len(ASSETS),'scan_error_count':len(SQL_FAILURES),'inventory_errors':sum(r.get('status')=='INVENTORY_ERROR' for r in SOURCE_QA),'scanner_errors':int(SCAN_AUDIT.status.eq('SCAN_ERROR').sum()) if len(SCAN_AUDIT) else 0,'real_sql_preflight':REAL_SQL_PREFLIGHT,'dimension_audit':DIMENSION_AUDIT.to_dict('records') if len(DIMENSION_AUDIT) else []}
(WORK/'ml03_data_smoke.json').write_text(json.dumps(DATA_SMOKE_REPORT,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
fatal_sql=len(SQL_FAILURES)>0 or DATA_SMOKE_REPORT['inventory_errors']>0 or DATA_SMOKE_REPORT['scanner_errors']>0
mark('DATA_SMOKE','FAIL' if fatal_sql else 'PASS',f"SQL runtime errors={len(SQL_FAILURES)}, inventory errors={DATA_SMOKE_REPORT['inventory_errors']}, scanner errors={DATA_SMOKE_REPORT['scanner_errors']}")
if fatal_sql and CFG['MODE']=='FULL' and CFG['FAIL_FAST_SQL']:
    raise RuntimeError('FAIL_PIPELINE_SQL_SCAN: see '+str(WORK/'ml03_data_smoke.json')+' and SCAN_SQL_ERRORS')

selected_by_group={g:[] for g in PATTERNS};SELECTION_BY_H=[]
for g in ('G1','G2','G3','G4'):
    selected_candidates={}
    for h in sorted(FACTOR_CUTOFF_BY_H):
        cutoff=int(FACTOR_CUTOFF_BY_H[h]);candidates=[]
        for c in FEATURE_CANDIDATES:
            if c['group']!=g:continue
            z=c['data'].loc[c['data'].observation_year.le(cutoff)&c['data']['value'].notna()]
            years=int(z.observation_year.nunique());num=int(len(z));mun=int(z.entity_id.nunique())
            recent=int(z.observation_year.ge(cutoff-CFG['CATALOG_RECENT_WINDOW']).sum())
            if num<CFG['MIN_INDICATOR_OBS_TRAIN'] or years<CFG['MIN_INDICATOR_YEARS_TRAIN'] or mun<CFG['MIN_INDICATOR_MUNICIPALITIES']:continue
            candidates.append({**c,'_h_score':(recent,num,mun),'_h_rows':num})
        candidates.sort(key=lambda c:c['_h_score'],reverse=True)
        source_counts=collections.Counter();domain_counts=collections.Counter();h_selected=[];pool=candidates.copy()
        for diversified in (True,False):
            while pool and len(h_selected)<CFG['MAX_FEATURES_PER_GROUP']:
                opts=[x for x in pool if source_counts[x['asset_id']]<CFG['MAX_FEATURES_FROM_ONE_ASSET'] and (not diversified or domain_counts[x['domain']]==0)]
                if not opts:break
                chosen=max(opts,key=lambda z:(z['_h_score'][0]/(1+source_counts[z['asset_id']]),z['_h_score'][1],-domain_counts[z['domain']]))
                pool.pop(next(i for i,x in enumerate(pool) if x is chosen))
                source_counts[chosen['asset_id']]+=1;domain_counts[chosen['domain']]+=1
                key=(chosen['asset_id'],str(chosen['code']),chosen['name'],chosen['unit'],json.dumps(chosen['dims'],ensure_ascii=False,sort_keys=True))
                if key not in selected_candidates:
                    feature=f'{g}_{len(selected_by_group[g])+1:02d}';data=chosen['data'].rename(columns={'value':feature})
                    ensure(not data.duplicated(['entity_id','observation_year']).any(),'FEATURE_UNIQUE_KEY_'+feature)
                    meta={'feature':feature,'group':g,'domain':chosen['domain'],'asset_id':chosen['asset_id'],'indicator_code':chosen['code'],'indicator_name':chosen['name'],'unit':chosen['unit'],'dimensions_json':key[-1],'observed_grain':'MUNICIPAL_OKTMO_YEAR_DIMENSION_SLICE','report_available_year':'NOT_VERIFIED','availability_provenance':'CONSERVATIVE_OBSERVATION_PLUS_LAG_NOT_OFFICIAL_PUBLICATION_DATE','effective_available_year_rule':f'observation_year+{CFG["FEATURE_PUBLICATION_LAG"]}','n_observed':int(data[feature].notna().sum()),'n_recent_train_keys':int(chosen['_h_score'][0]),'first_year':int(data.observation_year.min()),'last_year':int(data.observation_year.max()),'transform':'RAW_PRESERVED; UNIT_QA_REQUIRED','selected_horizons':[]}
                    selected_candidates[key]=meta;selected_by_group[g].append(meta);FEATURE_PARTS.append(data)
                meta=selected_candidates[key];meta['selected_horizons'].append(h);h_selected.append(meta['feature'])
            if not diversified:break
        SELECTION_BY_H.append({'group':g,'horizon':h,'max_observation_year':cutoff,'n_eligible_candidates':len(candidates),'n_selected':len(h_selected),'selected_features_json':json.dumps(h_selected,ensure_ascii=False),'domain_coverage_json':json.dumps(sorted(domain_counts),ensure_ascii=False),'status':'COMPLETE_SELECTION' if h_selected else 'NO_VALIDATED_CANDIDATES'})
    for meta in selected_by_group[g]:meta['selected_horizons_json']=json.dumps(sorted(set(meta.pop('selected_horizons'))))
SELECTION_BY_H=pd.DataFrame(SELECTION_BY_H)

from sklearn.neighbors import BallTree
SPATIAL_GEOMETRY_QA=[];SPATIAL_FEATURE_AUDIT=[];SPATIAL_FUNDAMENTAL_GATE='NOT_FEASIBLE'

def _is_municipal_grain(v):
    x=str(v).casefold().strip().replace('ё','е')
    return bool(re.search(r'(^municipality$|^municipal$|муниципальн.{0,15}(?:район|округ|образован)|городск.{0,8}округ)',x))

def _true_flag(v):return str(v).strip().casefold() in ('true','1','да','yes','истина')

def _spatial_snapshot_candidates(source,demography):
    audit={'n_source':int(len(source)),'n_matched_municipal':0,'n_verified':0,'n_distinct':0,'n_conflicting_codes':0,'n_ambiguous_regions':0,'status':'NOT_FEASIBLE'}
    required={'source_oktmo','object_level','region_name','latitude','longitude','coordinates_valid','census_year'}
    if not required.issubset(source.columns):
        audit['reason']='SPATIAL_COLUMN_CONTRACT_MISSING';return pd.DataFrame(),audit
    z=source[list(required)+(['population_original'] if 'population_original' in source else [])].copy()
    z=z.rename(columns={'source_oktmo':'entity_id','census_year':'snapshot_year'})
    z['entity_id']=z.entity_id.astype('string').str.strip()
    z=z[z.entity_id.notna()&z.entity_id.str.fullmatch(r'\d{5,11}',na=False)].copy()
    z=z[z.object_level.map(_is_municipal_grain)&z.coordinates_valid.map(_true_flag)].copy()
    for c in ['latitude','longitude','snapshot_year']:z[c]=pd.to_numeric(z[c],errors='coerce')
    z=z[z.latitude.between(-89.999,89.999)&z.longitude.between(-179.999,179.999)&z.snapshot_year.between(1990,max(YEARS))].copy()
    audit['n_matched_municipal']=int(len(z))
    if z.empty:
        audit['reason']='NO_VERIFIED_MUNICIPAL_GEOMETRY_OR_VINTAGE';return pd.DataFrame(),audit
    z['snapshot_year']=z.snapshot_year.astype(int)
    d=demography[['entity_id','region_name']].drop_duplicates()
    region_conflicts=d.groupby('entity_id').region_name.nunique(dropna=False)
    ambiguous=set(region_conflicts[region_conflicts.ne(1)].index);audit['n_ambiguous_regions']=len(ambiguous)
    d=d[~d.entity_id.isin(ambiguous)]
    z=z.merge(d,on='entity_id',how='inner',validate='many_to_one',suffixes=('_ref','_d01'))
    def norm_region(x):return re.sub(r'\s+',' ',str(x).casefold().replace('ё','е')).strip()
    z=z[z.region_name_ref.map(norm_region).eq(z.region_name_d01.map(norm_region))].copy()
    if z.empty:
        audit['reason']='NO_EXACT_REGION_AND_OKTMO_MATCH';return pd.DataFrame(),audit
    counts=z.groupby(['entity_id','snapshot_year']).agg(n_lat=('latitude','nunique'),n_lon=('longitude','nunique'))
    conflicted=set(counts.index[counts.n_lat.gt(1)|counts.n_lon.gt(1)].tolist())
    audit['n_conflicting_codes']=len(conflicted)
    if conflicted:z=z[~pd.MultiIndex.from_frame(z[['entity_id','snapshot_year']]).isin(conflicted)]
    z=z.drop_duplicates(['entity_id','snapshot_year','latitude','longitude']).sort_values(['entity_id','snapshot_year'])
    audit['n_verified']=len(z);audit['n_distinct']=z.entity_id.nunique()
    if z.entity_id.nunique()<max(30,int(CFG['MIN_INDICATOR_MUNICIPALITIES'])):
        audit['reason']='TOO_FEW_EXACT_VERIFIED_MUNICIPALITY_COORDS';return pd.DataFrame(),audit
    audit['status']='PASS';audit['reason']='TIME_STAMPED_VERIFIED_GEO_EXACT_MUNICIPAL_OKTMO_REGION'
    return z[['entity_id','snapshot_year','latitude','longitude','region_name_d01']],audit

def _snapshot_knn(z):
    out=[];earth=6371.0088
    for year in sorted(z.snapshot_year.unique()):
        available=z[z.snapshot_year.le(year)].sort_values('snapshot_year').drop_duplicates('entity_id',keep='last')
        this=z[z.snapshot_year.eq(year)].copy()
        if len(available)<5:continue
        latlon=np.radians(available[['latitude','longitude']].to_numpy(dtype=float))
        query=np.radians(this[['latitude','longitude']].to_numpy(dtype=float))
        tree=BallTree(latlon,metric='haversine');dist,idx=tree.query(query,k=min(5,len(available)))
        for i,v in enumerate(this.itertuples(index=False)):
            neighbours=[(float(dd)*earth,available.iloc[int(j)].entity_id) for dd,j in zip(dist[i],idx[i]) if available.iloc[int(j)].entity_id!=v.entity_id]
            if not neighbours:continue
            nearby=tree.query_radius(query[i:i+1],r=50/earth)[0]
            n_within_50=sum(available.iloc[int(j)].entity_id!=v.entity_id for j in nearby)
            out.append({'entity_id':v.entity_id,'snapshot_year':int(year),'nearest_municipality_km':neighbours[0][0],'municipalities_50km_n':int(n_within_50),'known_neighbour_universe_n':len(available)})
    return pd.DataFrame(out)

_SPATIAL_SOURCE_ID='06::D04::D04_member00::spatial_reference::derived'
if _SPATIAL_SOURCE_ID in LOCAL_ASSETS:
    try:
        spatial_source=pd.read_parquet(LOCAL_ASSETS[_SPATIAL_SOURCE_ID],columns=['source_oktmo','object_level','region_name','latitude','longitude','coordinates_valid','census_year'])
        geo,qa=_spatial_snapshot_candidates(spatial_source,base)
        SPATIAL_GEOMETRY_QA.append({'asset_id':_SPATIAL_SOURCE_ID,**qa})
        if not geo.empty:
            near=_snapshot_knn(geo)
            if not near.empty:
                pubyear=CFG.get('SPATIAL_PUBLICATION_YEAR')
                pubref=str(CFG.get('SPATIAL_PUBLICATION_EVIDENCE','')).strip()
                if pubyear is None or not pubref or not CFG.get('SPATIAL_BOUNDARY_CROSSWALK_VERIFIED',False) or not str(CFG.get('SPATIAL_BOUNDARY_CROSSWALK_EVIDENCE','')).strip():
                    SPATIAL_GEOMETRY_QA.append({'asset_id':_SPATIAL_SOURCE_ID,'status':'NOT_FEASIBLE','reason':'PUBLICATION_YEAR_OR_BOUNDARY_CROSSWALK_UNVERIFIED','n_verified_geometry':int(len(near))})
                    near=pd.DataFrame()
            if not near.empty and int(pubyear)>max(YEARS):
                SPATIAL_GEOMETRY_QA.append({'asset_id':_SPATIAL_SOURCE_ID,'status':'NOT_FEASIBLE','reason':'VERIFIED_PUBLICATION_AFTER_LAST_TARGET_YEAR'})
                near=pd.DataFrame()
            if not near.empty:
                yrs=pd.DataFrame({'observation_year':YEARS})
                snapshots=near.loc[near.snapshot_year.le(max(YEARS))].copy().merge(yrs,how='cross')
                snapshots=snapshots.loc[snapshots.observation_year.ge(np.maximum(snapshots.snapshot_year,int(pubyear)))]
                snapshots=snapshots.sort_values('snapshot_year').drop_duplicates(['entity_id','observation_year'],keep='last')
                for f,un,domain,col in (('G5_01_km','км','spatial_distance','nearest_municipality_km'),('G5_02_neighbors','число муниципалитетов','spatial_density','municipalities_50km_n')):
                    data=snapshots[['entity_id','observation_year',col]].rename(columns={col:f})
                    data[f]=pd.to_numeric(data[f],errors='coerce')
                    ensure(not data.duplicated(['entity_id','observation_year']).any(),'SPATIAL_TIME_KEY_DUPLICATE')
                    FEATURE_PARTS.append(data)
                    selected_by_group['G5'].append({'feature':f,'group':'G5','domain':domain,'asset_id':_SPATIAL_SOURCE_ID,'indicator_code':'D04_VERIFIED_GEO_'+domain.upper(),'indicator_name':'Подтверждённое пространственное расстояние' if domain=='spatial_distance' else 'Муниципалитеты в радиусе 50 км','unit':un,'dimensions_json':'{}','observed_grain':'MUNICIPAL_GEOMETRY_VINTAGE','report_available_year':f'max(snapshot_year,{pubyear})+publication_lag','effective_available_year_rule':f'max(census_year,{pubyear})+{CFG["FEATURE_PUBLICATION_LAG"]}<=origin_year','n_observed':int(data[f].notna().sum()),'n_recent_train_keys':int(data[data.observation_year.le(TRAIN_CATALOG_MAX_YEAR)][f].notna().sum()),'first_year':int(data.observation_year.min()),'last_year':int(data.observation_year.max()),'transform':'SOURCE_VINTAGE_GEOMETRY; NO FUTURE_COORDINATES'})
                    SPATIAL_FEATURE_AUDIT.append({'feature':f,'n_rows':len(data),'n_municipalities':data.entity_id.nunique(),'snapshot_min':int(near.snapshot_year.min()),'snapshot_max':int(near.snapshot_year.max()),'model_eligible':'PRELIMINARY_TIMING_GATE'})
                SPATIAL_FUNDAMENTAL_GATE='PASS'
    except Exception as ex:
        SPATIAL_GEOMETRY_QA.append({'asset_id':_SPATIAL_SOURCE_ID,'status':'NOT_FEASIBLE','reason':'SPATIAL_REFERENCE_REJECTED_'+type(ex).__name__})
        log('SPATIAL_DATA_GATE: '+type(ex).__name__+': '+str(ex)[:250],'WARN')
else:
    SPATIAL_GEOMETRY_QA.append({'asset_id':_SPATIAL_SOURCE_ID,'status':'NOT_FEASIBLE','reason':'SPATIAL_VINTAGE_FILE_NOT_DOWNLOADED'})

SPATIAL_GEOMETRY_QA.append({'asset_id':'06::TRANSPORT::TRANSPORT_member00::full','status':'NOT_FEASIBLE','reason':'TRANSPORT_OBSERVATION_YEAR_NOT_PROVIDED'})
SPATIAL_FEATURE_AUDIT=pd.DataFrame(SPATIAL_FEATURE_AUDIT);SPATIAL_GEOMETRY_QA=pd.DataFrame(SPATIAL_GEOMETRY_QA)
G5_DIAGNOSTICS.extend(SPATIAL_GEOMETRY_QA.to_dict('records'))

# V3.00: independent region-level economic CONTEXT. Never claim these are
# municipality-specific wages/employment. Strict object-level and exact region/year.
REGIONAL_CONTEXT_PARTS=[];REGIONAL_CONTEXT_META=[];REGIONAL_CONTEXT_AUDIT=[]
D05_ECONOMIC_SOURCES=('02::labor::','02::socio1::','02::socio2::','02::socio3::','02::social::')
def _region_norm(s):return re.sub(r'\s+',' ',str(s).casefold().replace('ё','е')).strip()
def _region_context_collect():
    if not CFG.get('ENABLE_REGIONAL_ECONOMIC_CONTEXT'):return
    for asset in ASSETS:
        aid=asset['asset_id']
        if aid not in LOCAL_ASSETS or not aid.startswith(D05_ECONOMIC_SOURCES):continue
        cols=set(asset.get('columns_manifest') or [])
        if not {'object_level','object_name','indicator_name','indicator_value','year','indicator_code','indicator_unit'}.issubset(cols):continue
        sql="""SELECT CAST(object_name AS VARCHAR) region, TRY_CAST(year AS INTEGER) yr,
             CAST(indicator_code AS VARCHAR) code, CAST(indicator_name AS VARCHAR) title,
             CAST(indicator_unit AS VARCHAR) unit, CAST(indicator_value AS VARCHAR) val,
             CAST(object_level AS VARCHAR) object_level
             FROM read_parquet(?) WHERE TRY_CAST(year AS INTEGER) BETWEEN ? AND ?
             AND regexp_matches(lower(coalesce(cast(indicator_name AS VARCHAR),'')),?)"""
        try:
            x=sql_run(sql,[str(LOCAL_ASSETS[aid]),min(YEARS),max(YEARS),PATTERNS['G1']],asset_id=aid,stage='REGION_CONTEXT_DISCOVERY')
        except Exception as exc:
            REGIONAL_CONTEXT_AUDIT.append({'asset_id':aid,'status':'SQL_FAILURE','reason':type(exc).__name__});continue
        if x.empty:
            REGIONAL_CONTEXT_AUDIT.append({'asset_id':aid,'status':'NO_G1_CANDIDATES'});continue
        lev=x.object_level.astype('string').str.casefold().fillna('')
        x=x[lev.str.contains(r'регион|субъект',regex=True,na=False)].copy()
        if x.empty:
            REGIONAL_CONTEXT_AUDIT.append({'asset_id':aid,'status':'NO_VALID_REGION_ROWS'});continue
        x['region_key']=x.region.map(_region_norm);x['yr']=pd.to_numeric(x.yr,errors='coerce')
        x=x[x.yr.notna()].copy();x['yr']=x.yr.astype(int)
        regions={_region_norm(v) for v in base.region_name.dropna().unique()}
        x=x[x.region_key.isin(regions)].copy()
        if x.empty:
            REGIONAL_CONTEXT_AUDIT.append({'asset_id':aid,'status':'REGION_NAMES_NOT_EXACT'});continue
        x['value']=pd.to_numeric(x.val.astype('string').str.replace(' ','',regex=False).str.replace(',','.',regex=False),errors='coerce')
        x=x[x.value.notna()];candidates=0
        for (code,title,unit),s in x.groupby(['code','title','unit'],dropna=False,sort=False):
            domain,_=classify_indicator('G1',aid,title,unit)
            if domain is None:continue
            # Only intensive regional quantities: an explicit rate, percentage,
            # average wage or average per-capita income; no regional totals.
            text=(str(title)+' '+str(unit)).casefold()
            intensive=bool(re.search(r'средн|на душу|на одного|в расчете на|уровень безработ|коэффициент|процент|%|дол[яи]',text))
            if not intensive:continue
            key=['region_key','yr'];span=s.groupby(key)['value'].agg(['size','nunique','first']).reset_index()
            clashes=int(span['nunique'].gt(1).sum())
            if clashes:
                REGIONAL_CONTEXT_AUDIT.append({'asset_id':aid,'code':str(code),'status':'CONFLICTING_REGION_YEAR','n_conflicts':clashes});continue
            z=span[['region_key','yr','first']].rename(columns={'yr':'observation_year'})
            before=z[z.observation_year.le(TRAIN_CATALOG_MAX_YEAR)]
            if before.region_key.nunique()<CFG['REGIONAL_ECONOMIC_MIN_REGIONS'] or before.observation_year.nunique()<2:continue
            f=f'G1R_{len(REGIONAL_CONTEXT_META)+1:02d}'
            z=z.rename(columns={'first':f})
            REGIONAL_CONTEXT_META.append({'feature':f,'group':'G1R','domain':domain,
                 'source_grain':'REGION_YEAR_ONLY','source_id':aid,'indicator_code':str(code),
                 'indicator_name':str(title),'unit':str(unit),'n_regions':int(z.region_key.nunique()),
                 'n_years':int(z.observation_year.nunique()),'n_conflicts':clashes,
                 'note':'NOT_MUNICIPAL_G1; no region-to-municipality economic reconstruction'})
            REGIONAL_CONTEXT_PARTS.append(z)
            candidates+=1
            if len(REGIONAL_CONTEXT_META)>=CFG['REGIONAL_ECONOMIC_MAX_FEATURES']:break
        REGIONAL_CONTEXT_AUDIT.append({'asset_id':aid,'status':'AUDITED_REGION_CONTEXT',
            'candidate_rows':int(len(x)),'accepted_features_from_source':candidates})
        if len(REGIONAL_CONTEXT_META)>=CFG['REGIONAL_ECONOMIC_MAX_FEATURES']:break
_region_context_collect()
REGIONAL_CONTEXT_AUDIT=pd.DataFrame(REGIONAL_CONTEXT_AUDIT)
REGIONAL_CONTEXT_METADATA=pd.DataFrame(REGIONAL_CONTEXT_META)
mark('G1_REGIONAL_CONTEXT_DISCOVERY','PASS' if REGIONAL_CONTEXT_META else 'NOT_FEASIBLE',
     f'{len(REGIONAL_CONTEXT_META)} exact-match region-year economic indicators, strictly separate from municipal G1')

# Stage I factor recovery and complete diagnostics are executed in R1 before creating the merged panel.
FEATURE_DICT=pd.DataFrame(sum(selected_by_group.values(),[]))
SOURCE_QA=pd.DataFrame(SOURCE_QA);CONFLICT_QA=pd.DataFrame(conflict_rows)
SPATIAL_SOURCE_QA=pd.DataFrame(G5_DIAGNOSTICS);SEMANTIC_QA=pd.DataFrame(CANDIDATE_AUDIT)
progress('FACTORS',0,4,'Первичный отбор; проверка сальдо, дорог и семантики впереди')
DOMAIN_COVERAGE=pd.DataFrame([{'group':g,'domain':d,'n_selected':sum(x['domain']==d for x in selected_by_group[g])} for g in SEMANTIC_DOMAINS for d,_ in SEMANTIC_DOMAINS[g]])
for g in PATTERNS:
    domains={r['domain'] for r in selected_by_group[g]}
    required=2 if g=='G3' else 1
    status='PASS' if len(domains)>=required else 'PARTIAL' if domains else 'NOT_FEASIBLE'
    if g=='G1' and not DISCOVERY_EXHAUSTIVE:status='PARTIAL'
    mark('FEATURE_'+g,status,f'{len(selected_by_group[g])} valid indicators; domains={sorted(domains)}')

#### V5: проверка бюджетного сальдо, дорожных показателей и исправление классификации.

In [6]:
# R1 · V5 · ДОИЗВЛЕЧЕНИЕ ИЗ СТАДИИ I: БЮДЖЕТ, ДОРОГИ, ПРЕДМЕТНАЯ КЛАССИФИКАЦИЯ
# Результат исследовательский: не приписывать доле субсидий статус «совокупные доходы».
V5_RECOVERY_ROWS=[]
V5_BUDGET_BALANCE_QA=[]
V5_ROADS_QA=[]
V5_SEMANTIC_FIXES=[]
V5_PUBLICATION_EVIDENCE=[]

# Контрактный тест по точным названиям, на которых выявлены ошибки V4.03.
assert classify_indicator('G1','SUPP::BDMO::9::000','Инвестиции в основной капитал, осуществляемые организациями','Тысяча рублей')[0]=='investment'
assert classify_indicator('G3','SUPP::BDMO::19::000','Численность лиц, обслуживаемых специализированными отделениями социального обслуживания','Человек')[0]=='social'

for g,records in selected_by_group.items():
    for meta in records:
        title=str(meta.get('indicator_name',''))
        old=meta.get('domain')
        dom,_=classify_indicator(g,meta.get('asset_id',''),title,meta.get('unit',''))
        if g=='G4':
            dims=json.loads(meta.get('dimensions_json','{}') or '{}')
            details=' '.join(str(v) for v in dims.values()).casefold()
            if re.search(r'субсид|дотаци|субвенци|трансферт',details):
                dom='intergovernmental'
                meta['disaggregated_budget_item']='INTERGOVERNMENTAL_TRANSFER_NOT_TOTAL_REVENUE'
            elif re.search(r'образован|здравоохран|культур|социал',details):
                meta['disaggregated_budget_item']='FUNCTIONAL_BUDGET_EXPENDITURE_NOT_TOTAL'
            elif any(not re.search(r'^(?:cd|∅|всего|итого|total)$',str(v).strip(),re.I) for v in dims.values()):
                meta['disaggregated_budget_item']='BUDGET_DIMENSION_SLICE_NOT_TOTAL'
        if dom and dom!=old:
            meta['domain']=dom
            V5_SEMANTIC_FIXES.append({'feature':meta['feature'],'old_domain':old,'new_domain':dom,
                'indicator_code':meta.get('indicator_code'), 'title':title,'reason':'EXACT_SEMANTIC_RECLASSIFICATION'})

progress('FACTORS',1,4,'Проверены предметные категории исходных показателей')

# Доходы/расходы местного бюджета: только строго подтверждённые итоговые срезы.
_BUDGET='02::budget::00::00'
_BUDGET_CODES={'Y48013001':'revenue','Y48013002':'expenditure'}
_TOTAL_RE=r'^(?:всего(?:\s+(?:доходов|расходов))?|итого|все(?:\s+(?:доходы|расходы))?|общая\s+сумма|суммарно)$'
_NULL_RE=r'^(?:cd|∅|nan|none|не\s+применимо|не\s+предусмотрено|нет|)$'

def _v5_budget_verified_total(code):
    if _BUDGET not in LOCAL_ASSETS:
        return None,'SOURCE_NOT_PRESENT',{}
    a=get_asset(_BUDGET)
    # Полный список размерностей: гарантирует, что мы не суммируем несовместимые бюджеты.
    raw,dims,with_unit=factor_raw_sql(a)
    if not set(dims).issuperset({'stdohod','strashod'}):
        return None,'UNEXPECTED_BUDGET_SCHEMA',{'dims':dims}
    sub=CATALOG.loc[CATALOG.asset_id.eq(_BUDGET)&CATALOG.indicator_code.astype(str).eq(code)]
    if sub.empty:return None,'INDICATOR_CODE_NOT_IN_CATALOG',{}
    sub=sub.sort_values('recent_keys',ascending=False).iloc[0]
    params=[str(LOCAL_ASSETS[_BUDGET]),str(code),str(sub.indicator_name)]+([str(sub.indicator_unit)] if with_unit else [])
    dd=['dim_'+str(i).zfill(2) for i in range(len(dims))]
    # Проверить только комбинации размерностей, представленные до тестового календаря.
    values=sql_run('WITH raw AS ('+raw+') SELECT '+','.join(qident(x) for x in dd)+', COUNT(*) n FROM raw WHERE observation_year<=? GROUP BY '+','.join(qident(x) for x in dd)+' ORDER BY n DESC',
            params+[TRAIN_CATALOG_MAX_YEAR],asset_id=_BUDGET,code=code,stage='V5_BUDGET_TOTAL_DISCOVERY')
    qualified=[]
    for z in values.to_dict('records'):
        d=dict(zip(dims,[str(z[x]) for x in dd]))
        nonnull=[v for v in d.values() if not re.match(_NULL_RE,v.strip(),re.I)]
        if not nonnull:  # Все `CD` и `∅` не означают автоматически «Всего».
            continue
        if all(re.match(_TOTAL_RE,v.strip(),re.I) for v in nonnull): qualified.append((d,int(z['n'])))
    if len(qualified)!=1:
        return None,'TOTAL_DIMENSION_AMBIGUOUS_OR_MISSING',{'n_qualified':len(qualified),'dimension_options':qualified[:12]}
    dims_sel,_=qualified[0]
    pred=' AND '.join(qident(dd[i])+'=?' for i in range(len(dims)))
    x=sql_run('WITH raw AS ('+raw+') SELECT entity_id,observation_year,MIN(val) v,COUNT(DISTINCT val) n_values FROM raw WHERE '+pred+' GROUP BY 1,2',
              params+[dims_sel[d] for d in dims],asset_id=_BUDGET,code=code,stage='V5_VERIFIED_BUDGET_TOTAL')
    if x.empty:return None,'EMPTY_VERIFIED_TOTAL',{}
    conflict=x.n_values.gt(1)
    if conflict.any():return None,'CONFLICTING_BUDGET_TOTAL_KEYS',{'n_conflicts':int(conflict.sum())}
    x=x[['entity_id','observation_year','v']].copy()
    x['v']=pd.to_numeric(x.v,errors='coerce')
    if x.loc[x.observation_year.le(TRAIN_CATALOG_MAX_YEAR),'v'].notna().sum()<CFG['MIN_INDICATOR_OBS_TRAIN']:
        return None,'LOW_PRETEST_SUPPORT',{'n_rows':len(x)}
    return x,'EXACT_TOTAL_DIMENSION',{'dims':dims_sel,'unit':str(sub.indicator_unit),'n_rows':len(x)}

def _v5_safe_budget_total(code):
    try:return _v5_budget_verified_total(code)
    except Exception as exc:
        # Failure of OPTIONAL source enrichment cannot erase the verified V4 model bank.
        log('V5_BUDGET_TOTAL_EXTRACTION_FAILED '+str(code)+' '+type(exc).__name__,'WARN')
        return None,'SOURCE_QUERY_ERROR',{'code':code,'exception':type(exc).__name__,'reason':str(exc)[:180]}

if CFG['ENABLE_STAGE1_BUDGET_BALANCE']:
    _income,_st_income,_meta_income=_v5_safe_budget_total('Y48013001')
    _outgo,_st_outgo,_meta_outgo=_v5_safe_budget_total('Y48013002')
    V5_BUDGET_BALANCE_QA.extend([{'role':'revenue','status':_st_income,**_meta_income},
                                  {'role':'expenditure','status':_st_outgo,**_meta_outgo}])
    if (_income is not None and _outgo is not None and
         _meta_income.get('unit')==_meta_outgo.get('unit')):
        v5b=_income.rename(columns={'v':'income'}).merge(_outgo.rename(columns={'v':'spending'}),on=['entity_id','observation_year'],how='inner',validate='one_to_one')
        v5b=v5b[v5b.income.notna()&v5b.spending.notna()].copy()
        v5b['G4_BALANCE']=v5b.income-v5b.spending
        count=v5b.observation_year.le(TRAIN_CATALOG_MAX_YEAR)&v5b.G4_BALANCE.notna()
        meta={'feature':'G4_BALANCE','group':'G4','domain':'budget_balance','asset_id':_BUDGET,
              'indicator_code':'Y48013001_MINUS_Y48013002',
              'indicator_name':'Сальдо местного бюджета: подтверждённые итоговые доходы минус итоговые расходы',
              'unit':_meta_income['unit'],'dimensions_json':json.dumps({'revenue':_meta_income['dims'],'expenditure':_meta_outgo['dims']},ensure_ascii=False),
              'observed_grain':'MUNICIPAL_OKTMO_YEAR_TWO_VERIFIED_TOTALS',
              'report_available_year':'NOT_VERIFIED','availability_provenance':'OBSERVATION_PLUS_LAG_NOT_OFFICIAL_RELEASE',
              'effective_available_year_rule':f'observation_year+{CFG["FEATURE_PUBLICATION_LAG"]}',
              'n_observed':int(v5b.G4_BALANCE.notna().sum()),'n_recent_train_keys':int(count.sum()),
              'first_year':int(v5b.observation_year.min()) if len(v5b) else 0,
              'last_year':int(v5b.observation_year.max()) if len(v5b) else 0,
              'transform':'DERIVED_DIFFERENCE_OF_TWO_VERIFIED_BUDGET_TOTALS',
              'selected_horizons_json':json.dumps(CFG['HORIZONS'])}
        if count.sum()>=CFG['MIN_INDICATOR_OBS_TRAIN'] and v5b.loc[count,'observation_year'].nunique()>=CFG['MIN_INDICATOR_YEARS_TRAIN']:
            selected_by_group['G4'].append(meta)
            FEATURE_PARTS.append(v5b[['entity_id','observation_year','G4_BALANCE']]);
            V5_BUDGET_BALANCE_QA.append({'role':'derived_balance','status':'ACCEPTED_VERIFIED_TOTAL_PAIR','n_rows':len(v5b)})
        else:V5_BUDGET_BALANCE_QA.append({'role':'derived_balance','status':'NOT_ELIGIBLE_LOW_SUPPORT'})
    else:
        V5_BUDGET_BALANCE_QA.append({'role':'derived_balance','status':'EXCLUDED_NO_VERIFIED_MATCHING_TOTAL_PAIR'})

progress('FACTORS',2,4,'Доходы и расходы бюджетов проверены по полным разрезам')

# Дорожные показатели: фактические наблюдения БДМО6 с датированием, не «координатная реконструкция».
_ROADS='06::BDMO6::BDMO6_member00::full'
if CFG['ENABLE_STAGE1_ROAD_DISCOVERY'] and _ROADS in LOCAL_ASSETS:
    try:
        aa=get_asset(_ROADS)
        inv=inventory_asset(aa,LOCAL_ASSETS[_ROADS],'G5')
        vr=inv.loc[inv.indicator_name.astype(str).str.contains(r'доро[гж]|автомобил.{0,14}дорог',case=False,regex=True,na=False)].sort_values('recent_keys',ascending=False)
        V5_ROADS_QA.append({'asset_id':_ROADS,'status':'SCANNED','n_catalog_candidates':len(vr)})
        used=0
        for rec in vr.head(15).to_dict('records'):
            dom,_=classify_indicator('G5',_ROADS,rec['indicator_name'],rec['indicator_unit'])
            if dom!='roads':continue
            dat,dim,detail=select_dimensional_slice(aa,LOCAL_ASSETS[_ROADS],rec['indicator_code'],rec['indicator_name'],rec['indicator_unit'],'G5')
            if dat is None or dat.empty:continue
            fname='G5_ROADS_'+str(used+1).zfill(2)
            dat=dat.rename(columns={'value':fname})
            years=sorted(dat.observation_year.unique()); train=dat.observation_year.le(TRAIN_CATALOG_MAX_YEAR)&dat[fname].notna()
            if train.sum()<CFG['MIN_INDICATOR_OBS_TRAIN'] or dat.loc[train,'observation_year'].nunique()<2:continue
            selected_by_group['G5'].append({'feature':fname,'group':'G5','domain':'roads','asset_id':_ROADS,
              'indicator_code':rec['indicator_code'],'indicator_name':rec['indicator_name'],'unit':rec['indicator_unit'],
              'dimensions_json':json.dumps(dim,ensure_ascii=False),'observed_grain':'BDMO6_MUNICIPAL_YEAR_DIMENSION_SLICE',
              'report_available_year':'NOT_VERIFIED','availability_provenance':'OBSERVATION_PLUS_LAG_RETROSPECTIVE_ONLY',
              'effective_available_year_rule':f'observation_year+{CFG["FEATURE_PUBLICATION_LAG"]}',
              'n_observed':int(dat[fname].notna().sum()),'n_recent_train_keys':int(train.sum()),
              'first_year':min(years),'last_year':max(years),'transform':'SOURCE_ROAD_INDICATOR_NO_SPATIAL_IMPUTATION',
              'selected_horizons_json':json.dumps(CFG['HORIZONS'])})
            FEATURE_PARTS.append(dat[['entity_id','observation_year',fname]])
            V5_ROADS_QA.append({'asset_id':_ROADS,'code':rec['indicator_code'],'feature':fname,
                'status':'SELECTED_RETROSPECTIVE_ROADS','n_rows':len(dat),'unit':rec['indicator_unit']})
            used+=1
            if used>=4:break
    except Exception as exc:
        V5_ROADS_QA.append({'asset_id':_ROADS,'status':'QUERY_FAILED','reason':type(exc).__name__+': '+str(exc)[:200]})

progress('FACTORS',3,4,'Дорожные наблюдения и временные ограничения проверены')

# Сохраняем редактируемые факторы в едином словаре перед unit-policy и расчётом панели.
FEATURE_DICT=pd.DataFrame(sum(selected_by_group.values(),[]))
DOMAIN_COVERAGE=pd.DataFrame([{'group':g,'domain':d,
    'n_selected':sum(x['domain']==d for x in selected_by_group[g])}
    for g in SEMANTIC_DOMAINS for d,_ in SEMANTIC_DOMAINS[g]])
V5_RECOVERY_ROWS=V5_BUDGET_BALANCE_QA+V5_ROADS_QA
V5_BUDGET_BALANCE_QA=pd.DataFrame(V5_BUDGET_BALANCE_QA)
V5_ROADS_QA=pd.DataFrame(V5_ROADS_QA)
V5_SEMANTIC_FIXES=pd.DataFrame(V5_SEMANTIC_FIXES,columns=['feature','old_domain','new_domain','indicator_code','title','reason'])
mark('V5_BUDGET_BALANCE','PASS' if 'feature' in FEATURE_DICT.columns and 'G4_BALANCE' in FEATURE_DICT.feature.values else 'SOURCE_LIMITED',
     'Only matched explicit total revenue and expenditure slices can produce a fiscal balance')
mark('V5_ROAD_RECOVERY','PASS' if any(x.get('status')=='SELECTED_RETROSPECTIVE_ROADS' for x in V5_RECOVERY_ROWS) else 'SOURCE_LIMITED',
     'Observed BDMO6 road features sought; unverified historical publication prevents confirmatory claims')
mark('V5_SEMANTIC_RECLASSIFICATION','PASS',f'{len(V5_SEMANTIC_FIXES)} classifications updated')
progress('FACTORS',4,4,'Бюджетные итоги, дороги, семантика проверены')


# Этап II / IV. Воспроизводимый дизайн прогнозирования

#### Муниципальная панель, история ОКТМО, лаги и целевые переменные.

In [7]:
# 06 · V5.00 · RESEARCH PANEL — VERIFIED G1R, UNITS, TIME AND LEVELS
PANEL=base[['entity_id','observation_year','region_name','mun_type','municipality','population','average_population','migration','migration_rate','births','deaths','migration_per_1000','boundary_uncertainty','source_vintage']].copy()
merge_audit=[]
for data in FEATURE_PARTS:
    f=data.columns[-1]; n=len(PANEL)
    ensure(not data.duplicated(['entity_id','observation_year']).any(),'SOURCE_JOIN_MANY_TO_MANY_'+f)
    PANEL=PANEL.merge(data,on=['entity_id','observation_year'],how='left',validate='one_to_one')
    ensure(len(PANEL)==n,'PANEL_CARDINALITY_CHANGED_'+f)
    merge_audit.append(dict(feature=f,before_rows=n,after_rows=len(PANEL),nonmissing=int(PANEL[f].notna().sum()),status='ONE_TO_ONE'))
PANEL=PANEL.sort_values(['entity_id','observation_year']).reset_index(drop=True); QA_KEYS=pd.DataFrame(merge_audit)

HISTORY_ID='06::BDMO6::BDMO6_member00::municipality_history_observed::derived'
PANEL['history_observed']=False; PANEL['observed_name_change']=False; PANEL['observed_type_change']=False; PANEL['boundary_change_not_verified']=True
if HISTORY_ID in LOCAL_ASSETS:
    try:
        hh=pd.read_parquet(LOCAL_ASSETS[HISTORY_ID],columns=['source_oktmo','observation_year','observed_name_change','observed_type_change','boundary_change_not_verified'])
        hh['entity_id']=hh.source_oktmo.astype('string').str.strip().astype(str); hh['observation_year']=pd.to_numeric(hh.observation_year,errors='coerce')
        hh=hh[hh.observation_year.notna()].copy(); hh['observation_year']=hh.observation_year.astype(int)
        flags=['observed_name_change','observed_type_change','boundary_change_not_verified']
        for c in flags: hh[c]=hh[c].astype('string').str.lower().isin(['true','1','да','yes'])
        hh=hh.groupby(['entity_id','observation_year'],as_index=False)[flags].max(); hh['history_observed']=True
        PANEL=PANEL.drop(columns=['history_observed']+flags).merge(hh,on=['entity_id','observation_year'],how='left',validate='one_to_one')
        for c in ['history_observed','observed_name_change','observed_type_change']: PANEL[c]=PANEL[c].astype('boolean').fillna(False).astype(bool)
        PANEL['boundary_change_not_verified']=PANEL['boundary_change_not_verified'].astype('boolean').fillna(True).astype(bool)
    except Exception as exc:
        PANEL['history_observed']=False; PANEL['observed_name_change']=False; PANEL['observed_type_change']=False; PANEL['boundary_change_not_verified']=True
        mark('HISTORY_BOUNDARY_QA','PARTIAL','Audit failed: '+type(exc).__name__)
PANEL['boundary_year_at_risk']=~PANEL.history_observed | PANEL.boundary_change_not_verified | PANEL.observed_name_change | PANEL.observed_type_change
PANEL['boundary_evidence_status']=np.select([~PANEL.history_observed,PANEL.observed_name_change|PANEL.observed_type_change,PANEL.boundary_change_not_verified],['NO_HISTORY_RECORD','OBSERVED_NAME_OR_TYPE_CHANGE','BOUNDARY_NOT_VERIFIED'],default='NO_OBSERVED_CHANGE_NOT_PROOF_OF_STABILITY')
BOUNDARY_QA=pd.DataFrame([{'metric':c,'n_yearly_observations_flagged':int(PANEL[c].sum())} for c in ['history_observed','boundary_change_not_verified','boundary_year_at_risk','observed_name_change','observed_type_change']])
mark('PASS_BOUNDARY_AUDIT','PASS',f'Unverified entity-years={int(PANEL.boundary_year_at_risk.sum())}/{len(PANEL)}; NEVER coerced to stable')

def mark_boundary_forecast_window(t,reference):
    z=reference[['entity_id','observation_year','boundary_year_at_risk']].sort_values(['entity_id','observation_year']).copy()
    z['cum_obs']=z.groupby('entity_id').cumcount()+1; z['cum_bad']=z.boundary_year_at_risk.astype(int).groupby(z.entity_id).cumsum()
    a=z.rename(columns={'observation_year':'origin_year','cum_obs':'start_obs','cum_bad':'start_bad'}); b=z.rename(columns={'observation_year':'target_year','cum_obs':'end_obs','cum_bad':'end_bad'})
    out=t.merge(a[['entity_id','origin_year','start_obs','start_bad']],on=['entity_id','origin_year'],how='left',validate='many_to_one')
    out=out.merge(b[['entity_id','target_year','end_obs','end_bad']],on=['entity_id','target_year'],how='left',validate='many_to_one')
    ensure(len(out)==len(t),'BOUNDARY_JOIN_CARDINALITY_CHANGE')
    miss=out[['start_obs','start_bad','end_obs','end_bad']].isna().any(axis=1)
    out['boundary_period_uncertain']=(miss | (out.end_obs-out.start_obs).ne(out.horizon) | (out.end_bad-out.start_bad).gt(0)).astype('boolean').fillna(True).astype(bool)
    return out.drop(columns=['start_obs','start_bad','end_obs','end_bad'])

RATIO_REGEX=r'на\s*(?:одного|одну|одно|1\b|1000\b|10\s*тыс|душу|жител|человек|учащ|предприяти)|в\s+расч[её]те\s+на|в\s+среднем\s+на|удельн|на\s+единиц'
RATE_REGEX=r'(?:%|процент|коэффициент|индекс|дол[яи]\s|отношение|промилле)'
def unit_policy(indicator_name,unit,group,year=2020):
    title=str(indicator_name).casefold().replace('ё','е'); raw=str(unit).casefold().replace('ё','е'); both=title+' '+raw
    m=re.search(r'с\s*(\d{4})\s*г?\.?\s*[-–—:, ]*',raw); eff=(raw[m.end():] if year>=int(m.group(1)) else raw[:m.start()]) if m and re.search(r'рубл|руб',raw) else raw
    rub=bool(re.search(r'рубл|руб\b',eff)); scale=1e6 if re.search(r'млн\.?\s*руб|миллион.{0,4}руб',eff) else 1e3 if re.search(r'тыс\.?\s*руб|тысяч.{0,5}руб',eff) else 1.
    def result(action,mult=1.,den='NONE',family='UNCLASSIFIED',status='PASS'):
        return dict(action=action,multiplier=mult,denominator=den,effective_unit=eff,unit_family=family,status=status)
    if group=='G5' and re.search(r'км|километр|число муниципалитетов|муниципалитет',raw): return result('KEEP_SPATIAL_MEASUREMENT',family='SPATIAL_REFERENCE')
    wage=bool(re.search(r'(средн.{0,20}(?:заработн|начисленн.{0,9}зарплат)|заработн.{0,9}плат.{0,8}одн.{0,9}работник)',both))
    if wage and rub:return result('KEEP_RAW_AVERAGE',scale,family='AVERAGE_WAGE_PER_WORKER')
    if re.search(RATIO_REGEX,both):return result('KEEP_ALREADY_NORMALIZED',scale,family='RATIO_OR_PER_ENTITY')
    if re.search(RATE_REGEX,both):return result('KEEP_RATE',family='RATE')
    if rub:
        action='PER_RESIDENT' if group in ['G1','G2','G3','G4'] else 'KEEP_RAW'
        return result(action,scale,'D01_population_same_year' if action=='PER_RESIDENT' else 'NONE','MONEY_NOMINAL_RUB')
    if re.search(r'человек|чел\b|единиц|ед\.|штук|штука|кв\.?\s*м|м2|квадрат|метр',eff) and group in ['G1','G2','G3']:
        if group=='G1' and re.search(r'средн|уровень|размер',title):return result('KEEP_RAW_AVERAGE',family='AVERAGE')
        return result('PER_1000_RESIDENTS',1000.,'D01_population_same_year','COUNT_OR_AREA')
    return result('KEEP_RAW_UNCLASSIFIED',family='UNCLASSIFIED',status='REVIEW')

ensure(unit_policy('Расходы ОМСУ на одного жителя','2008 г. — тысяча рублей, с 2009 г. — рубль','G4',2020)['action']=='KEEP_ALREADY_NORMALIZED','UNIT_CANARY_DOUBLE_DENOMINATOR')
ensure(unit_policy('Расходы местного бюджета','2008 г. — тыс. руб.; с 2009 г. — рубли','G4',2020)['multiplier']==1,'UNIT_CANARY_HISTORICAL_RUBLES')
ensure(unit_policy('Средняя начисленная заработная плата','тыс. рублей','G1',2020)['multiplier']==1000,'UNIT_CANARY_AVERAGE_WAGE_THOUSANDS')
ensure(unit_policy('Средняя начисленная заработная плата','рублей','G1',2020)['action']=='KEEP_RAW_AVERAGE','UNIT_CANARY_WAGE_NOT_PER_RESIDENT')
ensure(unit_policy('Расходы на одно малое предприятие','рублей','G4',2020)['denominator']=='NONE','UNIT_CANARY_PER_ENTERPRISE')
mark('UNIT_SEMANTICS_CANARY','PASS','No double ratio, mixed-year rubles correctly resolved')


# V3.00: add region-level G1R context under an explicit separate namespace.
# Exact region_key joins only; no organizational aggregation or apportionment to municipalities.
PANEL['region_key']=PANEL.region_name.map(_region_norm)
for part in REGIONAL_CONTEXT_PARTS:
    rows_before=len(PANEL)
    PANEL=PANEL.merge(part,on=['region_key','observation_year'],how='left',validate='many_to_one')
    ensure(len(PANEL)==rows_before,'REGIONAL_CONTEXT_ROW_MULTIPLICATION')
GROUP_FEATURES={g:[r['feature'] for r in selected_by_group[g]] for g in PATTERNS}
GROUP_FEATURES['G1R']=[r['feature'] for r in REGIONAL_CONTEXT_META]
# Regional quantities are not normalized by local municipal population.
# Unit conversion happens in the actual source observation year; noncomparable
# units are removed instead of filling unknown levels with guessed municipal values.
G1R_UNIT_QA=[]
for meta in REGIONAL_CONTEXT_META:
    feature=meta['feature'];unit=meta['unit'];name=meta['indicator_name']
    policies={int(y):unit_policy(name,unit,'G1R',int(y)) for y in PANEL.observation_year.unique()}
    accepted=all(p['status']=='PASS' for p in policies.values()) and len({p['action'] for p in policies.values()})==1
    action=next(iter(policies.values()))['action']
    # A regional total can never masquerade as a municipal income rate.
    accepted=bool(accepted and action in ('KEEP_RAW_AVERAGE','KEEP_ALREADY_NORMALIZED','KEEP_RATE'))
    if accepted:
        scale=PANEL.observation_year.map({y:p['multiplier'] for y,p in policies.items()})
        PANEL[feature]=pd.to_numeric(PANEL[feature],errors='coerce')*scale
    else:
        GROUP_FEATURES['G1R']=[c for c in GROUP_FEATURES['G1R'] if c!=feature]
    G1R_UNIT_QA.append({'feature':feature,'source_id':meta['source_id'],'unit':unit,
        'action':action,'decision':'ACCEPT_REGIONAL_CONTEXT' if accepted else 'EXCLUDE_REGIONAL_UNIT_AMBIGUITY',
        'source_grain':'REGION_YEAR_ONLY','n_values':int(PANEL[feature].notna().sum())})
G1R_UNIT_QA=pd.DataFrame(G1R_UNIT_QA)
audit=[]
for row in FEATURE_DICT.to_dict('records'):
    f,g,unit,title=row['feature'],row['group'],row['unit'],row['indicator_name']
    years=pd.to_numeric(PANEL.observation_year,errors='coerce'); policies={int(y):unit_policy(title,unit,g,int(y)) for y in years.dropna().unique()}
    pol=policies[int(years.max())]; ok={p['status'] for p in policies.values()}=={'PASS'} and len({p['action'] for p in policies.values()})==1
    suffix='_per_resident' if pol['action']=='PER_RESIDENT' else '_per_1000' if pol['action']=='PER_1000_RESIDENTS' else '_scaled_rub' if any(p['multiplier']!=1 for p in policies.values()) else ''
    derived=f+suffix
    if ok and suffix:
        value=pd.to_numeric(PANEL[f],errors='coerce')*years.map({y:p['multiplier'] for y,p in policies.items()})
        if pol['action'] in ['PER_RESIDENT','PER_1000_RESIDENTS']: value=value/pd.to_numeric(PANEL.population,errors='coerce').where(PANEL.population.gt(0))
        PANEL[derived]=value
    if ok: GROUP_FEATURES[g]=[derived if x==f else x for x in GROUP_FEATURES[g]]
    else: GROUP_FEATURES[g]=[x for x in GROUP_FEATURES[g] if x!=f]
    examples={int(y):float(pd.to_numeric(PANEL.loc[PANEL.observation_year.eq(y),f],errors='coerce').dropna().iloc[0]) for y in sorted(policies) if PANEL.loc[PANEL.observation_year.eq(y),f].notna().any()}
    audit.append(dict(parent_feature=f,model_feature=derived if ok else None,group=g,domain=row['domain'],indicator_name=title,asset_id=row['asset_id'],action=pol['action'],ratio_denominator=pol['denominator'],source_units=unit,effective_unit_recent=pol['effective_unit'],effective_unit_by_year_json=json.dumps({y:p['effective_unit'] for y,p in policies.items()},ensure_ascii=False),multiplier_by_year_json=json.dumps({y:p['multiplier'] for y,p in policies.items()}),yearwise_raw_value_exemplar=json.dumps(examples),unit_family=pol['unit_family'],decision='ACCEPT_DIMENSIONAL_CONVERSION' if ok else 'EXCLUDE_UNIT_AMBIGUITY',notes='Raw field preserved; per-resident conversion only where explicitly required.'))
TRANSFORM_QA=pd.DataFrame(audit,columns=['parent_feature','model_feature','group','domain','indicator_name','asset_id','action','ratio_denominator','source_units','effective_unit_recent','effective_unit_by_year_json','multiplier_by_year_json','yearwise_raw_value_exemplar','unit_family','decision','notes'])
for c in ['feature','group','domain','model_feature','unit_status']:
    if c not in FEATURE_DICT.columns: FEATURE_DICT[c]=pd.Series(dtype='object')
if len(FEATURE_DICT):
    lookup=TRANSFORM_QA.set_index('parent_feature'); FEATURE_DICT['model_feature']=FEATURE_DICT.feature.map(lookup.model_feature); FEATURE_DICT['unit_status']=FEATURE_DICT.feature.map(lookup.decision)
MODEL_DOMAIN_COVERAGE=pd.DataFrame([dict(group=g,domain=d,n_model_features=int(((TRANSFORM_QA.group==g)&(TRANSFORM_QA.domain==d)&TRANSFORM_QA.model_feature.notna()).sum())) for g in SEMANTIC_DOMAINS for d,_ in SEMANTIC_DOMAINS[g]])
for g in ['G1','G2','G3','G4']:
    valid=TRANSFORM_QA.loc[(TRANSFORM_QA.group==g)&TRANSFORM_QA.model_feature.notna(),'domain'].dropna().unique().tolist(); threshold=2 if g=='G3' else 1
    mark('FEATURE_'+g,'PASS' if len(valid)>=threshold else 'PARTIAL' if valid else 'NOT_FEASIBLE',f'Usable domains after strict unit QA: {sorted(valid)}; features={len(GROUP_FEATURES[g])}')
actual=set(MODEL_DOMAIN_COVERAGE.loc[(MODEL_DOMAIN_COVERAGE.group=='G1')&MODEL_DOMAIN_COVERAGE.n_model_features.gt(0),'domain'])
# Distinguish nominal domains from valid labour/income proxies; do not call sales 'employment'.
G1_DOMAIN_QA=pd.DataFrame([{'domain':d,'n_model_features':int(n),'adequate_for_claim':int(n)>0,
  'claim':'DESCRIPTIVE_ONLY' if d in ('business','retail','investment') else 'REQUIRES_SEMANTICALLY_VALID_DIRECT_INDICATOR'}
  for d,n in zip(MODEL_DOMAIN_COVERAGE.loc[MODEL_DOMAIN_COVERAGE.group.eq('G1'),'domain'],
                 MODEL_DOMAIN_COVERAGE.loc[MODEL_DOMAIN_COVERAGE.group.eq('G1'),'n_model_features'])])
econ_core=bool(actual & {'wages','employment','income'})
mark('G1_DOMAIN_COMPLETENESS','PASS' if len(actual)>=CFG['MIN_ECON_DOMAINS'] and econ_core else 'PARTIAL' if actual else 'NOT_FEASIBLE',
   f'observed={sorted(actual)}; direct wages/employment/income present={econ_core}; sales not employment')
HEALTH_DOMAIN_QA=pd.DataFrame([{
   'feature':r.model_feature,'name':r.indicator_name,'domain':r.domain,
   'hospital_specific':bool(re.search(r'больниц|больнич|стационар|койк|поликлиник|амбулатор',str(r.indicator_name).casefold())),
   'source_grain':'VALIDATED_MUNICIPAL' if pd.notna(r.model_feature) else 'NOT_SELECTED',
   'n_observed':int(PANEL[r.parent_feature].notna().sum()) if r.parent_feature in PANEL else 0}
    for r in TRANSFORM_QA.loc[(TRANSFORM_QA.group=='G3')&(TRANSFORM_QA.domain=='health')].itertuples()])
health=int(((HEALTH_DOMAIN_QA.hospital_specific)&(HEALTH_DOMAIN_QA.source_grain=='VALIDATED_MUNICIPAL')).sum()) if len(HEALTH_DOMAIN_QA) else 0
HEALTH_AVAILABLE=bool(health)
mark('HEALTH_INFRASTRUCTURE_GATE','PASS' if HEALTH_AVAILABLE else 'PARTIAL' if len(HEALTH_DOMAIN_QA) else 'NOT_FEASIBLE',
    f'Hospital/ambulatory indicators selected={health}, broad health candidates={len(HEALTH_DOMAIN_QA)}; horizon checks below')
mark('PASS_UNIT_AUDIT','PASS' if not TRANSFORM_QA.decision.eq('EXCLUDE_UNIT_AMBIGUITY').any() else 'PARTIAL',f"Excluded ambiguous units={int(TRANSFORM_QA.decision.eq('EXCLUDE_UNIT_AMBIGUITY').sum())}")

FACTOR_ELIGIBILITY=[]; GROUP_FEATURES_BY_H={}
for h,cutoff in FACTOR_CUTOFF_BY_H.items():
    GROUP_FEATURES_BY_H[h]={}
    for g,ff in GROUP_FEATURES.items():
        eligible=[]
        for f in ff:
            parent=next((r['parent_feature'] for r in audit if r['model_feature']==f),f)
            valid=PANEL.loc[PANEL.observation_year.le(cutoff)&PANEL[f].notna(),['entity_id','observation_year']]
            n,ny,nm=len(valid),valid.observation_year.nunique(),valid.entity_id.nunique()
            meta=next((r for r in selected_by_group.get(g,[]) if r['feature']==parent),None)
            selected=True if g in ('G5','G1R') else bool(meta is not None and h in json.loads(meta.get('selected_horizons_json','[]')))
            ok=bool(selected and n>=CFG['MIN_INDICATOR_OBS_TRAIN'] and ny>=CFG['MIN_INDICATOR_YEARS_TRAIN'] and nm>=CFG['MIN_INDICATOR_MUNICIPALITIES'])
            if ok:eligible.append(f)
            FACTOR_ELIGIBILITY.append(dict(horizon=h,group=g,feature=f,parent_feature=parent,observation_cutoff_year=cutoff,source_published_year='NOT_VERIFIED',effective_available_year_rule=f'observation_year+{CFG["FEATURE_PUBLICATION_LAG"]}',n_keys_before_cutoff=n,n_years_before_cutoff=int(ny),n_municipalities_before_cutoff=int(nm),eligible=ok,reason='TRAIN_SUPPORT_VERIFIED' if ok else 'INSUFFICIENT_PRE_ORIGIN_SUPPORT'))
        GROUP_FEATURES_BY_H[h][g]=eligible
FACTOR_ELIGIBILITY=pd.DataFrame(FACTOR_ELIGIBILITY,columns=['horizon','group','feature','parent_feature','observation_cutoff_year','source_published_year','effective_available_year_rule','n_keys_before_cutoff','n_years_before_cutoff','n_municipalities_before_cutoff','eligible','reason'])

ORIGINS=PANEL[['entity_id','region_name','mun_type','municipality','observation_year','boundary_uncertainty','source_vintage']].rename(columns={'observation_year':'origin_year'}).copy()
prior=PANEL[['entity_id','observation_year','population','migration_per_1000']].rename(columns={'observation_year':'known_year'})
lag0=CFG['FEATURE_PUBLICATION_LAG']
for lag in range(lag0,lag0+4):
    tmp=prior.rename(columns={'population':f'hist_population_l{lag}','migration_per_1000':f'hist_migration_l{lag}'})
    ORIGINS['known_year']=ORIGINS.origin_year-lag; ORIGINS=ORIGINS.merge(tmp,on=['entity_id','known_year'],how='left',validate='many_to_one').drop(columns='known_year')
G0_FEATURES=[c for c in ORIGINS if c.startswith('hist_')]+['origin_year']
ORIGINS['hist_growth_lagged_pct']=100*(ORIGINS[f'hist_population_l{lag0}']/ORIGINS[f'hist_population_l{lag0+2}']-1)
ORIGINS['hist_growth_lagged_pct']=ORIGINS.hist_growth_lagged_pct.replace([np.inf,-np.inf],np.nan); G0_FEATURES.append('hist_growth_lagged_pct')
all_factors=list(dict.fromkeys(sum(GROUP_FEATURES.values(),[])))
if all_factors:
    raw=PANEL[['entity_id','observation_year']+all_factors].copy()
    # last observed value carries forward at most MAX_FEATURE_STALENESS_YEARS,
    # with a separate age ledger and with the required publication lag.
    STALENESS_QA=[]
    for f in all_factors:
        observed=pd.to_numeric(raw[f],errors='coerce')
        lastyear=raw['observation_year'].where(observed.notna()).groupby(raw.entity_id).ffill()
        lag_age=raw.observation_year-lastyear
        stale=lag_age.gt(CFG['MAX_FEATURE_STALENESS_YEARS']) | lastyear.isna()
        filled=observed.groupby(raw.entity_id).ffill()
        raw[f]=filled.where(~stale)
        STALENESS_QA.append({'feature':f,'n_observed':int(observed.notna().sum()),
             'n_carried_valid':int((raw[f].notna()&observed.isna()).sum()),
             'max_age_years':CFG['MAX_FEATURE_STALENESS_YEARS'],'publication_lag':lag0})
    STALENESS_QA=pd.DataFrame(STALENESS_QA)
    ensure(STALENESS_QA.n_carried_valid.ge(0).all(),'STALE_QA_INVALID')
    raw=raw.rename(columns={'observation_year':'known_year'})
    ORIGINS['known_year']=ORIGINS.origin_year-lag0
    ORIGINS=ORIGINS.merge(raw,on=['entity_id','known_year'],how='left',validate='many_to_one').drop(columns='known_year')
else:STALENESS_QA=pd.DataFrame(columns=['feature','n_observed','n_carried_valid','max_age_years','publication_lag'])
LABELS=PANEL[['entity_id','observation_year','population','migration_per_1000']].rename(columns={'observation_year':'target_year','population':'future_population','migration_per_1000':'future_migration_per_1000'})
CURRENT=PANEL[['entity_id','observation_year','population']].rename(columns={'observation_year':'origin_year','population':'population_at_origin'})
ORIGINS=ORIGINS.merge(CURRENT,on=['entity_id','origin_year'],how='left',validate='one_to_one'); frames=[]
for h in CFG['HORIZONS']:
    t=ORIGINS.copy(); t['horizon']=h; t['target_year']=t.origin_year+h
    t=t.merge(LABELS,on=['entity_id','target_year'],how='left',validate='many_to_one')
    t['migration_per_1000']=t.future_migration_per_1000
    t['population_change_pct']=np.where(t.population_at_origin.gt(0)&t.future_population.gt(0),100*(t.future_population/t.population_at_origin-1),np.nan)
    t['information_cutoff']=t.origin_year; t['effective_available_year']=t.origin_year-lag0
    t['target_origin_unit']='ENTITY_YEAR'; t['source_vintage']=source_sha
    frames.append(mark_boundary_forecast_window(t,PANEL).drop(columns=['future_population','future_migration_per_1000']))
TASK_PANEL=pd.concat(frames,ignore_index=True)
ensure(not TASK_PANEL.duplicated(['entity_id','origin_year','horizon']).any(),'FORECAST_KEY_COLLISION')
ensure((TASK_PANEL.effective_available_year<TASK_PANEL.origin_year).all(),'FUTURE_COVARIATE_LEAK')
if CFG['MODE']=='SMOKE': TASK_PANEL=TASK_PANEL[TASK_PANEL.entity_id.isin(sorted(TASK_PANEL.entity_id.unique())[:CFG['SMOKE_MUNICIPALITIES']])].copy()

TARGET_AVAILABILITY=[]
for target in CFG['TARGETS']:
    for y in YEARS:
        src=base.loc[base.observation_year.eq(y)]
        valid_raw=(src.migration.notna()&src.average_population.gt(0)) if target=='migration_per_1000' else src.population.gt(0)
        for h in CFG['HORIZONS']:
            tp=TASK_PANEL.loc[TASK_PANEL.horizon.eq(h)&TASK_PANEL.target_year.eq(y)]
            available=tp[target].notna(); g0=tp[[c for c in G0_FEATURES if c!='origin_year']].notna().any(axis=1)
            g1=tp[GROUP_FEATURES['G1']].notna().any(axis=1) if GROUP_FEATURES['G1'] else pd.Series(False,index=tp.index)
            TARGET_AVAILABILITY.append(dict(target=target,target_year=int(y),horizon=int(h),raw_d01_rows=len(src),raw_d01_target_available=int(valid_raw.sum()),label_rows=int(available.sum()),g0_ready=int((available&g0).sum()),factor_rows_g1=int((available&g1).sum()),reason='D01_MIGRATION_UNOBSERVED' if target=='migration_per_1000' and not valid_raw.any() else 'OBSERVED_OR_OTHER_SUPPORT'))
TARGET_AVAILABILITY=pd.DataFrame(TARGET_AVAILABILITY)
GROUP_SUPPORT={g:TASK_PANEL[ff].notna().any(axis=1) if ff else pd.Series(False,index=TASK_PANEL.index) for g,ff in GROUP_FEATURES.items()}
G0_SUPPORT=TASK_PANEL[[c for c in G0_FEATURES if c!='origin_year']].notna().any(axis=1)
TASK_PANEL['common_support']=G0_SUPPORT.copy()
for g in ['G1','G2','G3','G4','G5']:
    if GROUP_FEATURES[g]: TASK_PANEL['common_support'] &= GROUP_SUPPORT[g]
TASK_PANEL['available_factor_groups']=sum(v.astype(int) for v in GROUP_SUPPORT.values())
coverage=pd.DataFrame([dict(group=g,n_features=len(ff),raw_panel_coverage=float(PANEL[ff].notna().any(axis=1).mean()) if ff else 0.,forecast_coverage=float(GROUP_SUPPORT[g].mean()),status='PRESENT' if ff else 'NOT_FEASIBLE') for g,ff in GROUP_FEATURES.items()])
mark('G1R_NONMUNICIPAL_GRAIN','PASS','Regional economic context is labeled G1R, NOT municipal G1; zero municipal claims')
mark('PASS_DATA',('PASS' if sum(map(len,GROUP_FEATURES.values()))>0 and not fatal_sql else 'PARTIAL' if len(PANEL) and not PANEL[['entity_id','observation_year']].duplicated().any() else 'FAIL'),f'panel={len(PANEL)}, features={sum(map(len,GROUP_FEATURES.values()))}, common_support={int(TASK_PANEL.common_support.sum())}')
quiet_print('Панель:',len(PANEL),'строк | label candidates:',len(TASK_PANEL),'| complete support:',int(TASK_PANEL.common_support.sum()))
def audited_stability_share(sample):
    if sample.empty:return {'n':0,'n_unverified':0,'n_verified':0,'verified_share':0.}
    flag=sample.boundary_period_uncertain.astype('boolean').fillna(True).astype(bool)
    return dict(n=len(sample),n_unverified=int(flag.sum()),n_verified=int((~flag).sum()),verified_share=float((~flag).mean()))
BOUNDARY_WINDOW_QA=TASK_PANEL.groupby(['horizon','target_year'],as_index=False).agg(n=('entity_id','size'),uncertain=('boundary_period_uncertain','sum'))

progress('PANEL',1,1,f'Наблюдений: {len(PANEL)}; прогнозных строк: {len(TASK_PANEL)}')

# CHECKPOINT I: полный срез фактических наблюдений и прогнозной панели.
if CFG.get('ENABLE_CHECKPOINTS'):
    cp=WORK/'checkpoints';cp.mkdir(parents=True,exist_ok=True)
    checkpoint_i=[]
    for label,frame in [('municipal_panel',PANEL),('supervised_origin_panel',TASK_PANEL)]:
        file=cp/(label+'.parquet');frame.to_parquet(file,index=False,compression='zstd')
        checkpoint_i.append({'file':file.name,'rows':len(frame),'columns':len(frame.columns),
                             'sha256':digest(file),'source_commit':source_sha})
    (cp/'stage_I_checkpoint.json').write_text(json.dumps({'version':CFG['VERSION'],
       'status':'PHYSICAL_PARQUET_CHECKPOINT','assets':checkpoint_i},ensure_ascii=False,indent=2),encoding='utf-8')
    progress('PANEL',1,1,'Исходная панель и checkpoint I зафиксированы')


#### Хронологический дизайн, гипотезы, тестовые окна и аудит исключённого горизонта 5 лет.

In [8]:
# 07 · PREREGISTERED ABLATIONS, COMMON-SUPPORT STUDIES, SPATIAL/TEMPORAL SPLITS
GROUP_ORDER=('G1','G2','G3','G4','G5')
PRIMARY_TEST_YEARS=sorted(set(sum(TARGET_TEST_WINDOWS.values(),[])))
SPATIAL_YEAR_COUNTS={int(y):int((TASK_PANEL.horizon.eq(1)&TASK_PANEL.target_year.eq(y)&
    GROUP_SUPPORT['G5']).sum()) for y in PRIMARY_TEST_YEARS}
G5_CANONICAL_ELIGIBLE=bool(GROUP_FEATURES.get('G5')) and bool(CFG.get('SPATIAL_BOUNDARY_CROSSWALK_VERIFIED')) and bool(CFG.get('SPATIAL_PUBLICATION_EVIDENCE')) and all(
    SPATIAL_YEAR_COUNTS[y]>=CFG['MIN_SPATIAL_CANONICAL_TEST_ROWS'] for y in PRIMARY_TEST_YEARS)
mark('SPATIAL_CANONICAL_GATE','PASS' if G5_CANONICAL_ELIGIBLE else 'NOT_FEASIBLE',
     f'vintage-gated support={SPATIAL_YEAR_COUNTS}; undated transport excluded')
CANONICAL=[{'arm':'M0','groups':[]}];prefix=[]
for j,g in enumerate(GROUP_ORDER,1):
    if not GROUP_FEATURES[g] or (g=='G5' and not G5_CANONICAL_ELIGIBLE):
        for k in range(j,6):mark('ARM_M'+str(k),'NOT_FEASIBLE','Missing or time-ineligible group '+g)
        break
    prefix=prefix+[g];CANONICAL.append({'arm':'M'+str(j),'groups':prefix.copy()})
AVAILABLE_GROUPS=[g for g in GROUP_ORDER if GROUP_FEATURES[g] and
                  (g!='G5' or G5_CANONICAL_ELIGIBLE)]
STUDIES=[]
if len(CANONICAL)>1:
    STUDIES.append({'id':'CANONICAL','required_groups':prefix.copy(),'arms':CANONICAL,
        'protocols':['ROLLING_TIME','TERRITORY_ONLY','CROSSED'],'confirmatory':len(CANONICAL)>=5})
else:
    STUDIES.append({'id':'CANONICAL','required_groups':[],'arms':CANONICAL,
        'protocols':['ROLLING_TIME'],'confirmatory':False})
# Separate, self-contained temporal experiments for the two nominal test years.
if CFG['MODE']=='FULL' and len(CANONICAL)>1:
    for target,years in TARGET_TEST_WINDOWS.items():
        for year in years:
            STUDIES.append({'id':f'YEAR_{target}_{year}','test_year':int(year),
               'allowed_targets':[target],'required_groups':prefix.copy(),'arms':CANONICAL,
               'protocols':['ROLLING_TIME'],'confirmatory':False})
for g in AVAILABLE_GROUPS:
    STUDIES.append({'id':'AUX_'+g,'required_groups':[g],
      'arms':[{'arm':'M0','groups':[]},{'arm':'ADD_'+g,'groups':[g]}],
      'protocols':['ROLLING_TIME'],'confirmatory':False})
if GROUP_FEATURES.get('G5') and not G5_CANONICAL_ELIGIBLE:
    STUDIES.append({'id':'AUX_G5_VINTAGED','required_groups':['G5'],
      'arms':[{'arm':'M0','groups':[]},{'arm':'ADD_G5','groups':['G5']}],
      'protocols':['ROLLING_TIME'],'confirmatory':False})
# Secondary observed migration window only if it differs from the predeclared migration calendar.
if CFG['MODE']=='FULL' and CFG.get('EXPLORE_SECONDARY_MIGRATION_WINDOW'):
    mig_years=sorted(base.loc[base.migration_per_1000.notna(),'observation_year'].unique())
    secondary=mig_years[-CFG['TEST_YEARS']:] if len(mig_years)>=CFG['TEST_YEARS'] else []
    if secondary and list(map(int,secondary))!=TARGET_TEST_WINDOWS['migration_per_1000']:
        STUDIES.append({'id':'MIGRATION_OBSERVED_LAST_TWO_YEARS','test_years':list(map(int,secondary)),
          'allowed_targets':['migration_per_1000'],'required_groups':[],
          'arms':[{'arm':'M0','groups':[]}], 'protocols':['ROLLING_TIME'],
          'confirmatory':False,'label':'SECONDARY_OBSERVED_WINDOW_NOT_PRIMARY'})
if GROUP_FEATURES.get('G1R'):
    STUDIES.append({'id':'AUX_G1_REGION_CONTEXT','required_groups':['G1R'],
       'arms':[{'arm':'M0','groups':[]},{'arm':'ADD_G1_REGION','groups':['G1R']}],
       'protocols':['ROLLING_TIME','TERRITORY_ONLY','CROSSED'],
       'confirmatory':False,'label':'REGION_CONTEXT_NOT_MUNICIPAL_G1'})
if len(AVAILABLE_GROUPS)>=2:
    all_g=AVAILABLE_GROUPS.copy()
    specs=[{'arm':'M0','groups':[]},{'arm':'FULL_AVAILABLE','groups':all_g}]
    specs += [{'arm':'WITHOUT_'+g,'groups':[x for x in all_g if x!=g]} for g in all_g]
    STUDIES.append({'id':'MAX_AVAILABLE','required_groups':all_g,'arms':specs,
       'protocols':['ROLLING_TIME','TERRITORY_ONLY','CROSSED'],'confirmatory':False})
for study in STUDIES:
    for arm in study['arms']:
        arm['features']=list(dict.fromkeys(G0_FEATURES+sum((GROUP_FEATURES[g] for g in arm['groups']),[])))
        arm['n_factors']=sum(len(GROUP_FEATURES[g]) for g in arm['groups'])
# ПРЕРЕГИСТРАЦИЯ до обучения: H3.1–H3.8 из научного протокола, без просмотра TEST.
_HYPOTHESES=[
 ('H3.1','CANONICAL','M1 → M3','Последовательный вклад жилищной и социальной инфраструктуры'),
 ('H3.2','INTERACTION','G1 × G2','Экономические условия × жилищная среда'),
 ('H3.3','AUX_G3','M0 → ADD_G3','Дополнительная ценность социальной инфраструктуры'),
 ('H3.4','AUX_G4','M0 → ADD_G4','Дополнительная ценность муниципальных финансов'),
 ('H3.5','SENSITIVITY','LAG_AND_PROXY','Лаги публикации и коррелированные прокси'),
 ('H3.7','BOUNDARIES','STABLE_VS_UNVERIFIED','Устойчивость к изменениям муниципальных границ'),
 ('H3.8','SIZE','SIZE_STRATA','Неоднородность эффекта по размеру муниципалитетов')]
HYPOTHESIS_REGISTRY=pd.DataFrame([{
    'hypothesis_id':h,'study_id':study,'contrast':contrast,'definition':definition,
    'test_family':('CONFIRMATORY_BY_TARGET_HORIZON_PROTOCOL' if h=='H3.1' else
                   'EXPLORATORY_BY_TARGET_HORIZON_PROTOCOL'),
    'sample_contract':'PAIRED_IDENTICAL_ENTITY_ORIGIN_TARGET_YEAR',
    'primary_metric':'MAE','multiplicity_method':('HOLM' if h=='H3.1' else 'BH_FDR'),
    'preregistered_before_test':True,'status':'PLANNED_CONDITIONAL',
    'version':CFG['VERSION'],
    'reason':'Requires verified source domain, sufficient paired support and temporal labels'}
    for h,study,contrast,definition in _HYPOTHESES])
STUDY_HYPOTHESIS_FAMILIES=pd.DataFrame([{
    'study_id':s['id'],'confirmatory_design':bool(s['confirmatory']),
    'family_scope':'TARGET_HORIZON_PROTOCOL',
    'multiplicity_method':'HOLM' if s['confirmatory'] else 'BH_FDR',
    'preregistered_before_test':True,'status':'DESIGN_FROZEN'} for s in STUDIES])
TASKS=[];TASK_RECORDS=[]
ALL_REGIONS=sorted(base.region_name.dropna().astype(str).unique())
rng=np.random.default_rng(CFG['SEED']);HELD_REGIONS=set(rng.choice(
   np.asarray(ALL_REGIONS,dtype=object),size=max(2,int(0.2*len(ALL_REGIONS))),replace=False)) if len(ALL_REGIONS)>8 else set()

def splits_for(d,protocol,h,test_year=None,test_years=None):
    # Real-time forecasting: all training AND DEV outcome labels must have been observed
    # before the earliest issuance (origin year) of the held-out test forecasts.
    yrs=[y for y in YEARS if y>=min(YEARS)+h]
    nt=1 if CFG['MODE']=='SMOKE' else CFG['TEST_YEARS']
    nd=1 if CFG['MODE']=='SMOKE' else CFG['DEV_YEARS']
    if len(yrs)<nt+nd+CFG['MIN_TRAIN_YEARS']:return None,'TOO_FEW_TARGET_YEARS'
    test_yrs=([int(test_year)] if test_year is not None else
              sorted(int(v) for v in test_years) if test_years else yrs[-nt:])
    first_test_origin=test_yrs[0]-h
    latest_known_label=first_test_origin-int(CFG['TARGET_LABEL_AVAILABILITY_LAG'])
    potential_dev=[y for y in yrs if y<=latest_known_label]
    if len(potential_dev)<CFG['MIN_TRAIN_YEARS']+nd:
        return None,'LONG_HORIZON_TARGET_LABELS_NOT_YET_KNOWN_AT_FORECAST_ORIGIN'
    dv=potential_dev[-nd:]
    tr=[y for y in potential_dev if y<min(dv)]
    if len(tr)<CFG['MIN_TRAIN_YEARS']:return None,'TOO_FEW_REALTIME_TRAIN_YEARS'
    held=d.region_name.astype(str).isin(HELD_REGIONS)
    if protocol=='ROLLING_TIME':
        return {'train':d.target_year.isin(tr),'dev':d.target_year.isin(dv),
                'test':d.target_year.isin(test_yrs)},''
    if not HELD_REGIONS:return None,'INSUFFICIENT_REGIONS_FOR_GROUP_HOLDOUT'
    if protocol=='TERRITORY_ONLY':
        # Pure geography generalization (historical evaluation on held regions).
        return {'train':(~held)&d.target_year.isin(tr),
                'dev':(~held)&d.target_year.isin(dv),
                'test':held&d.target_year.isin(dv)},''
    if protocol=='CROSSED':
        return {'train':(~held)&d.target_year.isin(tr),
                'dev':(~held)&d.target_year.isin(dv),
                'test':held&d.target_year.isin(test_yrs)},''
    raise ValueError('UNKNOWN_SPLIT_PROTOCOL')

# A missing late test year must NOT move the test window backwards.
_test_stub=pd.DataFrame({'target_year':[y for y in YEARS if y<max(YEARS)-2],
                         'region_name':['unit_test']*len([y for y in YEARS if y<max(YEARS)-2])})
_stub_masks,_=splits_for(_test_stub,'ROLLING_TIME',min(CFG['HORIZONS']))
if _stub_masks is not None:
    ensure(int(_stub_masks['test'].sum())==0,'UNIT_TEST_CALENDAR_SLIPPAGE')
mark('UNIT_SPLIT_CONTRACT','PASS','D01-frozen target years and issued-forecast cutoff; source gaps do not shift test years')

for st in STUDIES:
    for target in CFG['TARGETS']:
        if target not in st.get('allowed_targets',CFG['TARGETS']):continue
        if target=='migration_per_1000' and not valid_rate:
            for h in (CFG['HORIZONS'] if CFG['MODE']=='FULL' else [1]):
                for protocol in st['protocols']:
                    TASK_RECORDS.append({'study_id':st['id'],'target':target,'horizon':h,
                      'protocol':protocol,'status':'NOT_FEASIBLE','scientific_status':'NOT_FEASIBLE',
                      'reason':'MIGRATION_TARGET_SEMANTICS_FAILED','n_support':0,'n_train':0,'n_dev':0,'n_test':0,
                      'requested_test_years':','.join(map(str,TARGET_TEST_WINDOWS[target])),
                      'actual_test_year_counts_json':'{}','missing_test_years':','.join(map(str,TARGET_TEST_WINDOWS[target]))})
            continue
        for h in (CFG['HORIZONS'] if CFG['MODE']=='FULL' else [1]):
            support=G0_SUPPORT.copy()
            missing_required=[g for g in st['required_groups'] if not GROUP_FEATURES_BY_H[h].get(g)]
            for g in st['required_groups']:
                if GROUP_FEATURES_BY_H[h].get(g):
                    support &= TASK_PANEL[GROUP_FEATURES_BY_H[h][g]].notna().any(axis=1)
            d=TASK_PANEL[TASK_PANEL.horizon.eq(h)&TASK_PANEL[target].notna()&support].copy()
            local_st=dict(st)
            local_st['arms']=[{**a,'features':list(dict.fromkeys(
                G0_FEATURES+sum((GROUP_FEATURES_BY_H[h][g] for g in a['groups']),[]))),
                'n_factors':sum(len(GROUP_FEATURES_BY_H[h][g]) for g in a['groups'])}
               for a in st['arms']]
            for protocol in st['protocols']:
                mm,issue=splits_for(d,protocol,h,st.get('test_year'),st.get('test_years',TARGET_TEST_WINDOWS[target]))
                if missing_required:issue='GROUP_NOT_ELIGIBLE_AT_HORIZON_'+','.join(missing_required);mm=None
                counts={k:int(v.sum()) for k,v in mm.items()} if mm is not None else {}
                if mm is not None and (counts['train']<CFG['MIN_TRAIN_ROWS'] or
                      min(counts['dev'],counts['test'])<CFG['MIN_TEST_ROWS']):
                    issue='INSUFFICIENT_TRAIN_DEV_TEST';mm=None
                requested=([int(st['test_year'])] if 'test_year' in st else
                           st['test_years'] if 'test_years' in st else
                           TARGET_TEST_WINDOWS[target])
                test_by_year=(d.loc[mm['test']].groupby('target_year').size().to_dict()
                              if mm is not None else {})
                missing_test_years=[y for y in requested if test_by_year.get(y,0)==0] if protocol!='TERRITORY_ONLY' else []
                task_status=('FEASIBLE_COMPLETE' if mm is not None and not missing_test_years else
                             'FEASIBLE_PARTIAL_YEARS' if mm is not None else 'NOT_FEASIBLE')
                TASK_RECORDS.append({'study_id':st['id'],'target':target,'horizon':h,'protocol':protocol,
                  'status':task_status,'scientific_status':('COMPLETE_YEARS' if task_status=='FEASIBLE_COMPLETE' else 'PARTIAL_YEARS' if task_status=='FEASIBLE_PARTIAL_YEARS' else 'NOT_FEASIBLE'),
                  'reason':issue,'n_support':len(d),
                  'n_train':counts.get('train'),'n_dev':counts.get('dev'),'n_test':counts.get('test'),
                  'requested_test_years':','.join(map(str,requested)),
                  'actual_test_year_counts_json':json.dumps({int(k):int(v) for k,v in test_by_year.items()}),
                  'missing_test_years':','.join(map(str,missing_test_years)),
                  'evaluation_window':('HISTORICAL_DEV_YEARS_GEO_ONLY' if protocol=='TERRITORY_ONLY' else
                     'PRIMARY_TARGET_YEARS' if 'test_years' not in st else 'SECONDARY_OBSERVED_YEARS')})
                if mm is not None:
                    TASKS.append({'study':local_st,'target':target,'horizon':h,'protocol':protocol,
                                 'data':d,'masks':mm})
TASK_STATUSES=pd.DataFrame(TASK_RECORDS)
ensure({'status','scientific_status','study_id','target','horizon','protocol','n_test'}.
       issubset(TASK_STATUSES.columns),'TASK_STATUS_SCHEMA_INCOMPLETE')
ensure((TASK_STATUSES.loc[TASK_STATUSES.status.eq('FEASIBLE_PARTIAL_YEARS'),
       'scientific_status']=='PARTIAL_YEARS').all(),'INCOMPLETE_TEST_YEARS_MISLABELLED')

def _study_target_years(st,target):
    if 'test_years' in st:return list(map(int,st['test_years']))
    if 'test_year' in st:return [int(st['test_year'])]
    return TARGET_TEST_WINDOWS[target]

# Audit target availability per study with explicit source/target coverage.
TEST_YEAR_AUDIT_ROWS=[]
for rec in TASK_RECORDS:
    if rec['protocol']=='TERRITORY_ONLY':continue  # historical geo-only protocol, not the primary time calendar
    st=next(s for s in STUDIES if s['id']==rec['study_id'])
    required=list(st['required_groups'])
    for year in _study_target_years(st,rec['target']):
        mask=TASK_PANEL.target_year.eq(year)&TASK_PANEL.horizon.eq(rec['horizon'])&TASK_PANEL[rec['target']].notna()
        supported=mask&G0_SUPPORT
        for g in required:
            if GROUP_FEATURES_BY_H[int(rec['horizon'])].get(g):
                supported=supported&TASK_PANEL[GROUP_FEATURES_BY_H[int(rec['horizon'])][g]].notna().any(axis=1)
            else:supported=supported&False
        by_year=json.loads(rec.get('actual_test_year_counts_json','{}') or '{}')
        TEST_YEAR_AUDIT_ROWS.append({'study_id':rec['study_id'],'target':rec['target'],
            'horizon':rec['horizon'],'protocol':rec['protocol'],'target_year':int(year),
            'n_target_observations':int(mask.sum()),'n_with_study_factor_support':int(supported.sum()),
            'n_test':int(by_year.get(str(year),0)),
            'status':rec['status'],'reason':rec['reason']})
TEST_YEAR_AUDIT=pd.DataFrame(TEST_YEAR_AUDIT_ROWS)
CANONICAL_YEAR_QA=TASK_STATUSES.loc[TASK_STATUSES.study_id.eq('CANONICAL') &
    TASK_STATUSES.protocol.isin(['ROLLING_TIME','CROSSED'])].copy()
if CFG['MODE']=='FULL':
    primary=CANONICAL_YEAR_QA.loc[CANONICAL_YEAR_QA.horizon.eq(1) &
        CANONICAL_YEAR_QA.status.str.startswith('FEASIBLE')]
    missing_any=primary.missing_test_years.ne('').any() if len(primary) else True
    mark('TWO_YEAR_CALENDAR_QA','PARTIAL' if missing_any else 'PASS',
         'Two-year target-specific observed primary windows: migration 2021–2022, population 2022–2023; no silent sliding')
else: mark('TWO_YEAR_CALENDAR_QA','SMOKE_ONLY','Year coverage in FULL checked separately')
H5_CALENDAR_GAPS=TASK_STATUSES.loc[TASK_STATUSES.horizon.eq(5),
    ['study_id','target','protocol','horizon','status','reason','requested_test_years','actual_test_year_counts_json']].copy()
if len(H5_CALENDAR_GAPS):
    mark('H5_LONG_HORIZON','PASS' if H5_CALENDAR_GAPS.status.eq('FEASIBLE_COMPLETE').all() else 'NOT_FEASIBLE',
         f'{int(H5_CALENDAR_GAPS.status.eq("FEASIBLE_COMPLETE").sum())}/{len(H5_CALENDAR_GAPS)}; unmet label availability is never bypassed')
# V5: пятилетние модели исключены из исполняемой программы. Архив исключённого
# календаря должен оставаться доказуемым. Формируем динамическую запись для каждого
# дизайна (не обучаем H5), применяя тот же запрет на неизвестные целевые значения.
V5_H5_EXCLUSION=[]
for _study in STUDIES:
    for _target in CFG['TARGETS']:
        if _target not in _study.get('allowed_targets',CFG['TARGETS']):continue
        _years=([int(_study['test_year'])] if 'test_year' in _study else
            list(map(int,_study['test_years'])) if 'test_years' in _study else
            list(map(int,TARGET_TEST_WINDOWS[_target])))
        for _proto in _study['protocols']:
            _h=5;_first=min(_years);_cutoff=_first-_h-int(CFG['TARGET_LABEL_AVAILABILITY_LAG'])
            _eligible_years=[int(y) for y in YEARS if int(y)<=_cutoff and int(y)>=min(YEARS)+_h]
            _reason=('LONG_HORIZON_TARGET_LABELS_NOT_YET_KNOWN_AT_FORECAST_ORIGIN'
                if len(_eligible_years)<CFG['MIN_TRAIN_YEARS']+CFG['DEV_YEARS'] else
                'EXCLUDED_FROM_V5_SCOPE_NO_H5_MODELING')
            V5_H5_EXCLUSION.append({'study_id':_study['id'],'target':_target,
                'protocol':_proto,'horizon':5,'status':'EXCLUDED_NOT_FITTED',
                'reason':_reason,'requested_test_years':','.join(map(str,_years)),
                'actual_test_year_counts_json':'{}','last_pre_origin_known_label':_cutoff,
                'n_eligible_pre_origin_label_years':len(_eligible_years)})
H5_CALENDAR_GAPS=pd.DataFrame(V5_H5_EXCLUSION)
ensure(not TASK_STATUSES.horizon.eq(5).any(),'V5_H5_INADVERTENTLY_TRAINED')
mark('H5_SCOPE_EXCLUSION','PASS',f'{len(H5_CALENDAR_GAPS)} audit records; H5 excluded from model fitting')
# A historical time-window redefinition is NOT substituted for preregistered h5 tests.
ARM_SPECS=CANONICAL
if not TASKS:mark('PASS_SPLITS','NOT_FEASIBLE','No feasible task after paired common-support gating')
else:mark('PASS_SPLITS','PASS',f'{len(TASKS)} dataset/protocol tasks, {len(STUDIES)} studies')
# One fixed work budget before the first model; never grow denominator during run.
RUN_COMPLETION_STATUS='IN_PROGRESS'
_TICK_PHASE=None;_TICK_DONE=0;_TICK_TOTAL=0
def set_tick(phase,total):
    global _TICK_PHASE,_TICK_TOTAL,_TICK_DONE
    _TICK_PHASE=phase;_TICK_TOTAL=max(0,int(total));_TICK_DONE=0
    if _TICK_TOTAL==0:progress(phase,1,1,'Нет доступных единиц работы')
    else:progress(phase,0,_TICK_TOTAL,'Выполнение')
def tick(stage,n=1,detail=''):
    global _TICK_DONE
    ensure(_TICK_PHASE is not None,'PROGRESS_PHASE_NOT_SET')
    _TICK_DONE+=int(n)
    ensure(_TICK_DONE<=_TICK_TOTAL,'PROGRESS_WORK_COUNT_OVERFLOW')
    progress(_TICK_PHASE,_TICK_DONE,_TICK_TOTAL,detail)

quiet_print('Studies:',[(s['id'],len(s['arms'])) for s in STUDIES], '| feasible tasks:',len(TASKS),
      '| frozen regions held out:',len(HELD_REGIONS))

progress('TASKS',1,1,f'Задач в плане: {len(TASKS)}')


#### V5: диаграмма потерь строк и воспроизводимое сравнение с V4.02/V4.03.

In [9]:
# R2 · V5 · СКВОЗНОЙ АУДИТ ВЫБОРОК И ПАРНОЕ СОПОСТАВЛЕНИЕ С V4
# Замороженные проверенные агрегаты V4.02/V4.03 — только ориентиры; ключи сравниваются
# по первичным D01, а не выдумываются по агрегатам.
V5_SAMPLE_AUDIT=[]
for target in CFG['TARGETS']:
    years=set(map(int,TARGET_TEST_WINDOWS[target]))
    for h in CFG['HORIZONS']:
        subset=TASK_PANEL[TASK_PANEL.horizon.eq(h)&TASK_PANEL.target_year.isin(years)].copy()
        labelled=subset[subset[target].notna()].copy()
        latest=labelled[['entity_id','origin_year','target_year','horizon']].drop_duplicates()
        if len(latest):
            ensure(not latest.duplicated(['entity_id','origin_year','target_year','horizon']).any(),'V5_LABEL_KEY_DUPLICATED')
        # Только разные строки, а не искусственное восполнение отсутствующих исторических данных.
        for y in sorted(years):
            sample=subset[subset.target_year.eq(y)]
            rec={'target':target,'horizon':int(h),'target_year':int(y),
                 'n_origin_target_candidates':int(len(sample)),
                 'n_actual_target_labels':int(sample[target].notna().sum()),
                 'n_missing_labels':int(sample[target].isna().sum()),
                 'n_g0_support':int((sample[target].notna()&G0_SUPPORT.reindex(sample.index).fillna(False)).sum()),
                 'n_uncertain_history':int(sample.boundary_period_uncertain.sum()),
                 'source':'D01_AND_V5_TASK_PANEL'}
            V5_SAMPLE_AUDIT.append(rec)
V5_SAMPLE_AUDIT=pd.DataFrame(V5_SAMPLE_AUDIT)

# Данные V4 подтверждены предыдущим архивом, но в нём агрегаты, а не все ключи теста.
V5_FROZEN_PREVIOUS_COUNTS=pd.DataFrame([
 {'version':'V4.02','target':'population_change_pct','horizon':1,'study_id':'CANONICAL','protocol':'ROLLING_TIME','n_test':4344},
 {'version':'V4.03','target':'population_change_pct','horizon':1,'study_id':'CANONICAL','protocol':'ROLLING_TIME','n_test':3943}])
latest=(TASK_STATUSES.loc[TASK_STATUSES.study_id.eq('CANONICAL')&TASK_STATUSES.protocol.eq('ROLLING_TIME')&
    TASK_STATUSES.target.eq('population_change_pct')&TASK_STATUSES.horizon.eq(1),['n_test','status']].head(1))
if len(latest):
    count=int(latest.n_test.iloc[0]) if pd.notna(latest.n_test.iloc[0]) else None
    V5_FROZEN_PREVIOUS_COUNTS['v5_n_test']=count
    V5_FROZEN_PREVIOUS_COUNTS['v5_minus_reference']=(count-V5_FROZEN_PREVIOUS_COUNTS.n_test) if count is not None else None
    V5_FROZEN_PREVIOUS_COUNTS['comparison_level']='AGGREGATE_ONLY_NOT_ROW_PARITY'
else:
    V5_FROZEN_PREVIOUS_COUNTS['comparison_level']='NO_VALID_CANONICAL_TASK'

V5_ROW_REFERENCE_QA=pd.DataFrame(columns=['reference_version','n_reference_keys','n_v5_keys',
    'only_reference_keys','only_v5_keys','paired_keys','status'])
V5_ROW_EXCLUSION_REASONS=pd.DataFrame(columns=['entity_id','origin_year','target_year','horizon','reason'])
mark('V5_SAMPLE_COVERAGE','PASS','Source D01 labels and each test-window exclusion counted separately')
progress('TASKS',1,1,'Выборка и исторический дизайн проверены')


#### Обучение, девелопмент-подбор и фиксация моделей до теста.

In [10]:
# 08 · DEV-LOCKED MODEL BANK; TRUE PAIRED TEST PREDICTIONS, SEEDS, NO OUTCOME-BASED SELECTION
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
PRED_ROWS=[];METRIC_ROWS=[];DEV_ROWS=[];SPLIT_ROWS=[];IMPORTANCE_ROWS=[];MODEL_REGISTRY={}
try:
    from catboost import CatBoostRegressor
    CB_AVAILABLE=True
except ImportError:CB_AVAILABLE=False
try:
    from lightgbm import LGBMRegressor
    LGBM_AVAILABLE=True
except ImportError:LGBM_AVAILABLE=False
BANK=[f for f in CFG['MODEL_FAMILIES'] if f in ('RIDGE','HGB','ELASTICNET') or (f=='CATBOOST' and CB_AVAILABLE) or
      (f=='LIGHTGBM' and LGBM_AVAILABLE)]
mark('MODEL_BANK','PASS' if 'RIDGE' in BANK and 'HGB' in BANK else 'FAIL',
      f'available={BANK}; CatBoost={CB_AVAILABLE}; LightGBM={LGBM_AVAILABLE}')

def design(train,test,features,keep_reference=None):
    refs=sorted(train.region_name.dropna().astype(str).unique()) if keep_reference is None else list(keep_reference)
    usable=[f for f in dict.fromkeys(features) if f in train and
            pd.to_numeric(train[f],errors='coerce').notna().any()]
    ensure('origin_year' in usable,'ORIGIN_YEAR_MISSING')
    def mx(df):
        xx=df[usable].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan).reset_index(drop=True)
        for f in usable:
            if f.startswith(('G1_','G2_','G3_','G4_','G5_')):
                xx[f]=np.sign(xx[f])*np.log1p(np.abs(xx[f])) # fixed transform; no fitted test statistics
        reg=df.region_name.astype(str).reset_index(drop=True)
        # One-shot matrix construction: no repeated __setitem__ / pandas frame fragmentation.
        region_columns={'region_'+hashlib.sha256(name.encode()).hexdigest()[:10]:
                        reg.eq(name).to_numpy(dtype=float) for name in refs}
        if region_columns:
            region_matrix=pd.DataFrame(region_columns,index=xx.index,dtype=float)
            xx=pd.concat([xx,region_matrix],axis=1,copy=False)
        ensure(not xx.columns.duplicated().any(),'DUPLICATE_REGION_DUMMY_COLUMNS')
        return xx.astype(float)
    return mx(train),mx(test),usable,refs

def estimator(family,seed,params):
    if family=='RIDGE':
        return make_pipeline(SimpleImputer(strategy='median',add_indicator=True),StandardScaler(),
                             Ridge(alpha=float(params['alpha'])))
    if family=='ELASTICNET':
        return make_pipeline(SimpleImputer(strategy='median',add_indicator=True),StandardScaler(),
                ElasticNet(alpha=float(params['alpha']),l1_ratio=float(params['l1_ratio']),
                           max_iter=5000,random_state=int(seed)))
    if family=='HGB':
        return HistGradientBoostingRegressor(max_iter=int(params['iterations']),
            max_leaf_nodes=int(params['leaves']),learning_rate=0.045,min_samples_leaf=18,
            l2_regularization=2.0,early_stopping=False,random_state=int(seed))
    if family=='CATBOOST':
        return CatBoostRegressor(iterations=int(params['iterations']),depth=int(params['depth']),
             learning_rate=0.05,l2_leaf_reg=4,loss_function='RMSE',verbose=False,
             thread_count=int(CFG['DUCKDB_THREADS']),random_seed=int(seed),allow_writing_files=False)
    if family=='LIGHTGBM':
        return LGBMRegressor(n_estimators=int(params['iterations']),num_leaves=int(params['leaves']),
             max_depth=6,learning_rate=0.04,reg_lambda=3.0,min_child_samples=18,
             n_jobs=int(CFG['DUCKDB_THREADS']),verbosity=-1,random_state=int(seed))
    raise ValueError('UNKNOWN_MODEL_FAMILY')
PARAM_GRID={
 'RIDGE':[{'alpha':2},{'alpha':20}],
 'ELASTICNET':[{'alpha':0.2,'l1_ratio':0.25},{'alpha':2.,'l1_ratio':0.55}],
 'HGB':[{'iterations':125,'leaves':15},{'iterations':175,'leaves':23}],
 'CATBOOST':[{'iterations':220,'depth':4},{'iterations':300,'depth':6}],
 'LIGHTGBM':[{'iterations':150,'leaves':15},{'iterations':200,'leaves':23}]}
def fit_predict(family,train,test,target,features,seed=1729,params=None):
    Xtr,Xte,used,refs=design(train,test,features)
    model=estimator(family,seed,params or PARAM_GRID[family][0]);model.fit(Xtr,train[target].to_numpy(float))
    pred=np.asarray(model.predict(Xte),dtype=float)
    ensure(len(pred)==len(test) and np.isfinite(pred).all(),'PREDICTION_NOT_FINITE')
    return pred,model,used,refs

def baseline_predict(train,test,target,h):
    la=CFG['FEATURE_PUBLICATION_LAG'];hist=test.get('hist_migration_l'+str(la))
    if target=='migration_per_1000':
        return pd.to_numeric(hist,errors='coerce').fillna(float(train[target].median())).to_numpy(float)
    p1=pd.to_numeric(test[f'hist_population_l{la}'],errors='coerce')
    p3=pd.to_numeric(test[f'hist_population_l{la+2}'],errors='coerce')
    recent_growth=np.power(p1/p3,0.5*h)
    series=100*(recent_growth-1)
    return series.replace([np.inf,-np.inf],np.nan).fillna(float(train[target].median())).to_numpy(float)

def regional_baseline_predict(train,test,target):
    """Train-only regional tendency; unknown region falls back to overall median."""
    med=train.groupby('region_name')[target].median()
    glob=float(train[target].median())
    return test.region_name.map(med).fillna(glob).to_numpy(float)

def deterministic_family(family):
    # HGB and LightGBM used here have neither row nor column sampling, and no early stopping.
    return family in ('RIDGE','ELASTICNET','HGB','LIGHTGBM')

def family_seeds(family):
    return [CFG['SEED']] if deterministic_family(family) else list(CFG['SEEDS_FINAL'])

def metrics(y,p,w):
    y=np.asarray(y,float);p=np.asarray(p,float);w=np.asarray(w,float)
    ensure(len(y)==len(p) and np.isfinite(y).all() and np.isfinite(p).all(),'INVALID_EVALUATION_VECTORS')
    w=np.where(np.isfinite(w)&(w>0),w,0)
    err=y-p
    return {'mae':float(np.mean(np.abs(err))), 'rmse':float(np.sqrt(np.mean(err**2))),
       'weighted_mae':float(np.average(np.abs(err),weights=w)) if w.sum()>0 else np.nan,
       'n_test':int(len(y))}

def add_result(task,arm,model,seed,train,test,pred,n_feat):
    target,h,protocol=task['target'],task['horizon'],task['protocol'];study=task['study']['id']
    y=test[target].to_numpy(float)
    w=test[f'hist_population_l{CFG["FEATURE_PUBLICATION_LAG"]}'].to_numpy(float)
    met=metrics(y,pred,w)
    METRIC_ROWS.append({'study_id':study,'target':target,'horizon':h,'protocol':protocol,
         'arm':arm,'model':model,'seed':seed,'n_train':len(train),'n_features_used':n_feat,**met})
    rows=test[['entity_id','origin_year','target_year','region_name','mun_type','information_cutoff',
       'source_vintage','horizon','population_at_origin']].copy()
    rows['target']=target;rows['study_id']=study;rows['split_id']=protocol
    rows['arm']=arm;rows['model']=model;rows['seed']=seed;rows['y_true']=y;rows['y_pred']=pred
    rows['population_weight']=w
    rows['absolute_error']=np.abs(y-pred);rows['squared_error']=(y-pred)**2
    PRED_ROWS.append(rows)

# CPU-only, isolated contract test (synthetic values never added to evidence or metrics).
def _model_unit_check():
    rng=np.random.default_rng(CFG['SEED']);n=100
    fake=pd.DataFrame({'region_name':np.repeat(['R1','R2'],n//2),
       'origin_year':np.tile(np.arange(2010,2020),n//10),
       'G1_01_per_resident':rng.lognormal(size=n),'G2_01_per_1000':rng.random(n)*20,
       '_test_target':rng.normal(size=n),'hist_population_l1':rng.integers(100,10000,n)})
    for fam in ('RIDGE','HGB','ELASTICNET'):
        yp,_,_,_=fit_predict(fam,fake.iloc[:80],fake.iloc[80:],'_test_target',
            ['origin_year','G1_01_per_resident','G2_01_per_1000'],CFG['SEED'],PARAM_GRID[fam][0])
        ensure(np.isfinite(yp).all() and len(yp)==20,'MODEL_UNIT_TEST_FAIL_'+fam)
_model_unit_check()
mark('UNIT_MODEL_CONTRACT','PASS','Ridge and HGB synthetic function-only smoke; excluded from scientific results')

# DEV count is known after TASKS and exact parameter grids are registered.
DEV_RUNS=[(t['study']['id'],t['target'],t['horizon']) for t in TASKS
     if t['protocol']=='ROLLING_TIME']
DEV_WORK=len(set(DEV_RUNS))

DEV_TOTAL=len(set(DEV_RUNS))*sum(len(PARAM_GRID[f]) for f in BANK
                  if f in PARAM_GRID and (CFG['MODE']=='FULL'))
if CFG['MODE']!='FULL':DEV_TOTAL=len(set(DEV_RUNS))*len(BANK)
set_tick('DEV',DEV_TOTAL)
# Tune on a held-out historical DEV block, prior to FINAL test. No use of final target values.
SELECTED_FAMILY={}
for st in STUDIES:
    for target in CFG['TARGETS']:
        for h in CFG['HORIZONS'] if CFG['MODE']=='FULL' else [1]:
            runs=[t for t in TASKS if t['study']['id']==st['id'] and t['target']==target and
                  t['horizon']==h and t['protocol']=='ROLLING_TIME']
            if not runs:continue
            task=runs[0];df=task['data'];tr=df[task['masks']['train']];dv=df[task['masks']['dev']]
            maxarm=max(task['study']['arms'],key=lambda a:len(a['features']))
            for family in BANK:
                for params in PARAM_GRID[family] if CFG['MODE']=='FULL' else PARAM_GRID[family][:1]:
                    try:
                        yp,_,used,_=fit_predict(family,tr,dv,target,maxarm['features'],CFG['SEED'],params)
                        sc=metrics(dv[target],yp,dv[f'hist_population_l{CFG["FEATURE_PUBLICATION_LAG"]}'])
                        DEV_ROWS.append({'study_id':st['id'],'target':target,'horizon':h,
                            'family':family,'params_json':json.dumps(params,sort_keys=True),
                            'dev_mae':sc['mae'],'dev_rmse':sc['rmse'],'n_dev':len(dv),
                            'features_used':len(used)})
                    except Exception as ex:
                        log(f'DEV model failure {st["id"]} h{h} {family}: {type(ex).__name__}', 'WARN')
                    finally:tick('DEV model',1,f'{st["id"]} h{h} {family}')
            v=[r for r in DEV_ROWS if (r['study_id'],r['target'],r['horizon'])==(st['id'],target,h)]
            if v:
                choice=min(v,key=lambda r:(r['dev_mae'],r['dev_rmse']))
                SELECTED_FAMILY[(st['id'],target,h)]=(choice['family'],json.loads(choice['params_json']))
            # Fits, not tasks, are the exact DEV work units.

ensure(_TICK_DONE==_TICK_TOTAL,'DEV_PROGRESS_MISMATCH')
TRAIN_TOTAL=sum(2+(len(t['study']['arms'])*len(family_seeds(SELECTED_FAMILY[(t['study']['id'],t['target'],t['horizon'])][0]))
                   if (t['study']['id'],t['target'],t['horizon']) in SELECTED_FAMILY else 0)
                for t in TASKS)
set_tick('TRAIN',TRAIN_TOTAL)
for task in TASKS:
    study=task['study'];d=task['data'];m=task['masks'];protocol=task['protocol']
    tr=d[m['train']|m['dev']].copy();te=d[m['test']].copy()
    if protocol in ('ROLLING_TIME','CROSSED'):
        ensure(tr.target_year.max()<te.origin_year.min(),'TRAIN_LABEL_NOT_KNOWN_AT_EARLIEST_TEST_ORIGIN')
    if protocol in ('TERRITORY_ONLY','CROSSED'):
        ensure(set(tr.region_name.astype(str)).isdisjoint(set(te.region_name.astype(str))),'REGIONAL_CROSS_FOLD_LEAK')
        ensure(set(tr.entity_id).isdisjoint(set(te.entity_id)),'ENTITY_CROSS_FOLD_LEAK')
    SPLIT_ROWS.append(pd.concat([tr[['entity_id','origin_year','target_year','region_name']].assign(
           study_id=study['id'],target=task['target'],horizon=task['horizon'],protocol=protocol,part='train'),
           te[['entity_id','origin_year','target_year','region_name']].assign(
           study_id=study['id'],target=task['target'],horizon=task['horizon'],protocol=protocol,part='test')]))
    bp=baseline_predict(tr,te,task['target'],task['horizon'])
    add_result(task,'M0','HISTORICAL',CFG['SEED'],tr,te,bp,1)
    rb=regional_baseline_predict(tr,te,task['target'])
    add_result(task,'M0','REGIONAL_MEDIAN',CFG['SEED'],tr,te,rb,1)
    tick('historical baselines',2,study['id'])
    choice=SELECTED_FAMILY.get((study['id'],task['target'],task['horizon']))
    if choice is None:continue
    family,params=choice
    for arm in study['arms']:
        seeds=family_seeds(family)
        for si,seed in enumerate(seeds):
            try:
                yp,model,used,refs=fit_predict(family,tr,te,task['target'],arm['features'],seed,params)
                add_result(task,arm['arm'],family,seed,tr,te,yp,len(used))
                if si==0 and protocol=='ROLLING_TIME' and arm==max(study['arms'],key=lambda a:len(a['features'])):
                    key=(study['id'],task['target'],task['horizon'])
                    MODEL_REGISTRY[key]={'model':model,'features':arm['features'],
                       'used':used,'refs':refs,'train':tr,'test':te,'pred':yp,
                       'family':family,'params':params,'arm':arm['arm']}
                    name=f'{study["id"]}_{task["target"]}_h{task["horizon"]}_{family}.joblib'
                    joblib.dump({'model':model,'features':used,'refs':refs,'target':task['target'],
                        'horizon':task['horizon'],'study':study['id'],'source_commit':source_sha,
                        'model_family':family,'params':params},WORK/'models'/name,compress=3)
            except Exception as ex:
                mark(f'FIT_{study["id"]}_{task["target"]}_{task["horizon"]}_{protocol}_{arm["arm"]}_{seed}',
                     'FAIL',type(ex).__name__+': '+str(ex)[:100])
                log(f'FIT {study["id"]} {arm["arm"]} {family}: {type(ex).__name__}','ERROR')
            tick('training',1,
                 f'{study["id"]} {arm["arm"]} h={task["horizon"]} s={seed}')
ensure(_TICK_DONE==_TICK_TOTAL,'TRAIN_PROGRESS_MISMATCH')
ensure(DEV_WORK==len(set(DEV_RUNS)),'DEV_WORK_COUNT_CONTRACT')
# Единственный _PROGRESS_BAR сохраняется на весь прогон.
RUN_COMPLETION_STATUS='MODELS_COMPLETED_EXPORT_PENDING'
METRICS=pd.DataFrame(METRIC_ROWS);DEV_METRICS=pd.DataFrame(DEV_ROWS)
PREDICTIONS=pd.concat(PRED_ROWS,ignore_index=True) if PRED_ROWS else pd.DataFrame()
SPLITS=pd.concat(SPLIT_ROWS,ignore_index=True) if SPLIT_ROWS else pd.DataFrame()
SEED_QA_ROWS=[]
if len(PREDICTIONS):
    for (sid,targ,h,proto,arm,model),g in PREDICTIONS[PREDICTIONS.model.isin(BANK)].groupby(
          ['study_id','target','horizon','split_id','arm','model']):
        hashes=[]
        for seed,z in g.groupby('seed'):
            arr=z.sort_values(['entity_id','origin_year','target_year']).y_pred.to_numpy(float)
            hashes.append(hashlib.sha256(np.asarray(arr,dtype='<f8').tobytes()).hexdigest())
        SEED_QA_ROWS.append({'study_id':sid,'target':targ,'horizon':h,'protocol':proto,
            'arm':arm,'model':model,'n_requested_seeds':len(family_seeds(model)),
            'n_configured_seeds':len(CFG['SEEDS_FINAL']),
            'n_executed_seeds':len(hashes),'n_distinct_prediction_hashes':len(set(hashes)),
            'n_effectively_distinct_runs':len(set(hashes)),
            'seeds_are_independent_samples':False})
SEED_QA=pd.DataFrame(SEED_QA_ROWS)
if len(PREDICTIONS):
    cols=['entity_id','origin_year','target_year','horizon','target','study_id','split_id','arm','model','seed']
    ensure(not PREDICTIONS.duplicated(cols).any(),'DUPLICATE_TEST_PREDICTIONS')
    for keys,p in PREDICTIONS[PREDICTIONS.model.ne('HISTORICAL')].groupby(
           ['study_id','target','horizon','split_id','model','seed']):
        ensure(p.groupby('arm').size().nunique()==1,'ABLA_SAMPLE_SIZE_MISMATCH')
        key_hashes=[hashlib.sha256(a[['entity_id','origin_year','target_year']].sort_values(['entity_id','origin_year']).to_csv(index=False).encode()).hexdigest() for _,a in p.groupby('arm')]
        ensure(len(set(key_hashes))==1,'ABLA_TEST_KEYS_MISMATCH')
mark('PASS_TRAINING','PASS' if len(METRICS) and len(SELECTED_FAMILY)==DEV_WORK and DEV_WORK>0 and not any(v['status']=='FAIL' for k,v in STATUS.items() if k.startswith('FIT_')) else 'NOT_FEASIBLE' if not DEV_WORK else 'PARTIAL',
     f'{len(METRICS)} model evaluations; {len(PREDICTIONS)} test predictions; dev-selected families={len(SELECTED_FAMILY)}')
quiet_print('Bank:',BANK,'| evaluated models:',len(METRICS),'| test forecasts:',len(PREDICTIONS))


# ADVANCED · V4.00 · DEV-LOCKED NON-NEGATIVE SUPER LEARNER, NO TEST-TUNING
# Additional experiment: all feasible TASKS; legacy preregistered comparisons remain intact.
from scipy.optimize import minimize as _minimize
from sklearn.ensemble import ExtraTreesRegressor as _ExtraTreesRegressor
try:
    from xgboost import XGBRegressor as _XGBRegressor
    XGB_AVAILABLE=True
except Exception:
    XGB_AVAILABLE=False

def _advanced_estimator(family,seed,params=None):
    z=CFG['ADVANCED_N_JOBS'];p=params or {}
    if family=='XGBOOST':
        ensure(XGB_AVAILABLE,'XGBOOST_NOT_INSTALLED')
        return _XGBRegressor(n_estimators=p.get('n_estimators',260),max_depth=p.get('max_depth',5),
            learning_rate=p.get('learning_rate',0.045),min_child_weight=6,subsample=1.,colsample_bytree=1.,
            reg_lambda=7.,reg_alpha=0.3,tree_method='hist',n_jobs=z,random_state=seed,objective='reg:squarederror')
    if family=='EXTRA_TREES':
        return make_pipeline(SimpleImputer(strategy='median',add_indicator=True),
          _ExtraTreesRegressor(n_estimators=p.get('n_estimators',240),max_features=0.84,
             min_samples_leaf=8,max_depth=16,n_jobs=z,random_state=seed))
    if family=='RIDGE':return estimator('RIDGE',seed,{'alpha':20.})
    if family=='HGB':return estimator('HGB',seed,{'iterations':175,'leaves':23})
    if family=='CATBOOST':return estimator('CATBOOST',seed,{'iterations':320,'depth':6})
    if family=='LIGHTGBM':return estimator('LIGHTGBM',seed,{'iterations':250,'leaves':23})
    raise ValueError('ADVANCED_FAMILY_NOT_REGISTERED '+str(family))

_ADV_REQUEST=['XGBOOST','CATBOOST','LIGHTGBM','HGB','EXTRA_TREES','RIDGE']
_ADV_READY={'XGBOOST':XGB_AVAILABLE,'CATBOOST':CB_AVAILABLE,'LIGHTGBM':LGBM_AVAILABLE,
            'HGB':True,'EXTRA_TREES':True,'RIDGE':True}
ADV_BANK=[x for x in _ADV_REQUEST if _ADV_READY[x]]
if CFG['MODE']=='FULL' and CFG['ADVANCED_REQUIRE_ALL_FAMILIES']:
    ensure(all(_ADV_READY.values()),'ADVANCED_FULL_MODEL_DEPENDENCIES_INCOMPLETE')
ensure(len(ADV_BANK)>=3,'ADVANCED_ENSEMBLE_REQUIRES_THREE_MODEL_FAMILIES')
ADVANCED_DEV_ROWS=[];ADVANCED_TEST_ROWS=[];ADVANCED_INTERVAL_ROWS=[];ADVANCED_ENSEMBLE_REGISTRY={};ADVANCED_MODEL_REGISTRY={}

def _advance_design_model(family,tr,te,target,features,seed=1729):
    xtr,xte,used,refs=design(tr,te,features)
    # Missing predictors are supported by all configured estimators through native missing
    # handling (XGBoost, HGB, LGBM, CatBoost) or train-only median imputation.
    mod=_advanced_estimator(family,int(seed));mod.fit(xtr,tr[target].to_numpy(float))
    yp=np.asarray(mod.predict(xte),float).reshape(-1)
    ensure(len(yp)==len(te) and np.isfinite(yp).all(),'ADVANCED_PREDICTION_NONFINITE_'+family)
    return {'family':family,'model':mod,'prediction':yp,'used':used,'refs':refs,'cols':list(xte.columns)}

def _simplex_weights(y,prediction_columns):
    P=np.column_stack(prediction_columns).astype(float);v=np.asarray(y,float)
    ensure(np.isfinite(P).all() and np.isfinite(v).all(),'ENSEMBLE_DEV_NONFINITE')
    n=P.shape[1];baseline=int(np.argmin(np.mean(np.abs(P-v[:,None]),axis=0)))
    if n==1:return np.ones(1,dtype=float)
    # Smoothed absolute loss with weak shrinkage toward equal weights; DEV only.
    fun=lambda w:float(np.mean(np.sqrt((P@w-v)**2+0.03**2))+CFG['ADVANCED_WEIGHT_RIDGE']*np.sum(w**2))
    res=_minimize(fun,x0=np.full(n,1./n),method='SLSQP',bounds=[(0.,1.)]*n,
        constraints={'type':'eq','fun':lambda w:float(np.sum(w)-1.)},options={'maxiter':200,'ftol':1e-9})
    w=np.maximum(0,np.asarray(res.x if res.success else np.eye(n)[baseline],float))
    if not np.isfinite(w).all() or w.sum()<=0:w=np.eye(n)[baseline]
    w=w/w.sum()
    if mean_absolute_error(v,P@w)>mean_absolute_error(v,P[:,baseline])+1e-9:
        w=np.eye(n)[baseline]  # no regressive weights chosen on DEV
    ensure(np.all(w>=0) and abs(w.sum()-1)<1e-9,'ENSEMBLE_WEIGHT_SIMPLEX_BROKEN')
    return w

def _adv_fit_one_task(task):
    sid=task['study']['id'];target=task['target'];h=int(task['horizon']);proto=task['protocol']
    d=task['data'];m=task['masks'];tr=d[m['train']].copy();dv=d[m['dev']].copy();te=d[m['test']].copy()
    ensure(len(tr)>=CFG['MIN_TRAIN_ROWS'] and len(dv)>=CFG['MIN_TEST_ROWS'],'ADVANCED_DEV_TOO_SMALL')
    ensure(len(te)>=CFG['MIN_TEST_ROWS'],'ADVANCED_TEST_TOO_SMALL')
    if proto in ('ROLLING_TIME','CROSSED'):
        ensure(tr.target_year.max()<te.origin_year.min(),'ADVANCED_ORIGIN_CAUSAL_ORDER_BROKEN')
    if proto in ('TERRITORY_ONLY','CROSSED'):
        ensure(set(tr.region_name.astype(str)).isdisjoint(set(te.region_name.astype(str))),'ADVANCED_REGIONAL_LEAK')
        ensure(set(dv.entity_id).isdisjoint(set(te.entity_id)),'ADVANCED_GEOGRAPHICAL_DEV_LEAK')
    arms=task['study']['arms'];full_arm=max(arms,key=lambda a:len(a['features']));features=full_arm['features']
    supported=[f for f in features if f in tr and pd.to_numeric(tr[f],errors='coerce').notna().any()]
    if len(supported)<2:return {'status':'NOT_FEASIBLE','reason':'TOO_FEW_TRAIN_PREDICTORS'}
    seed=int(CFG['SEED']);devmods=[]
    for fam in ADV_BANK[:CFG['ADVANCED_MAX_CANDIDATE_FAMILIES']]:
        try:
            r=_advance_design_model(fam,tr,dv,target,features,seed)
            loss=float(mean_absolute_error(dv[target],r['prediction']))
            devmods.append((loss,fam,r))
            ADVANCED_DEV_ROWS.append({'study_id':sid,'target':target,'horizon':h,'protocol':proto,
                'family':fam,'dev_mae':loss,'n_train':len(tr),'n_dev':len(dv),'n_columns':len(r['cols']),
                'selection':'PRE_TEST_HISTORICAL_DEV_ONLY','status':'PASS'})
        except Exception as ex:
            ADVANCED_DEV_ROWS.append({'study_id':sid,'target':target,'horizon':h,'protocol':proto,
                'family':fam,'dev_mae':np.nan,'n_train':len(tr),'n_dev':len(dv),
                'selection':'PRE_TEST_HISTORICAL_DEV_ONLY','status':'FIT_FAILED_'+type(ex).__name__})
            log(f'ADV DEV {sid}/{h}/{fam}: {type(ex).__name__}: {str(ex)[:120]}','WARN')
    if len(devmods)<2:return {'status':'NOT_FEASIBLE','reason':'LESS_THAN_TWO_DEV_FAMILIES_FIT'}
    devmods.sort(key=lambda z:(z[0],z[1]));top=devmods[:min(CFG['ADVANCED_ENSEMBLE_K'],len(devmods))]
    weights=_simplex_weights(dv[target], [v[2]['prediction'] for v in top]);dev_mix=sum(float(w)*entry[2]['prediction'] for w,entry in zip(weights,top))
    dev_abs=np.abs(dv[target].to_numpy(float)-dev_mix)
    q90=float(np.quantile(dev_abs,min(1.,math.ceil((len(dv)+1)*0.9)/len(dv)),method='higher'))
    full=pd.concat([tr,dv],ignore_index=True);members=[];testpred=[]
    for (dev_mae,fam,_),weight in zip(top,weights):
        r=_advance_design_model(fam,full,te,target,features,seed)
        members.append({'family':fam,'weight':float(weight),'model':r['model'],'used':r['used'],
                        'refs':r['refs'],'columns':r['cols'],'dev_mae':dev_mae})
        testpred.append(r['prediction'])
    best_single_family=str(top[0][1]);best_single_prediction=np.asarray(testpred[0],float)
    pred=sum(member['weight']*v for member,v in zip(members,testpred))
    ensure(np.isfinite(pred).all(),'ADVANCED_FINAL_NONFINITE')
    arm='SUPERLEARNER_DEV';add_result(task,arm,'SUPERLEARNER',seed,full,te,pred,len(supported))
    for member in members:
        r=next(v for v,memb in zip(testpred,members) if memb is member)
        if CFG['ADVANCED_EXPORT_MEMBERS']:add_result(task,arm,'SL_'+member['family'],seed,full,te,r,len(member['used']))
    n_test=len(te);covers=np.abs(te[target].to_numpy(float)-pred)<=q90
    ADVANCED_INTERVAL_ROWS.append({'study_id':sid,'target':target,'horizon':h,'protocol':proto,
        'n_dev':len(dv),'n_test':n_test,'q90_abs_dev':q90,'empirical_test_coverage':float(covers.mean()),
        'avg_width':2*q90,'method':'DEV_RESIDUAL_QUANTILE_AFTER_REFIT_HEURISTIC','nominal':0.9,
        'formal_coverage_guarantee':False,'reason':'Temporal shift and refitting after DEV; not formal split-conformal coverage'})
    result={'study_id':sid,'target':target,'horizon':h,'protocol':proto,
        'n_train':len(tr),'n_dev':len(dv),'n_test':len(te),'arm':arm,
        'members':[m['family'] for m in members], 'weights':[round(m['weight'],8) for m in members],
        'dev_mae':float(mean_absolute_error(dv[target],dev_mix)),
        'test_mae':float(mean_absolute_error(te[target],pred)),
        'best_single_family_dev_locked':best_single_family,
        'best_single_dev_mae':float(top[0][0]),
        'best_single_test_mae':float(mean_absolute_error(te[target],best_single_prediction)),
        'ensemble_minus_single_test_mae':float(mean_absolute_error(te[target],pred)-mean_absolute_error(te[target],best_single_prediction)),
        'status':'PASS','selection':'DEV_ONLY','n_factors':len(supported)}
    if proto=='ROLLING_TIME':
        key=(sid,target,h)
        ADVANCED_ENSEMBLE_REGISTRY[key]={'members':members,'study_id':sid,'target':target,'horizon':h,
            'train':full,'test':te,'features':features,'pred':pred,'dev_predictions':dev_mix,
            'dev_target':dv[target].to_numpy(float),'q90_dev':q90,
            'best_single_family':best_single_family,'best_single_pred':best_single_prediction}
        base=WORK/'models'/f'SUPERLEARNER_{sid}_{target}_h{h}.joblib'
        joblib.dump({'members':members,'source_sha':source_sha,'model_stage':'DEV_FROZEN_FULL_REFIT',
                     'target':target,'horizon':h,'features':features},base,compress=3)
    return result

if CFG['ADVANCED_ENABLE']:
    progress('ADVANCED',0,len(TASKS),'Ансамбль: проверка по DEV')
    _adv_task_done=0
    for t in TASKS:
        sid=t['study']['id'];h=t['horizon'];target=t['target'];proto=t['protocol']
        try:ADVANCED_TEST_ROWS.append(_adv_fit_one_task(t))
        except Exception as ex:
            ADVANCED_TEST_ROWS.append({'study_id':sid,'target':target,'horizon':h,'protocol':proto,
                'status':'FAIL','reason':type(ex).__name__+': '+str(ex)[:250]})
            log('ADVANCED ENSEMBLE '+sid+'/'+target+': '+type(ex).__name__+': '+str(ex)[:200],'ERROR')
        _adv_task_done+=1
        progress('ADVANCED',_adv_task_done,len(TASKS),f'Задач: {_adv_task_done}/{len(TASKS)}')
    progress('ADVANCED',1,1,'DEV-ансамбль завершён')
if not CFG['ADVANCED_ENABLE']:progress('ADVANCED',1,1,'Ансамбль отключён настройкой')
ADVANCED_DEV=pd.DataFrame(ADVANCED_DEV_ROWS);ADVANCED_ENSEMBLES=pd.DataFrame(ADVANCED_TEST_ROWS)
ADVANCED_INTERVALS=pd.DataFrame(ADVANCED_INTERVAL_ROWS)
if CFG['ADVANCED_ENABLE']:
    allpass=bool(len(ADVANCED_ENSEMBLES)==len(TASKS) and ADVANCED_ENSEMBLES.status.eq('PASS').all()) if len(TASKS) else False
    mark('ADVANCED_ENSEMBLE','PASS' if allpass else 'PARTIAL',
        f'completed={int(ADVANCED_ENSEMBLES.status.eq("PASS").sum()) if len(ADVANCED_ENSEMBLES) else 0}/{len(TASKS)}; ' +
        f'families={ADV_BANK}; dev-only weight calibration')
else:mark('ADVANCED_ENSEMBLE','NOT_FEASIBLE','Explicitly disabled')
# Rebuild canonical output structures; the next cell computes all independent metrics/CI from combined predictions.
METRICS=pd.DataFrame(METRIC_ROWS)
PREDICTIONS=pd.concat(PRED_ROWS,ignore_index=True) if PRED_ROWS else pd.DataFrame()
if len(PREDICTIONS):
    _id_cols=['entity_id','origin_year','target_year','horizon','target','study_id','split_id','arm','model','seed']
    ensure(not PREDICTIONS.duplicated(_id_cols).any(),'ADVANCED_OUTPUT_DUPLICATE_TEST_KEYS')
quiet_print('Advanced:',len(ADVANCED_ENSEMBLES),'tasks | pass',int(ADVANCED_ENSEMBLES.status.eq('PASS').sum()) if len(ADVANCED_ENSEMBLES) else 0,
      '| full prediction rows:',len(PREDICTIONS))

# CHECKPOINT II: полные модельные предсказания и реестр выборки, включая ансамбли.
if CFG.get('ENABLE_CHECKPOINTS'):
    cp=WORK/'checkpoints';cp.mkdir(parents=True,exist_ok=True)
    checkpoint_ii=[]
    for label,frame in [('all_test_predictions',PREDICTIONS),('model_test_metrics',METRICS),
                        ('dev_model_selection',DEV_METRICS),('test_split_registry',SPLITS),
                        ('ensemble_model_tests',ADVANCED_ENSEMBLES),('ensemble_dev_selection',ADVANCED_DEV),
                        ('seed_independence_QA',SEED_QA)]:
        file=cp/(label+'.parquet');frame.to_parquet(file,index=False,compression='zstd')
        checkpoint_ii.append({'file':file.name,'rows':len(frame),'columns':len(frame.columns),'sha256':digest(file)})
    (cp/'stage_II_checkpoint.json').write_text(json.dumps({'version':CFG['VERSION'],
       'status':'PHYSICAL_PARQUET_CHECKPOINT','source_commit':source_sha,
       'assets':checkpoint_ii,'saved_models':[x.name for x in sorted((WORK/'models').glob('*.joblib'))]},
       ensure_ascii=False,indent=2),encoding='utf-8')
    progress('ADVANCED',1,1,'Модели и checkpoint II зафиксированы')


#### V5: независимое сопоставление ключей прогнозов с подтверждённым Evidence V4.03.

In [11]:
# R4 · V5 · ПАРНОЕ ВОСПРОИЗВОДИМОЕ СОПОСТАВЛЕНИЕ КЛЮЧЕЙ ПРЕДСКАЗАНИЙ С V4.03
# Сравниваем именно те наблюдения, по которым сделаны тестовые предсказания
# исторической моделью CANONICAL / ROLLING_TIME, а не всю исходную D01.
V5_REFERENCE_DOWNLOAD_QA=[]
V5_ROW_REFERENCE_QA=pd.DataFrame(columns=['reference_version','target','horizon',
    'n_reference_keys','n_v5_keys','only_reference_keys','only_v5_keys','paired_keys','status'])
V5_ROW_EXCLUSION_REASONS=pd.DataFrame(columns=['entity_id','origin_year','target_year','horizon','reason'])

def _v5_reference_zip():
    local=str(CFG.get('REFERENCE_V4_ARCHIVE_LOCAL') or '').strip()
    if local:
        path=Path(local)
        if path.is_file() and zipfile.is_zipfile(path):return path,'USER_LOCAL_ARCHIVE'
        V5_REFERENCE_DOWNLOAD_QA.append({'status':'INVALID_LOCAL_ARCHIVE','path_name':path.name})
    if not CFG.get('V403_REFERENCE_GITHUB') or CFG['MODE']!='FULL':return None,'AUTOMATED_DOWNLOAD_DISABLED'
    cache=WORK/'cache'/'ML03_V403_VERIFIED_REFERENCE.zip'
    expected=str(CFG.get('V403_REFERENCE_SHA256'))
    if cache.exists():
        if digest(cache)==expected:return cache,'GITHUB_CACHE_SHA256_VERIFIED'
        cache.unlink()
    try:
        tag=CFG['V403_REFERENCE_TAG']
        release=api('GET','/releases/tags/'+tag).json()
        a=next(a for a in release.get('assets',[]) if a.get('name','').endswith('.zip') and 'TECHNICAL_EVIDENCE' in a.get('name',''))
        if int(a['size'])>int(CFG['MAX_REFERENCE_ARCHIVE_MB'])*2**20:
            raise ValueError('V403_REFERENCE_SIZE_LIMIT')
        url='https://api.github.com/repos/'+CFG['REPO']+'/releases/assets/'+str(a['id'])
        # Github REST returns redirect to a short-lived signed URL for binary asset.
        with SESSION.get(url,headers={'Accept':'application/octet-stream'},stream=True,timeout=(30,240)) as response:
            response.raise_for_status()
            if 'json' in response.headers.get('content-type','').lower():
                raise ValueError('GITHUB_REFERENCE_RETURNED_JSON_NOT_ZIP')
            h=hashlib.sha256();got=0
            with open(str(cache)+'.part','wb') as f:
                for chunk in response.iter_content(2**20):
                    if not chunk:continue
                    got+=len(chunk)
                    if got>int(CFG['MAX_REFERENCE_ARCHIVE_MB'])*2**20:
                        raise ValueError('V403_REFERENCE_EXCEEDED_SIZE_LIMIT')
                    h.update(chunk);f.write(chunk)
            ensure(h.hexdigest()==expected,'V403_REFERENCE_SHA256_MISMATCH')
        Path(str(cache)+'.part').replace(cache)
        ensure(zipfile.is_zipfile(cache),'V403_REFERENCE_CRC_INVALID')
        return cache,'GITHUB_RELEASE_SHA256_VERIFIED'
    except Exception as exc:
        Path(str(cache)+'.part').unlink(missing_ok=True)
        V5_REFERENCE_DOWNLOAD_QA.append({'status':'REFERENCE_FETCH_FAILED','exception':type(exc).__name__,
            'detail':str(exc)[:240]})
        return None,'REFERENCE_FETCH_FAILED'

_v4ref,refmode=_v5_reference_zip()
if _v4ref is not None:
    try:
        with zipfile.ZipFile(_v4ref) as z:
            members=[n for n in z.namelist() if n.endswith('test_predictions.parquet')]
            ensure(len(members)==1,'V403_REFERENCE_PREDICTION_MEMBER_AMBIGUOUS')
            # Only key columns required. No previous model training and no imputed labels.
            with z.open(members[0]) as f:
                old=pd.read_parquet(io.BytesIO(f.read()),columns=['entity_id','origin_year','target_year','horizon',
                    'study_id','target','split_id','model','arm','seed'])
        _required={'entity_id','origin_year','target_year','horizon','study_id','target','split_id','model'}
        ensure(_required.issubset(old.columns),'V403_REFERENCE_PREDICTION_SCHEMA_FAILED')
        _keys=['entity_id','origin_year','target_year','horizon']
        missing_records=[]
        for target in CFG['TARGETS']:
            for h in CFG['HORIZONS']:
                where=lambda d:(d.study_id.eq('CANONICAL')&d.target.eq(target)&d.horizon.eq(h)&
                    d.split_id.eq('ROLLING_TIME')&d.model.eq('HISTORICAL'))
                ok=old.loc[where(old),_keys].drop_duplicates()
                cur=PREDICTIONS.loc[where(PREDICTIONS),_keys].drop_duplicates()
                oldsize,newsize=len(ok),len(cur)
                matched=ok.merge(cur,on=_keys,how='outer',indicator=True,validate='one_to_one')
                V5_ROW_REFERENCE_QA.loc[len(V5_ROW_REFERENCE_QA)]={
                    'reference_version':'V4.03','target':target,'horizon':h,
                    'n_reference_keys':oldsize,'n_v5_keys':newsize,
                    'only_reference_keys':int(matched._merge.eq('left_only').sum()),
                    'only_v5_keys':int(matched._merge.eq('right_only').sum()),
                    'paired_keys':int(matched._merge.eq('both').sum()),
                    'status':'IDENTICAL_KEYS' if matched._merge.eq('both').all() else 'KEYS_DIFFER_DIAGNOSED'}
                absent=matched.loc[matched._merge.eq('left_only'),_keys].copy()
                if not absent.empty:
                    subset=TASK_PANEL.loc[TASK_PANEL.horizon.eq(h),['entity_id','origin_year','target_year','horizon',target]].copy()
                    absent=absent.merge(subset,on=_keys,how='left',validate='one_to_one')
                    absent['reason']=np.where(absent[target].isna(),'D01_TARGET_OR_ORIGIN_LABEL_UNAVAILABLE',
                        'D01_LABEL_EXISTS_BUT_TEST_DESIGN_OR_FACTOR_SUPPORT_CHANGED')
                    missing_records.append(absent.drop(columns=[target]))
        if missing_records:V5_ROW_EXCLUSION_REASONS=pd.concat(missing_records,ignore_index=True)
        V5_REFERENCE_DOWNLOAD_QA.append({'status':'EXACT_COMPARISON_COMPLETE','archive_mode':refmode,
            'n_comparisons':len(V5_ROW_REFERENCE_QA),'missing_keys':len(V5_ROW_EXCLUSION_REASONS)})
    except Exception as exc:
        V5_REFERENCE_DOWNLOAD_QA.append({'status':'REFERENCE_COMPARISON_FAILED','exception':type(exc).__name__,
            'detail':str(exc)[:250]})
else:
    V5_REFERENCE_DOWNLOAD_QA.append({'status':'NOT_COMPARED_ROW_LEVEL','reason':refmode})
# V5: детальная декомпозиция перехода V4.02 → V4.03 → V5 без приписывания
# различий в числе моделей «потерянным муниципалитетам».
V5_PREDICTION_VOLUME_QA=pd.DataFrame([
 {'version':'V4.02','n_predictions':4183063,'source':'V402_TECHNICAL_EVIDENCE_PREVIOUS_AUDIT'},
 {'version':'V4.03','n_predictions':4149839,'source':'V403_TECHNICAL_EVIDENCE_PREVIOUS_AUDIT'},
 {'version':'V5.00','n_predictions':len(PREDICTIONS),'source':'CURRENT_EXECUTED_MODEL_PREDICTIONS'}])
V5_PREDICTION_VOLUME_QA['difference_from_v403']=V5_PREDICTION_VOLUME_QA.n_predictions-4149839
V5_SEED_FAMILY_PARITY=pd.DataFrame(columns=['study_id','target','horizon','split_id','arm','model','seed',
    'n_v403','n_v5','delta_n','status'])
if 'old' in locals() and isinstance(old,pd.DataFrame) and len(old) and len(PREDICTIONS):
    labels=['study_id','target','horizon','split_id','arm','model','seed']
    a=old.groupby(labels,dropna=False).size().reset_index(name='n_v403')
    b=PREDICTIONS.groupby(labels,dropna=False).size().reset_index(name='n_v5')
    V5_SEED_FAMILY_PARITY=a.merge(b,on=labels,how='outer',validate='one_to_one')
    V5_SEED_FAMILY_PARITY[['n_v403','n_v5']]=V5_SEED_FAMILY_PARITY[['n_v403','n_v5']].fillna(0).astype('int64')
    V5_SEED_FAMILY_PARITY['delta_n']=V5_SEED_FAMILY_PARITY.n_v5-V5_SEED_FAMILY_PARITY.n_v403
    V5_SEED_FAMILY_PARITY['status']=np.where(V5_SEED_FAMILY_PARITY.n_v403.eq(0),'NEW_MODEL_OR_SEED',
        np.where(V5_SEED_FAMILY_PARITY.n_v5.eq(0),'REMOVED_MODEL_OR_SEED',
            np.where(V5_SEED_FAMILY_PARITY.delta_n.eq(0),'SAME_COUNT','CHANGED_TEST_SUPPORT')))
    ensure(int(V5_SEED_FAMILY_PARITY.n_v403.sum())==len(old),'V403_SEED_FAMILY_ROW_TOTAL_MISMATCH')
    ensure(int(V5_SEED_FAMILY_PARITY.n_v5.sum())==len(PREDICTIONS),'V5_SEED_FAMILY_ROW_TOTAL_MISMATCH')
mark('V5_PREDICTION_VOLUME_AUDIT','PASS',
    f'V403={4149839:,}; V5={len(PREDICTIONS):,}; detailed seed/family comparison rows={len(V5_SEED_FAMILY_PARITY)}')

V5_REFERENCE_DOWNLOAD_QA=pd.DataFrame(V5_REFERENCE_DOWNLOAD_QA)
mark('V403_ROW_KEY_RECONCILIATION','PASS' if len(V5_ROW_REFERENCE_QA)==len(CFG['TARGETS'])*len(CFG['HORIZONS']) else 'SOURCE_LIMITED',
     f'Row-level comparisons={len(V5_ROW_REFERENCE_QA)}; unpaired prior keys={len(V5_ROW_EXCLUSION_REASONS)}; archive={refmode}')


# Этап III / IV. Статистика, объяснение прогнозов и устойчивость

#### Парные сравнения, региональный bootstrap, абляции, SHAP.

In [12]:
# 09 · COMMON-KEY ABLATIONS, REGION-BLOCK BOOTSTRAP, HOLM/FDR, ALE, INTERACTIONS
progress('STATS',0,1,'Парные сравнения и статистика')
ABLA_ROWS=[];LOO_ROWS=[];CORR_ROWS=[];INTERACTION_ROWS=[];TRANSFER_ROWS=[]
GEO_ROWS=[];IMP_ROWS=[];ALE_ROWS=[];SHAP_ROWS=[];FE_ROWS=[];SENSITIVITY_ROWS=[];ERROR_YEAR_ROWS=[]
def holm_adjust(p):
    arr=np.asarray(p,float);order=np.argsort(arr);out=np.empty(len(arr));running=0.
    for rank,j in enumerate(order):
        running=max(running,min(1.0,float(arr[j])*(len(arr)-rank)));out[j]=running
    return out
def bh_adjust(p):
    arr=np.asarray(p,float);ordr=np.argsort(arr);out=np.empty(len(arr));running=1.
    for rank in reversed(range(len(arr))):
        j=ordr[rank];running=min(running,float(arr[j])*len(arr)/(rank+1));out[j]=running
    return out
def block_delta(z,old,new,boots,seed):
    v=z[['region_name','target_year',old,new]].dropna()
    groups=[g for _,g in v.groupby('region_name',dropna=False)]
    if len(groups)<CFG['MIN_BOOTSTRAP_REGIONS']:return None
    d=np.array([(g[old]-g[new]).sum() for g in groups]);n=np.array([len(g) for g in groups])
    point=float(d.sum()/n.sum())
    rng=np.random.default_rng(seed);ids=rng.integers(0,len(groups),(boots,len(groups)))
    vals=(d[ids].sum(axis=1)/n[ids].sum(axis=1))
    ci=np.quantile(vals,[0.025,0.975])
    # Monte Carlo finite-sample p with +1 correction; no p=0 due to bootstrap discretization.
    # Random region-level sign-flips approximate a paired null test; CI remains a block bootstrap.
    signs=rng.choice(np.array([-1.,1.]),size=(boots,len(groups)),replace=True)
    null_dist=(signs*d).sum(axis=1)/n.sum()
    p=(1+int((np.abs(null_dist)>=abs(point)).sum()))/(boots+1)
    return {'delta_mae':point,'ci_low':float(ci[0]),'ci_high':float(ci[1]),
         'p_cluster_signflip':float(p),'n_regions':len(groups),'n_common_keys':len(v),
         'bootstrap_unit':'region','n_bootstraps':boots}

# Isolated, inexpensive sign-flip / regional block unit test.
_check=pd.DataFrame({'region_name':list('ABCDEFGH')*2,'target_year':[2022]*16,
                      '_baseline_error':[2.0]*16,'_improved_error':[1.0]*16})
_st=block_delta(_check,'_baseline_error','_improved_error',99,CFG['SEED'])
ensure(_st is not None and abs(_st['delta_mae']-1.0)<1e-12 and _st['n_regions']==8,
       'UNIT_CLUSTER_INFERENCE_FAILURE')
mark('UNIT_STAT_CONTRACT','PASS','Synthetic sign-flip/bootstrap arithmetic; excluded from scientific findings')

if len(PREDICTIONS):
    selected=PREDICTIONS[~PREDICTIONS.model.isin(['HISTORICAL','REGIONAL_MEDIAN'])].copy()
    keys=['entity_id','origin_year','target_year','region_name']
    for (sid,target,h,proto,model),p in selected.groupby(['study_id','target','horizon','split_id','model']):
        # Average losses over stochastic seeds BEFORE block resampling; seeds are not independent municipalities.
        grp=p.groupby(keys+['arm'],dropna=False).agg(absolute_error=('absolute_error','mean')).reset_index()
        wide=grp.pivot(index=keys,columns='arm',values='absolute_error').reset_index()
        st=next(s for s in STUDIES if s['id']==sid)
        if st['id']=='CANONICAL' or st['id'].startswith('YEAR_'):
            pairs=[(a['arm'],b['arm']) for a,b in zip(st['arms'][:-1],st['arms'][1:])]
            pairs+=([('M1','M3')] if 'M3' in wide and 'M1' in wide else [])
        elif st['id'].startswith('AUX_'):
            pairs=[('M0',st['arms'][-1]['arm'])]
        else:
            pairs=[('M0','FULL_AVAILABLE')]
            pairs += [('WITHOUT_'+g,'FULL_AVAILABLE') for g in AVAILABLE_GROUPS if 'WITHOUT_'+g in wide]
        for old,new in pairs:
            if old not in wide or new not in wide:continue
            stat=block_delta(wide,old,new,CFG['BOOTSTRAPS'],CFG['SEED']+h)
            if stat:
                row={'study_id':sid,'target':target,'horizon':h,'protocol':proto,
                     'model':model,'old_arm':old,'new_arm':new,
                     'comparison':old+' → '+new,'confirmatory':(st['confirmatory'] and
                     len(TASK_STATUSES.loc[(TASK_STATUSES.study_id==sid)&
                      (TASK_STATUSES.target==target)&(TASK_STATUSES.horizon==h)&
                      (TASK_STATUSES.protocol==proto)&
                      (TASK_STATUSES.scientific_status=='COMPLETE_YEARS')])>0)}
                ABLA_ROWS.append({**row,**stat})
        best=p[p.arm.eq(st['arms'][-1]['arm'])].drop_duplicates(
            ['entity_id','origin_year','target_year','arm'],keep='first')
        if len(best):
            for reg,v in best.groupby('region_name'):
                if len(v)>5:TRANSFER_ROWS.append({'study_id':sid,'target':target,'horizon':h,
                    'protocol':proto,'region':reg,'n':len(v),'mae':float(v.absolute_error.mean())})
            for (reg,mt),v in best.groupby(['region_name','mun_type'],dropna=False):
                if len(v)>5:GEO_ROWS.append({'study_id':sid,'target':target,'horizon':h,
                    'protocol':proto,'region':reg,'mun_type_original':mt,'n':len(v),
                    'mae':float(v.absolute_error.mean())})
            for year,v in best.groupby('target_year'):
                ERROR_YEAR_ROWS.append({'study_id':sid,'target':target,'horizon':h,'protocol':proto,
                   'year':int(year),'n':len(v),'mae':float(v.absolute_error.mean())})
ABLA=pd.DataFrame(ABLA_ROWS)
if len(ABLA):
    ABLA['p_holm']=np.nan;ABLA['q_fdr_bh']=np.nan
    for keys,inds in ABLA.groupby(['target','horizon','protocol']).groups.items():
        for is_confirm in (True,False):
            ix=[i for i in inds if bool(ABLA.loc[i,'confirmatory'])==is_confirm]
            if ix:
                p=ABLA.loc[ix,'p_cluster_signflip'].tolist()
                if is_confirm:ABLA.loc[ix,'p_holm']=holm_adjust(p)
                else:ABLA.loc[ix,'q_fdr_bh']=bh_adjust(p)
    ABLA['finding']=np.where((ABLA.ci_low>0)&(
          np.where(ABLA.confirmatory,ABLA.p_holm,ABLA.q_fdr_bh)<0.05),
          'PREDICTIVE_IMPROVEMENT',np.where(ABLA.ci_high<0,'DEGRADATION_OR_NEGATIVE','NOT_CONFIRMED'))
    mark('PASS_VALIDATION','PASS',f'{len(ABLA)} paired group contrasts; region-block CI; Holm confirmatory/FDR exploratory')
else:mark('PASS_VALIDATION','NOT_FEASIBLE','Not enough regions/pairs for block bootstrap')
LOO=ABLA[ABLA.study_id.eq('MAX_AVAILABLE')&ABLA.old_arm.str.startswith('WITHOUT_',na=False)].copy() if len(ABLA) else pd.DataFrame()
REGIONAL_ERRORS=pd.DataFrame(TRANSFER_ROWS);MUNICIPAL_TYPES=pd.DataFrame(GEO_ROWS)
ERROR_YEARS=pd.DataFrame(ERROR_YEAR_ROWS)

# Непосредственно из сохранённых test predictions: независимый пересчёт каждой MAE/RMSE.
PREDICTION_METRIC_QA=pd.DataFrame()
if len(PREDICTIONS) and len(METRICS):
    groups=['study_id','target','horizon','split_id','arm','model','seed']
    pp=PREDICTIONS.copy()
    pp['weight_eff']=pd.to_numeric(pp.population_weight,errors='coerce').fillna(0).clip(lower=0)
    pp['weighted_loss']=pp.absolute_error*pp.weight_eff
    calc=pp.groupby(groups,dropna=False).agg(
      independent_mae=('absolute_error','mean'),
      independent_rmse=('squared_error',lambda z:float(np.sqrt(np.mean(z)))),
      independent_n=('entity_id','size'),sum_weight=('weight_eff','sum'),
      sum_weighted_loss=('weighted_loss','sum')).reset_index().rename(columns={'split_id':'protocol'})
    calc['independent_weighted_mae']=np.where(calc.sum_weight.gt(0),
         calc.sum_weighted_loss/calc.sum_weight,np.nan)
    cmp=METRICS.merge(calc,on=['study_id','target','horizon','protocol','arm','model','seed'],
        how='outer',validate='one_to_one',indicator=True)
    cmp['mae_abs_difference']=(cmp.mae-cmp.independent_mae).abs()
    cmp['rmse_abs_difference']=(cmp.rmse-cmp.independent_rmse).abs()
    cmp['n_difference']=(cmp.n_test-cmp.independent_n).abs()
    cmp['weighted_mae_abs_difference']=(cmp.weighted_mae-cmp.independent_weighted_mae).abs()
    PREDICTION_METRIC_QA=cmp[['study_id','target','horizon','protocol','arm','model','seed',
       '_merge','mae_abs_difference','rmse_abs_difference',
       'weighted_mae_abs_difference','n_difference']].copy()
    passed=(cmp['_merge'].eq('both').all() and
       cmp[['mae_abs_difference','rmse_abs_difference','weighted_mae_abs_difference']].max().max()<1e-10 and
       cmp.n_difference.eq(0).all())
    mark('INDEPENDENT_METRIC_RECONCILIATION','PASS' if passed else 'FAIL',
         f'rows={len(cmp)}; max MAE diff={cmp.mae_abs_difference.max()}')
    ensure(passed,'INDEPENDENT_METRIC_RECONCILIATION_FAILED')

# Correlation diagnostics use TRAIN ONLY and the ORIGINAL training measurements (no post-hoc test selection).
for task in [t for t in TASKS if t['protocol']=='ROLLING_TIME' and t['study']['id']=='MAX_AVAILABLE']:
    tr=task['data'][task['masks']['train']]
    ff=list(dict.fromkeys(G0_FEATURES+sum(GROUP_FEATURES.values(),[])))
    for i,one in enumerate(ff):
        for two in ff[i+1:]:
            if one=='origin_year' or two=='origin_year':continue
            sample=tr[[one,two]].apply(pd.to_numeric,errors='coerce').dropna()
            if len(sample)<50 or min(sample[one].nunique(),sample[two].nunique())<5:continue
            r=sample[one].corr(sample[two],method='spearman')
            if np.isfinite(r) and abs(r)>=0.7:
                CORR_ROWS.append({'target':task['target'],'horizon':task['horizon'],
                  'feature_A':one,'feature_B':two,'n_train_pairs':len(sample),'spearman_r':float(r)})
CORRELATIONS=pd.DataFrame(CORR_ROWS)

# Interpretation of the fitted predictive function, not causal effects.
for (sid,target,h),registry in MODEL_REGISTRY.items():
    if sid not in ('MAX_AVAILABLE','CANONICAL'):continue
    tr=registry['train'];te=registry['test'];model=registry['model']
    Xtr,Xte,used,_=design(tr,te,registry['features'],registry['refs'])
    n=min(len(te),CFG['MAX_IMPORTANCE_ROWS'])
    if n<40:continue
    indices=np.sort(np.random.default_rng(CFG['SEED']).choice(len(te),n,replace=False))
    x=Xte.iloc[indices].copy();y=te[target].to_numpy(float)[indices]
    try:
        benchmark=mean_absolute_error(y,model.predict(x))
        for col in [u for u in used if u!='origin_year'][:CFG['MAX_IMPORTANCE_FEATURES']]:
            losses=[]
            for s in (11,37,71):
                z=x.copy();z[col]=np.random.default_rng(s).permutation(z[col].to_numpy())
                losses.append(float(mean_absolute_error(y,model.predict(z))-benchmark))
            IMP_ROWS.append({'study_id':sid,'target':target,'horizon':h,'feature':col,
                'mae_increase':float(np.mean(losses)),'sd_permutation':float(np.std(losses)),
                'n_test_sample':n,'scope':'predictive_unconditional_permutation'})
        # First-order ALE on observed TRAIN support; percentiles computed without final test outcomes.
        for col in [u for u in used if u.startswith(('G1_','G2_','G3_','G4_'))][:4]:
            observed=Xtr[col].dropna()
            if observed.nunique()<12:continue
            edges=np.unique(np.quantile(observed,np.linspace(0,1,11)))
            if len(edges)<5:continue
            accumulated=0.;raw_rows=[]
            for k in range(len(edges)-1):
                lo,hi=edges[k:k+2]
                mask=Xtr[col].between(lo,hi,inclusive='both')
                if mask.sum()<15:continue
                xl=Xtr.loc[mask].copy();xh=xl.copy();xl[col]=lo;xh[col]=hi
                inc=float(np.mean(model.predict(xh)-model.predict(xl)))
                accumulated+=inc
                raw_rows.append((k,(lo+hi)/2.,inc,accumulated,int(mask.sum())))
            if raw_rows:
                center=float(np.average([v[3] for v in raw_rows],weights=[v[4] for v in raw_rows]))
                for k,middle,inc,total,nn in raw_rows:
                    ALE_ROWS.append({'study_id':sid,'target':target,'horizon':h,'feature':col,
                      'bin':k,'feature_mid_logscale':middle,'ale_centered':total-center,'n_train':nn,
                      'scope':'train_support_first_order_ALE_not_causal'})
    except Exception as ex:log('importance/ALE '+sid+': '+type(ex).__name__,'WARN')
IMPORTANCE=pd.DataFrame(IMP_ROWS);ALE=pd.DataFrame(ALE_ROWS)

# Additional interpretable evidence: SHAP for best available tree model, on held-out data.
try:
    import shap
    SHAP_AVAILABLE=True
except Exception as ex:
    SHAP_AVAILABLE=False;log('SHAP import unavailable: '+type(ex).__name__,'WARN')
if SHAP_AVAILABLE:
    for (sid,target,h),r in MODEL_REGISTRY.items():
        if sid!='MAX_AVAILABLE' or r['family'] not in ('HGB','CATBOOST','LIGHTGBM'):continue
        try:
            xtr,xtest,used,_=design(r['train'],r['test'],r['features'],r['refs'])
            take=min(len(xtest),CFG['MAX_SHAP_ROWS']);xtest=xtest.iloc[:take]
            tree=shap.TreeExplainer(r['model'])
            values=tree.shap_values(xtest)
            if isinstance(values,list):values=values[0]
            values=np.asarray(values)
            ensure(values.shape==(len(xtest),xtest.shape[1]),'SHAP_SHAPE_MISMATCH')
            for j,name in enumerate(xtest.columns):
                SHAP_ROWS.append({'study_id':sid,'target':target,'horizon':h,'model':r['family'],
                  'feature':name,'mean_abs_shap':float(np.abs(values[:,j]).mean()),
                  'n_test_sample':take,'scope':'explanation_of_prediction_not_causal'})
        except Exception as ex:log('SHAP computation '+sid+': '+type(ex).__name__+' '+str(ex)[:120],'WARN')
SHAP_IMPORTANCE=pd.DataFrame(SHAP_ROWS)
mark('SHAP_QA','PASS' if len(SHAP_IMPORTANCE) else 'NOT_FEASIBLE',
     f'tree SHAP cells={len(SHAP_IMPORTANCE)}; optional where compatible')

# Train-only two-way fixed effects: identifiability / cluster diagnostics, never causal.
FE_QA_ROWS=[]
try:
    from linearmodels.panel import PanelOLS
    PANEL_OLS_AVAILABLE=True
except ImportError:
    PANEL_OLS_AVAILABLE=False

def _fit_fe_train_only(task):
    tr=task['data'].loc[task['masks']['train']].copy()
    report={'target':task['target'],'horizon':task['horizon'],
      'n_initial':int(len(tr)),'n_complete':0,'n_entities':0,'n_regions':0,
      'n_years':0,'n_regressors':0,'status':'NOT_FEASIBLE','reason':''}
    if not PANEL_OLS_AVAILABLE:
        report['reason']='LINEARMODELS_NOT_INSTALLED';return pd.DataFrame(),report
    available=[]
    for g in ('G1','G2','G3','G4','G5'):
        ff=[f for f in GROUP_FEATURES.get(g,[]) if f in tr]
        ff.sort(key=lambda f:tr[f].notna().sum(),reverse=True)
        if ff:available.append(ff[0])
    if not available:
        report['reason']='NO_FE_COVARIATES';return pd.DataFrame(),report
    columns=['entity_id','origin_year','region_name',task['target']]+available
    w=tr[columns].copy()
    for f in available:w[f]=pd.to_numeric(w[f],errors='coerce')
    w=w.dropna()
    counts=w.groupby('entity_id').origin_year.nunique()
    w=w[w.entity_id.isin(counts[counts.ge(CFG['MIN_FE_ENTITY_OBSERVATIONS'])].index)].copy()
    report.update(n_complete=int(len(w)),n_entities=int(w.entity_id.nunique()),
        n_regions=int(w.region_name.nunique()),n_years=int(w.origin_year.nunique()))
    if len(w)<400 or w.entity_id.nunique()<20 or w.region_name.nunique()<8 or w.origin_year.nunique()<4:
        report['reason']='INSUFFICIENT_CONNECTED_FE_PANEL';return pd.DataFrame(),report
    # Drop regressors absorbed by entity or time effects: residual variance of two-way demeaned x.
    keys=['entity_id','origin_year']
    X=pd.DataFrame(index=w.index)
    for f in available:
        x=np.sign(w[f])*np.log1p(np.abs(w[f]))
        residual=x-x.groupby(w.entity_id).transform('mean')-x.groupby(w.origin_year).transform('mean')+x.mean()
        if residual.std()>1e-8:X[f]=x
    if X.empty:
        report['reason']='ALL_COVARIATES_ABSORBED_BY_FIXED_EFFECTS';return pd.DataFrame(),report
    residualized=pd.DataFrame(index=w.index)
    for f in X:
        a=X[f]
        residualized[f]=a-a.groupby(w.entity_id).transform('mean')-a.groupby(w.origin_year).transform('mean')+a.mean()
    rank=np.linalg.matrix_rank(residualized.to_numpy(float),tol=1e-9)
    if rank<len(residualized.columns):
        selected=[]
        for f in residualized.columns:
            candidate=selected+[f]
            if np.linalg.matrix_rank(residualized[candidate].to_numpy(float),tol=1e-9)==len(candidate):
                selected.append(f)
        X=X[selected]
    report['n_regressors']=len(X.columns)
    if X.empty:
        report['reason']='COLLINEAR_WITHIN_FE';return pd.DataFrame(),report
    aligned=w.set_index(keys).sort_index()
    x=X.loc[aligned.reset_index().index] if False else pd.DataFrame(
       X.to_numpy(float),index=pd.MultiIndex.from_frame(w[keys]),columns=X.columns).sort_index()
    y=aligned[task['target']].astype(float)
    groups=pd.DataFrame({'region_cluster':pd.factorize(aligned.region_name.astype(str))[0]},index=aligned.index)
    try:
        model=PanelOLS(y,x,entity_effects=True,time_effects=True,drop_absorbed=True,check_rank=False)
        fit=model.fit(cov_type='clustered',clusters=groups)
        rows=pd.DataFrame([{'target':task['target'],'horizon':task['horizon'],'parameter':f,
           'coefficient':float(fit.params[f]),'std_error_region_cluster':float(fit.std_errors[f]),
           'p_value':float(fit.pvalues[f]),'n_train':int(len(w)),
           'n_municipalities':int(w.entity_id.nunique()),'n_clusters':int(w.region_name.nunique()),
           'interpretation':'train-only two-way FE; predictive association, NOT causal'}
          for f in fit.params.index])
        report.update(status='PASS',reason='TWO_WAY_FE_TRAIN_ONLY_REGION_CLUSTERED')
        return rows,report
    except Exception as ex:
        report['reason']='FIT_EXCEPTION_'+type(ex).__name__+': '+str(ex)[:200]
        return pd.DataFrame(),report

if PANEL_OLS_AVAILABLE:
    for task in TASKS:
        if task['study']['id']!='MAX_AVAILABLE' or task['protocol']!='ROLLING_TIME' or task['horizon']!=1:
            continue
        rows,qa=_fit_fe_train_only(task)
        FE_QA_ROWS.append(qa)
        if not rows.empty:FE_ROWS.extend(rows.to_dict('records'))
else:
    FE_QA_ROWS.append({'status':'NOT_FEASIBLE','reason':'LINEARMODELS_UNAVAILABLE'})
FE_QA=pd.DataFrame(FE_QA_ROWS)
FE_DIAGNOSTICS=pd.DataFrame(FE_ROWS)
mark('FE_DIAGNOSTICS','PASS' if len(FE_DIAGNOSTICS) else 'NOT_FEASIBLE',
     f'train-only FE parameter rows={len(FE_DIAGNOSTICS)}, audited attempts={len(FE_QA)}')
# Exploratory interactions (H3.2), identical test labels/keys, no post-hoc causal claim.
# Non-arbitrary G6: semantically identified wage/income by housing access or stock.
def _feature_for_domain(group,domains):
    rows=FEATURE_DICT[FEATURE_DICT.group.eq(group)&
       FEATURE_DICT.domain.isin(domains)&FEATURE_DICT.model_feature.notna()]
    for record in rows.to_dict('records'):
        if record['model_feature'] in GROUP_FEATURES[group]:return record['model_feature']
    return None
G6_ECON_FEATURE=_feature_for_domain('G1',('income','wages'))
G6_HOUSING_FEATURE=_feature_for_domain('G2',('housing_access','housing_stock','housing_construction'))
if G6_ECON_FEATURE and G6_HOUSING_FEATURE:
    g1,g2=G6_ECON_FEATURE,G6_HOUSING_FEATURE
    for (sid,target,h),reg in MODEL_REGISTRY.items():
        if sid!='MAX_AVAILABLE':continue
        tr=reg['train'].copy();te=reg['test'].copy()
        inter='G6_income_housing_interaction'
        signed_log=lambda x:np.sign(pd.to_numeric(x,errors='coerce'))*np.log1p(np.abs(pd.to_numeric(x,errors='coerce')))
        tr[inter]=signed_log(tr[g1])*signed_log(tr[g2]);te[inter]=signed_log(te[g1])*signed_log(te[g2])
        if tr[inter].notna().sum()<CFG['MIN_TRAIN_ROWS']:continue
        try:
            pred,_,_,_=fit_predict(reg['family'],tr,te,target,reg['features']+[inter],CFG['SEED'],reg['params'])
            ref=metrics(te[target],reg['pred'],te[f'hist_population_l{CFG["FEATURE_PUBLICATION_LAG"]}'])
            cur=metrics(te[target],pred,te[f'hist_population_l{CFG["FEATURE_PUBLICATION_LAG"]}'])
            INTERACTION_ROWS.append({'target':target,'horizon':h,'study_id':sid,
              'factor_A':g1,'factor_B':g2,'base_mae':ref['mae'],'interaction_mae':cur['mae'],
              'delta_mae':ref['mae']-cur['mae'],'n_test':len(te),
              'status':'EXPLORATORY_PREDICTIVE_NOT_CAUSAL'})
        except Exception as ex:log('interaction '+str(h)+': '+type(ex).__name__,'WARN')
INTERACTIONS=pd.DataFrame(INTERACTION_ROWS)
mark('H3.2','PASS' if len(INTERACTIONS) else 'NOT_FEASIBLE',
     f'{len(INTERACTIONS)} semantically anchored income/wages × housing comparisons')
mark('CAUSAL_MODULE','CAUSAL_NOT_IDENTIFIED','No validated treatment register with dates, exposure, and controls')


# ADVANCED · V4.00 · ENSEMBLE SHAP, ADDITIVITY, GROUP PERMUTATION, ALE/PDP/ICE, REGION STABILITY
# All explanations refer to frozen predictors trained without TEST labels. Predictive, never causal.
ADVANCED_SHAP_ROWS=[];ADVANCED_SHAP_LOCAL_ROWS=[];ADVANCED_SHAP_QA_ROWS=[]
ADVANCED_GROUP_PERM_ROWS=[];ADVANCED_PDP_ROWS=[];ADVANCED_ICE_ROWS=[];ADVANCED_SHAP_CI_ROWS=[];ADVANCED_SHAP_STABILITY_ROWS=[];ADVANCED_SHAP_INTERACTION_ROWS=[];ADVANCED_LOO_ROWS=[]
try:import shap as _shap
except Exception as _e:
    _shap=None;log('Advanced SHAP import failed: '+str(_e)[:150],'WARN')

def _explain_one(m,x,bg):
    model=m['model'];family=m['family'];cols=list(x.columns)
    if family=='RIDGE':
        prep=model[:-1];head=model[-1];tx=np.asarray(prep.transform(x),float);tb=np.asarray(prep.transform(bg),float)
        mean=tb.mean(axis=0);co=np.asarray(head.coef_,float)
        v=(tx-mean)*co;bias=np.full(len(x),float(head.intercept_+mean@co))
        names=list(cols)+[f'missing_indicator_{int(k)}' for k in prep[0].indicator_.features_] if v.shape[1]>len(cols) else list(cols)
        return v,bias,names,'ANALYTICAL_LINEAR'
    if family=='EXTRA_TREES':
        prep=model[:-1];tree=model[-1];xx=np.asarray(prep.transform(x),float)
        names=list(cols)+[f'missing_indicator_{int(k)}' for k in prep[0].indicator_.features_] if xx.shape[1]>len(cols) else list(cols)
        if _shap is None:raise RuntimeError('SHAP_UNAVAILABLE_FOR_EXTRA_TREES')
        expl=_shap.TreeExplainer(tree,feature_perturbation='tree_path_dependent')
        values=np.asarray(expl.shap_values(xx),float)
        bias=np.broadcast_to(np.asarray(expl.expected_value,float).reshape(-1)[0],len(x))
        return values,bias,names,'TREE_PATH_DEPENDENT'
    if family=='XGBOOST':
        import xgboost
        dm=xgboost.DMatrix(x,feature_names=cols)
        raw=np.asarray(model.get_booster().predict(dm,pred_contribs=True),float)
        ensure(raw.shape==(len(x),len(cols)+1),'XGB_SHAP_CONTRIBS_WRONG_SHAPE')
        return raw[:,:-1],raw[:,-1],cols,'NATIVE_TREE_SHAP'
    if family=='LIGHTGBM':
        raw=np.asarray(model.predict(x,pred_contrib=True),float)
        ensure(raw.shape==(len(x),len(cols)+1),'LGB_SHAP_CONTRIBS_WRONG_SHAPE')
        return raw[:,:-1],raw[:,-1],cols,'NATIVE_TREE_SHAP'
    if family=='CATBOOST':
        from catboost import Pool
        raw=np.asarray(model.get_feature_importance(Pool(x),type='ShapValues'),float)
        ensure(raw.shape==(len(x),len(cols)+1),'CATBOOST_SHAP_CONTRIBS_WRONG_SHAPE')
        return raw[:,:-1],raw[:,-1],cols,'NATIVE_TREE_SHAP'
    if _shap is None:raise RuntimeError('SHAP_NOT_INSTALLED_FOR_'+family)
    expl=_shap.TreeExplainer(model,feature_perturbation='tree_path_dependent')
    values=np.asarray(expl.shap_values(x),float)
    return values,np.broadcast_to(float(np.asarray(expl.expected_value).reshape(-1)[0]),len(x)),cols,'TREE_PATH_DEPENDENT'

def _adv_predict(reg,x):
    pp=[]
    for m in reg['members']:
        pp.append(m['weight']*np.asarray(m['model'].predict(x),float))
    return np.sum(pp,axis=0)

def _adv_feature_group(feature):
    if feature.startswith('G1R_'):return 'G1R_REGIONAL_NOT_MUNICIPAL'
    for g in ('G1','G2','G3','G4','G5'):
        if feature.startswith(g+'_'):return g
    if feature.startswith('region_'):return 'REGION_FIXED_DUMMIES'
    if feature.startswith('hist_'):return 'G0_DEMOGRAPHIC_HISTORY'
    if feature=='origin_year':return 'G0_TIME'
    return 'OTHER'

def _within_strata_permute(X,source,cols,seed):
    rng=np.random.default_rng(seed);res=X.copy();groups=source[['region_name','target_year']].reset_index(drop=True)
    ensure(len(groups)==len(X),'SHUFFLE_STRATA_ALIGNMENT_INVALID')
    # Joint permutation preserves correlations WITHIN each named feature block.
    # Within region-year strata when >=2; isolated observations remain unchanged.
    for _,group in groups.groupby(['region_name','target_year'],dropna=False).groups.items():
        ids=np.asarray(list(group),dtype=int)
        if len(ids)<2:continue
        shuffle=rng.permutation(ids)
        res.iloc[ids,res.columns.get_indexer(cols)]=X.iloc[shuffle,X.columns.get_indexer(cols)].to_numpy()
    return res

def _adv_explain_registry(reg):
    sid=reg['study_id'];target=reg['target'];h=reg['horizon'];tr=reg['train'];te=reg['test']
    cols=reg['members'][0]['columns']
    # Design by the exact fitted train population; no fitted statistics from target TEST.
    _,XX,_,_=design(tr,te,reg['features'],reg['members'][0]['refs'])
    indices=np.sort(np.random.default_rng(CFG['SEED']+h).choice(len(XX),
                  size=min(CFG['ADVANCED_SHAP_ROWS'],len(XX)),replace=False))
    x=XX.iloc[indices].reset_index(drop=True);meta=te.iloc[indices].reset_index(drop=True)
    background=tr.sample(n=min(CFG['ADVANCED_BACKGROUND_ROWS'],len(tr)),random_state=CFG['SEED'])
    # Design shares the same fitted train population for test and background.
    _,BACK,_,_=design(tr,background,reg['features'],reg['members'][0]['refs'])
    ensure(list(x.columns)==list(BACK.columns),'SHAP_BACKGROUND_COLUMN_DIFFERENCE')
    contributions=np.zeros((len(x),len(cols)),float);bline=np.zeros(len(x),float);ok_members=0
    fam_rows=[]
    for member in reg['members']:
        fam=member['family'];weight=member['weight']
        try:
            XX_m=X if False else x
            vals,baseval,names,method=_explain_one(member,XX_m,BACK)
            ensure(vals.shape==(len(x),len(names)),'SHAP_MEMBER_MATRIX_SHAPE')
            reconstructed=np.asarray(baseval,float)+vals.sum(axis=1)
            direct=np.asarray(member['model'].predict(XX_m),float)
            discrepancy=np.max(np.abs(reconstructed-direct))
            ensure(discrepancy<CFG['ADVANCED_SHAP_TOLERANCE'],'SHAP_MEMBER_ADDITIVITY_'+fam)
            for j,feature in enumerate(names):
                fam_rows.append({'study_id':sid,'target':target,'horizon':h,'method':'SHAP_MEMBER',
                    'model':fam,'feature':feature,'group':_adv_feature_group(feature),
                    'mean_abs_shap':float(np.abs(vals[:,j]).mean()),
                    'mean_signed_shap':float(vals[:,j].mean()),'ensemble_weight':weight,
                    'n_explained':len(x),'mode':method,'scope':'PREDICTIVE_NOT_CAUSAL'})
                if feature in cols:contributions[:,cols.index(feature)]+=weight*vals[:,j]
                else:
                    # Imputer missing indicators contribute jointly to their original input.
                    if feature.startswith('missing_indicator_'):
                        nmissing=int(feature.split('_')[-1]);
                        if nmissing<len(cols):contributions[:,nmissing]+=weight*vals[:,j]
                    else:raise RuntimeError('UNRECOGNIZED_SHAP_INTERNAL_FEATURE_'+feature)
            bline+=weight*np.asarray(baseval,float)
            ADVANCED_SHAP_QA_ROWS.append({'study_id':sid,'target':target,'horizon':h,'member':fam,
                'max_additivity_abs_error':float(discrepancy),'n_explained':len(x),'status':'PASS','mode':method})
            ok_members+=1
        except Exception as ex:
            ADVANCED_SHAP_QA_ROWS.append({'study_id':sid,'target':target,'horizon':h,'member':fam,
                'max_additivity_abs_error':np.nan,'n_explained':len(x),'status':'FAIL',
                'reason':type(ex).__name__+': '+str(ex)[:160]})
            log('SHAP member '+sid+'/'+fam+': '+str(ex)[:160],'ERROR')
    if ok_members!=len(reg['members']):return 0
    direct=_adv_predict(reg,x)
    diff=float(np.max(np.abs(bline+contributions.sum(axis=1)-direct)))
    ensure(diff<CFG['ADVANCED_SHAP_TOLERANCE'],'SHAP_ENSEMBLE_ADDITIVITY_FAILED')
    ADVANCED_SHAP_QA_ROWS.append({'study_id':sid,'target':target,'horizon':h,'member':'WEIGHTED_ENSEMBLE',
        'max_additivity_abs_error':diff,'n_explained':len(x),'status':'PASS','mode':'WEIGHTED_MEMBER_SUM'})
    for j,feature in enumerate(cols):
        rho=np.nan
        if x[feature].nunique(dropna=True)>5:
            rho=float(pd.Series(x[feature]).corr(pd.Series(contributions[:,j]),method='spearman'))
        fam_rows.append({'study_id':sid,'target':target,'horizon':h,'method':'SHAP_ENSEMBLE',
            'model':'SUPERLEARNER','feature':feature,'group':_adv_feature_group(feature),
            'mean_abs_shap':float(np.abs(contributions[:,j]).mean()),
            'mean_signed_shap':float(contributions[:,j].mean()),'shap_value_spearman_r':rho,
            'ensemble_weight':1.,'n_explained':len(x),'mode':'WEIGHTED_MEMBER_SUM',
            'scope':'PREDICTIVE_NOT_CAUSAL'})
        for i in range(len(x)):
            if abs(contributions[i,j])>1e-9:
                ADVANCED_SHAP_LOCAL_ROWS.append({'study_id':sid,'target':target,'horizon':h,
                    'entity_id':meta.entity_id.iloc[i],'target_year':int(meta.target_year.iloc[i]),
                    'region_name':meta.region_name.iloc[i],'feature':feature,
                    'feature_value':float(x[feature].iloc[i]) if pd.notna(x[feature].iloc[i]) else np.nan,
                    'shap_value':float(contributions[i,j]),'baseline':float(bline[i]),
                    'prediction':float(direct[i]),'method':'WEIGHTED_MEMBER_SUM'})
    ADVANCED_SHAP_ROWS.extend(fam_rows)
    # Region-clustered confidence intervals for mean |SHAP|; seeds never treated as independent units.
    rng=np.random.default_rng(CFG['SEED']+h+19)
    region=meta.region_name.fillna('UNKNOWN').astype(str).to_numpy()
    ids=[np.flatnonzero(region==s) for s in np.unique(region)]
    if len(ids)>=CFG['MIN_BOOTSTRAP_REGIONS']:
        boot=np.empty((CFG['ADVANCED_EXPLAIN_BOOTSTRAPS'],len(cols)))
        for j in range(len(boot)):
            draw=rng.integers(0,len(ids),len(ids));part=np.concatenate([ids[k] for k in draw]);boot[j]=np.mean(np.abs(contributions[part]),axis=0)
        center=np.abs(contributions).mean(axis=0);top_k=min(5,len(center));global_top=set(np.argsort(center)[-top_k:].tolist())
        for j,feature in enumerate(cols):
            probability=float(np.mean([j in np.argsort(boot[b])[-top_k:] for b in range(len(boot))]))
            ADVANCED_SHAP_STABILITY_ROWS.append({'study_id':sid,'target':target,'horizon':h,
                'feature':feature,'top_k':top_k,'bootstrap_in_top_k_share':probability,
                'base_rank':int(len(center)-np.argsort(center).tolist().index(j)),
                'n_region_resamples':len(boot),'n_regions':len(ids),'scope':'SHAP_RANK_STABILITY_NONCAUSAL'})
        for j,feature in enumerate(cols):
            ADVANCED_SHAP_CI_ROWS.append({'study_id':sid,'target':target,'horizon':h,'feature':feature,
                'mean_abs_shap':float(np.abs(contributions[:,j]).mean()),
                'lo95':float(np.quantile(boot[:,j],0.025)),'hi95':float(np.quantile(boot[:,j],0.975)),
                'n_regions':len(ids),'n_bootstrap':len(boot),'unit':'REGION'})
    # Model-native interactions. XGBoost returns true pairwise SHAP interactions;
    # CatBoost's Interaction metric is a different, structural split score (marked explicitly).
    for member in reg['members']:
        try:
            if member['family']=='XGBOOST':
                import xgboost
                matrix=xgboost.DMatrix(x.iloc[:min(45,len(x))],feature_names=cols)
                shap_pairs=np.asarray(member['model'].get_booster().predict(matrix,pred_interactions=True),float)
                ensure(shap_pairs.shape[1:]==(len(cols)+1,len(cols)+1),'XGB_INTERACTION_SHAPE')
                for jj in range(len(cols)):
                    for kk in range(jj+1,len(cols)):
                        strength=float(2*np.mean(np.abs(shap_pairs[:,jj,kk])))
                        if strength>0:
                            ADVANCED_SHAP_INTERACTION_ROWS.append({'study_id':sid,'target':target,'horizon':h,
                                'family':'XGBOOST','feature_a':cols[jj],'feature_b':cols[kk],
                                'interaction_strength':strength,'definition':'TWO_TIMES_MEAN_ABS_PAIRWISE_SHAP',
                                'n_sample':len(shap_pairs),'predictive_only':True})
            elif member['family']=='CATBOOST':
                rank=np.asarray(member['model'].get_feature_importance(type='Interaction'),float)
                if rank.ndim==2 and rank.shape[1]>=3:
                    for pair in rank[:100]:
                        jj,kk=int(pair[0]),int(pair[1])
                        if 0<=jj<len(cols) and 0<=kk<len(cols) and jj!=kk:
                            ADVANCED_SHAP_INTERACTION_ROWS.append({'study_id':sid,'target':target,'horizon':h,
                                'family':'CATBOOST','feature_a':cols[jj],'feature_b':cols[kk],
                                'interaction_strength':float(pair[2]),
                                'definition':'CATBOOST_NATIVE_SPLIT_INTERACTION_NOT_SHAP',
                                'n_sample':len(x),'predictive_only':True})
        except Exception as ex:log('Interaction audit '+sid+'/'+member['family']+': '+type(ex).__name__,'WARN')
    # Grouped conditional permutation on held-out TEST; no re-training and no test selection.
    n=min(len(te),CFG['ADVANCED_PERM_ROWS']);take=np.sort(np.random.default_rng(CFG['SEED']+37).choice(len(te),n,replace=False))
    sample=te.iloc[take].reset_index(drop=True)
    _,mx,_,_=design(tr,sample,reg['features'],reg['members'][0]['refs'])
    before=np.abs(sample[target].to_numpy(float)-_adv_predict(reg,mx))
    groups={g:[f for f in mx.columns if _adv_feature_group(f)==g] for g in
            ('G1','G2','G3','G4','G5','G1R_REGIONAL_NOT_MUNICIPAL','G0_DEMOGRAPHIC_HISTORY')}
    for g,ff in groups.items():
        if not ff:continue
        altered=[];effective_counts=[]
        for rep in range(CFG['ADVANCED_PERM_REPEATS']):
            perm=_within_strata_permute(mx,sample,ff,CFG['SEED']+h*31+rep)
            changed=int((perm[ff].fillna(-999998)!=mx[ff].fillna(-999998)).any(axis=1).sum())
            effective_counts.append(changed)
            if changed:altered.append(np.abs(sample[target].to_numpy(float)-_adv_predict(reg,perm)))
        if not altered:
            ADVANCED_GROUP_PERM_ROWS.append({'study_id':sid,'target':target,'horizon':h,'group':g,
                'status':'NOT_FEASIBLE','reason':'NO_EXCHANGEABLE_REGION_YEAR_STRATA','n_permuted':0});continue
        pert=np.mean(np.column_stack(altered),axis=1)
        z=pd.DataFrame({'region_name':sample.region_name,'target_year':sample.target_year,
                        'perturbed_error':pert,'base_error':before})
        delta=float(np.mean(pert-before))
        stat=block_delta(z,'perturbed_error','base_error',CFG['ADVANCED_EXPLAIN_BOOTSTRAPS'],CFG['SEED']+h)
        ADVANCED_GROUP_PERM_ROWS.append({'study_id':sid,'target':target,'horizon':h,'group':g,
            'n_test_sample':len(sample),'n_permuted':int(np.mean(effective_counts)),
            'n_features':len(ff),'delta_mae':delta,'ci_low':stat['ci_low'] if stat else np.nan,
            'ci_high':stat['ci_high'] if stat else np.nan,'n_regions':stat['n_regions'] if stat else 0,
            'repetitions':len(altered),'status':'PASS','method':'JOINT_REGION_YEAR_STRATIFIED_PERMUTATION',
            'interpretation':'Predictive reliance, not causal importance'})
    # PDP/ICE on training domain only to minimize post-test interpretation bias.
    strongest=np.argsort(np.abs(contributions).mean(axis=0))[::-1]
    train_sample=tr.sample(n=min(CFG['ADVANCED_PDP_TRAIN_ROWS'],len(tr)),random_state=CFG['SEED'])
    _,xt,_,_=design(tr,train_sample,reg['features'],reg['members'][0]['refs'])
    for j in strongest[:CFG['ADVANCED_PDP_TOP']]:
        feature=cols[j]
        if feature.startswith('region_') or not xt[feature].notna().any() or xt[feature].nunique()<8:continue
        knots=np.unique(np.quantile(xt[feature].dropna().to_numpy(float),np.linspace(.05,.95,CFG['ADVANCED_PDP_BINS'])))
        chosen=np.sort(np.random.default_rng(CFG['SEED']).choice(len(xt),min(12,len(xt)),replace=False))
        for k,v in enumerate(knots):
            xx=xt.copy();xx[feature]=float(v);yp=_adv_predict(reg,xx)
            ADVANCED_PDP_ROWS.append({'study_id':sid,'target':target,'horizon':h,'feature':feature,
                'bin':k,'feature_value_transformed':float(v),'mean_prediction':float(yp.mean()),
                'n_train_background':len(xt),'method':'TRAIN_BACKGROUND_PDP_MARGINAL_NOT_CAUSAL'})
            for ix in chosen:
                ADVANCED_ICE_ROWS.append({'study_id':sid,'target':target,'horizon':h,'feature':feature,
                    'train_sample_index':int(ix),'feature_value_transformed':float(v),
                    'prediction':float(yp[ix]),'method':'ICE_TRAIN_BACKGROUND_NONCAUSAL'})
    return len(x)

progress('STATS',1,1,'Статистика и факторные вклады рассчитаны')
if CFG['ADVANCED_EXPLAIN_ENABLE']:
    keys=list(ADVANCED_ENSEMBLE_REGISTRY)
    if CFG['MODE']=='SMOKE':keys=keys[:2]
    else:keys=[k for k in keys if k[0] in ('MAX_AVAILABLE','CANONICAL') or k[0].startswith('AUX_')]
    progress('SHAP',0,len(keys),'SHAP, permutation, PDP/ICE')
    _shap_done=0
    for key in keys:
        try:_adv_explain_registry(ADVANCED_ENSEMBLE_REGISTRY[key])
        except Exception as ex:
            ADVANCED_SHAP_QA_ROWS.append({'study_id':key[0],'target':key[1],'horizon':key[2],
                'member':'ENSEMBLE_OVERALL','status':'FAIL','reason':type(ex).__name__+': '+str(ex)[:200]})
            log('ADV EXPLANATION '+str(key)+': '+type(ex).__name__+': '+str(ex)[:200],'ERROR')
        _shap_done+=1
        progress('SHAP',_shap_done,len(keys),f'Моделей объяснено: {_shap_done}/{len(keys)}')
    progress('SHAP',1,1,'Объяснения сохранены')
if not CFG['ADVANCED_EXPLAIN_ENABLE']:progress('SHAP',1,1,'Объяснения отключены настройкой')
ADVANCED_SHAP=pd.DataFrame(ADVANCED_SHAP_ROWS);ADVANCED_SHAP_LOCAL=pd.DataFrame(ADVANCED_SHAP_LOCAL_ROWS)
ADVANCED_SHAP_QA=pd.DataFrame(ADVANCED_SHAP_QA_ROWS);ADVANCED_GROUP_PERM=pd.DataFrame(ADVANCED_GROUP_PERM_ROWS)
ADVANCED_PDP=pd.DataFrame(ADVANCED_PDP_ROWS);ADVANCED_ICE=pd.DataFrame(ADVANCED_ICE_ROWS)
ADVANCED_SHAP_CI=pd.DataFrame(ADVANCED_SHAP_CI_ROWS)
ADVANCED_SHAP_STABILITY=pd.DataFrame(ADVANCED_SHAP_STABILITY_ROWS)
ADVANCED_SHAP_INTERACTIONS=pd.DataFrame(ADVANCED_SHAP_INTERACTION_ROWS)
# V4.02: independent, paired test against the strongest SINGLE family selected on DEV.
# Only predictions with the same municipality, target year and task enter inference.
ADVANCED_SINGLE_DUEL_ROWS=[]
if len(PREDICTIONS) and len(ADVANCED_ENSEMBLES):
    for row in ADVANCED_ENSEMBLES.loc[ADVANCED_ENSEMBLES.status.eq('PASS')].itertuples():
        family=getattr(row,'best_single_family_dev_locked',None)
        if not family:continue
        subset=PREDICTIONS.loc[
          PREDICTIONS.study_id.eq(row.study_id)&PREDICTIONS.target.eq(row.target)&
          PREDICTIONS.horizon.eq(row.horizon)&PREDICTIONS.split_id.eq(row.protocol)&
          PREDICTIONS.model.isin(['SUPERLEARNER','SL_'+family])]
        if subset.empty:continue
        z=subset.pivot_table(index=['entity_id','origin_year','target_year','region_name'],
              columns='model',values='absolute_error',aggfunc='first').reset_index()
        single='SL_'+family
        if single not in z.columns or 'SUPERLEARNER' not in z.columns:continue
        stat=block_delta(z,single,'SUPERLEARNER',CFG['BOOTSTRAPS'],
                         CFG['SEED']+int(row.horizon)+sum(ord(c) for c in str(row.study_id)))
        if stat:ADVANCED_SINGLE_DUEL_ROWS.append({
            'study_id':row.study_id,'target':row.target,'horizon':int(row.horizon),
            'protocol':row.protocol,'best_single_family_dev_locked':family,
            'comparison':'DEV_BEST_SINGLE → SUPERLEARNER','confirmatory':False,
            'model_selected_without_test_labels':True,**stat})
ADVANCED_SINGLE_DUEL=pd.DataFrame(ADVANCED_SINGLE_DUEL_ROWS)
if len(ADVANCED_SINGLE_DUEL):
    ADVANCED_SINGLE_DUEL['q_fdr_bh']=np.nan
    for _,inds in ADVANCED_SINGLE_DUEL.groupby(['target','horizon','protocol']).groups.items():
        ADVANCED_SINGLE_DUEL.loc[list(inds),'q_fdr_bh']=bh_adjust(
             ADVANCED_SINGLE_DUEL.loc[list(inds),'p_cluster_signflip'])
mark('ADVANCED_VS_BEST_SINGLE','PASS' if len(ADVANCED_SINGLE_DUEL)==int(ADVANCED_ENSEMBLES.status.eq('PASS').sum()) else 'PARTIAL',
     f'paired={len(ADVANCED_SINGLE_DUEL)}/{int(ADVANCED_ENSEMBLES.status.eq("PASS").sum())}; regional bootstrap; DEV locked')

# Independently test SuperLearner vs historical M0 on identical keys/regions; exploratory only.
if len(PREDICTIONS):
    subset=PREDICTIONS[PREDICTIONS.model.isin(['HISTORICAL','SUPERLEARNER'])]
    for (sid,targ,h,proto),grp in subset.groupby(['study_id','target','horizon','split_id']):
        z=grp.pivot_table(index=['entity_id','origin_year','target_year','region_name'],columns='model',
                          values='absolute_error',aggfunc='first').reset_index()
        if 'HISTORICAL' not in z or 'SUPERLEARNER' not in z:continue
        stat=block_delta(z,'HISTORICAL','SUPERLEARNER',CFG['BOOTSTRAPS'],CFG['SEED']+h)
        if stat:ADVANCED_LOO_ROWS.append({'study_id':sid,'target':targ,'horizon':h,'protocol':proto,
            'comparison':'M0_HISTORICAL → SUPERLEARNER','confirmatory':False,**stat})
ADVANCED_COMPARISONS=pd.DataFrame(ADVANCED_LOO_ROWS)
if len(ADVANCED_COMPARISONS):
    ADVANCED_COMPARISONS['q_fdr_bh']=np.nan
    for _,inds in ADVANCED_COMPARISONS.groupby(['target','horizon','protocol']).groups.items():
        ADVANCED_COMPARISONS.loc[list(inds),'q_fdr_bh']=bh_adjust(ADVANCED_COMPARISONS.loc[list(inds),'p_cluster_signflip'])
    ADVANCED_COMPARISONS['significant_improvement']=(ADVANCED_COMPARISONS.delta_mae.gt(0)&
        ADVANCED_COMPARISONS.ci_low.gt(0)&ADVANCED_COMPARISONS.q_fdr_bh.lt(0.05))
mark('ADVANCED_SHAP_QA','PASS' if len(ADVANCED_SHAP_QA) and ADVANCED_SHAP_QA.status.eq('PASS').all() else 'PARTIAL',
     f'additive explainers={int(ADVANCED_SHAP_QA.status.eq("PASS").sum()) if len(ADVANCED_SHAP_QA) else 0}; '
     f'failures={int(ADVANCED_SHAP_QA.status.eq("FAIL").sum()) if len(ADVANCED_SHAP_QA) else 0}; '
     f'region-block CI={len(ADVANCED_SHAP_CI)}; interactions={len(ADVANCED_SHAP_INTERACTIONS)}; perm={len(ADVANCED_GROUP_PERM)}')
quiet_print('Advanced explanations: SHAP global',len(ADVANCED_SHAP),'local',len(ADVANCED_SHAP_LOCAL),
      'perm group',len(ADVANCED_GROUP_PERM),'PDP',len(ADVANCED_PDP),'ICE',len(ADVANCED_ICE))


#### Лаги публикаций, чувствительность к прокси и тесты на утечки.

In [13]:
# 10 · V3.02 · REPLAY-CONSISTENT PUBLICATION LAG / PROXY SENSITIVITY / FUTURE CANARY

# Functions must exist BEFORE the sensitivity loop; previous V3.00 swallowed NameError.
def shifted_predictors(frame,year_lag,features,panel_override=None):
    """Rebuild the exact factor availability transform used in cell 06.

    Origin t uses the factor value at t-publication_lag. A missing value at that
    source year is carried forward from the latest *past* observation only if
    its age <= MAX_FEATURE_STALENESS_YEARS. Future data are never backfilled.
    Unlike the former direct-year join, this mirrors cell 06's ffill+age guard.
    """
    source=PANEL if panel_override is None else panel_override
    ff=list(dict.fromkeys(features))
    if not ff:return frame.copy()
    ensure(set(ff).issubset(source.columns),'CANARY_FEATURE_NOT_IN_SOURCE_PANEL')
    ensure(not source.duplicated(['entity_id','observation_year']).any(),'CANARY_SOURCE_DUPLICATE_KEYS')
    known=source[['entity_id','observation_year']+ff].sort_values(['entity_id','observation_year']).copy()
    max_age=int(CFG['MAX_FEATURE_STALENESS_YEARS'])
    for f in ff:
        observed=pd.to_numeric(known[f],errors='coerce')
        source_year=known.observation_year.where(observed.notna())
        last=source_year.groupby(known.entity_id,sort=False).ffill()
        filled=observed.groupby(known.entity_id,sort=False).ffill()
        age=known.observation_year-last
        known[f]=filled.where(last.notna()&age.le(max_age))
    known=known.rename(columns={'observation_year':'available_year'})
    result=frame.drop(columns=ff,errors='ignore').copy()
    result['available_year']=result.origin_year.astype(int)-int(year_lag)
    result=result.merge(known,on=['entity_id','available_year'],how='left',validate='many_to_one',sort=False)
    ensure(len(result)==len(frame),'CANARY_JOIN_CARDINALITY_CHANGED')
    return result.drop(columns='available_year')

def _temporal_replay_unit_test():
    """Small real-pandas contract: bounded carry, future isolation, past sensitivity."""
    yrs=[2014,2018,2019,2020,2021]
    toy=pd.DataFrame({'entity_id':['TEST_A']*len(yrs),'observation_year':yrs,
                      'G2_TEST':[1.,10.,np.nan,333.,444.]})
    frame=pd.DataFrame({'entity_id':['TEST_A'],'origin_year':[2020],'G2_TEST':[10.]})
    f=['G2_TEST'];replay=shifted_predictors(frame,1,f,toy)
    ensure(float(replay.G2_TEST.iloc[0])==10.,'UNIT_CANARY_CARRY_FORWARD_MISSING')
    future=toy.copy();future.loc[future.observation_year.ge(2020),'G2_TEST']=1e12
    after=shifted_predictors(frame,1,f,future)
    ensure(float(after.G2_TEST.iloc[0])==10.,'UNIT_CANARY_FUTURE_MUTATION_LEAK')
    past=toy.copy();past.loc[past.observation_year.eq(2018),'G2_TEST']=12.
    changed=shifted_predictors(frame,1,f,past)
    ensure(float(changed.G2_TEST.iloc[0])==12.,'UNIT_CANARY_PAST_MUTATION_NOT_OBSERVED')
    stale=toy.copy();stale.loc[stale.observation_year.eq(2018),'G2_TEST']=np.nan
    ensure(pd.isna(shifted_predictors(frame,1,f,stale).G2_TEST.iloc[0]),'UNIT_CANARY_STALE_VALUE_NOT_EXCLUDED')
    mark('UNIT_TEMPORAL_REPLAY','PASS','Bounded ffill, lag, future invariance, past mutation and stale guard')
_temporal_replay_unit_test()

def rebuild_historical_g0(sample,source):
    """Reconstruct demographic lags from source observations, never from target labels."""
    out=sample.copy();lag=CFG['FEATURE_PUBLICATION_LAG']
    history=source[['entity_id','observation_year','population','migration_per_1000']].copy()
    for k in range(lag,lag+4):
        p='hist_population_l'+str(k);m='hist_migration_l'+str(k)
        values=history.rename(columns={'observation_year':'lookback_year','population':p,
                                        'migration_per_1000':m})
        out=out.drop(columns=[p,m],errors='ignore')
        out['lookback_year']=out.origin_year-k
        out=out.merge(values,on=['entity_id','lookback_year'],how='left',validate='many_to_one')
        out=out.drop(columns='lookback_year')
    out['hist_growth_lagged_pct']=100*(out['hist_population_l'+str(lag)]/
                                         out['hist_population_l'+str(lag+2)]-1)
    out['hist_growth_lagged_pct']=out.hist_growth_lagged_pct.replace([np.inf,-np.inf],np.nan)
    ensure(len(out)==len(sample),'G0_HISTORY_REBUILD_CHANGED_ROWS')
    return out



for (sid,target,h),r in MODEL_REGISTRY.items():
    if sid!='MAX_AVAILABLE' or not AVAILABLE_GROUPS:continue
    features=list(dict.fromkeys(sum((GROUP_FEATURES[g] for g in AVAILABLE_GROUPS),[])))
    try:
        alt_train=shifted_predictors(r['train'],CFG['FEATURE_PUBLICATION_LAG']+1,features)
        alt_test=shifted_predictors(r['test'],CFG['FEATURE_PUBLICATION_LAG']+1,features)
        pred,_,_,_=fit_predict(r['family'],alt_train,alt_test,target,r['features'],CFG['SEED'],r['params'])
        alt=metrics(r['test'][target],pred,r['test'][f'hist_population_l{CFG["FEATURE_PUBLICATION_LAG"]}'])
        original=metrics(r['test'][target],r['pred'],r['test'][f'hist_population_l{CFG["FEATURE_PUBLICATION_LAG"]}'])
        SENSITIVITY_ROWS.append({'study_id':sid,'target':target,'horizon':h,'test_keys':len(r['test']),
           'check':'publication_lag_plus_1_year','original_mae':original['mae'],
           'alternative_mae':alt['mae'],'delta_mae':original['mae']-alt['mae'],
           'interpretation':'different information timing; test keys held fixed'})
        # Remove correlated proxies, preferentially retaining more complete training measurements.
        ff=[f for f in r['features'] if f.startswith(('G1_','G2_','G3_','G4_','G5_'))]
        tr=r['train'];removed=[];retained=[]
        for f in sorted(ff,key=lambda c:tr[c].notna().sum(),reverse=True):
            redundant=False
            for stay in retained:
                sample=tr[[stay,f]].dropna()
                if len(sample)>50 and abs(sample[stay].corr(sample[f],method='spearman'))>=CFG['CORRELATED_PROXY_CUTOFF']:
                    redundant=True;break
            if redundant:removed.append(f)
            else:retained.append(f)
        if removed:
            feats=[f for f in r['features'] if f not in removed]
            pred,_,_,_=fit_predict(r['family'],tr,r['test'],target,feats,CFG['SEED'],r['params'])
            teststat=metrics(r['test'][target],pred,r['test'][f'hist_population_l{CFG["FEATURE_PUBLICATION_LAG"]}'])
            SENSITIVITY_ROWS.append({'study_id':sid,'target':target,'horizon':h,
               'test_keys':len(r['test']),'check':'remove_correlated_proxies_on_train',
               'original_mae':original['mae'],'alternative_mae':teststat['mae'],
               'delta_mae':original['mae']-teststat['mae'],
               'excluded_features':','.join(removed),'interpretation':'exploratory; no causal attribution'})
    except Exception as ex:
        mark('SENSITIVITY_'+sid+'_'+target+'_h'+str(h),'FAIL',type(ex).__name__+': '+str(ex)[:180])
SENSITIVITY=pd.DataFrame(SENSITIVITY_ROWS)

# Two independent canary contracts:
# (1) SOURCE_REPLAY reproduces *exactly* the model's actual feature matrix.
# (2) FUTURE_INVARIANCE changes all source rows t >= issue origin and reruns
#     the SAME transformation. A breach is genuine only after replay parity.
CANARY_RECORDS=[]

def _canary_model_matrix(registry,frame):
    return design(registry['train'],frame,registry['features'],registry['refs'])[1]

def _canary_comparison(before,after,rtol=0.,atol=1e-10):
    same_shape=(list(before.columns)==list(after.columns) and before.shape==after.shape)
    if not same_shape:
        return False,['SHAPE_OR_COLUMN_DIFFERENCE'],float('inf')
    lhs=before.to_numpy(dtype=float);rhs=after.to_numpy(dtype=float)
    matches=np.isclose(lhs,rhs,equal_nan=True,rtol=rtol,atol=atol)
    changed=[str(c) for c in before.columns[np.any(~matches,axis=0)].tolist()]
    different=np.abs(lhs-rhs);good=np.isfinite(different)
    max_delta=float(np.max(different[good])) if np.any(good) else 0.
    return bool(matches.all()),changed,max_delta

for (sid,target,h),r in MODEL_REGISTRY.items():
    if sid not in ('MAX_AVAILABLE','CANONICAL','AUX_G1_REGION_CONTEXT'):continue
    covariates=[f for f in dict.fromkeys(r['features']) if f in PANEL.columns]
    sample=r['test'].drop_duplicates('entity_id').head(CFG['MAX_FUTURE_MUTATION_ROWS']).copy()
    if sample.empty:continue
    record={'study_id':sid,'target':target,'horizon':h,'n_test_rows':len(sample),
            'n_model_source_factors':len(covariates),'status':'NOT_RUN'}
    try:
        train=r['train'];refs=r['refs']
        x_original=_canary_model_matrix(r,sample)
        # Fidelity contract: replay without modifying a single source value.
        replay=shifted_predictors(sample,CFG['FEATURE_PUBLICATION_LAG'],covariates)
        replay=rebuild_historical_g0(replay,PANEL)
        x_replay=_canary_model_matrix(r,replay)
        faithful,replay_cols,replay_delta=_canary_comparison(x_original,x_replay)
        record.update(source_replay_exact=faithful,
          replay_mismatch_columns_json=json.dumps(replay_cols,ensure_ascii=False),
          replay_max_finite_delta=replay_delta)
        if not faithful:
            record['status']='FAIL_SOURCE_REPLAY_NOT_IDENTICAL'
            raise AssertionError('CANARY_SOURCE_REPLAY_MISMATCH: '+','.join(replay_cols[:8]))
        y_original=np.asarray(r['model'].predict(x_original),float)
        y_replay=np.asarray(r['model'].predict(x_replay),float)
        ensure(np.allclose(y_original,y_replay,rtol=0,atol=1e-9,equal_nan=True),
               'CANARY_UNMUTATED_MODEL_REPLAY_DIFFERENCE')
        mutated=PANEL.copy(deep=True)
        origins=dict(zip(sample.entity_id,sample.origin_year))
        at_or_after=mutated.observation_year.ge(mutated.entity_id.map(origins))
        mask=mutated.entity_id.isin(origins)&at_or_after.fillna(False)
        mutated_source_rows=int(mask.sum())
        ensure(mutated_source_rows>0,'CANARY_HAS_NO_FUTURE_SOURCE_ROWS')
        for f in ('population','migration_per_1000'):
            if f in mutated.columns:mutated.loc[mask,f]=1e12
        for f in covariates:mutated.loc[mask,f]=1e12
        after=shifted_predictors(sample,CFG['FEATURE_PUBLICATION_LAG'],covariates,mutated)
        after=rebuild_historical_g0(after,mutated)
        x_after=_canary_model_matrix(r,after)
        invariant_x,changed_columns,changed_max=_canary_comparison(x_replay,x_after)
        y_after=np.asarray(r['model'].predict(x_after),float)
        invariant_y=bool(np.allclose(y_replay,y_after,rtol=0,atol=1e-9,equal_nan=True))
        record.update(n_future_source_rows_mutated=mutated_source_rows,
          n_model_features=len(x_original.columns),invariant_features=invariant_x,
          invariant_predictions=invariant_y,
          mutated_mismatch_columns_json=json.dumps(changed_columns,ensure_ascii=False),
          mutated_max_finite_delta=changed_max,
          status='PASS' if invariant_x and invariant_y else 'FAIL_FUTURE_DEPENDENCE')
        if not invariant_x or not invariant_y:
            raise AssertionError('FUTURE_MUTATION_CHANGES_FINAL_FORECAST: '+','.join(changed_columns[:8]))
    except Exception as ex:
        if record['status']=='NOT_RUN':record['status']='FAIL_CANARY_RUNTIME'
        record['exception_type']=type(ex).__name__;record['reason']=str(ex)[:400]
        CANARY_RECORDS.append(record)
        CANARY_QA=pd.DataFrame(CANARY_RECORDS)
        (WORK/'ml03_canary_failure.json').write_text(json.dumps(CANARY_RECORDS,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
        mark('TEMPORAL_CANARY','FAIL',record['status']+' '+record['reason'])
        raise
    CANARY_RECORDS.append(record)
CANARY_QA=pd.DataFrame(CANARY_RECORDS)
mark('TEMPORAL_CANARY','PASS' if len(CANARY_QA) and CANARY_QA.status.eq('PASS').all() else 'NOT_FEASIBLE',
     f'Source replay + future mutation; passing models={int(CANARY_QA.status.eq("PASS").sum()) if len(CANARY_QA) else 0}/{len(CANARY_QA)}')

# Same fitted models, objectively restricted boundary and municipality-size subsets.
BOUNDARY_ROWS=[];SIZE_ROWS=[]
for (sid,target,h),r in MODEL_REGISTRY.items():
    if sid not in ('MAX_AVAILABLE','CANONICAL'):continue
    test=r['test'];y_true=test[target].to_numpy(float)
    err=np.abs(y_true-np.asarray(r['pred'],float))
    if 'boundary_period_uncertain' in test:
        safe=~test.boundary_period_uncertain.fillna(True).astype(bool)
        verified=test.loc[safe]
        if safe.sum()>=CFG['MIN_TEST_ROWS'] and verified.region_name.nunique()>=CFG['MIN_BOUNDARY_SENSITIVITY_REGIONS']:
            BOUNDARY_ROWS.append({'study_id':sid,'target':target,'horizon':h,
                'n_test_all':len(test),'n_boundary_verified':int(safe.sum()),
                'n_regions_verified':int(verified.region_name.nunique()),
                'mae_all':float(err.mean()),'mae_verified_boundary':float(err[safe.to_numpy()].mean()),
                'status':'EXPLORATORY_FIXED_MODEL_STRATUM_NONCAUSAL'})
    pop='hist_population_l'+str(CFG['FEATURE_PUBLICATION_LAG'])
    train_pop=pd.to_numeric(r['train'][pop],errors='coerce')
    train_pop=train_pop[train_pop.gt(0)].dropna()
    thresholds=np.quantile(train_pop,[1/3,2/3]) if len(train_pop)>100 else None
    if thresholds is not None and thresholds[0]<thresholds[1]:
        values=pd.to_numeric(test[pop],errors='coerce').to_numpy(float)
        bins=np.digitize(values,thresholds)
        for group in range(3):
            flag=np.isfinite(values)&(bins==group)
            if flag.sum()>=15:
                SIZE_ROWS.append({'study_id':sid,'target':target,'horizon':h,
                    'size_group':('small','medium','large')[group],
                    'n_test':int(flag.sum()),'mae':float(err[flag].mean()),
                    'train_size_thresholds_json':json.dumps(thresholds.tolist()),
                    'scope':'train-defined lagged-population strata; prediction not causal'})
BOUNDARY_SENSITIVITY=pd.DataFrame(BOUNDARY_ROWS)
SIZE_SENSITIVITY=pd.DataFrame(SIZE_ROWS)
mark('BOUNDARY_SENSITIVITY','PASS' if len(BOUNDARY_SENSITIVITY) else 'NOT_FEASIBLE',
    f'Boundary-confirmed test strata={len(BOUNDARY_SENSITIVITY)}; undocumented does not imply stable')
mark('SIZE_HETEROGENEITY','PASS' if len(SIZE_SENSITIVITY) else 'NOT_FEASIBLE',
    f'Train-derived municipality-size strata={len(SIZE_SENSITIVITY)}')
# H3.5 can only be marked tested when publication lag AND correlated-proxy comparisons are present.
lag_exists=(len(SENSITIVITY)>0 and SENSITIVITY.check.eq('publication_lag_plus_1_year').any())
proxy_exists=(len(SENSITIVITY)>0 and SENSITIVITY.check.eq('remove_correlated_proxies_on_train').any())
mark('H3.5','PASS' if lag_exists and proxy_exists else 'NOT_FEASIBLE',
     f'lag sensitivity={lag_exists}; correlated-proxy sensitivity={proxy_exists}')
# Scientific interpretation is predictive only; lack of causal data is not, by itself, a fatal gate.
if len(ABLA) and len(SENSITIVITY):
    mark('PASS_INTERPRETATION','PASS',
       f'factor tests={len(ABLA)}; improved={int(ABLA.finding.eq("PREDICTIVE_IMPROVEMENT").sum())}; uncertainty and negative findings retained')
else:
    mark('PASS_INTERPRETATION','NOT_FEASIBLE','No paired factor tests or inadequate sensitivity checks')
quiet_print('Independent factor comparisons:',len(ABLA),'| ALE rows:',len(ALE),
      '| sensitivity:',len(SENSITIVITY),'| future-mutation tests:',len(CANARY_QA))

CELL10_COMPLETE=True
mark('CELL10_PIPELINE','PASS','Sensitivity, temporal canary, boundary and size tables constructed')


# V4 · ADVANCED temporal anti-leakage replay for the actual fitted SuperLearner.
# Validates source replay, model matrix and future-mutation invariance, fails closed.
ADVANCED_TEMPORAL_CANARY_ROWS=[]
for (sid,target,h),r in ADVANCED_ENSEMBLE_REGISTRY.items():
    if sid not in ('MAX_AVAILABLE','CANONICAL','AUX_G1_REGION_CONTEXT'):continue
    source_cols=[f for f in dict.fromkeys(r['features']) if f in PANEL.columns]
    small=r['test'].drop_duplicates('entity_id').head(CFG['MAX_FUTURE_MUTATION_ROWS']).copy()
    if small.empty:continue
    rec={'study_id':sid,'target':target,'horizon':h,'n_test':len(small),'n_source_features':len(source_cols)}
    try:
        orig=design(r['train'],small,r['features'],r['members'][0]['refs'])[1]
        replay=shifted_predictors(small,CFG['FEATURE_PUBLICATION_LAG'],source_cols)
        replay=rebuild_historical_g0(replay,PANEL)
        rebuilt=design(r['train'],replay,r['features'],r['members'][0]['refs'])[1]
        faithful,missing,delta=_canary_comparison(orig,rebuilt)
        ensure(faithful,'ADVANCED_SOURCE_REPLAY_MISMATCH: '+','.join(missing[:8]))
        mutated=PANEL.copy(deep=True)
        first=dict(zip(small.entity_id,small.origin_year))
        mask=mutated.entity_id.isin(first)&mutated.observation_year.ge(mutated.entity_id.map(first)).fillna(False)
        ensure(mask.sum()>0,'ADVANCED_FUTURE_MUTATION_NO_ROWS')
        for f in ['population','migration_per_1000']+source_cols:mutated.loc[mask,f]=1e12
        altered=shifted_predictors(small,CFG['FEATURE_PUBLICATION_LAG'],source_cols,mutated)
        altered=rebuild_historical_g0(altered,mutated)
        after=design(r['train'],altered,r['features'],r['members'][0]['refs'])[1]
        stable,changed,change=_canary_comparison(rebuilt,after)
        y0=_adv_predict(r,rebuilt);y1=_adv_predict(r,after)
        ok=bool(stable and np.allclose(y0,y1,rtol=0,atol=1e-9))
        rec.update({'n_mutated_source_rows':int(mask.sum()),'source_replay_exact':True,
            'prediction_invariant':ok,'matrix_invariant':stable,
            'max_predictive_diff':float(np.abs(y0-y1).max()),'status':'PASS' if ok else 'FAIL'})
        ensure(ok,'ADVANCED_ENSEMBLE_FUTURE_DEPENDENCE')
    except Exception as ex:
        rec.update({'status':'FAIL','exception':type(ex).__name__+': '+str(ex)[:250]})
        ADVANCED_TEMPORAL_CANARY_ROWS.append(rec)
        mark('ADVANCED_TEMPORAL_CANARY','FAIL',rec['exception'])
        raise
    ADVANCED_TEMPORAL_CANARY_ROWS.append(rec)
ADVANCED_TEMPORAL_CANARY=pd.DataFrame(ADVANCED_TEMPORAL_CANARY_ROWS)
mark('ADVANCED_TEMPORAL_CANARY', 'PASS' if len(ADVANCED_TEMPORAL_CANARY) and ADVANCED_TEMPORAL_CANARY.status.eq('PASS').all() else 'NOT_FEASIBLE',
     f'passing weighted-ensemble replay/future tests={len(ADVANCED_TEMPORAL_CANARY)}')

PUBLICATION_AVAILABILITY_QA=pd.DataFrame([{
  'feature':r.get('feature'),'indicator_name':r.get('indicator_name'),'group':r.get('group'),
  'documented_publication_date':r.get('report_available_year','NOT_VERIFIED'),
  'availability_provenance':r.get('availability_provenance','UNKNOWN'),
  'lag_assumption_years':CFG['FEATURE_PUBLICATION_LAG'],
  'status':'DOCUMENTED_SOURCE_DATE' if str(r.get('report_available_year','')).isdigit() and str(r.get('availability_provenance','')).startswith('OFFICIAL') else 'LAG_ASSUMPTION_ONLY'}
  for r in FEATURE_DICT.to_dict('records')])
if len(PUBLICATION_AVAILABILITY_QA):
    real=int(PUBLICATION_AVAILABILITY_QA.status.eq('DOCUMENTED_SOURCE_DATE').sum())
    mark('HISTORICAL_PUBLICATION_DATES','PASS' if real==len(PUBLICATION_AVAILABILITY_QA) else 'PARTIAL',
         f'documented={real}/{len(PUBLICATION_AVAILABILITY_QA)}; future-mutation and lag sensitivity do not prove original release dates')

progress('SENSITIVITY',1,1,'Canary и чувствительность завершены')


#### V5: решения по гипотезам на основании фактических прогнозных контрастов.

In [14]:
# R3 · V5 · РЕШЕНИЯ ПО ГИПОТЕЗАМ И КАЧЕСТВУ МОДЕЛЕЙ, БЕЗ ПРИЧИННЫХ ЗАКЛЮЧЕНИЙ
# TEST-результаты не используются для переобучения или выбора семейства моделей.
V5_HYPOTHESIS_DECISIONS=[]
_available_comparisons=(ABLA if isinstance(ABLA,pd.DataFrame) else pd.DataFrame())
for row in HYPOTHESIS_REGISTRY.to_dict('records'):
    h=row['hypothesis_id'];sid=row['study_id']
    z=(_available_comparisons.loc[_available_comparisons.study_id.astype(str).eq(str(sid))]
       if not _available_comparisons.empty and 'study_id' in _available_comparisons else pd.DataFrame())
    # H3.1 касается точного контраста (M1→M3): он не проверен, если записаны лишь шаги M1→M2 и M2→M3.
    if h=='H3.1' and len(z) and 'comparison' in z.columns:
        z=z.loc[z.comparison.astype(str).eq('M1 → M3')]
    decision='EVALUATED_WITH_COMPUTED_CONTRAST' if len(z)>0 else 'NOT_EVALUATED_IN_OBSERVED_DATA'
    # Сравнение корреляционных / прогнозных зависимостей НЕ тождественно проверке причинного механизма.
    V5_HYPOTHESIS_DECISIONS.append({'hypothesis_id':h,'study_id':sid,'contrast':row['contrast'],
        'result_status':decision,'n_observed_contrasts':len(z),'causal_interpretation_allowed':False,
        'reason':'AVAILABLE_PREDICTIVE_ASSOCIATION_ONLY' if len(z) else 'NO_ADMISSIBLE_PAIRED_CONTRAST'})
V5_HYPOTHESIS_DECISIONS=pd.DataFrame(V5_HYPOTHESIS_DECISIONS)
HYPOTHESIS_REGISTRY=HYPOTHESIS_REGISTRY.drop(columns=['status'],errors='ignore').merge(
    V5_HYPOTHESIS_DECISIONS[['hypothesis_id','result_status','n_observed_contrasts']],on='hypothesis_id',how='left')
# Model beating a historical baseline is an empirical outcome, never an unconditional PASS gate.
V5_BASELINE_COMPARISONS=pd.DataFrame()
if isinstance(ADVANCED_COMPARISONS,pd.DataFrame) and len(ADVANCED_COMPARISONS):
    V5_BASELINE_COMPARISONS=ADVANCED_COMPARISONS.copy()
mark('V5_HYPOTHESIS_ADJUDICATION','PASS',f'{V5_HYPOTHESIS_DECISIONS.result_status.eq("EVALUATED_WITH_COMPUTED_CONTRAST").sum()} / {len(V5_HYPOTHESIS_DECISIONS)} hypotheses have observed predictive tests')


#### Все итоговые таблицы, графики и книга Excel.

In [15]:
# 11 · V3.01 · ANALYTICAL WORKBOOK + OBSERVED FIGURES · STAGE CONTRACTS
ensure(globals().get('CELL10_COMPLETE',False),
       'CELL10_NOT_COMPLETE: execute cell 10 before workbook; do not manufacture missing diagnostics')
_REQUIRED_11=('SENSITIVITY','BOUNDARY_SENSITIVITY','SIZE_SENSITIVITY','CANARY_QA','MODEL_REGISTRY','REGIONAL_CONTEXT_METADATA')
_MISSING_11=[k for k in _REQUIRED_11 if k not in globals()]
ensure(not _MISSING_11,'CELL11_INPUTS_MISSING: '+','.join(_MISSING_11))
# H3.5 tests are derived from the actual sensitivity ledger, never stale variables.
_SENS_CHECKS=set(SENSITIVITY['check'].dropna().astype(str)) if isinstance(SENSITIVITY,pd.DataFrame) and 'check' in SENSITIVITY.columns else set()
lag_exists='publication_lag_plus_1_year' in _SENS_CHECKS
proxy_exists='remove_correlated_proxies_on_train' in _SENS_CHECKS
CHARTS=[]
plt.rcParams.update({'figure.dpi':125,'savefig.dpi':160,'font.size':9,
    'axes.spines.top':False,'axes.spines.right':False})
def figure_file(name):
    path=WORK/'charts'/(name+'.png');plt.savefig(path,bbox_inches='tight',dpi=160)
    plt.close();CHARTS.append(path)
def safe_plot(label,fn):
    try:fn()
    except Exception as ex:log('PLOT '+label+': '+type(ex).__name__+' '+str(ex)[:150],'WARN');plt.close('all')

def chart_ablation():
    if ABLA.empty:return
    p=ABLA[(ABLA.protocol=='ROLLING_TIME')&(ABLA.target=='population_change_pct')].head(16)
    if p.empty:return
    plt.figure(figsize=(11,5));x=np.arange(len(p));plt.bar(x,p.delta_mae,yerr=[p.delta_mae-p.ci_low,p.ci_high-p.delta_mae],capsize=3)
    plt.axhline(0,color='black',lw=.7);plt.xticks(x,[f"h{r.horizon}:{r.comparison}" for r in p.itertuples()],rotation=45,ha='right')
    plt.ylabel('Δ MAE, п.п. (положительное = улучшение)');plt.title('Парный прирост качества: общие тестовые наблюдения и региональные интервалы')
    figure_file('01_ablation_region_CI')
safe_plot('ablation',chart_ablation)
def chart_loo():
    if LOO.empty:return
    p=LOO[(LOO.target=='population_change_pct')&(LOO.protocol=='ROLLING_TIME')].head(12)
    if p.empty:return
    plt.figure(figsize=(9,4));plt.barh(np.arange(len(p)),p.delta_mae)
    plt.yticks(np.arange(len(p)),[f'{r.horizon} г. {r.old_arm}' for r in p.itertuples()])
    plt.xlabel('Вклад исключённой группы в MAE, п.п.');plt.title('Leave-one-group-out: прогнозный вклад группы')
    figure_file('02_leave_one_group_out')
safe_plot('loo',chart_loo)
def chart_importance():
    if IMPORTANCE.empty:return
    p=IMPORTANCE.sort_values('mae_increase',ascending=False).head(15)
    plt.figure(figsize=(9,5));plt.barh(p.feature[::-1],p.mae_increase[::-1])
    plt.xlabel('Ухудшение MAE при перестановке');plt.title('Важность для прогностической функции, не причинный эффект')
    figure_file('03_permutation_importance')
safe_plot('importance',chart_importance)
def chart_ale():
    if ALE.empty:return
    p=ALE[ALE.feature.eq(ALE.feature.iloc[0])];plt.figure(figsize=(8,4))
    for h,a in p.groupby('horizon'):
        plt.plot(a.feature_mid_logscale,a.ale_centered,marker='o',label=f'{h} год(лет)')
    plt.legend();plt.ylabel('Центрированная ALE');plt.xlabel('Признак в логарифмической шкале')
    plt.title('Условная нелинейность на обучающей области наблюдений')
    figure_file('04_train_ALE')
safe_plot('ale',chart_ale)
def chart_heterogeneity():
    if REGIONAL_ERRORS.empty:return
    p=REGIONAL_ERRORS[(REGIONAL_ERRORS.protocol=='ROLLING_TIME')&(REGIONAL_ERRORS.horizon==1)]
    p=p.groupby('region',as_index=False).agg(mae=('mae','mean')).sort_values('mae').head(18)
    if p.empty:return
    plt.figure(figsize=(10,5));plt.barh(p.region,p.mae)
    plt.xlabel('MAE');plt.title('Территориальная неоднородность качества')
    figure_file('05_regional_errors')
safe_plot('regional',chart_heterogeneity)
def chart_times():
    if ERROR_YEARS.empty:return
    p=ERROR_YEARS[(ERROR_YEARS.study_id=='MAX_AVAILABLE')]
    if p.empty:p=ERROR_YEARS
    p=p.groupby(['target','horizon','year'],as_index=False).agg(mae=('mae','mean'))
    plt.figure(figsize=(9,4))
    for (target,h),x in p.groupby(['target','horizon']):plt.plot(x.year,x.mae,marker='o',label=target[:10]+' h'+str(h))
    plt.legend(fontsize=7);plt.xlabel('Год целевого наблюдения');plt.ylabel('MAE')
    plt.title('Стабильность ошибки по временным отложениям')
    figure_file('06_out_of_time_stability')
safe_plot('time',chart_times)
def chart_coverage():
    if coverage.empty:return
    plt.figure(figsize=(7,3));plt.bar(coverage.group,coverage.forecast_coverage)
    plt.ylim(0,1);plt.ylabel('Доля прогнозных наблюдений с доступными факторами')
    plt.title('Наблюдаемость блоков после публикационного лага')
    figure_file('07_factor_availability')
safe_plot('coverage',chart_coverage)
def chart_sensitivity():
    if SENSITIVITY.empty:return
    p=SENSITIVITY.groupby('check',as_index=False).agg(delta_mae=('delta_mae','mean'))
    plt.figure(figsize=(8,3));plt.barh(p.check,p.delta_mae)
    plt.axvline(0,color='black',lw=.8);plt.xlabel('Средняя разница MAE (оригинал − вариант)')
    plt.title('Проверка задержки публикации и коррелированных показателей')
    figure_file('08_sensitivity')
safe_plot('sensitivity',chart_sensitivity)
def chart_migration_gate():
    if MIGRATION_QA.empty:return
    p=MIGRATION_QA.head(8);plt.figure(figsize=(8,3))
    plt.barh([f'{r.denominator}, ×{r.numerator_scale:g}' for r in p.itertuples()][::-1],
             p.agreement_5pct.iloc[::-1]);plt.xlim(0,1)
    plt.xlabel('Доля согласованных значений');plt.title('Аудит семантики миграционного показателя D01')
    figure_file('09_migration_semantics')
safe_plot('migration',chart_migration_gate)
def chart_interactions():
    if INTERACTIONS.empty:return
    p=INTERACTIONS.head(8);plt.figure(figsize=(8,3))
    plt.bar([str(h) for h in p.horizon],p.delta_mae)
    plt.axhline(0,color='black',lw=.8);plt.ylabel('ΔMAE взаимодействия, п.п.')
    plt.title('G1 × G2: исследовательская проверка добавочной прогностической ценности')
    figure_file('10_income_housing_interaction')
safe_plot('interactions',chart_interactions)

# Advanced figures from real fitted ensembles/explanations; no decorative plots.
# Scientific visualization labels from source dictionary and transformations (never generic ID alone).
_VIS_META={str(r.model_feature):str(r.indicator_name) for r in TRANSFORM_QA.itertuples()
           if pd.notna(r.model_feature)}
_VIS_META.update({str(r.feature):str(r.indicator_name) for r in FEATURE_DICT.itertuples()})
def feature_label(feature):
    f=str(feature);label=_VIS_META.get(f)
    if not label and f.startswith('hist_'):
        lab=re.sub(r'_l(\d+)$',r' (лаг \1 г.)',f[5:]).replace('_',' ')
        label='История: '+lab
    if not label and f.startswith('G1R_'):label='Региональный контекст: '+f
    if not label:label=f.replace('_',' ')
    return str(label).strip()[:74]
def _uniquify_axis_labels(features):
    labs=[feature_label(f) for f in features]
    if len(labs)!=len(set(labs)):labs=[f'{lab} [{f}]' for lab,f in zip(labs,features)]
    return labs

def chart_adv_global():
    if ADVANCED_SHAP.empty:return
    p=ADVANCED_SHAP[(ADVANCED_SHAP.method=='SHAP_ENSEMBLE') & ADVANCED_SHAP.study_id.eq('MAX_AVAILABLE')]
    if p.empty:return
    ss=p.groupby(['feature','group'],as_index=False).mean_abs_shap.mean().sort_values('mean_abs_shap').tail(18)
    plt.figure(figsize=(11,8));plt.barh(_uniquify_axis_labels(ss.feature),ss.mean_abs_shap)
    plt.xlabel('Среднее абсолютное SHAP, единицы целевого прогноза');plt.title('SuperLearner: объяснение предсказаний на тестовых наблюдениях')
    figure_file('11_ensemble_shap_global')
safe_plot('advanced_shap_global',chart_adv_global)

def chart_adv_permutation():
    if ADVANCED_GROUP_PERM.empty:return
    p=ADVANCED_GROUP_PERM[(ADVANCED_GROUP_PERM.status=='PASS')&ADVANCED_GROUP_PERM.study_id.eq('MAX_AVAILABLE')]
    if p.empty:return
    p=p.groupby('group',as_index=False).delta_mae.mean().sort_values('delta_mae')
    plt.figure(figsize=(9,4));plt.barh(p.group,p.delta_mae);plt.axvline(0,color='black',lw=.8)
    plt.xlabel('Рост MAE после совместной перестановки блока');plt.title('Зависимость прогноза от групп факторов (не причинный эффект)')
    figure_file('12_conditional_group_permutation')
safe_plot('advanced_group_permutation',chart_adv_permutation)

def chart_adv_comparison():
    if ADVANCED_COMPARISONS.empty:return
    p=ADVANCED_COMPARISONS[(ADVANCED_COMPARISONS.protocol=='ROLLING_TIME') & ADVANCED_COMPARISONS.study_id.eq('MAX_AVAILABLE')]
    if p.empty:return
    lab=[('миграция' if 'migration' in x.target else 'население')+f' {int(x.horizon)} г.' for x in p.itertuples()]
    plt.figure(figsize=(9,4));xx=np.arange(len(p));plt.bar(xx,p.delta_mae)
    plt.axhline(0,color='black',lw=.8);plt.xticks(xx,lab,rotation=25,ha='right')
    plt.ylabel('ΔMAE (M0 − SuperLearner); положительное = улучшение');plt.title('Независимая проверка ансамбля по регионам')
    figure_file('13_superlearner_vs_historical')
safe_plot('advanced_superlearner',chart_adv_comparison)

def chart_adv_pdp():
    if ADVANCED_PDP.empty:return
    p=ADVANCED_PDP[ADVANCED_PDP.study_id.eq('MAX_AVAILABLE')]
    if p.empty:return
    row=p.groupby(['target','horizon','feature']).size().sort_values(ascending=False).index[0]
    p=p[(p.target==row[0])&(p.horizon==row[1])&(p.feature==row[2])]
    plt.figure(figsize=(8,4));plt.plot(p.feature_value_transformed,p.mean_prediction,marker='o')
    plt.xlabel(feature_label(row[2])+' (преобразованная шкала)');plt.ylabel('Прогноз модели')
    plt.title('PDP на обучающей области, не причинная зависимость')
    figure_file('14_ensemble_train_PDP')
safe_plot('advanced_pdp',chart_adv_pdp)

def chart_adv_best_single():
    if ADVANCED_SINGLE_DUEL.empty:return
    d=ADVANCED_SINGLE_DUEL[(ADVANCED_SINGLE_DUEL.protocol=='ROLLING_TIME') &
                          ADVANCED_SINGLE_DUEL.study_id.eq('CANONICAL')].copy()
    if d.empty:return
    labels=[('Миграция' if 'migration' in v.target else 'Численность')+f' · {int(v.horizon)} г.'
            for v in d.itertuples()]
    plt.figure(figsize=(10,5));x=np.arange(len(d));plt.bar(x,d.delta_mae)
    plt.errorbar(x,d.delta_mae,yerr=[d.delta_mae-d.ci_low,d.ci_high-d.delta_mae],
        fmt='none',ecolor='black',capsize=4)
    plt.axhline(0,color='black',lw=.8);plt.xticks(x,labels,rotation=20,ha='right')
    plt.ylabel('MAE одиночной − MAE ансамбля (плюс означает выигрыш ансамбля)')
    plt.title('Сравнение с лучшим одиночным методом по DEV · один тестовый набор')
    figure_file('15_superlearner_vs_best_single')
safe_plot('advanced_vs_best_single',chart_adv_best_single)

PASSPORT=pd.DataFrame([
 {'Параметр':'Исследование','Значение':'ML-03: экономика, инфраструктура и демографическая устойчивость'},
 {'Параметр':'Версия','Значение':CFG['VERSION']},{'Параметр':'Режим','Значение':CFG['MODE']},
 {'Параметр':'Источник Stage I','Значение':CFG['REPO']+'@'+source_sha},
 {'Параметр':'SHA-256 handoff','Значение':hashlib.sha256(raw_index).hexdigest()},
 {'Параметр':'Верифицированных Stage I файлов','Значение':sum(a['asset_id'] in LOCAL_ASSETS for a in ASSETS)},
 {'Параметр':'Муниципалитетов D01','Значение':int(base.entity_id.nunique())},
 {'Параметр':'Муниципалитет×год','Значение':len(PANEL)},
 {'Параметр':'Диапазон лет','Значение':str(min(YEARS))+'–'+str(max(YEARS))},
 {'Параметр':'Дополнительные БДМО Parquet','Значение':len(BDMO_EXTERNAL_ASSETS)},
 {'Параметр':'Advanced SuperLearner experiments','Значение':int(ADVANCED_ENSEMBLES.status.eq('PASS').sum()) if len(ADVANCED_ENSEMBLES) else 0},
 {'Параметр':'SHAP additivity checks','Значение':int(ADVANCED_SHAP_QA.status.eq('PASS').sum()) if len(ADVANCED_SHAP_QA) else 0},
 {'Параметр':'Факторных признаков','Значение':sum(len(x) for x in GROUP_FEATURES.values())},
 {'Параметр':'Тестовых прогнозов','Значение':len(PREDICTIONS)},
 {'Параметр':'Модельных оценок','Значение':len(METRICS)},
 {'Параметр':'Вариантов факторных исследований','Значение':len(STUDIES)},
 {'Параметр':'Миграционная цель проходит проверку','Значение':bool(valid_rate)},
 {'Параметр':'Причинная идентификация','Значение':'НЕ ПРОВЕДЕНА — нет реестра вмешательств'},
 {'Параметр':'Прогнозные признаки доступны','Значение':f'не позднее origin_year−{CFG["FEATURE_PUBLICATION_LAG"]}'},
 {'Параметр':'Разбиение','Значение':'rolling-time / territory-only / crossed; раздельное DEV и TEST'},
 {'Параметр':'Семантика оценок','Значение':'ассоциации и прогнозы; НЕ причинные эффекты'}])
HYPOTHESES=pd.DataFrame([
 {'Гипотеза':'H3.1','Критерий':'M1→M3; region-block CI, Holm; same test keys',
    'Статус':'TESTED' if len(ABLA) and ABLA.comparison.eq('M1 → M3').any() else 'NOT_FEASIBLE'},
 {'Гипотеза':'H3.2','Критерий':'G1×G2 interaction; exploratory',
    'Статус':'PRELIMINARY_PENDING_SEMANTICS' if len(INTERACTIONS) else 'NOT_FEASIBLE'},
 {'Гипотеза':'H3.3','Критерий':'G3 incremental on common support',
    'Статус':'PRELIMINARY_PENDING_HEALTH_COVERAGE' if len(ABLA) and ABLA.study_id.eq('AUX_G3').any() else 'NOT_FEASIBLE'},
 {'Гипотеза':'H3.4','Критерий':'G4/G5 availability and transfer',
    'Статус':'EXPLORATORY_PARTIAL' if len(ABLA) and ABLA.study_id.eq('AUX_G4').any() else 'NOT_FEASIBLE'},
 {'Гипотеза':'H3.5','Критерий':'publication lag + correlated-proxy sensitivity',
    'Статус':'EXPLORATORY_TESTED' if lag_exists and proxy_exists else 'NOT_FEASIBLE'},
 {'Гипотеза':'H3.6','Критерий':'health separately identified from education',
    'Статус':'EXPLORATORY_ELIGIBLE' if HEALTH_AVAILABLE else 'NOT_FEASIBLE'},
 {'Гипотеза':'H3.7','Критерий':'municipality boundary-consistency strata',
    'Статус':'EXPLORATORY_TESTED' if len(BOUNDARY_SENSITIVITY) else 'NOT_FEASIBLE'},
 {'Гипотеза':'H3.8','Критерий':'municipality size strata using train thresholds',
    'Статус':'EXPLORATORY_TESTED' if len(SIZE_SENSITIVITY) else 'NOT_FEASIBLE'}])
LIMITATIONS=pd.DataFrame([
 {'Ограничение':'Неполнота предметного поиска','Следствие':'Если сохранены неразобранные коды, SCIENTIFIC_PASS запрещен'},
 {'Ограничение':'Муниципальные границы','Следствие':'ОКТМО не означает неизменных границ; spatial history requires validated crosswalk'},
 {'Ограничение':'Отбор источников по семантике','Следствие':'Владение всеми 52 активами не означает применимость всех; широкие таблицы без проверенной единицы остаются в аудите'},
 {'Ограничение':'Региональные и организационные источники','Следствие':'Indexed and downloaded but not joined without verified grain/crosswalk'},
 {'Ограничение':'Срок публикации','Следствие':'Отсутствует конкретный available_at; применяется conservative publication lag'},
 {'Ограничение':'Миграционная цель','Следствие':'Defined only when independent numerical audit validates scale and denominator'},
 {'Ограничение':'Финансы','Следствие':'Local budget spending is not treated as a treatment assignment'},
 {'Ограничение':'Инфраструктура','Следствие':'Counts are different from access; age-cohort denominator not inferred'},
 {'Ограничение':'Денежные показатели','Следствие':'Nominal source-year amounts; no unreliable municipal deflator imputed'},
 {'Ограничение':'Персональные траектории','Следствие':'Population change is municipal stock variation, not individual retention'},
 {'Ограничение':'Коррелированные признаки','Следствие':'Permutation and ALE explain prediction, not structural/causal mechanisms'},
 {'Ограничение':'Пространственные признаки','Следствие':'Static transport/settlement crosswalk withheld if source-vintage unavailable'},
 {'Ограничение':'Множественная проверка','Следствие':'Holm for preregistered, BH FDR for exploratory; regional cluster bootstrap'},
 {'Ограничение':'Устойчивость модели','Следствие':'Stochastic seeds not counted as independent test municipalities'}])
SHEETS={
 'Паспорт':PASSPORT,'Гипотезы':HYPOTHESES,'Источники':input_map,'Загрузки_52':DOWNLOAD_QA,
 'Аудит_миграции':MIGRATION_QA,'Окна_целей_V3':TARGET_CALENDAR_PREFLIGHT,
 'Реестр_физ_52':STAGE1_PHYSICAL_AUDIT,'Манифест_экономики':ECONOMY_SOURCE_MANIFEST_QA,
 'Региональный_G1R':REGIONAL_CONTEXT_METADATA,'G1R_аудит':REGIONAL_CONTEXT_AUDIT,
 'Единицы_G1R':G1R_UNIT_QA,
 'Давность_факторов':STALENESS_QA,'Наличие_целей_по_годам':TARGET_AVAILABILITY,'Факторы':FEATURE_DICT,'Преобразования':TRANSFORM_QA,
 'Каталог_индикаторов':CATALOG,'Поиск_Г1_все':SCAN_AUDIT,'Паспорт_поиска_Г1':G1_DISCOVERY,
 'Измерения_кодов':DIMENSION_AUDIT,'Уровни_источников':pd.DataFrame(SOURCE_ADAPTER_AUDIT),
 'Паспорт_52_источников':SOURCE_GRAIN_AUDIT,'SQL_ошибки':SCAN_SQL_ERRORS,'SQL_запросы':SCAN_SQL_AUDIT,'SQL_предпроверка':REAL_SQL_PREFLIGHT_QA,
 'Факторы_по_горизонтам':FACTOR_ELIGIBILITY,'Отбор_по_горизонтам':SELECTION_BY_H,'Семантика_отбора':SEMANTIC_QA,'Домены':DOMAIN_COVERAGE,'Домены_модели':MODEL_DOMAIN_COVERAGE,'Коллизии_разрезов':CONFLICT_QA,'QA_источников':SOURCE_QA,
 'Дата_публикации':PUBLICATION_AVAILABILITY_QA,'Экономика_домены':G1_DOMAIN_QA,'Медицина_покрытие':HEALTH_DOMAIN_QA,'H5_исключен_протокол':H5_CALENDAR_GAPS,'Исключения_организаций':pd.DataFrame(ORGANIZATION_GRAIN_EXCLUSIONS),
 'Пространственные_данные':SPATIAL_SOURCE_QA,'Геометрия_G5':SPATIAL_GEOMETRY_QA,'Геопризнаки_G5':SPATIAL_FEATURE_AUDIT,'История_муниципалитетов':BOUNDARY_QA,'Неопределенность_границ':BOUNDARY_WINDOW_QA,'Покрытие':coverage,'JOIN_аудит':QA_KEYS,
 'План_задач':TASK_STATUSES,'Годы_раздельно':TEST_YEAR_AUDIT,'Годы_CANONICAL':CANONICAL_YEAR_QA,'Модели_DEV':DEV_METRICS,'Метрики_TEST':METRICS,
 'Абляции_CI':ABLA,'Вклад_LOO':LOO,'Реальные_сиды':SEED_QA,
 'Сверка_метрик':PREDICTION_METRIC_QA,'Реестр_гипотез':HYPOTHESIS_REGISTRY,'Семейства_тестов':STUDY_HYPOTHESIS_FAMILIES,'Корреляции':CORRELATIONS,'Взаимодействия':INTERACTIONS,
 'Региональные_ошибки':REGIONAL_ERRORS,'Типы_муниципалитетов':MUNICIPAL_TYPES,
 'ALE':ALE,'SHAP':SHAP_IMPORTANCE,'FE_Панель':FE_DIAGNOSTICS,'FE_QA':FE_QA,'Важность':IMPORTANCE,'Сенситивити':SENSITIVITY,'Ошибки_по_годам':ERROR_YEARS,
 'Canary_контроль':CANARY_QA,'Границы_сенситивити':BOUNDARY_SENSITIVITY,'Размер_территорий':SIZE_SENSITIVITY,'Приемка':pd.DataFrame([{'Этап':k,**v} for k,v in STATUS.items()]),
 'БДМО_доп_план':BDMO_EXTERNAL_DESIGN,'БДМО_доп_загрузки':BDMO_EXTERNAL_QA,'БДМО_неуспехи':BDMO_EXTERNAL_RECOVERY,
 'Ансамбль_DEV':ADVANCED_DEV,'Ансамбль_TEST':ADVANCED_ENSEMBLES,'Ансамбль_интервалы':ADVANCED_INTERVALS,
 'Ансамбль_сравнения':ADVANCED_COMPARISONS,'Ансамбль_лучший_один':ADVANCED_SINGLE_DUEL,'SHAP_ансамбли':ADVANCED_SHAP,
 'SHAP_локальные':ADVANCED_SHAP_LOCAL,'SHAP_аддитивность':ADVANCED_SHAP_QA,'SHAP_регионы_CI':ADVANCED_SHAP_CI,
 'SHAP_устойчивость':ADVANCED_SHAP_STABILITY,'SHAP_взаимодействия':ADVANCED_SHAP_INTERACTIONS,
 'Canary_ансамбль':ADVANCED_TEMPORAL_CANARY,'Важность_групп':ADVANCED_GROUP_PERM,'Ансамбль_PDP':ADVANCED_PDP,'Ансамбль_ICE':ADVANCED_ICE,
 'Ограничения':LIMITATIONS,
 'V5_бюджет_сальдо':V5_BUDGET_BALANCE_QA, 'V5_дороги_БДМО':V5_ROADS_QA,
 'V5_семантика_исправления':V5_SEMANTIC_FIXES,
 'V5_тестовая_выборка':V5_SAMPLE_AUDIT,
 'V5_сопоставление_с_V403':V5_ROW_REFERENCE_QA,
 'V5_причины_потерь_строк':V5_ROW_EXCLUSION_REASONS,
 'V5_загрузка_эталона':V5_REFERENCE_DOWNLOAD_QA,
 'V5_гипотезы_результаты':V5_HYPOTHESIS_DECISIONS,
 'V5_прогнозы_объем':V5_PREDICTION_VOLUME_QA,
 'V5_модели_сиды_сверка':V5_SEED_FAMILY_PARITY}

def write_xlsx(path,tables):
    # One-pass streaming workbook. No head() truncation and no multi-gigabyte cell cache.
    from openpyxl.cell import WriteOnlyCell
    wb=Workbook(write_only=True);count=0;sheet_index=[];used=set()
    for name,df in tables.items():
        if df is None or len(df)==0:continue
        maxrows=min(int(CFG['EXCEL_MAX_ROWS_PER_SHEET']),1048575)
        chunks=max(1,math.ceil(len(df)/maxrows))
        for chunk in range(chunks):
            offset=chunk*maxrows
            d=df.iloc[offset:offset+maxrows].copy().replace([np.inf,-np.inf],np.nan)
            sheetname=(name[:25]+f'_{chunk+1:02d}') if chunks>1 else name[:31]
            ensure(sheetname not in used,'XLSX_DUPLICATE_SHEET_NAME_'+sheetname);used.add(sheetname)
            ws=wb.create_sheet(sheetname);ws.freeze_panes='A2';ws.sheet_view.showGridLines=False
            cols=list(d.columns)
            for j,col in enumerate(cols,1):
                sample=d[col].dropna().head(90)
                widths=[len(str(v)) for v in sample]
                w=max(13,len(str(col))+2,int(np.percentile(widths,80))+2 if widths else 13)
                ws.column_dimensions[get_column_letter(j)].width=min(48,w)
            hdr=[]
            for c in cols:
                cell=WriteOnlyCell(ws,value=str(c));cell.fill=PatternFill('solid',fgColor='173653')
                cell.font=Font(color='FFFFFF',bold=True,size=10)
                cell.alignment=Alignment(vertical='center',wrap_text=True);hdr.append(cell)
            ws.append(hdr)
            for row in d.itertuples(index=False,name=None):
                vals=[]
                for v in row:
                    if isinstance(v,(list,dict,tuple)):v=json.dumps(v,ensure_ascii=False,default=str)
                    elif isinstance(v,np.integer):v=int(v)
                    elif isinstance(v,np.floating):v=float(v)
                    elif isinstance(v,(pd.Timestamp,datetime.datetime)):v=str(v)
                    elif v is pd.NA or v is pd.NaT:v=None
                    elif isinstance(v,float) and not np.isfinite(v):v=None
                    if isinstance(v,str) and v.startswith(('=','+','@')):v="'"+v
                    vals.append(v)
                ws.append(vals)
            ws.auto_filter.ref=f'A1:{get_column_letter(len(cols))}{len(d)+1}'
            sheet_index.append({'source':name,'sheet':sheetname,'rows':len(d),
                                'offset':offset,'full_source_rows':len(df)})
            count+=1
    ensure(count>5,'XLSX_HAS_TOO_FEW_REAL_SHEETS')
    index=wb.create_sheet('Индекс_листов')
    index.append(['Источник','Лист','Строк','Смещение','Всего строк'])
    for r in sheet_index:index.append([r['source'],r['sheet'],r['rows'],r['offset'],r['full_source_rows']])
    wb.save(path)
    exported=sum(r['rows'] for r in sheet_index if r['source']=='SHAP_локальные')
    ensure(exported==len(ADVANCED_SHAP_LOCAL),'XLSX_SHAP_LOCAL_TRUNCATED')
    return sheet_index
# Единый сквозной бар создан в ячейке 02. Второй tqdm запрещён.
XLSX=WORK/'ML3_FACTORS_RESULTS.xlsx';EXCEL_SHEET_INDEX=write_xlsx(XLSX,SHEETS)
progress('EXCEL',1,1,'Книга записана');quiet_print('Excel:',XLSX.name,'sheets=',sum(len(v)>0 for v in SHEETS.values() if v is not None),
                   'graphs=',len(CHARTS))


# Этап IV / IV. Evidence, независимый контроль и выпуск результатов

#### Полный контроль Parquet, контрольных сумм и Evidence.

In [16]:
# 12 · REPRODUCIBLE EVIDENCE, REAL PARQUET QA, XLSX QA, NO PREMATURE GITHUB WRITE
EVD=WORK/'evidence'
TABLE_EXPORTS={
 'bdmo_external_source_plan':BDMO_EXTERNAL_DESIGN,'bdmo_external_download_qa':BDMO_EXTERNAL_QA,
 'advanced_single_vs_ensemble_paired':ADVANCED_SINGLE_DUEL,
 'bdmo_section_recovery':BDMO_EXTERNAL_RECOVERY,'economic_domain_claims':G1_DOMAIN_QA,
 'health_hospital_coverage':HEALTH_DOMAIN_QA,'publication_availability_audit':PUBLICATION_AVAILABILITY_QA,
 'h5_design_excluded_not_fitted':H5_CALENDAR_GAPS,
 'v5_semantic_corrections':V5_SEMANTIC_FIXES,'v5_road_source_qc':V5_ROADS_QA,
 'v5_budget_balance_qc':V5_BUDGET_BALANCE_QA, 'v5_sample_window_qc':V5_SAMPLE_AUDIT,
 'v5_frozen_previous_counts':V5_FROZEN_PREVIOUS_COUNTS,'v5_v4_exact_key_comparison':V5_ROW_REFERENCE_QA,
 'v5_missing_reference_keys_reasons':V5_ROW_EXCLUSION_REASONS,
 'v5_v403_archive_download_qa':V5_REFERENCE_DOWNLOAD_QA,
 'v5_prediction_volume_qa':V5_PREDICTION_VOLUME_QA,
 'v5_seed_family_parity':V5_SEED_FAMILY_PARITY,
 'v5_hypothesis_adjudication':V5_HYPOTHESIS_DECISIONS,
 'organization_grain_exclusions':pd.DataFrame(ORGANIZATION_GRAIN_EXCLUSIONS),
 'scientific_blockers_v5':pd.DataFrame([{'blocker':k,'status':v['status'],'reason':v['reason']} for k,v in STATUS.items() if v['status'] not in ('PASS','NOT_FEASIBLE','CAUSAL_NOT_IDENTIFIED')]),
 'excel_sheet_index':pd.DataFrame(EXCEL_SHEET_INDEX),'one_widget_progress_ledger':pd.DataFrame(PROGRESS_LEDGER),
 'advanced_dev_family_evaluation':ADVANCED_DEV,'advanced_superlearner_test':ADVANCED_ENSEMBLES,
 'advanced_interval_diagnostics':ADVANCED_INTERVALS,'advanced_test_comparisons':ADVANCED_COMPARISONS,
 'advanced_shap_global':ADVANCED_SHAP,'advanced_shap_local':ADVANCED_SHAP_LOCAL,
 'advanced_shap_additivity_audit':ADVANCED_SHAP_QA,'advanced_shap_region_ci':ADVANCED_SHAP_CI,
 'advanced_shap_topk_region_stability':ADVANCED_SHAP_STABILITY,
 'advanced_shap_interactions':ADVANCED_SHAP_INTERACTIONS,
 'advanced_ensemble_temporal_canary':ADVANCED_TEMPORAL_CANARY,
 'advanced_grouped_permutation':ADVANCED_GROUP_PERM,'advanced_pdp':ADVANCED_PDP,'advanced_ice':ADVANCED_ICE,
 'municipal_source_panel':PANEL,'forecast_supervised_panel':TASK_PANEL,
 'stage1_physical_qa':STAGE1_PHYSICAL_AUDIT,'economy_manifest_structural':ECONOMY_SOURCE_MANIFEST_QA,
 'regional_economic_dictionary':REGIONAL_CONTEXT_METADATA,'regional_economic_qa':REGIONAL_CONTEXT_AUDIT,
 'regional_economic_unit_qa':G1R_UNIT_QA,
 'target_evaluation_calendar_preflight':TARGET_CALENDAR_PREFLIGHT,'carry_forward_age_qa':STALENESS_QA,
 'test_predictions':PREDICTIONS,'split_membership':SPLITS,
 'factor_provenance':FEATURE_DICT,'factor_transforms':TRANSFORM_QA,
 'factor_eligibility_by_task':FACTOR_ELIGIBILITY,'feature_selection_by_horizon':SELECTION_BY_H,'target_availability_matrix':TARGET_AVAILABILITY,
 'G1_source_discovery':G1_DISCOVERY,'G1_per_code_scan':SCAN_AUDIT,'dimension_scan_qa':DIMENSION_AUDIT,
 'stage1_source_grain_full_audit':SOURCE_GRAIN_AUDIT,'sql_error_audit':SCAN_SQL_ERRORS,
 'sql_execution_audit':SCAN_SQL_AUDIT,'real_stage1_sql_preflight':REAL_SQL_PREFLIGHT_QA,
 'all_stage1_source_adapters':pd.DataFrame(SOURCE_ADAPTER_AUDIT),
 'hypothesis_registry_preregistered':HYPOTHESIS_REGISTRY,'study_hypothesis_test_families':STUDY_HYPOTHESIS_FAMILIES,
 'seed_distinctness_qa':SEED_QA,'independent_metric_qa':PREDICTION_METRIC_QA,
  'factor_semantic_selection':SEMANTIC_QA,'factor_domain_coverage':DOMAIN_COVERAGE,'factor_model_domain_coverage':MODEL_DOMAIN_COVERAGE,
 'source_asset_catalog':input_map,'source_download_qa':DOWNLOAD_QA,
 'source_semantics_audit':MIGRATION_QA,'source_qa':SOURCE_QA,
 'raw_indicator_catalog_train_only':CATALOG,'dimension_conflicts':CONFLICT_QA,
 'spatial_source_qa':SPATIAL_SOURCE_QA,'spatial_geometry_gate':SPATIAL_GEOMETRY_QA,'spatial_feature_audit':SPATIAL_FEATURE_AUDIT,'municipal_history_qa':BOUNDARY_QA,'boundary_window_qa':BOUNDARY_WINDOW_QA,'panel_join_qa':QA_KEYS,
 'common_support_coverage':coverage,'task_plan':TASK_STATUSES,'separate_test_years':TEST_YEAR_AUDIT,'canonical_test_year_qa':CANONICAL_YEAR_QA,
 'dev_model_selection':DEV_METRICS,'holdout_metrics':METRICS,
 'ablation_block_bootstrap':ABLA,'leave_one_group_out':LOO,
 'factor_correlations_train':CORRELATIONS,'factor_importance':IMPORTANCE,
 'ale_on_train':ALE,'shap_tree_explanations':SHAP_IMPORTANCE,
 'two_way_FE_train_only':FE_DIAGNOSTICS,'FE_diagnostic_qa':FE_QA,'regional_transfer':REGIONAL_ERRORS,
 'municipality_types':MUNICIPAL_TYPES,'exploratory_interactions':INTERACTIONS,
 'lag_correlation_sensitivity':SENSITIVITY,'target_year_errors':ERROR_YEARS,
 'future_mutation_canary':CANARY_QA,'boundary_sensitivity':BOUNDARY_SENSITIVITY,'size_strata':SIZE_SENSITIVITY}
EVID_FILES=[];EVIDENCE_INDEX=[]
for name,frame in TABLE_EXPORTS.items():
    if frame is None or frame.empty:continue
    fp=EVD/(name+'.parquet');frame.to_parquet(fp,index=False,compression='zstd')
    verify=pq.ParquetFile(fp)
    ensure(verify.metadata.num_rows==len(frame),f'PARQUET_ROW_MISMATCH {name}')
    EVID_FILES.append(fp)
    EVIDENCE_INDEX.append({'file':name+'.parquet','sha256':digest(fp),
       'n_rows':len(frame),'columns':list(frame.columns),'bytes':fp.stat().st_size})
methods={'version':CFG['VERSION'],'stage1_source_sha':source_sha,'stage1_assets':len(ASSETS),
   'downloaded_SHA_verified_assets':sum(a['asset_id'] in LOCAL_ASSETS for a in ASSETS),
   'publication_lag_years':CFG['FEATURE_PUBLICATION_LAG'],
   'feature_selection_cutoffs_by_horizon':FACTOR_CUTOFF_BY_H,
   'feature_selection_cutoff_year':TRAIN_CATALOG_MAX_YEAR,
   'target_specific_test_windows':TARGET_TEST_WINDOWS,
   'regional_economic_context':'Separate group G1R, region/year exact matched; NOT municipal economic G1',
   'max_feature_staleness_years':CFG['MAX_FEATURE_STALENESS_YEARS'],
   'discovery_exhaustive':DISCOVERY_EXHAUSTIVE,
   'data_smoke_report':DATA_SMOKE_REPORT,
   'sources_measured_level':'D01 municipality-year exact OKTMO, BDMO sliced on all named dimensions; no region-to-municipality cloning',
   'target_semantics_confirmed':valid_rate,'targets':CFG['TARGETS'],'horizons':CFG['HORIZONS'],
   'supplementary_official_bdmo_sources':BDMO_EXTERNAL_RECEIPTS,
   'ensemble_method':'DEV selected non-negative SuperLearner, fixed sum-to-one weights; TEST never used for tuning',
   'shap_method':'native TreeSHAP / linear analytical; additivity tested per member and weighted ensemble',
   'conditional_feature_importance':'joint within-region-year block permutation, region bootstrap',
   'predictive_intervals':'DEV residual quantile, descriptive only after train+DEV refit; no formal guarantee',
   'group_features':GROUP_FEATURES,'group_features_by_horizon':GROUP_FEATURES_BY_H,
   'paired_study_design':STUDIES,
   'independent_split_holdouts':sorted(HELD_REGIONS),
   'model_selection':'Train→DEV frozen family/parameters; TEST is evaluation only',
   'statistics':'MAE, RMSE, population weighted MAE; region-block bootstrap; Holm confirmatory, BH exploratory',
   'causal_inference':'OUT_OF_SCOPE; no ATT/DR-DID/event-study in this notebook',
   'github_colab_secret_name':CFG['GITHUB_COLAB_SECRET'],
   'historical_publication_dates_documented':int(PUBLICATION_AVAILABILITY_QA.status.eq('DOCUMENTED_SOURCE_DATE').sum()) if len(PUBLICATION_AVAILABILITY_QA) else 0,
   'shap_excel_export_rows':sum(r['rows'] for r in EXCEL_SHEET_INDEX if r['source']=='SHAP_локальные'),
   'excluded_research_directions':['ATT','DR-DID','EVENT_STUDY','H5_MODEL_FITTING','HISTORIC_BOUNDARY_RECONSTRUCTION','UNVERIFIED_MUNICIPAL_AREA'],
   'evidence_index':EVIDENCE_INDEX}
# Avoid serialising internal live study objects to JSON; keep only prescribed static definitions.
methods['paired_study_design']=[{'study_id':s['id'],'required_groups':s['required_groups'],
 'arms':[{'name':a['arm'],'groups':a['groups']} for a in s['arms']],
 'confirmatory':s['confirmatory'],'protocols':s['protocols']} for s in STUDIES]
with open(EVD/'methods_and_evidence_index.json','w',encoding='utf-8') as f:
    json.dump(methods,f,ensure_ascii=False,indent=2,default=str)
EVID_FILES.append(EVD/'methods_and_evidence_index.json')
if TRACE.exists():
    shutil.copy2(TRACE,EVD/'ml03_technical.log');EVID_FILES.append(EVD/'ml03_technical.log')
EVID_FILES.extend(CHARTS)
EVID_FILES.extend(sorted((WORK/'models').glob('*.joblib')))
ZIP=WORK/'ML3_EVIDENCE.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6,allowZip64=True) as z:
    for f in EVID_FILES:
        sub='charts' if f.parent.name=='charts' else 'models' if f.parent.name=='models' else 'evidence'
        z.write(f,sub+'/'+f.name)
ensure(ZIP.stat().st_size<CFG['MAX_EVIDENCE_GB']*2**30,'EVIDENCE_ASSET_LIMIT_EXCEEDED')
with zipfile.ZipFile(ZIP) as z:
    ensure(z.testzip() is None,'EVIDENCE_CRC_FAIL')
    required_names={'evidence/municipal_source_panel.parquet','evidence/forecast_supervised_panel.parquet',
                    'evidence/methods_and_evidence_index.json'}
    ensure(required_names.issubset(set(z.namelist())),'EVIDENCE_REQUIRED_FILE_MISSING')
    with z.open('evidence/municipal_source_panel.parquet') as f:
        ensure(pq.ParquetFile(io.BytesIO(f.read())).metadata.num_rows==len(PANEL),'EVIDENCE_PANEL_SIZE_FAIL')
    # Independently check that bytes in the archived Parquet match the saved evidence index.
    for item in EVIDENCE_INDEX:
        arc='evidence/'+item['file']
        ensure(arc in z.namelist(),'EVIDENCE_INDEX_MISSING_FILE_'+item['file'])
        h=hashlib.sha256()
        with z.open(arc) as stream:
            for piece in iter(lambda: stream.read(4*1024*1024),b''):
                h.update(piece)
        ensure(h.hexdigest()==item['sha256'],'EVIDENCE_INDEX_SHA_MISMATCH_'+item['file'])
    if len(PREDICTIONS):
        ensure('evidence/test_predictions.parquet' in z.namelist(),'PREDICTIONS_NOT_ARCHIVED')
        with z.open('evidence/test_predictions.parquet') as f:
            ensure(pq.ParquetFile(io.BytesIO(f.read())).metadata.num_rows==len(PREDICTIONS),'EVIDENCE_PRED_SIZE_FAIL')
w=load_workbook(XLSX,read_only=True,data_only=True)
try:
    expected={'Паспорт','Метрики_TEST' if len(METRICS) else 'План_задач'}
    ensure(expected.issubset(set(w.sheetnames)),'EXCEL_CRITICAL_SHEETS_MISSING')
    actual_empty=[]
    for sn in w.sheetnames:
        vals=w[sn].iter_rows(min_row=1,max_row=2,values_only=True)
        header=next(vals,None);data=next(vals,None)
        if header is None or data is None or not any(v is not None for v in data):actual_empty.append(sn)
    ensure(not actual_empty,'EXCEL_EMPTY_SHEET: '+', '.join(actual_empty[:8]))
finally:w.close()
mark('PASS_OUTPUTS','PASS',f'Excel={XLSX.stat().st_size:,} B; ZIP={ZIP.stat().st_size:,} B; charts={len(CHARTS)}; parquet tables={len(EVIDENCE_INDEX)}')
ensure(not PANEL.duplicated(['entity_id','observation_year']).any(),'PANEL_NOT_UNIQUE')
ensure(not TASK_PANEL.duplicated(['entity_id','origin_year','horizon']).any(),'TASK_NOT_UNIQUE')
ensure((TASK_PANEL.information_cutoff<TASK_PANEL.target_year).all(),'FORECAST_ORDER_INVALID')
ensure((TASK_PANEL.effective_available_year==TASK_PANEL.origin_year-CFG['FEATURE_PUBLICATION_LAG']).all(),'CUTOFF_INCONSISTENT')
progress('EVIDENCE',1,1,'Архив, Parquet и контрольные суммы')


#### Восстановление полной программы из файла, манифесты, статус науки.

In [ ]:
# 13 · V5.00 · САМОСОСТОЯТЕЛЬНЫЙ ИСХОДНЫЙ КОД, СТРОГИЕ QA, МАНИФЕСТ
# Без зависимости от файлов /content и истории Jupyter для восстановления полной V5.
NB_V5_SOURCE_B85 = 'c-qvxYjYb{mLT|jenplX-3drZ5a2@(&?yH*f)vUmKm(v;6^YGC0tpgD5P*>hN;Jg@Kgv~hS9N>4w<5NqW4fwm=fm!oiI8Q<mSjne*k3^Xh503W9`|P6e1Vkhn3%0@Q^?HwJonu5y62vA@}IOzZKvONFBP@z|D<Ur@E^aiyyKArZ!aC%<Acud{@{{9-+CQ<*}k^B*l`vY#mhs-vpcqD!>5x|e&r4)<F+H3h39v)jrE!Qy!QWouiaV9<@4HVrCM6oR=3vIwR*WxF4dQB?}$#I^gG(we?NP1_WJA<{P*(g*_%I}J$>^jym@x^R6BbH|9|%C&F|lQa`x!Wr*A%c^U0gf&tAa4pTnD1Z~g>j-h2U{;4{94s-Hj;P!OuUz}hdN{403%2YB%#6#4;vW3kVmx5w1}6YK%Lyre3RG<<*oJcXi9sp{9kXoFvuFQ5Gln)rbR@m!-$Kcy;9uw`iCGi>6`pT%6BYs<sFy*q=`gu#Dy_6<&?sL^ykd-Eq4H%#Rp;lm>wr=dYBG%}4Ecnpu<6NsJ@MCjWK>gDyDPqeegFp3{}JaW{z%a>CI{(oaF?d(VB6Nd^gd-FAo;!~=no&7sO02c)&fd3Lu&c1=ie}bMLK^HFx_Hpn+d9f^?!pQKoczO1E=FLC;tc9it!khsR0=iHOO8o$3{`lrgc==CnK9ke>mb&>~qh<Y;TE)RU6Fq$o@1F*+!ew|S*6$T95us25F2K6L7b?M+Bxk1yyn9U;DCdR)d5wc$j04y$!H~J&*>mg_mzE~{XPTgf6$r*J4Gob3o}UrW<eNu4cm&Wn=nY8pEy3|e`u0+L^Lw#PUh~o;Flc%RjOR3}$5aqr;2?P3&vV+@|H1Quu3;#MuSM<h<pS29_eXsBa&9`pJtL3+4k361%y5Yi$gqSGy~P63R9_;ZAx@T>O+#C$&#Y8d49z)m2HuS8O*+R0mw!f}d?hgjQHhBGenfDH<)?l14fRF~N@(@wYoZrFP(g&|7ZCs$V(heX3?;^sM~FJONQBo4-u{7cC1rbH2RuLeNI3Z;boKhpSMUzDb`JO8)6lMdCbwK{xVYJHve^5dfE1`Oh8Khj{yv5^|Bm+*4kxrGZ~mEapHcP+t@fj{=U5WP^Di{dm&(ME(ov^~V6Or!eiq&eQ1Gv3|919&o&A@yzeb1-ZdfEb8g1(rn!oI?cnirbCpwZ9gAED`Lj+q8hbKHfwPD=i^odsuDt`yR|5>2QOWpvFiENAR@MRhz|ErkS9|`U+h*<uOHU?9CW(I3L@AhO@`)z694F@a3kM}#hgW=KaigP&J8{1vW9UeK}pm*?F82<N!<h&ohW8hL7`}UwS>~=NkkZ=NqTG29#bDG%0)MwVvt`K|>ImPVZ6JeoGr17(lMgh%YGR*{qP#v-7pU?h+3xx~LNXM&j_OvMA{1^#_hQoMCl=@R1h(=|fG6_!O3eUxKXo$3c3KMv)5njIL&GqKbgmYiu(iNFVz2L3Pj8yw=vtB8!%+$AL==rxA(mr5OueIN<RhqZAZduLR`{n9ym&CZgC&r51{)vd(Q;8vuc?W+_oaE0uJfJx#81NqnMG*`BjiB@`0HXW=q$)$BW&4H*@*e=O%Z)pR*z`b!NjNbyxhfYGA%0WY1W;&U83MVf<e#*TFOc;Tvp}lwh1^VjE9IkJ#~HL8+1#c*{%GQOvjNTHO<*vudlJHz4@%URKns|PpgXWy|0mG}W!n<hquud>rV7nI#>V~KdHHgfIQ>X$^#?>t$*E$jl-3amG180>%h4|3zb*JflIcYy+(P^6^=sX{{WcQr*?(t24R++LR=ER{k5C4Gb)Vp<1?eK`a{4OsB&n^Zw49HA+N3&#8}swdjlx?@>Kzot|0ZN<v2;igLI6*tSWig|2+IhBDO{a*ZkSaQw@>4qK!eYnd3)G#!~~rOPJ80@h6AfToD4kpI+*nPAw_1l(4DvEr!4bu;*BOADM+ofh0hmIui01!{f3nrcp}Y)WH%zgh|hl%+vA@Mji*HmOVS3?=<nYA1MC9aDNhk$7<?}T*n}$c=1&1;FR!j?rgoyBI{Z)HsI8RsqK@6`hL-TZcQn1R0Yp$o7Vrb^$ZtUQkmGqCV#cH`&1&45Up|JcQH6*Gvbl9@z0$aCVO?vp)M)6%RlDyxhCdDk2#=U6ONoyj0lk=1l1d|FrvP!jA}oWw^9>STF>GQD-vw-&m*Im_WNh{FW(|OJ9xK+3NWDgFtG-;e>g83ay?A}$+I;uMjm3q9e8;xib2k>-_Wa!9Jp8!PE#wQkUAwU0*s)e_m#QnZRcu1=^V$40>u4_CTX}7xTx~)**R%JWx!J>he%|WZZErYsW2Ntu>y1jS>X)85?9a~}0XpXLGu__6?#F7}t<~ROsnnsyY<oEHKzg3uL!(??(Tjy^b2p-2R%&;v>$TE~Rf4r{G)rscf?jNnCt|VRQ3Qm>?b6JVGwyYJPDg9tNP%3U0Q!T_;{vguZMXNGCGFR>N>z)o1d!KA41g{36$|H2h0+eJHM%ECA6kt{b#1+D0Unl{*4iz-cs-YoPGh4|wN@&P_pQ}>xlDzwCKQ6k%6BR&<?1q*Ddb}CxLvDPep9Og%5N8p`BwCEvs7OzH}R`}*aM6s5;bzhRw19y>xMoWjwXG6+XfWbbF5L@(__u7mP^g8dfB3#wp?n$>RaojH5gHBead&Nd*xCcx}1v*rrc<zenf~g>m~RrO3bI1sMqd7fx=uOSg98-v5;8cR;|`-Ktr3byVfcMyX#4y7n5#So8`K-R<CVsVxLn>uGVWCR;?-)UoS2um0pHTSy?GH%hobbP;Oq7pUWp#XqMJ%Yw_aM;Q;ae9TqW3L;QrrOV)&55oKj58Cm9M#O+?G6#t1z=Yt-hC#wNVt@dZF)HVS6qT2+(mX`r7xKP>(H)_k3()u}t06?`WpxL^0F2*!A0U4p-Mya`cyNp;9gP<&;;x0CsJddQ(at{Le($mV{s%=$QO7(lza=q4Q+%2uYZvmOCtX7baF*mw+&1oH~vyK2N73mR0w`c=zY&CgKTy{QXgNP9|%GE}tS-Deb-kY{ppvr350{(&JYPTBY`W@;dZ4*}j{{ghMIt&MfU)e0JLoc{`bBoi<L*q;WU?=AD)5{Xt$kK_(Q!g*Wpw&U#lgM^0*0|h+qS9UHsCGAHhp$vN5OTFDtV{zI35nGrjF-0RQ<7+&u^2%%mN#DtbJYL%<S$ya*0jnW-Y#u50Ptl)Xpg?YLhK4<ob$R3Ht#aYX>)lo0M&%2Qi7Kt-jF3)8Y+`bC5;c(XGnu24U~$!Bo&_(;FqM~KS~0(4`%|h)!5qHtkr>vUCk$RE#NS>%Ij9?PN@RFDzrSwOa~`lU0D^gw(gYHx5~5=1jrN2I!G_2&GnIQzfvt1$mq#(A?&5k#7l-B>+HhHNlS!U7Fmbsr;tNfXMDj1$|zHXd0DF^*th^wA;d&!#0Nl$Kn7f|tlftFUB*3+Odz>*`NPfi8f@|g5@OoG8<n*>?{|r>N%c1Cl?^OSM1%`M4<|_yLved9KUXl|&s-}_dZ03MSZ2O;I)z=L__kiImK%*!mS3)xZmpLE?SaYT#A+LrWq?ApS^lt@JfC=Rl}RV;*@Oy29t6cJ<Wnn%aafgxg&Q|O@wGzo*o5HMtW=j_pChIQ94V2>5jU;^20(d`XzQ?*LC(&{3NF{`_3}DM6)S)-wGV-vY&A*r&*yH$i4-6a;@$@AsxnE3^~%ax83e-HAZHp#>_AFtG~p+q-rB7V_zBI!z?Z8adbE-xc94JYs0~F?l-vu~=I7zh!ZrA_h`+Akuj}~h27JAqFTkGy{+XLkY!6movkXjy_*I#SMIzoamVf}T{Qf40@XeI{;R}998*bc40(E5ryD6c-?F*Uta|#BCHJ7-2uxgv4wYfO0BMmvm2p8kMu52y8zjDjkC~ws2_dqhZiYt{|rg<Ak#R|?cP9m1e>$DwkK*Syzx63F?7I1$@JAFrMkO@L_2IF3P{}2?nqGoqGJ+F7<XrNE_+C96k-CEhGX|B^oO5ke4iRbn@j^^z<TJP|1;@P`>$5+4>=W>xT33KUk4X8ZQt~w4!51UtYCgezizYbYr$s{BUtcv;X?CFv)s}h@gOsrM4%SJc!7K)u)4Wt1d0#quj4hu5^^!zW{oyEkX9z0~kE?@8=3R}6wS;U)viXA$|#BC`h22$5;AP=y_Aj)iRHAw`70Sre7T!8h+_Vx{pEkm=%jy-m@!-?x@y8!G##~C^BXW;dZwYD?%NcT<aP!)0Sz^(yZx{=Z;wA^dkpsG`k+Gw)d@3m*fPT#RzM;kkBr#JH4w2q{>DmqE&W54G@2hdGl+jE9!TXQvg(7}g8#~Z_h#zWV=Z}$%v6%tN6pGh;eVhJ^**(Tl67sba#pf*hq#y88$D1%wdS5c_E=i3_I39LcN!Ox!}$G>PaVr$Wtc)Wp<5}_%h=-JDtGQS8&2rLoMz$$I6RGMmcDk4QrejmYXzQA*kg!42OWFC`U6%WH-u`zf_Jz{(&Bt0=M?e=2a{PjB7K&2v4Y{*aefFi-bCQiW<kB<DIh_Oc&RLEQc`dA<3vQNpX^EjC^0EaHukzFI9O5qLgB`8N^n<q$im$|rcCSGByx>;I&AMhQk%;hgq1&H~ra$V42#5T05fvONA!*$1njo03v0f`JqtMg#gw|fI3tj@r74&g@;C~zNWu+stR(070W?82Ve9}fq^{&0`>)k^tJl=zlb?jRFeu^3dz!0`*!%O7l2KypLlTOd75tx~mq4O&|QEAmT(Z6ZOlSB><sHUMz2u<s|xj6tankwr@wpEJz6WfrkX3_=NOZd$wRXob2E#1!Ta5>INeTYx!eVydpB8@Ix3Hf!tUdWnn#^Uln|xuaEupSig3<Ch0i+z?~9md~G44CJ#?voUQEHdi)7M!SW@{PeQ5&1txTMQ)Mhs1WT_{!d2-FCg2<%a+%-RzQ*~-Icp}xdc+8!q><4eP1#mBQX3OATK>~faCP-@tzYM3W(K0R*|-7h2({e$E;TWD2@`vF>6?r4`??CET6sk`s_O(+ds&XY}6yM=9zkGoGd{#%geXf93)Oe<iOq$P4gkUdIWnV#0KKN8b8LI3+jUD6FdEpnB23#*CM7TvA>-q)`W@PEkpx@t@XamAnb;NYBL@WryLw1b7I$`?~(1N^}1d|V9<7Hlr}bTvlbF0YXLrzSMCX0CseX3tJ3(8w#`8`FW1RDe`mp;BlXJSjWErU&(GI^-<W^yK*j|!)xL-vOAuT8Jn0CXKlm$nX8~x{EdHJMq4gtiduE?L*j_}%vxWS;6-<DE5Ma!FeoouCl{|oNLqZ1#In=!oK_4Cr^p6IU$Evei{BWI)RRS^j`Sh_i(F&yF3DV3Fd;vk!o%B6-W-dQ>wNRLwYZmhJ3x&Dg#80lm?Z6_>EnY=kYGHTIxjx^S?_9IzuN4Xlh3?gBcBkF8^BvonZ`=0V?%YD*s^eV0alO!9Y!|wP#T(tLH+I|CyZMEJo}~8${rK3Am>(zXLQ3J&EbG?R3L3v|mDZWMPtkZqjXIu(tyCJzH9+Be$u;gQS|DA5`iB$|xN;42^%Qj)xC9y-$Q*5nvp%HW)(55FVAYqv<x%?6*h;fu(O=S#lW5@ZX($#^)u9WT2nn`qEq%iDkO%;F5k-{7as}<stCeK{|9ZL9AX%k$x13I-Q@pciN)c&Zyi7cmMvYTFAXd~w62R$P<~7#{27Ma*v|Sf8Hf?!zZJUqQTcCDn8F7Se@X2^KR^IOSGy6C5q$>5?-oW*2APbp&qc`xfT6Z|s_OTsm);}z4Wt00&JLhZNGz*H)G&*OYRrpgrDSyVy;5#Jcn(#LHhH&9Kc8X<3C!G8%Q3k$$#Xn=t#B5N0tVs|9h5RSdL7r>EY2i5(eu{$ZPjXGUI=_(Ly<S*+Ti2BT$_NP5jw}+w6Xh{!T+iruO}Ye%<6+@d<-39gZgW2ImyV`reE#$ob~w3qeZijVb{Br8!^s@Fn3St`D)m|wjUtA&QK?ooO6wX4L7-D?g2D=94rF#iBNb3vudM~x(mNat0l(a1*U*OWpSus^?i(6tFdm@iA-tLFj>f~b<HA=i-S6$@xYY05;lR+m-XUz=ecRoKFNQXTheNW1^gO`b;jj-i@7v=+Z?FfyKJJaWy*`xb4GnD<gq5oc(8SnoJG*xK0Q&Q6ujhKbHk1XBINW-B*zXGymZ43?eH_Ec9wW80T@PBNu1BLWY^-2*-SO~{Y%=gw3|`xW$DjoIKA89_Lt9L`LbS3=_%nBS(CLjc(2+CnT$5;&^8iL?4G-u^*{QghJ4_TvRt2Kq2r&SDw)gF^=faOez>>bJXNy_}dzkHHMJH+PPFxUxO<tE=r{^A6^d{qaV`?Fr4ekA4BS`Pn*XZ|1yZg0pF=CZnBag=~Nyd@Bj>$fOs#CDyRQp&VoKd&KM~W&7I`CxdSU7f*#-8hr9cOm#^5yybO_ND#)Z?P1plDYc3_VS0r8u^GAmr9zYXNeU$K&BRql4bDwF;~SS&UQxu*KyPDE^h^J(W6F%VprR=84{!v=2JFAY<?kAf&m1@$MY@UL4!d?tS<qe!;hE@M!_R0Xer_{6s%)nsZodUQf_@^ugqCbPPr5Uu=KwDizh;c8@;Lztpkl**yRmemvk$2R(0w9<js%6-ej=w%e#b#EprPm;Z3noX=f1-o+sgN6uh${Gbot<xl)tpsz{8-o@Q-dpL6O^QO6&yJ5V06<QlRA59$BLlDTH2!#1mD8Ll&_d0tHRS6z$nlQW@(BSWeyF7UQfs>ub$toPUw%Mdp9ay!t?d=YSE)GuqB*-%S`n|n<Z*TVyzRI6BO#t1Zo~V;Tp#cC`?8{>4@uqo|$LH?bBU}*p=cbv@Ex>CKEA<C^avb6pLLrYZRR(_8bAj{kncGLi@DGe5q!NsRIUtdTZYHY%l81?FuG<@QEO+FzGY9b5)zBXzyemY*#qg|H+%^NUof}UEnQfr(Ic7^}mooI3Lqz1>2>#A|1b;|2>GyTxvS_wtwC$0HTaGyqQ$Npn;6*7E{(<3e$GNcsl?KQR6SwI`sk%bOT5GeiY5Dx=o`nalAl<0jR9c^{WDrV>_*p=V_VG(YGur+y0M<AbLor1%HBI5DiBc!MkJ=Gi37-<Jhpl3=87FCF{g%knQ(0M{%uQ~!vR+;(YWfv1C)evZ<MH+k9&Qy|*?3nY5PwOV`ES0B*;ig(v+|yB%>)=99w?END%|itC$~eqVg^5c<com5RXb#|{+{l1y%xS$EI~^l5gpp`CRB*K39*9cJIEs1U<7gq@dbPubwG+C-hzS=lS=r`EQJc#{v!%B=xa}}KZ^Ab(DkL@XKvy;0Q$W>b>GA)<VMH%iyH1n<s)K1QaVEE%Z(ObhLi*Rc91%1e)|WGC10l0fua-7uhD>i`%Z=4jnjG=J7TNBZt*L62+?t;(YE`kHJNf-M7m|gIx-rg+_zDptU6vwRcWS|)oz0<wl=mqJ&^luAz_EH)SW%RTX&e&J)VPs)apaWu?LpD>kj)cS<AsaV$eJHqY2299ex#QOY&yf$lc-i02s@G(I46!>%MqOsRaA!xbOG_SRD>L14LT@^<cFJ>PHFK;Q&vG)9NG1^s57Z2JNLkIUHDh#|y2OM0p_?7|gVT@1stR$)20T#mKdXqhkw{q5ET?)I{cjvOpS|TMhH%bm?xXUaeHu8s>IusaY>AmrW9o$&zGwPJ4fV$0WJ_a8F-?TJ@%h#0Po+>2L_&GT9JerzUbFRReD*^5bLxNOGW->^NNwn#>%!dxq<vN3wTpf-t#S0~tL`k%&m6*bQZ~37^JJ4){c7tY_ZqWi#&;_xANG`p9~?*Yox#yYSlrRm93}+cO{MXKq-nE5InW>*bZwa<jbB(uoN~+Px1|X?-#@enYo)!$B^9+)$r*-I?op7PoabflzmI1n`X1qH_L^!SH_Y7XN$<d=Zwvoz3;!A+KsCd!?&u+b0qzr!64dIK(UZ&dxxOfDo1W%)ka!4JpXPHO7t&%Au*ppvVipjb&RP%W-kND5~jP51+>w{+-<}E?m#IPQ%UMnqeDPbYv)8TmFvJjPq13PHF`yY;-Nb(9ua~wOndckVm4Q4Y?fPTlDyBvcG6H0n!q6bM^?2_%k>y2Lwg}$XiS?0#sIcGO+y-<BG9q;P{798!C~d2sRzF<1mh5Yn|R6aGwmS!_l_(O`*N#K*F6{yqZZxLvnD?$GcOo4g|Da1S@L)nZA9v+p&vXkYDUhX5n%nKgWNv#xCN<er_^COw8<Nqk<R8JKBD3-+3TLu9U#l*UN*W-grn9O{=|M)HaX3eMGq{D4hY)x&VxvJL>nK>UJKso3^~UWio;c-CTP#Ve?ETn}u(vlah96?e%s=sWE)+WTHB&QOJgs?p3?j*LEG4IUWAOvY`VGQfpi#^t3_N-vgqrjag(uSrk{#K7y(!LdMkP3=mPv60@)@F%Lbb<5U=HLD|U~8Xzx7)D3H+^y^yPw+JKBqQ@X2EDl2FR&}ho*B|cMeH55M#c@a`PP)&S^h<_spzBRV?d(6!{(AN=XaA0OcmI0!7yj`WeFs1u3(iVVT048e-VhP9+Swy56X=7v+s%#jY%XHVd<H_^cYH-GV!jOSY5MUHK4B-%#~Rw^wESF7JNqwbm#WY*@`{YaKZNcj%1gqUa1lMS`yoSo;L`t^ZpS{NGdQ}r^qO3TztC2JVSAh7A&Crz=6%#T1VOL|t7sns;OT}o-INsPY2+~_&hcUvX|`$RB5b?D_?{E3O1>>0VS8FALiGu&@l{aY8y`oh30Dr5g6fOk3T{4VJ0nl4P-!AGk#=MiGmKT%=54h0lNS{*d%aw}qZc!IWA19!z(d>H@HBVTShxm{!T~9Ru8G$R@Phq9GVox`<*!qTJ3x-f;U@!cjJf$6@D_c!@x}ZNV?K`!tpa1Mp>IJ3o{jnWd3Xn#rM!-B=I4$1Ym4xvS!%pbWv(q6^Ea^hl`;+vej5wdv5xRk&%l$hcw+%aEL0tM2``MRR1Fy$HUv+`wFPV%&o<!&JQ&yWbNC`~*@yS=&bYpSRmt%W-z;1=u3tr%1eXjl@WQyk6JiI>3_KY(=CB8Ob|nv0@#Ea0abs~_`G_aa_+fG0xN&2SCb(YV!>A0rFi`!<p2ildS!u49{l(HNvz0q&Z4NAqx?DQ{o4DJKn9U~!kLcX=Tist(zbM`jkNOT_i@exDZ-1F`Jzd^M*Fz`~)csee^H?Z(kHqa$af48o_!5-<o0u1e;}E9+NN~PEbx>FEDX8<$6!L_Klffrs;xeaz!4L8lytvjK^ezxl>7GuiXnMMjm3l=1gdcK2Mp20RlEHv)FX{q;m8yRuLh=m<g9*j1K&K<!$8e>}U!!p_p80P0x{CiwmH!=m_vrdBIg{!Bj({@iEZLIGhtHww{sIGd*C>_n1q~o2G>RVCSZaU&jYj=Pap(F6^i05C?4C-#e`F7khd*!rr2D&3Z6<PoUspG%Ub!QRl>>Xf*s8%dXoAe-&omymV1&4)j|KO7sZ1!ov6ZI&^&+tV^e6!c8W7u58E!xkA|rk!{1bkl*c}9Y<W`SW182LBW*VzDa3sTvFa?x8Evq-^d6t!Noqjh-hQ{}~ftg>TAKuWj`zBH*e(pGK+tg`CKa+ujLi?GEjX+iJ_8x%R!Os>d{$@U>JmGbTHi(SoOd(|u%)(4d6-qKB1DM>*zIWIMp%LXOStv=AQmk`IyHfPSvc(98&M4)w2{5FN(*P(~IF?DUq_sV>2cj-Zv;f!34DZ<h+S%+9>MJHTkOOerBPZj-KEXH#x^BFzSb&7$LZHv+?{<1e8ks80olA%IgBh5ww_m(5mmfV?8lnICV6RwM96itq*Wk_WaNKdm#R5FL!+x)$z0+Cfw4KEzJ<*!a#b?lVGjWTBd2GRMAMDY@i|^!j3+?&BQhV43(dC`)V)sV3yVM;HykcPidMJ&1cHeOAfji^k<>Z}9H<P;wO+u_*vD<eZEd6fcdcE%PjJQf&bVqjEnb~!``;IeM+OtR0(-J7md%eNT?$Gmwhs6TSJ-x5pn~K!!ljnc%kFo>uuf}7*g8CKKk>8!&y$Q{*Rs*;I3eVrsyIGjWv?-2<G{ceD*t_<4=aOavlxAo{42JjNtJ-Yn^I>lQpY!2&_5*ka7zp(e`}M)P@8Izbp~6ylS#LyEb$)SSVR3h92<lR|KfGVu?{$E(U2O3f`$CXel0E11f2a1t21fGoMZ+M-VI61BcxSF)FWOz>ovVe;_1%2-LWp^T)|<gE{)V>&VTU&y6|ciw1-jbT+lw7W*BQ6>u~RI}rK8Oi*q^xYK@WNYNfFvN?p>odXpbG#rl*i+SkX*CG7MNukmC8<6{Igg8$eFz{n!D`+1<}1bnZHyg~VmtBWt0BOF$z*^AXiD$)hP-40r&Qi8Kw0ciS<=Q%XOAFrC+jWFSAK8(EH;=fa}tI5=_RM9r5=l1L|$=S0SWoP@ff6CG%ECg61M=H}@wHfzQiV38?u%2G4uga(sXC$#uhY)*Nfi!;)AQ%)Zm<7o}YIR3vY<V`bQM6*v<->lcxKrU)%C-CL;@3a#iJ5I%Kv5Kcj7~k#>T}Lp{gyoGK!dbF2>sG05sxl6&SMU+<-bGa2;9GfRtz0*^TVZpYB&nIvzU?|j2P&HdL&!Zw$MNi5Kf!#;#v3i%b<8qIZA$xQNck4l!uOqHact)|v{ik-TDw~nr3FF6cCryDx}H7u3}?_WLoIJp;nosQ0VzWP-P|%#xS$y_!9Xg4pYT=I5CtOh96}@16Va6nbT>nf*~?gKHfc{#9qa*r7@?)h4YG8Q_>x0Rk|68H==+jAdqrSP;masOT}ca5l3Ogf!bM|i;?1ETR%h(ii&$6$F8|B|0)q6=b~0pFH*`(UZWpgFwqii{95WNr%>~2>cNmHSZ;~zE&accc+9c4WFwg|Wg{s^6)>8PJq?BF#1p7aoJ)setqS$f5!#LHabwGQD8jp<@*Y9Rz{fgmd!ML#GOmwBg?V)5Ishq--pl3o-lV(zk?BKw-Phi)Jym9sR5Dzsyr=u5XA#|#(7kNQ+5Sc8Zd3t0JgVSV_cbYvGiN;5h9`Z8ua?4x>8NoF!Up}~xKcaL3aQ&}<gFjGsz}LRblWdmk;28~|$VccDBvOl?xzR<=*HObKSxM=<oPtd%{_I+U*^*Am#>Yg)0nKy28=^D;JHcjOSR1JoO(wd8Y<4WuKOiX-(J>&|K3%4}A;R4d5Xo^hZ%8(He>xgX2Z<xT?xU?99vnPb)z~Q`7j)&qvK-nluA`pY3mfW5B_l|Q?ksO;(ODvpb$buJ76C=~hT~&HLzhQ74bo7E91Qu5$uVN^75$7ZBJ;Zw*-Xx|uno&PEovvClT&T`g!~s}57`|4JuRMaN2e_yDR@4m@<Qp2nREAt_r-&mG?gS6`;3bWgGK@VPY=5tCi~vTL*D(s*D}?Sg^cX~&i;yz#3cj~{2_cvhc5YkNx>bVOi_07h=U0R<U8b5?wj4^%|xL-3BQd^P$TdRbkKJ0GB|Pxumga5&R9NX^xocJICgYH6DwotE4F8^j%_~f*$0)Pf2?)K_8!W|PA7L+4=}_Wb#mko;O64JXMd_UwS6PM+i@22Z|hC{mw+PB<(JS!BXP$?7?Coe=$lL)q?^raSTa9q2QNhSm-eH+w!7#QuK(P?eH4(qQC{Avmo>C$YqJ_&(x0hS*Y9b5jE~3x3`n!&pcFVvgvru7A{eRbfKKFT1Az$PyYoxh01wzTI;z)to~u!S6eL=(|6GGES^;g86iUHFzAJCNU%p4;#PaG|4#tp4ys~dZuk5QNmf8VXbJ_tB&v-c8>!UvhPD@OC;yPmld89ZD*%A}S!^&W+hOXZ9PdrY?-g!(oFWCW2PPCatkbrQUg95!#q6)z*BO?V};|}s*{^=9<Y%xA{MV+CK#oe)EA51+03opCS%D8g4UG5DJpB+BC<^`jrvFcmAWI9bQS|$v%)ai#h=k5evW_B)#E$a;roB?fIib#f+!07@I8={pfKM8?#{eVYj&pEncwy5n~I&UCJ{(?J~ZmrkoMmBD^q9)>>eG||MCfYCfvQd--;khi&2f7csEp1lJ?#`us&l|bL*;#wk%dz_nAnoBSNZLbp_JkZCDfVOQbVp%14G_&LwW=u{^>PjLLK_a$d5JQ)eaG%NK&^!wda`q=H1YO_<KD*<a%bmK5hm!^W7yg|m#z>P3@Fn^<e`rsP4+Qt0~DS;8g#@kui%x6ol7aVXVw4hT>5Z^0Web<^=9s{rvNpL5q@S0g_*hQ*wUpf*g!L-J#_rSuQI&lj0|r%gAq%2E}e!i68sCVh&;dgL<FgQ=10_iDp|gIiHOaT6rohR-E3~o_}m#~LJUt#w?j!`5>apQhUx59eH}wrKbDtOrDZ=9Z5stHSW%F#0MZ@QKt~`!dd6h~y)N)_y_vy?Fxgb8crxyrvKsDi`8!9drHg(p8C@ceE*vBPr>@9?5sBTD=pax=4h{M-5Y>VCm^>2Dmqr;~yweRBr*R{nRYn?j{LQh$kcA9U>hf};2~-n{nSf%;%cx<jl&h8U%Fd;DOGu6v@&#idKQAV@bBR2k=z@E#j@J+C$nF{mwM}y&zaVSj0!zPbbstUf5{A8bg6)6=E2t4RqMd{~IbA}pXX4W4Nw7rh0rMVtz-$)_^XL<HDOrb7oE34;Tf^PonTR?18u3D)oVd@_eJ#9hWN9Y4plBVz?$*h7oNr@ZI{Pc#rF%mDEuV6L+UPaxr0dsb&zD5T2zgUl*-gUfgrbh9Q(_=k?!K)IA7D5O)a($11wqS{c(eo)F}rQNu%KS0k@xIt8ygk6MQsKhpvs7$gn2-I5)4I#RwHl+cm1Ae`9g8Zp=J=iTsi7b({Bm<yT_j6Ci<y`myzK>rz`*n^-@^wqX~#h?5yV6T@b}#GoQW(YQK3B%vga`Z1edjAUa5tQv|Z{CAQq|)rF3O=$nDc+kVBU`yq+fbP@c8NWZ}@KS2Mr-TrXb6@iLK!=Yp<xKrqp45su%iTwAqhEhMd2Ll9%!ai1$F^D_{;?}tzyPL^wL^sDi)NL{gW16Kg&FWWZQiM;bK*OR8HR+=YJLgxXrZa@}6gfZ|9zn#hKqk0lm8;9Ol}dFjZFk22jvXgM90NjDHk&4nIEPSjGYXFJy;~86xphUK<=9CaG*1LhqREED<*|L=!e~wp%xO)b$chIKR8lmeQYC6!v@>uXc$v%tlA0eNffmEw)={f>@v?DOd$6q|LhCIM3_F5VCH9gv$d-^m+mt?4`#hZzL_9$KNoG|KOw$ZCLI{DE^jmR(MFInZewHcDnzd1BP$*ax5cD;)JbuSZ8by0(m{ew4$Ggy;tI{(uvA#3Na8VfG;v7u}-<{Z^lGW<~41E(5mpz5TO3j2BP^u%?qZ|S>7KA;VRN+xkqnP8*LLw22hSOcBF|9d;jFKNik1rP99JeKWEg30W+}wc={UL??(p%d)D$epgI_y}wpGRRp^lWs_iL?CIU>0oJdm=8|i@+aG#qE1?Vj_2oP!y=Mr?XUusXkvWm6>*;c@NZylFPUr1ke;QT*)^I-LZkZNAj60DG}?KjH7RmrjT?H3+5tHfod}C>O$y1<-VxZ-d~cbRLm1bvYksuQjl(e7{V>P=81wR@ESR1tXFkzfsKqj<b;vz-&CJCG?XzHPl$3L4P!se?6Z4g?uK*Y`t`Ro`(RXdj<!=&ah;uiFkU-gqA?{&OC<QZmY88pxCB2Zj>Haczj!u>!u)Q#P`I}EGmSzEl%;N7_*}?<c?2esqcq>?(9rx4k+U-T=xnXNR;pHhLvb3kHT1BH-JtQDL%Iwf6ZmA%KD-N~`BC*hDA6+q&Jncn<TNZnp|}Ip4?+I1q-2mGA{domKw^jg3&r6Vk@m$VLGCXo$rA7i4S+qKxER6=wKbqAXrRh*GvTJ$k0{JgXbdVn&XJJdWU?U+vt68<1NJHJ!bJ=-9oG}Lg7}X8EcO&X#Bn|U;t$bF(Kx&ZnlA<-Ial&^%GhY70(SBs##4x6M;7ih<>F+dSa7D~w^@vJZOE|}BSSZc3-nn6(Tg~3)S@zIj<L`|1Zv9ux_U7nSO$a7+6QhykylO=ENCj4ZN~=H;v-DgOLz5xkFnL0H?YKQSU=vulI9(bOe}zcjpa_d-QI%?p#jL=OYkwRX7t3QEf?75kkB?AN$2AyQHg606(v(OP2#92i;3cxnV@%211zfr>PK=D5sgMy6dG(GuCwUL(zpEI1x0w@IFXsp8*>Z!?DSTJOp+wu1B8ThPlnyF0B>zoMIdp!hmtblN!NS}fnKc|TuTD{F!cNWtt5H#4cbnr${2xMJ-G}W8Yfqsm}n5vC24eLSuUMrj|mhrX#In&l1&*yU_C;ZP7GmEG*5Ik6Y51>H4+Zno`ZP}$BuodpBnr_Tpw3f6JsK$^%%A0{K|yF#dKTaILF>H*`=fl-+l}ywm6YBNZgxJJDD;VofH-%pJU&kO8B+RD?ii3T}bwt{_wst&Sa+y_IKu3An{T#Zpi{oq0aA)hxc7)Y^h2!b&%ff{t%c>XqUqS{m95xrqmW&2OG(`Ky!gHnrva@`}y<%|M2mUkl}^F8b%hjVC#6{3&f7ndS%6rp>-Q$flnDi7|-OGe-Z+vA`u-({F1BbTQoy`EnN&FZ&j)kt2w~PX^5E22+k!k)5lKNAb}7ZFg)-wq$TKC=o)14d`mxDR@w&_DC(28D@@z4TJ~bq1fo*ycgYQrLrNs%Oln=Gdysg=h+eH%(42_ZCNQRne!&iXB0Xi0l(4FQT=bVcfl~9hg@^}h6c_Qt?W2$l0`MXx(<HQxf`|0aLSbj=r%Ofu3yOZ^o_hKg5?ss?XNq9PJ2cJGi{S-mP8qw`iS7bhSj0Ho%TZvzz`by!0tnU`sz(i8Qizu!Ut&&=(&Q*@RMff#DADUAn;h+=c}R2mI#vf&OsT13iivGKKUXaF?Ol*U4f^d4+r42yeV9`p=EcL}pwsP(-`D7O$iSwr&B@01hZ6+4C<_Dz@NYtK((u*QViE7t$$zXv=Wy2<1IsUVL1m6MIxib_Iun*Tl(E^d;tR6)?gSlsqn{V)Gj!dom#PiQ55Xf^$uH0!Wi&3xy3hmSsx!tbNyQ>?kKPd)yrTLoV={W=k1}?)n7CSuJ$r;bd&X#l2ozo{HD-`KIL?GDh3aYGh;3-p#AuWSnkTc0pl^su>u!-OgotG7`9iUn*F)#2=@%KF^TFvCd_gS8Ohab;r{cWmkz$*9LU*71BnRnf)<t6W*PN^K6=hrqd1lcC$Ivi;B)8QUXU}Hi*E5$A0?!ATlJKnfE0NqRlABx^(KC&X!@d=9Ips0$>tH&5E?&ATr{ssu#dcn4rL@^Bi*WoQ4{Od9bI;l+RV%CIM%n>@erOMRT|Dcb&Cgk&dklt$#};l5>!=X7{L5BF_p)&`%#1oY-`y*-eIlvvfpe@E4<s?xi$^RGiN}yRR!7m)>J2P8CciX(fU>#>$3kW=y@(9{NFGw*2bcXqwp-z5_iYzXG@Z6*^_?T9uNO0V=$WL>l$PR69Zm+lc5ei`mFBiP9QJ{S4G+A-A;y{GhWYXI?~%S=sAXJ}u?O4Q?qTFR5sa$ADSWC;izHq$rRk(NTzCR}R;>rQf7WZDm--RyMQb$N3p6A6hI;!^jYt;7I&O>4aEz1FtfV@vY#Lj1`n^4LGgZLc9|BV))xnMvyms3p9vT{TfI)Kjkz>>YGo#G8-6rWhv~>KQt!F2?pJ9ZAaEfmxU!d9&dcnTVhK{*N^wS&&1(RJIV!NGkn1={7JV=EI-7f8Qk6Bu<0NMZydk_aj$AO|_r(;G~MBF?m3jWFr)AFK}dc^lw#q$?Ow4nR(o~@0o^=3unW=x!acnnEU;CtuN;Ja-{4|Rx#sr>6#k^$lJr3&NefQ<sCrE-YUK=b7Tz?(flHFkN@U9I+hbEAermrd3~vV7N8HPsT%j@P5{aM;y`UA18!sI0*tP6n@nWM-fh9yw!-$O7`5GlH&=T%%;#pWwlbOn9Sk3$=fXEb)}Lt@0rVZ5bWKkuu6LD4#6i?*-*I{85>FLQ@pB8`W!An~*Rc667PHuw@J&J=EZO`A&H~lB=w=wpK5%0jXNjsx?Um%jl@1RBYCi8nAZDwfYJLD3A6lV-Zn0Iw;><zZb+Erv#}%JCuh^KaHYWT+B|Bou<G&tKigDmHGIJ()ZoUN*Pdn+WD<A4j4qhf}(`yBJ?pE*(nro_PpILk75%xp3)TG%kzFVl#)fBbcE9-8Dm1|>wuChhS)D|d|57?E@04FoI9P$p%_8D)L%6HLNrq8mU(Kmre>c6LWgT_ZB_jYtqI}`Ep-u%6z9ty=dOLYeE?ez=?Kn`LWBrSa-fCiuoyf$Hz1X?Lc3GRHAt~FvQddaLF85v(GOB-Guh7sv_3O6)@M=<ZXv&fckXtef(NQgwsj>m&9FyrxMPleA64D^{o#G{MC=1aorT9d{->(7!Mj8*rE0j5>OKpKyk}kXBJGk|9;wZWZset^R9~cF*6|Zs9?m7boIkhCx7L{>q4J`rq)HHxZ!vOCH?nX^TS?9h9Ms3i8>ov!n#$s>AmP#Q0<*aOo}9+X?HAqDghN3nZs}k&@nD0FV%k(Hk&}8Ay2>zpi*j35s{)-0O41v<?M=w2p<jePf@BLK2i1;w-UJc;m>dAfQtWnt%`V1CTPYWxN+8dm8xx%?X+Xm%dEf)5W55p(bMWL%%g@i`7qtwsmP1S};W^{c7_T9sbvh87hQI^IN3>PkaAY;48@}_Pt2uOWj{3yhWFUImlfGwY_xC}4(T0QmF~&jbp|^WS!?Uj=d*De284YhVjYV*{U)L&CHy4hKZ}$wExHFg>g7oG&QFJj1v2dG-X8Dns?BS4L#8mty5{<1%n3TpWgC#~IGCL?;%*?In!bUraS!n6P7v>^5J?Fk7E~)dN_&;lk<$AqVf9v_>{R!c=$e`z-yssDp=7hKCEftP+DO;j@p>j#l8mkVU+lVzCzuZVz5+zZ9e~C1zi|bj`PTa|1=4gB862oBU67V<jjiII9xfIx;6^Lg~IZfh8Qp+KCFx7)6MB3EWROe8R7HMt^Hrl$9CJKjT5-F)f2GdFiMsN2z+)9s<R`>xbv58(sdb_Bn9dW-)sbxkA8oIpM!^bWw`2y)QLAtKE7a-IWnMZ~DpA2px1}wxaylfCd%r0W?o>K^-X|lhbI;QODZ1OTK1yh%>p@QNeypBBBK(Gpu%qgROd+E$~SPhLh<ee(!zgendvepC;pd+N*I)FK;00`#ItOVml2bpOV)m~C|)KP{L4`@5E8zk-eG@0-s(~ZzE#fdk4{wcB%g_;*@Q30(N)5tZ;{<1W1b&q<^eRbQi+BWg_r3Z&#w8__yeVJri(#Mvmyi-&<c+Q-R)mO|cd_9Kt+c?QtEQ^<k`vW_p>JzZ7K{HO;Gh)J<x9>G5p<vwn9T7C5GJ3CLhUm%C$W%d33RC<jOzg-}0v}2oQ6fj`b>iX@Q3ts~C_N<xP?A@lEHZ}|?Z7rJWjWws{7V#}+x5D1|Fon{SG)(YEbyyNKMKV|``%#UTu3<Snxl_$o9w!{3UX6slwBH(%~3ZeN09>oh=#W`XcL0NH`a~m-oBf3aLh`#7(Us1qm`DVd>bnDc2;3w8HVASNwB4D!{TP7FS67{`!~a*&i-OJ<I~yUfgp;F<=f?r67CmDo9^E-_@&TjFit|EK|6{<3F-PopJ7<YR3JK98AT9`;TVRLi9&HE0>tda1jTmo7ld~GgRL^=>1~!)O3jirMK=7UcW5#w;~>`^IelT65<+4-kP=VNrJf<NEU}2{-oJnBVk8mH;ozFdz-1ujP&vzNr(<avj9Z{7fK(b2KN34r1(4|I%zUc!)YVBSNS^O0Fwec73yK{nmB=HPv;@Jz?@+@a?@&grCrVU@wg))unkQ(ii;OHRT*XHo$Pr=rA~CU0;v7WInPk=ul-=aqEIIf&0Mzx9s&UqdoJbF*9sxs1k@*MYUvwIADABhiH4ban9&4u7MRuZd%&BO`!Ei=J%2g8zPZA0fl-zR{loROt5VYX&vDNP#_B>PPyPzvu%kQt;!o=Db4Y_ry?_A0*CDiryX|;G^)SAuPcsmBuP=+tkpp!0c6Rc(l_xunh_}YpgnZUMhmqkn>f0DzA*roK7jTAlaSlaEQ=M}-Z5Bfs#!H*SsdDz<<$9`LAVt^OBy|K5CY4<vgjXzqEgne!HxV-Qvqc#eJ$#%PCihpyqOSDJFa0uufjd2m;u@MN8_y=H#^eH3ZCGFTn9b|Fn4+*kAs-pNIk6Z~}MO`uQSbX%6JjY)cl4+0t1o4IzS$TTxjCgAZN(7>aLw*loSZv29k=Rf|v2$AHg&|P;5Y2%M1EO~J@1w6_-E{Fb+EGr#Z_N~L+{hc4$vE4hv^JTGOeMKVfh(8^Hrht8`l50$N>K}NDr7Ji!K%2*Y&#Qn90#FdJA_OLAUtrKk%&|opA_DQfO(4^qN`4a{eZo=)qi0GGa~5jZ^=ktyiSj9?0Ltg7-NnsTD!;RPCUEq55EOs`Je|BEz=nfM*~}cBT-THJIsN6U&!Y0rl$C7=xFO$u2ompE6Yv10D);^n)j@G<x)LrqY9`1_G$Sm=AJ+BFuvsQzQKQ{UjevtdtL@Tbd?2YnS&AN3xgy=gaRnLZd{;yAY1;OV**(8CJs)wVG5x(8T$pev#`ngJ-qmpVhgV*#tF|JV`Nc#(&>3cypcvBJ3AU0aA53|ab0XwjU9@}s-q$rAICc0jQ?6AcPiChy-``K^J%P9td~GjAC5Z=s&Ipm1hCFvc!>0NJ9!7Ft5J+cOcKU%K5^~Sb%5<c&M4&ZKmL=?CEZwS!?egW9~t5-Q3M)Q>vE*^cq@CuVaMdDN+PNKebBb@PHC1AE%721w#PE;yD;)%vL?;0JAKbKGu-tyR%^}p-SRW+uw6Hkte?9mqz*hYL+!kaXntrv=p91O1!rc_$mjBlm(|9(t!y@j3E7ewji5lFC()dH13CaI(u1K#;0mf?_(h8CY5+o1Z|w3hb^t0=kwB89Aj;O_sO{;+E+&)71QX4o%GBv`emP4R-MJ7!Aje-b*)+46sLh)a^))2HqJ<Eg{DxGaA^=X*oE-sMHbPb+^D=x8B+RE}*lrtOPR44^YtMg?)U@IZN*z>8HUX8|R14&KNHX)SB|6cf>6_$8fF*O`VPd}58=S$NzKSv5H0Sdw+S7$Gr8ydGXq3UnnBy9tk%^6Ohw!VlMCc>FM6s-uAb%-bb>xgKBo{#SiE898r&9*hQlPwx(JtX%4J%*-9!w8`I-j^}AEBh==G7hBbTF>n-v^@0<BjB1V68O(f3s9y18uhf+AuK3<wkJfUq7MooEpA?6uaa3nkV$nDW(D+0BJ=t1IUMRntY8(8<;2RB0*_?F~yhg;udrB-H;NRaPMylWTX7_g42!UnXAWGmBs)b1Y4f3A%TzqZ(X*sXbP^lvjiP~t!6q-7xQh9NeJ1cf65V;y>~ITy$x#Q1o-FJvw<8P!`$@11IJpf*FYOZuFkQZOLa^R8*NA|+7UE>XXPy=xKOw#dJYR=^&D9vECjY%F;OX^nM;F3M)&*C4gY6KR0#d-FBC6_(~>=+C>LLAAjuVVIyA)k=*_^+^gmF_9}e~We2H6EwFoKLKn34(9g1PdYZ8m#!;DUW^Z7AF?r8$D!lDm4!~5(4Al<o8C_o9bvbKa{oS~EX5lUnz{<Qa<_JNDfM~F49(+6Au8mcyp9+Q}XSVIsqVu36bAYb3!g>8`28eDle5Fsj^2R6*YaWP14dw2*0!QKNL!!smBdJ_1K$>*%QmFi0EZo@qBiX`OW5eWutBXh%My|RI~^wg4iYuiI_JQRlL0Fp{_la0__5Yf$YX}yg2X!&tOLgygwAc(XMMgd04HA)~D%Qq0itoFV=Kq(Dqf#!Sq$dkk#D*YV#a_rcBwOz#_)%GP*0w^6XdD?K?+v^QX{xY*~_y=hY@<SUJB><or)pN79DJ~e8zI2}~+@HI7RB<z|+NnKh%EQmmM`+<_$_9SJ!$mCtBcvmI@=>6GCYdb~=&m|1W&P-Gti0HTdYI^!9S^w8=oL^LE-f1LMdMA7t#^;D{ZN<<^@``dkEFM2^~!H*)%2X%m<o~xVVX&^nsU2S%4rdNi0M?$M;ZJ#Yb0_jDZZ!oGoWjqvNs`)rbGltzaLLR`D8pso~vRNj$3_BAAZOH(wYJn-eNzclNp@pe)i&hF>J)VKrV)b%y9n(<{VWv$o}xYsrLp02X>(Jm@@U@z(IKh{z0qYzCFT_`e`8%0OPiKfFNVU%e<r2Nl4cmyS=Yb&`~_FcSbhK4gF(^*5YD3aOC?F*U^SuZNDHiUB=t_+5N>_3QijX#A+Nx%089<OC`dA!o(=A7X?ue&rCQ-^AQ*;X2I|Fv#HKUy6@y+F`wcZ184$s72%Opy^Jw<8cq0T#cJNIvEU-uO}blCaOhRL)hO5R;E6pRdEb=BCmWRp-h|fA9UHL?OL${i$UdM^q)35OxLL0Awmgq4h`zBlp!L=|R|i_NRsk)mx+17t$m*NU8$;S!YOzQn?bJoa&6fUEtSIidLf~E2!e<|ppw$nKW9TI#g9-V7dq4=_H3&+?i04@P=GajVs1z2=0@Gw5DH(f#02oK(p*L&~@wNyux1QHOHiU4)+XX}^cfZ&5^tkK8MbDv9hArcyq7z#>%z#2r2nzrml24tLaW`}VMWpvU5J`=yCP3iJY`kgcHtV<ttsHCnW<d&?MeQVkS)WFZ&wf%&vi8;axjCoZe%ny=k(g;B<7ZJO5)S-})zD*x$OBB#E<d4!FQ3wt+vlA6Q3*u$E#(79iBk40-;n)61b2P(3&$y2Y|l9z`&#E`#wlB*@MJPKTx09juglBL8UHSV@DU86l~Jh=)@SLqk+yuhjJ|poysJF@x^XPtpR!=#3=GQZaoAhVPLh;lhXS7Zu?EA*q5p$cnRE60Nuv<7oY2`{y-*(7;1;!(S)9vi0xy&Z)-sS`Bzo#I&IJ2JLIPWU1aNr5*$rg<Tr9#HX1JnM%Hgz(q*74wL<%G`mcjHf)#N6>!AilSa?N~Yqh^&q1nyQXty>?I$ejRf)Zq`(*FXARL>6s?)FdiWTV1UzSMdHA2V&eRn|>&4pql%GVb1krz)^R0w{tku<HM?#Fy%m#uEU*x2EA6$jXC~jzKAgfjH^Wq4q#j_Vi*JC27fEu;E(gNtSCG$zAxbW1>Lxor`o4!@K`A~Dr?pFZERL#^`dK_t4nWh5ZsH}3vLlm_Tkz3&_!mY9rbL@pLpzKnjBW-eP#8Km}DEe^kgU=^U&T$6ZEi0RuR9JCsl2X`PA=I<Qo^0gV_Dy-UJ>0rJApBJl8q2f%CLMzr_qSn5hP}+xP_#6zW`K4~c%lGEKjfERwsf_XY>c!#<7@n4E%`+4FO~x|0zG`^ovQMP9xAvD0-!cqkjdpqwD!(Iq+9ubfd2ei*jrjooZ=*f#hMjM+|8q&jA&*Y+~pLaK(=VV@-&nlAFg(J-YzOZzxC8Hk*xooz1K%BJ35wqW|BKA9ZJ$TsfB<>_#Qrj*?n1ZgTP8dvjK40A;S2w%}7@hYm_F?BH`%a>i^_qnd)bTZ?d=RAl8teLQzAr~VEWFO+2v6Jh9tbnJTW2O%80eRT|-*$G~ot>Ra@7`=(c`x%`ac3v@ac`8>jR?F%U)k7IaCg%8VoEmQt<z4y+a&)>SY2m4Fvnee7StCQh}ifK{rl-zqF|PIe**uWTDi;b@qiQ3fJpzPZ6y}XgkoShITtp1dp{$3HICRvGC5S(QPz+01z0NX6XlL=JVR~whZFQa@bp%avbPG*#w?_0amS2RE=CGts^U`!&G%*QqkqmZTYcGDC|I76)HdBr#jD`GLulLPlvsSZhB8!X97nUrVfta6M#BO0ayW54pSWo~FPEB}Qot`|Q<7;GlQM`1*lKhP3eA=cB5VeC9Kyvw=f)_6lsKLifYx3t@GRX<Km$-h+u6jxL2B47F+6G(TKts>{3+c=aCm42STWXBMBCZvT(N{H8XBA)UhVL(=%byiv0Xeo<Xgi)sjr0EW&;;IWUuI!e#Nf=6_8|Qi>)BVpjxjmZ=s41-w%4WmAIaAEqS|cL)jLkTNLk&1LxT6+lRXyTf;&DB#tt3d4Ryl)l(n|Zk9bXKr0OIqgXki`|2q<9%vL7AiHREkrPBPQFaa*LX-}VCMkjb;!7q>m7_^wR6lrqf!!{!m1T?>*P9XBSxO*`ywDaUGSI`4&Q=}%t2XOfjfCKuA5&=3P^8?hjuM*%`wVE6-M46|iRd9W@Jy<b7=1O8h_csl5(@`AE9X5((xACgC@?SilVgj)Yoz)3g4_&|TV!OgQB$9I-I?op_E)${Nm2m!AmWiF-(Cn0B9VuT)1%Z5#%6wLKN{LNo1DTbCF*A+?5C0`hN3TC&u7)pjKJm3XY=t`g*Xc(SYc-h)uKp+lP=u?`{{uS1$d#MDSjRygX;Lol=!O&kqHOmcTF<Fg!cgjDCF1(L1@D0X8BEJ(`r<HQ|3^G=S3*Qxs3lVvsjo@=Kro7>;=LwiV?f}p#S%Cot}GOO)w=O`4(q$-Ld1`WP4JjYOPcn?^~<&a%lV)yn6|%Lqz|<#u}y7a`T?mXx28*o9ezyV1+YV>UPczoI4I8aEe(12|-CA*K;lLldvxvT?0JzhIL3A5$!w4r)Zagnt+6WM#Pi=H%p8fq~?B8(}hmi(~&TR%K9bDj0YB-+Hz_6b~$Z=N)8_pq*BaIT<VU-rKU|RyZ%jYv>Nn_d->yHY7ESeLArLmkI~s!Mm|j12+Q^5B&hWU-604ZJt7D_BnKbo@EyD7$qn8^XmomGLBM-CnRisibXBsG_(9wg<q7PL9E388-0-0Iz;7`<<2lt$*@ubtk>ETcumm`5^;S1ikkwj!qtv986t^;_U4ssy_%uOkR&;%kW-zO)IBv$XXAM9GJ!|4_n8tA&mNZ?va)GU`gZ=~!&n`Fa`1dk}Lx+FtBd&?js*qVi4v0Jtr%Zx};##0qjPL-3BIR`^ha)$$9V~A$R%{pN@~xJ^_-EN}yVn!0v!`kFgN3(zDiR}^rS;9GlKYMF#;tO_L5C<p8^r7F;eyRt4dbxglckeTp61kfU@!>6@d<bg#i;?&xgNoYuJpf~GT^Q1W{ECDr*kO32oHBUqv|-(JP3)>opl0}E9U1rrzz<aBHId^4GCiOA{@2gS7vn5Q}`jp^mkH%`cJ7gH6U|9BXcC`@OX$ey5qJH57pdpy`o^n=mLQQAMXAxgN5M}y7=sYu&l~D7m`OX^lcb6W(+pbRThhT>t3_mIFB`lLJvEMA%#;J-dlv_4dDUceDnq@b-hGB{`fg`?=-oUf#c8>baA8SCq)kw0i33yEBB*B7=$rSlg{3XGw@34MP)@Pf49Oi_(GqW;=u6ZMg~_qaa7yI`NbBH#HHz?G2YAW>ew6y>@5~)8^b?xAM0u42)2wFzs2brNZ7Ki`)B0Q;9S10Je5<p4yCRZt@)q~rmck8!D$<zyPriupV&R<sT~H!nj*@6dLT<ta2++QwR|UYy&&37Yg>l#DV}f`pQ0|Wl?GIliL1f6Jc<<B8=E~ny@==`?$QhVq9uxk5AStE!0`0#9S&Q(O=gRhY&3pHU)dtmqYUH`iW5={J%vYjk(tReq}1!f@?-`7G2Uhsr>vLYV16wct{M<sG?Th<`NpxR77hV5rCWbPqg+B1CK3zCcalYC3_%mbnW}!8{RL#y^C^)CpEz~zMpkGFFhZ-BHUpu%T!}h+3-L~TP7J6aTq(*`3~LcUfe2J9^~&!>!Y|<2ikW!cvUE|1=M}K}CofiQvs@+eKA`rjF1k!JO5k&%uuB=xcY8)gv@I--VIgzMaa=qgmai%M!Ht-xLoYK0@>EV}4nrKN$K=|H7|*GmPC8ua_shfs=lIL;RRTnmJV<z>MUx_dh~T-jfzDx>(?!LBImHZ2W5KCyi-(q2qN%p|fobN87^tdC(Dy;*gOx6(`5D!s`kn|!1ITe`i<@kEatJH+f8iX4(?iJ%hYRTe6It}4b}~4XqoPEHVNaT;XRp!*a^aw4zcgM57D(ZH#C`sA>|?%SeqJQ9xrO9NWNJjRu3i@MfR?=Cu&BS|q}<v`tV4u&5cupmbeFQ;7Y6T7A!SmE@F|)aSttdVKO2Y?&W%7O1uEsdax(B8IWHL3ElTz<rzUKMzlE`v^CtM1kDrPhv?7T(hKOKH$e{?{QD2gn=Y{|`?oAX96t5Bi{qWf)zf{gO@v6qybvl{Eg-tes$;L$IS<}R>Dacy0S>7ONARBWs3Cm%eoXN#}OcVqkU+J9Iv27#Ml*ez;cJT1m0@h&Bv9ZMw$1HKZ0p&B1Q}E|Hp+w;kUWxOd#U7F&@j{*x#3H7s=%EVl*_-D4xslg=>JTOR=1FS1c*Tv^&c*D0bWa*pSVAI^>3-)@vAA<7yFJ%pWO1W$_J~ufk)(Gnsk$T7&0?H4irEEa&<UJ9)@=+gs$#zIC&p;S<)+AC;ztL5i@ZG~^VsSVtFuma+p+9eJ|7uI-?9^h%Y@NJG@;<vChmuNwV|0>EIp4-*+)JBR{8@v2|g5$oG}F+U&}AK6U>Tsig{Sb=kL)&Ou^0xTON|nsmy!)?9Csj8DPd=lPBb-qFsK0sh=N0^H0ydS3BYiDo^NN@?fRx(8SjxAF0)6I5f)Y5>$UJ(l28kVI0Wghn#o?y&fOF`CQV-vsXBm>!*~b`g`o^Ar$8h=u=_w6k7QF&6j#C;gC+1U_|r}jOD9`XD`Sz_7j@<Cl5ucn%99)BFBQ+gkK8J(MQ+@ja2w+A{@T+Uy-9F4)w`HY6aIIRPI}%?v#w0ygUE!@XbF$8O&czy{lzkkmKkh%9lf_W)S+Yrsx|Avy|g~uE#PV>6DX)If+Pi{qW6~G?5>8`NhMVKR={|*`J}iDnl2ufg@(TWS?a6lzsJ3q{#W>Ll`+rY)>$;IKT}TQ}~q<0<mm|v-?5d!;6Q|;!7CN=ZulK_zy8xH?>U=#lbwszld*7XylIxnqScGS6KW<N(BTc^8y=!<@pRzLyzTk(xE53<@g=W>md#MQ;F_RXn`ne4=(uUP!y91JrNI#W%QNi|6Qo504WiugmC#&bu+ynfWOjXxukS43U)C5no*ZivA<Gh_JlVb4CP1Z_bqmaOr(V62`dr3pwcIVuDF|?JrvmX6|NtR?)quwy&{YV?}q}AL%z>hT$IP;*Zh)pBy0;oDrgv=(B`Ane1RF5aF@it!{8soe7*-f;#~uS`aR}DBFe&JM7QKOpQ+|hd7aqe)FO%WZ9AjixA!phj2<R!Vygq>CFzS00T^I}G=^y5HI)?dU<J6KNxvirAQ{~`9QB9CY+2Ob{E^Z+{lLD}ydt6+Ls^i?hssdVH~TB~1#O4yL%*%pKe+{kD!%!%9DbBINBWsdC?S9TQP9?x%nTH|FB^Y_D2!`Nu-V1H_8`#W`n<$_A$y3C{Dk$1mzl~&#7_MOCX=#}m$-96+(dqf<>->!k<xmJt@KO)keu@+_lVV`x%LJ}j_V0Lh_lQWBInm<@;-rt*khs=-?3f_?U0KzrpBYLiG$&711i37D)uoEm-w_6<KHoh8B+tsNjV8Q7VdNe(uIZhXnJklO#;LH)j(DP<Nlm^t@tTH=`XrJ3B1@(=N{rBIa0}2DrI0|NNWaMGwBP#{Wa0?9|LG9gj6N8uYIQX2nkudbbwy?7ui2%bc^>J{7jtO*)u(xkf<=DiyRJ?>kZonIPdQytpsK%=*Pq0<&9I74)a1jB_jV6?}hjaHOF!cvEV4-mfr~Avj0eaL;&GqIhlE;AgYLi&4vV=>_e*Yl$Q-_&4p@33a5!f0$F9PAAQ#JIy|>fbD??I?Kaa5V&k8``6DdYGhicMhNjLU+AApY9a3r}gYoaQdO)#aU#LSoH0^mk#`<4WW=j&yVp3v~tQKGXRQWH4jq{9+j@|e6X-26MS)vFQ5!2MiKc-1tQ6Vrc#;KAU6%Kd{vrr`)=Pw|XVheyVk}dMjuW-gs>7Er5jBlx9k@gd?Oc6AF;=FL0A<X-P$QSJhZk)*<uFPX*8eicCPzYm|h8obdkT9x<sn~)~+lb`7CaEL7u8Xv;poWio9;mo>-?GP!Eh+U|$a9h6*H7azBC{b(nSt$JNC0FW1VY|-+tAuSw<6L{xU~P%=k%)lNWF__rM)8#oEa7y69h+J#@vS0d)7)6k|ihm6e}bU@&y8MaBl35I5-ot;y^#3b$UX5e<!2>Nn4^K7-I*?<O0=$<|%&3MDWN6&pAh-h;zzsVk}<eeu5<f_UvJ1xdELV^<~dtRY%coLb^|as-q|yzb72}ObC6Lpvc;_`}UwMWdI5}^9R}{0$Y?9PlyzeqVNq(mNlM`@Ff);WdN{(ioW3Ic%vx!WmAXNeQ;bqC6a_PC?O#3yJtc+d4hWDpXfR!iJ_kng?pj!e_>=GsW2qq$IC<|8l^T7T%tzGun=!0yJb50gS;B*Gt#+mDLDj|Bf<^5CJ6%3?GKU%EAxIKaqPN^903UHWxQ8wRcqCaa;dRZFK?8qO;aav|1&~*#Azg7k`16_s<eAe>}4X~zJ2Ez>!U>3UeKC<%~<_RTtbzL3>%53@Qg^gc=3Wb$)|)6zgOc5>*pra^u2B}eBvUla0VSO!&dZSDS3Kc-=RA$likc%-`R1m00wJ1I;>~NWi!{dU8g(jcQV<m41BvZ8IYF)wWt(G^v|a*(Dy937m^5gg9`c?H`&*fQpp8n^jHrxQF*Kx)uE$pSxOj*#0;LYLDNKin~tu7QAc)M@_0-2dhM>&EPaS^!J+WhGDnA~*FL;=erGzR2B#EVV(Fw4ZZXw$R=sZXzli#wC&OlCm5+lNepYFtwiR1aF)?*gZ@whP2HZsn2#`QL9)@XypLJRX%ugE1GIp;FC^C|Fv^$GM&DX)1D4=%qwY-B)NxfNH$AsUqBnqBiv#16iXluX3y|2ZI1wzlzFxJ7Vq}%o=QoRL&j$?W_uFN89G|El-h<2WRm{g5)^$Y6pa=GJ6DO2KxtOa)#Eeu`IXyTT(@Jy~=St&QHTU(7vwcKb-*(dYyIjgd)(BR@yYve_Mw+j+FA{GE^y&+3%_My;J#WSflF-ysOyxR-F#h1tdCbw}@G-~3GaGkFo+^#h?E6vh+3_0aFDMwOx&CwW0y`+46nWSR|Mn@?7B}qXf=4v+!i&<SebZqp2b+txmz1-lOzW0aY183~MB?|tSGOUNnm+xqy*nA_u;u3tz>VQ1<P5k;j%PO$U!S@S5TCvU16s<e8^{ovCa(QEOy>>5w!MvlDIK~nlwP8e4%!$BjCZU7(GU(3j#2X%hRtF{e$J+kT9rZBhac|Ha+iao$@aE#DYn|btA?_O|u`uBuSUiVH${WjJx{Gb=$w{hxme|<3)o5aucghy$uSgOznzbg~McTlOs9WWRd4jt3r%VFL07#1=9_WkzBStPk88LK935{{Z`+cbfe}w@Ug9o6r>X)agwVVie9j`{gn95a7)|?@?<D*^&(~*c^|Gs6egIc#xx;^`W3{o!A)cd77&I7!bcPL6>Rw>XwF;T#|3Etg)ukB=fnXA(~j5#mksXViIA+f~RyjUJigR)#|me*?adsbt!yv#m{uwu)#_4U$bBg(HRYdyA^1e;|1q2f4Bz7CCktMIUW&{BmuDfG!9M^T|T^;!sr_$4{c_p9U<Xqf&8E&a+tBBC-Rs4%7+i_FWCS+7N-=mUGJ5#GqLEO!a=GKC~bOl^DiZ&5h|DU%{;Aku0WKE@Y+0CT^<n*R^WEoYDP>~_8tT;THe_;RVr!Ja;3VI*{e;39*il(*Q`f0gE5AoLXU=27HmhQyZ0nGwH~rte3j_>)eIm7dd}fym1fr|78fme$G_9`Jl9A8#61rTLD!r_&|j@PoEMIUe8G7ah>;a%sJJ`)xoNI9K3^rt(v=yTnWnGA2>tfjx+I89rOm!iP27;LAXu)~22GNF(#vqpYC?c70mtr1Q5of7YUQ8Sd}}y?9NAr!QoeXbSie#i2K!X@x~C0`g>xP+4hY4Vbk?cxrq_LfP1vZYFKM6G^^{`rKT8L8aV9{f2&u<B5-fXhj34LKr<ouBC8cKd_qKv>%9YiiU<Lk9YmhanE4B4-@801NL}4yiYV%qk(IC6Pp6MB|58w!?}gBC{QN*$zdibn5qb{o;4_&dC;D4^IzU-)>c=oTY&SN?Potr-f9?<82>vyW(T9=<Fsh4vV-i*{tP*1ua-+qP_nJfty=&&j!C#)TGNHAQc_e@HFhs0xfjlebU_ZMN}(j;Lk>D$iqwPOR4=(X9;keaR~D*rLTZ>5zkrCM<<G2C=kQsuuw7U=k1ng=3*kieK}jtR__0Fa)~HUSyk1^zmRIn%|0T#Kv9<_2skBzDH83DJRe(~#cL9X*dS$J0tFm5c-ov*6^NZ(uTra_D5fZ~pOO0C9BxgVn)#zSQnlSPaGVvo{jM%xP?_9dV@BFK0^wNO8m`LghJw+O!F!7mqZWuE9xc)tH3a~F?<mfv)4=}i;5E70I9aKCFSSMZdmQmcUrbPaLA?R0*f}kewXyC9e>DepzkTl)K2kSBfBSRdX%O92j(L#wn0D3?T;9GJ$>F~_mkNVblGRVNc242EFbWLK}mwi*3sp|$_dc%JyNiF^uu@ktsC>zbqIo@+kx?IaGid2nDUf#qo5xK)5z-&0^wWId4WKVbzSxL?Rg`>q@vP*am%j6QduH&`$Veov@TMUtXA4Q2yH!18y0vr>s1xYzqr#pajmkNLJZJIrL1OG(xJQ}*ww@0qiu?|I`*Gv=&GhRM>8AGLFQ?MZNeo!Pv2Ykv==l};Tll#sCaZfDq?uJx|06sr8-y&rxhfV`)Y399NHnX=svL5dBy#2|p1zMPu-L_{w&d=PiT4+h#u9sIzNH|)6E*~XD3F?slgZcA7OVmara=bwDM7)CJx}v2qeZza-dE}-rN_Z>gN}0Xo>$hGAyk>`<y&hgh$5Fsgw`nv-!B~PIT!_N0x5AIpWbBnxL82{#IJ-pd>sTNFm!@A)ZpWV-6geo^fk7kUiqeyrG^Ssf1fkQU8}jMbJ|Z`CBj$U$kcRmTGIo@BwA<!6IbW0MDc`mweN+A%Kx4x7dAN($8!WX@_)Sn2$zR&QL0JSSpvyC=%s*xDOR>boY5FqaM_f(bKA5id`buuQ9cg)p2^LS3d&<IlJAR{Cy5zszwP!pU3B4FO<KD27cr`kl7(>YrD4X@O7|OJUq2&^<F_I97)BfXr52I8a`_B}_l0sL9-grZiimmS)IeqcF(?g}7GWtYvI)?Ko(oZWgDlusBB%AezY*UXSKqr#8Ltz%CBy2G0PRKG7ehR}#eI<Iuh`bC~OWbxjP23^dZOu=m2)GV-(w4FdqDHEOaD*r2AN4Z;58K7yl(EQ<+<tclgyMT3MK+q5dU<obv|Nt;+XU?(LnThAxzk((H?&eiyHlz!-!9d&m<vJE4Gf-UXzl%Rrf@Z%6^}4)_>+TO1r=6mAi|fks!$gRq_#I6S-$p#7-8{h#CPzX>FdtCMZ27>XaGf%!ODs?#(`y}e{Df;k;xeJMtKeJp0%;H-mHidIHoh(dt5BZ1}aLVt_dD43=r4lJqClqt0e}6!^kutg1ZD%>g`i&BVBQhrAY^iAo862q{s-I<5P)x(DvzD=zdHzpeIlm6VPnAhB|i$o$TC+kl~{gDSH~lDPUKZh|Wezs!V`&g$N)`(FMREyL3D@bsyjDoPI_2Pm=>79=okno8`4~JsUAx5{0lpK&Ad26BHEK)6RpDCBhJ9nCiEOm=Fojv~9b%UZB~?UP816bsBY+cpnO98jo}RP*|kNWFy_Ggmcd4=rbii6KX!hNt^Ypax6R)$cSB3UN9{$s*Tz?D?_LfSs8o?mxx{JJEv8`x2dg?SHKCb_G$P7zKV567Rn6l;bx^8ozp?M8|r6ZDXgRCxaT!^ay&j26)EKZdF7$+vUc}&xk@#)Tjl26GCm~R6kveWmDopNS%}GzO>B1B3?DuOsr%tWCf?Y+@=+^SS7;(*r;WFiF;P`)GBB=;KjXt5&)I+hDR1ou^fp>`wO-rMLOGk?107uZb*)m>{LEO|u2!2q1#>q@a+P4hCvd?~pjb5JO0+aALcL6d-!;=9%<V<CdR%_TQWx6%7ejd7`tM~E`<c2ID-Eq$YiiZ4_4R1wwR&x9Q@eFfD;RUeymq^Er&3*uVkf~nd(%AeqXJf{D`;4&)q(p}E6bJ5(t4$dS=U-IFx2WR<vR8dp#YJs8{V)?aT2+*YS!3BjgMpA$*XkYjIk9MIYJ`2v~&3B2J|VRvC$-IDT)E!)@OUCeZdnkwd>?#F+5BPpB|Nj!8gKLh6e@<foAlOE;K(HSEBFI@Sqqv$KuvvMKR?`aF*t4B$;zhALHkhsppZ|DTmw28Cm2!jF0H1oEpYk*s3;~nA1icH<s4c>g6@E2&bAIyEcX)hgscc;rJJmx|R^4W=zpc4DKidGF&$j%FrzNi%DdGJyY0nIWSC=_hYD`5y(O^R+K8AAuXw_CW>jDs9c6(z5YMG{DMV#pH2ZFME0{@HurJ2*Y9WMve{D*Qz-DA(Fq45A;eA{;T8T#LGt;RxE>ytGodHX8KYD~l=EkaoG>->z4K*GJyA*&1IdVi{QpKyjM=^I`xlTAQM8T=a6Z?Lhj~y$@RP6veyR%K_&dtfjT5Qba#?sfVYG;I=?h9A2~tH=CP|&>1*DS;PWA!<jF4b{f|w4NoHn?(7S7H0w3A@7ol0p*JWgTnfK0^03_6ernn-2SoSY<J)41@=7g=*AVZ-+sPFAG%`omqjuf@)(!le?Au`DK}{yCZ++Z#L`iZRJ*Zj`^o6Z(qd8b!Pb%;vr`bL&cG+jm?hgM;6PG&01<C0ly7r<33ynCcj_S5nMVroPc+C>8@hU=gB+EID5Hx-3H^lEBf>HT6LJ)UWt$U>a5!lEIyG=qT#`N&&U8sja$`L(CdxDQfH8O7pgMgvgmW0hoh2B8b$99DbZ8q-tl;&3BHD0CZ=X@ExdUK6y}ev1n(BdfCA8Fr03(Vs`O~{IV*ipc|3t|NN;=Q$qhIE?P4Yg3%S>Tz|w~j<gfN_e8Q6J{3sR7ZRek5lOV*Kco^}olAP--eKlMY{Bhb>lAhiFtLoXw?5)S?;v7}@ZX5~!DV3wXZ0z97;VF}lZu3R+JQw(Cm>6aQke*~slHInQVEG~>8cXBWfTlNTVv%EmNM{}5wi&=K$?;uRv9Jrx|H5YOkLxE&&w1SFxnmYhZ0fJ1@U2Iy_{wIfF)8W7l2;Y=+m9<gvKb}t5k*k=8b@Pr0KtO-cLMHXu=FA8c%TePqBF8DPSpP_gl?_*saqMLWzFGhj4T~Sfve3%kS-{Pm7&Y)-O6g&pvQWbUKpbF&F_N#Uj2gi5>-^(M1KXB8^n%<(2UoX%*q6$eV_)*Z#?dUi3!<ZMh=~=!R))4={A7g9+B~wqp?yRhFEgujqInBfwc|+oHpnpke<6-j`~aEU-1=0+fohRDUT=nVB~5F}k(2g5Fy)S{22C6~%8dq(znThqp^x4f5lrL-5LaQ$$=$G;PR661Ma-q3vy_$76%)0Rjluyi{ujdEEGaq8oV>+Dn(;dJ|pI1M^ylJ8CPS(i*C4KyNgP+pfJAqbMUYpe3Womb<czh}3$^omVea-=`e9u?2%3a?at%J1$-TM#zECM-kQp>v=<q^5g}nXOo;;`sPW8oI#-GcCwHwTmh4rTY@|@JTUI>ADh!{f^I)ODa7y~#kV}-fp-C7qLL8Oy^0)hx8t;XE=Bu;p$h5}N|@mG-zFDAdT7%g9#eV@T@}ZG@x7a1BCg$PSR`=iEwqHis*<3TQkT3;W$h`sdvHrcL<?ONh0OQ#bhRd`R750>#H@Ff4~yE$q<ye*OXj-Z^mm*SSEG)#ol9he+qneDtL-}4N0T97N<un13bi^_YcbeEw*#+|mMbG@NMKKi*+fvT71F3U<UpV$um1V1CF2a91VKL}2m6yxDI1a3Wi@oz=yn}UWIwgyWvLU`_mhgzOa;h%Ei-*SQPDhQiF`8YFIb3By8Z1=i*G_n3;M<BCWV|_7R*XdrX<v@pq}ZQK{^GSTwH-@ZqLgUvP-=04sG{96p4Z(*LO0>I#OabF>DPxHD%9#AM~)~?%QL>?DmJYmkEmI&|N$`%aV0anqOy=XD?e%Ov@gfJb6S%sOfZ43ghbciGm7mJ-<I56h-kLqn{!Eb8!NzQf584F1}T*e6UsK<zBJgFW(~>gxbF8s~O9+>M9fftN>LQGz(uoqU+oKXK0{YZC0B1m|!M`c0a%IDJaF*&&bjrwzvPYgpIMS`@4!fqr!-Uu26Ik(P~k8lT0f9up|B3<Ky2`jcwUY4w&u0&SX=$mfD+G*mIcH6tE~)KzQJt$*51FStoN2U|Ly5V?F{D$RHi|Yg^6Dt;7U73G%E|a?+(x8XJfw2JDG*ZJI(lS1t_zSn$#w`s$~P?7mdZt(8`rnE!~Ak8sL?DVH}Bq`$D0cZ`mFz-~LcNOUO$;Mi^_BuT`iKB3%0`e8pmZxM~157qUv4lO|)3q}7a74dVtX2dxjp`53*feC)+bo|F(Bl;snk?-=S5cP`s;*hlx<Q)H4g0^Dt6x%hsj(6XIR+DXdM#_#lZ%e~g?u<cA@Nqqs@zFn_?sZ&K4x96;?jC29*0b?34<>^i0C#GW>ch7BaMPS?<>;V!yI9EQTPXmPKeqj``9sP9p$_zaY%|+R9{IjwcR-U!Jif)Hlx=5knlKcRq*rv8K5LjkxM(mkrOX8!F%0q!hnG!IpFbq30^4|t5_jWeqg$m$8M#g@(^I$i0Izok>o%s09vN}2ZJsDlC@NmO5iBW+ljm$A-4~r~M=c{QQ@^)xJ&zvo1$`-@6e&DV>g;bpFswI!)`G}z{zXO^y6@~!VncLDiw(Uvdz{nGUh3x#P;eMiFKBb~3&x_-EQeLc*8YXAiw9poOLfNb3*K^!h3tCsb$EQvgPlvPC{VS(a9D;ne}ZNsF6kh5pu@21IpYjdE76YC>FS2~eW8mu>J1!yxwd)FS15#0BGP(RYt}UV3Y(MRKcJv!GS)JjqEHj*3A0HY%o~Q2LD6MN5j(J0Y}~6hZ<o<w7SR_Z6cUOPlQ_{mBr&F(_?)D>_XbC1)U6f<VNe>jwYkI`odI8mnk;a0MoD-IqioLkD5K!4&(;0i=eru=HPYt2r0zO4ac5ymFjJ2-{ZR+~8E>}}i~!%ynLCUr=7B;gv+XfOQ>G|Y7mP|&yW5!FYD`qi+5NsS9u7wMz$siqMZMwwN8P(`w{e~Mg0E6`x?L!%2$8as*r3Z<5D7^<=0zq*$!-`24H5u}2m}}alqiwR@+Ft<ai*Q5lj%&lJKag2Ip>dFXWBO<iZAk7Jug7MLY`#4%f8pH0w^h&GtP<#pmy#1{`PnMeT_b_*f_IuaQKQ>%o!%w`ATE4vaqyLUY@X?Q0)_OX{xow&6^Ze>ZhJ&g591b42m)^aqnrukWU#@ahEVSl)>zu;x0h!U0vK*x0Am5v>7YXxpjuj^nqk_N<lMNoV_yB>)ydYJB&>rZ5$;yY;wkRIJLpr>l-U;zKSs!Logb5dp%TtV0Q6?R=X7kh@VFc`FIqNEgDl341-+~*G8x<*&cNJ+n8Y2lewdPSCXPc9{(<KLZwSm$RGwq-B@xNVlnBlJDX8%BQ-ljrJfkiiu=%xxCE!Hg*|HO(5usbWx!%Qop>_;$@SGyd2=WT{tr~H?x)_)MIhZR^mSl##otkMW=-W~H*1@~{!V|7xSexv6Wy8VFO|$pOE_+#`RPlC5GTwlXSs5#vYdU_nCqT%-%zV}yQw=z^(58<%p$@LFA3E}uO@SGNIf*Jl+D%3n?f|pN@?4?#}ghm;@#%R=Ypa-mX-W^pIuRmb$N;_2>n*{EkuqhqF)(qD|$|q`YmOCL{*?NpQs_D@q{hPaLVI-4I-GRA^V(z*RGO|hUXe-s$)3JT`try^zNc<F~J2xls+;Ni%JnuZ@E%_ouKzL)80UTGj9@LOo-CEpRMdF9`q-j{b*jVTvp=Mbp0d|+w^tr+0j?5-N*gw!$@X)GSP+eZfPY2{92+b1riVt0q6|mK*k5ra5j%mKM5!IMhXWOVu<Pm`9Z_eSKvSSt8#=!`QRl_6C<FSqc&y<LNJczxKP6vix`o%gE^(KQo<y_Lp6J$oA$%HG&egJ)nh0=fOVxQ&RRS65s-^_&c(`B)M2gDFIuYjvtG4S1O8=G#pJpfXkYT7xqQasj8kDOA}?!9)l9cGp4*zVOXg*bKSWP@y*jmdWHl;s9)XS3W2p;NK-`>HW}}UtbAh5I^-}yGR}P{c#d0jMe+$M>xYZn}pcn%|v-$j{S59iCq$GhEYPlWcNe`Ej!~hMlAnFY0AmjIwRAY?b${N|FRfc@^#`KdpJ#iG1wVAnkX8-t=n(f3=e+?lP1dgJ3?)7H@0cOrBroCEHkF<XaJW5R{ximD$4Z=4vfs_QVEN(8mzIc`2H1;SXd2}$JU{6|ffx8`c+oJ%)9hHod$)f-`7|jRdGSJ<|<Yjn~&Ihj@i4MR0yg4UtKVLsFAeKhK%xl4%QfD#^3$sx^j#8K-GR*=f$(15`OwKC}b_LUqn(~3|Yvf-%S=8@=G#Aqx2{k-t(Tl^;{r-5j4W%1{J~|coa_eQXFvzugSs06X@s@B&rtcuRdFS_me0e&NlZ{@|m+85t7KqOt?SGAp1Eu;r7I4oQS_Z@;%K%U#!$7H?1sNo^wA{^&^$J>Bl5As%i5LFH<`QB@+i*fhwl+?Hmz8Q3yHB5i8JWc`gXL8ch}0XGjW^))OY#ji9i@uDWE6+bm#la~2da|q!3OO~944^nFmpS*<AN{oALf-tc7?oN{ezd-D1LlNs6dAGnQ0C)7<0m7faWN02AeIVv70gzlv_T6+9_|NvAA}76%);hY|nhmXV34z(@g9{iT?c1cK@Fw!!dL)TV(SF-nhXlI{4m=UEwf%Bj6<DaL`l^`-s-v^urKa!ScSK<xxxH^%u>{<Sd<<K<t};+Lcy`Kl`uEYU3(;7PEF$FMU5RJ1~ZxG^UR`G5<r;RvRa}H%c7-Jq}WFZvNP}W-ZPBN%sXuB*^JVYna+V;RU06unV5`J`zrnbI)`?-Kdl^7!-rV;Drq%j!OCYqjAniQiQTBn9tVJVwS?X&998MMowicQfc5^E6fPFG?0Z@kDp(GKVHZEPbiu)=A6Q~P<@1TP8IRw{>S=#v;gN)CDtLK_DEzUZXY)H_7RV0|9s27UcOD59VFOO(h~d+9T=gLNK|BxGbjtP=;%qwZvcbLAM&e*i4!d(v!$#$1v#pdzchhq!+ojsRqSV!-%du9h5H{`zqWS=qc%FK?Tupc`zLlH$qDXD?GZQ9Pb?j2JAm3nlRJW5pltP$KsabDVsE?2Z{vScO(wa?l+kBvQq2Fkn<4LHcK1gUmOzcFClewY{fo!b@dU=QXEs^d)Mj|Mh12wXs*RW13E}-99GmGZN?9v<MpgF*eJN0|KbmOAYb0e?wP)0_;lTbK-XVq;qj?rR?u^rQLAbs9?Y*=~raCLDn=6&|r3D$tmEATi`BYVkt2jf6?}gCJ*-KB$>{)1+ilxs4zRbbBjA|tind&EgFf$8%E8S_YfjC{Pv3M=X6TI}hu>S0%L;-P05?<R}US@MP{ccxFEg^?n4c+t*oo2TOH<h<9weNX<F7oV>X9h9TQ5uR7&oWo-BUgiiv9N=(6QDWD2EK;rMT&HE=Amry-VkKR#<+brE*J+X-8s*BLgX8us>!7&Cg4$^21$4@?##TLt<K_#H_F#5b9Vi(hY&Cn^;y?wZ~>{cpiD3Z27FY)KzudulZ)Rh28)CXicfGLxVKI+Q9pzAgFYgdR)#~woyVpcH<xZ!mVv{uJ5mZb<c3?S8CEuN4%88-*F8}^Q3~rkZ^%kunsPS?hZE*VWiVRYL8vxLiGq9sOTXK@?d?0GgeJ{fd2|v7s8)g<O5o+w_JFDR$-LA!N`Ho<Y_*OiV<&Gv?>(>k!@&;h%Td9AH;QK|fL;K-FuN4R;ptB(LH47w_X0|e_z+V%Jqie~|1&dD8|lN1)=)9f=yc<z9XDmwAJ3(5quua_d1A{(NgjTq0~p{N9QMRByiRxoPfl<+zvmrq7dWAe1)P_TvK)NkiBp?i;iE9uGf6+2+)55hnyuYAA+4uJd*cK=YFN5PZ-6O`VZ-h<)0joisF#crL@5!<i{+aeq=ynrW+}YAv{=EUC6wn*2@0F5ir>v17Vav7%j9iQI*)^g&%Mtl9_(arE>q9TjxiH*4<fFV4pyZrd>igKlwaCH1rL3LIfN^7(CG-**bMlAj1G$13hT*XMBuS`z)(D$r{AOt+RnV0O+XG7Yn`k*&t-$2Y69QVP~Ui|WOF<GIL-5*Qo5csb%c`bGzB*kbm~HgS5&H<3!%8UsP4-I7w!UKPawE+@W1f%mz;%;Qr&(;iBNvexmrc;8cI?34y7{x3X|%jay}SFW^>&<CT83qW)Z&Fvfx(5ZW1q~5q}Ahzvf9ZbEvjbd}w*i9Hx~|_HR-Jgs1$Fl5cS1o{QgM)I`Br*czY>5WVKmwMOHtbARjes-QGp`J`ZYcP{;Ini5%kM2(jG%)KafM<~YBr;A3OBEI*~7nF9GcC?V;zP<@JV$2S6XF<iSh3RN|w!vcQt@}*>5$UeSzm19um{JBK;qBK5M7g11DSR_Wm2qrE!2_!tMK3sWEM&zx0`u>;(@+iEXlu}KC$m|M0?Jm%02v*O(diPCR({C|sL^?ry(@oDDTIE1_JKdO<sVXE67tIOj>m^@*zDf{HZgm4fIZyFnFoYY6xn9bCaz-r>C&;`Y3U~sRvm+^!O5^Q<#+NkY8}QR&|Y(O++`nso^ciERk@JNW!`4G)L&jfm<VJk^>QZ0a^>82?lS8dtn~&~aOVVs8~ym?;*n9{B+R~bJ?EjeQ<u><E(i?tA}zgI_oParqltjN88tbe;P^B_pr6wu!D1m*pdK(6H%;Dby&`UDhTy<Qu(n|dmOamglx;%HpcgPdu=4Y!fzcUG^dyEvR1uRtYGR0l%lVm2*JJ-2I=>2v>io)=SSA<AmnY$~>VQl`qpbXc=yE=XK6ct&(|m$G!yTS-FWB1gQvTK;veIsncg6^aMmr(LSh0L)u`nrEJ&K*D7N|r{r_vd&e6u9D2c8beN<8W0@g3~nGa>Dgl7H(U5(!K??c-346&P9m08fEigQu|5Lz=O$WKCe!(wtjk`KR1awmMT%9}kyZO0(HhZ9-z`4fq{s4*7Yrt53n%Gf~?fpsjD*PEb!h)M^*L?0sa`7&UXbLrVRiCfZCIh<a+H4Z4$k5(VFCEKK~1owu7pLMQ<u=0$wRk|)l7dis@LSTVA=S+arA%~TdQtDqaUJgrQzGndcwkawjY9ssjw%w<~OuFo}88V<XwwH){yNesfxRlF6KZ;=+G)rC`b7Xt-oQ?nA@_)#h5c~c7<FQ`K(LJW-u>OqL~TMgrm$SSS9Jjl?xw0gY(V^guc)kb;4uo~)juM<GQ2|GAp1AObH+I}txXXi?#+0^a3GY8unc_E;*s3MpvswBS65gj2oG0}zaIJ^MQoWRH25_ss$5@|^zF$fXv3*)PQYe+?iv0?K?T|;3u?Jk0HGT7VFS*uF}ed+s6!AcjKJ39q;=Ol$RPPYi%4x`xFQb{3=;XUd;z>>he_kfol6C)4hWMT!(Rs>y%;vVXl^kb1>0UtLH;W7STiv>{1BPmd6%&16%MYV!Ehol83e}vJ#j)m8=W6z0Dc5~h$&O%y<iU)o8eftg~K8l=1$!w%mTciEy){-z1xI128OgIR==2pAMv|2G@u^;p;o(8odwlB3b{m&AH%#DRSO?ZL2CgJ4jiCI7&JjA~hLnJ_XdMSo^sghH2p@nA(1?>n{4W|?2!`VolHT>P~j@=F9GJsWQWlCs(2C#9xQF#+B4iwPIx?F?ug!xCyZpfsQ^`yUFytfOxzF>{n(x-3+cbP7U)3?VVopXt>xx_a<AO6t2RkMeE{>y=rTR-8Y;B}bM)I3eF>)s0+MnDncu*o*770#K8R@Ao~JQmGOh<haWQkr}R>6@mqu(rCk0;{2LX;#?=*H|+4f=-2{HQ@<cPE^MzxEp@TY#j@@9EOOy7YV7{hlWut6Jb*L8bqpnV(B<fgJA2TBuOcw4IWPFwsjR|Rz}xK>6?nx3c2koBc-oQ$`+~PA=s|y+<Ry_Jm@!D&3*Ld&_<wp-Bz2em01&wZyN095zlP8M<*FfB+myi9-d6U&c2>#1r!RUq&D<SIZl>XEA<Mj78-L>AnWXY*GB`qyS}>SVH?x(wOb-~g%O9{(sXg8trK3;SEd!nG3P}Qfg8DaKGf%eTR6w|DDeq73HIXboGhG322Yg0t+FcX(p%n{Mvk&cod&JG@Q5-#WuVchJU~y<uKE7?S~k}KiSOEC#l&@0`y<I(F*yR>dSb2hP&^eVQX$DE?2DV^t%h*J%$QGYtu-o<7hoCb{CefqQss6!WGj6NGfi);!M%QO&}<?4?{1TRbaz!cPOs4y(L)n5+Q`(Gf}tpBexk9w211OONTALln3{onEflfrNugBSsWc=LJ0&L5-xb&$DytAj`%l*wy+A%W3|k1@gvL_)AvX_p&oQl^OiN>Jfj#v+;uP%^cZ9pL>bzjwSVoKA2sU=x!Ieuh!sA65(NV|OKHO~{jK<x&aCeU}h`xPsuRU%e`^aDsF>URBdmAmX!*1_BmJ_bK6ergoLu~==jAGv4Py+2T1jsh|F#}sg5rUpwP7_HSa<93!)oKQ#lAwkRo(KojB#z#lpi_0wD@^bu!Hn~0i6%FTl{bMGf)k26LO6Iwj;aM`-M4x^XgH7C&>Q7sdY?TQPPtKCS$n-AW5AAN3fizl<_fspk$2K+Pm;{|m@BU!QU$P~-FR$m9}E|lz7|w+geuIvCuFL)$$opN>OyA6p!o)qXDZo*|5T<%U{=H`T8?|6$3K*mvTQ<VP>5bwa{0{kwIiE#6=E_7!H8Ka7+!=VdX4r5ciQYZ7-hH(rWP~LYcOuLJ54m%qmm`hgRs+-IS5i~`Wl(wQq9)VOI^<_>Z~IaO!3C*CF!L|2_Iy)6bnq@xNl>8BceW{yJ5hUGGlA?{3JH0mMk^ZK;TW$<-GUYQQ4k%R`-=>YnI0=c}rd+!_OPbDMR6O{*}1VOTnn!7P{=b-F1Q6c5VhgTdy3$mEHenr&OzJBezVicrwocGq~xmzSX#q;{K;LiK`~Hzf^K`lD4t$8fi*|7a!2hwQJX?eV^`yU9HF!A^k#aw;s(KP1foe!H?K%+PYNAG!}0w_n7CBoe9>l2gx>TVko)5c1?ft0pId?q8C)Dlt+Et-wb&G&^$cIL7a@{`v-d}zTp9U2QEmw<pH0}f*B%nURGl}auhRk6qL78WdWfmZ^MPUcAKKia)KO?nLgN?0$`qoJrPVEOTs{3odQa8!fNW>q`2@GU)+wv#%@Cx@(MgVG4wx%HY0i_LO2iheD*G*PTc#LR4x%L&vpqmf~vb73#m3Mu1v6;tcdly^USl>E5FQcL&2xk>ht@9K`${nnFSd`Sm<41MLOL!i5>+44q#<*#&Np03;bq)Z74Vm-MoZrF25<&Qk;5+P~LMLFb4ZDy4s=k@uC)x2{h-ZX1Qb4J*{DkAh3TGmY-lNk@AYdU5ugP$jy1hdZifwfh@94t(i0eI$YWUPUcRIiv~sOP>Lcf9veI5L8<V-+$y!%`o+TB1$(p)tkFiX{l;89p0P*qz!*iIWgh32(D6tq(-rA94s8b`EJW<oA<7&)l)Osqu5)>-|LA;T_g5BDozC=3lF3~Q>3TW;p7@zn+EFaO3){c^Dy;pffT|kX|LZ<(j-0#D+ksDUz841Oh84;o&m}q`?oId<B;*&>I5L5XO(&k{!dz+gQtKp~$15jMBj8&qg$T^|k#tx%nGhppqj5H$2#!-;%>p?6^~zg~@PcT6Lh~rcLZ3xp)$7>fFyd|=(?7zqOVW9stSytu*IRP7kZ>Pyc2ej@>y=;!U|^e^AXvJz0)|aEWYcld73s{(&AZKRuZfBe0ohCnj><aaONL_NdAElvvOgRE6W#2S@94tXYPGU{t4t}H;Qr;$(8A5-&1z%0e4QlYHA0)li0I9$6v&AsNc4`wcBg}v%-y!ri^lK((PlcS#lJx5$KQTluDn?%G>AAm<o77&gk?$GU^TJAZZKu6-H;4_BToGwmcSyO@It3M6ed?)eVX0D6k=F?uQ^R&pxjs(y3ij{dhAk&apNGP;<Q#4=L5nldZXNsxFx7-l81J8Fzh~{lohr5N&d$7P>UK2xt`#OnD>zZ$<SbdXkN{Hd3{6qS>d2^4jdh}tcK_DY0Wbh)$p##c$mGZsq<UU;TP>IIqNF5MH-)1l;~Jk+9i8=BaL>>>@DsnrA%}KLG*Xpd2&?z(BN;3#T8?e=iazeboq>wBk~f2(KmP6=?a8MWnGq$zb@t7wXn8wbGfnsvZ$OflHqk<d`3hqJf$4Ykte{#w5~E^`;{{GhbL%R3<n)f@g>IPO>eBo3N3VGSzAwMJ@ZP<7_JtK?(|>-^o#v=cV~BNFdPNlJtVHd)n>1^F>D(`LlvZGq=wfkYbzDZ`20p0d1?8xJYLeQrXOssE|!6WqvOJLm`Gs?u9T}z+7BBcsnH!73T?U}3aJpHLjwP}8x^;kqjqP|YZVmM2VC>}z2<hi5T5-Mq4XYx{?KB|)I4k#M=%fDyM<v`_^*bO$^E}L)?fePIGvCjL5?vC`P`M0!naEr>G)VDCC85+vzI;w>mlxSG$D_96vl^9X3}V&_Xq7pr`O!U84=?U6Cp>hCxlfR_zFw|;oUNn`hD`Bfw8=M_8vt}eiUZ6INAqtfx4o8bAPlu7#rlFz-(>YYVQqTnf7<@r{X6m{mv0m4Jyj_2{CW<o$oSE9dFo!$v2_j01`m-Pke24M<C5^kHuGsIEF9;+RLHe02!jYb1*m<H5mDuf4AJt)fPN)HwI-yYeA0%$F9>v80H-0L$GG)_|$_L8Xu6u6q;Y;=k38@*y{FirW#~mz~7KijhObHc->NDABx@4=wJ)!0(V6LPnI&W97)#|0yY&VaA9qEb7fUW<gP4LtC-zz-WOdT#Z-@7{ey&?uMq;K>|BIc$Uv$NdVH4vnc6|LJcdQ~K4Enj8Ehoxjo)|jLGmEYs9VcXvY2giN=u>IrXq`nQ@6>d*Ar7wB>a#<JVyPUkj|4IiU>Mbvs6sXoV{TFhA(#xdOh0OKw92@yLEIWo|^+!PkEeW4!?$)D81jvZsZ|~*kWprabZO!G8$2u!dlkmwdb*3dQjW0<3+;>4G=-HfIr1T2@RXQjpy4t7}<o%>ofE6E5%n{c_rpQI1bgcE;IKnE+qb9DXUYe<lHN-%%ay|f%Ifi#*oDs@ll()sYOg3$_JSqn5koj{DY7}OmA0tZLY01ZY`~%AJTaaV9i~N#GQyt7rIuWNpsB>QB4XqIWdwbElKVu$;gUi)T%VEmY1coCX!Sq6ma7-QgeR*UZpS>i9|`!yjsyT#jlgGNd2CkmfH7Hxs!D1dYT}?j^BZy#26-HOyln09s<qv5szBhkQ3mv`}Z4gPzHTCNuz^Kr+Y|FtkAs<Onsw;`BvOMwUPV}_rwqihw6Gq4&`8rXvhdNq6Z;qbq+1oC(ZIidBn?DC+rjv4Wso44VQK)=yR%6w)6Y>&J6o*GU7ZrU9b*-T`8wODMAhW!0?1Ncf50o*a^Dd0OABoV^Bfu^V~;o3VdkLMfEgFB{iZ=N?E!v1ta!Fr1TPBs4dQ9d8~f|Nq~OBuw1%hqb$OeW_apW?zMI+y%73Y30MAmFpDb(y>WL5J60sziVn@_gIexv!F16fG1~H+5|@hPRhhvyp)tjZ7|mP4jEPcwBhSwRgH$LU(!TsNlDF|rPM3NWZ=089Dfnv&J4`ec&$h9)28`)#aT?P12)a5itgWptR?l%JL@OFditN<$>Bg5<D(E7+aub?VZNtqay5s8S_^|pjBj#_VOm^9!HW>Ek49JyCWW9I#xX8{NBNP(IqR`}cH~HpE-<m7V&c6I|nhnBE8KzqIZdK+(-x5j%AkkF&y<WS3f1LnmXD*`Di7_YcPP@IIkS@-PZok_ddD;cSE!xd8Z3!n3)%obvD>;TjSS%!h{r%#w+3Gg?BmH76gL$kQX|LGnpOWbMtOE#Ge&4JdNA(Qq9S_^>L@F{0Jcf9$4Hx2e67Duhvj&$l=C&M-<8JFP!8eN3odW$E-$_s#dL_2!D#{8&kMG(RA{+<;GY`kGxUjH{d%;XlmibVqVh(_xt`#`(aH|!)NLx5oePFOsBt74ORaX0j{e@et_&eB<2XOP_*lOK64iEm2xoT1+&n0*DmwK*#Qg?QOnYj=B-@0#1a03_nV8ruW>z29~=IZe<xiWi^MjF*-Gg|I95I42kZ(Ny8P!oQx(a`kJ+;E37w66#zrWC10!bqbgm!gqQgAN_A2k-|SQB6fs<IY}K($j@Fvlzr_2Q<N$?(CueDpqC;6zaUwAKU}FC@a9QT(X=iZ&$TDr@_#J0eY1^4XHB_UT#Z^NjUq>QfYDaGX2xoYwvBfhX{NwRYJfyVs`HBcB#~YD;I4SmXlGs$_b!U{$P5gfg&9M3ZWKivK2R<ZW8fTWd@l)gHp$*)!%6k;_o(-d{vq?NXpLcsmY^>^?6XIXCZ#^#qTyVvTIW?RC|yM7>oNlv2R>>C7M7X@IIP=Lc4#kMS+08MzJTR?7C?tBly%Pw-VTkq!G@or*dm)v9d~*#7qIn8Hnhg39gG{;}+$&RHIihgJBugXJLJ<TD@Ife%&xn@+BbO^>*6xzQ6AitmaR?UosS?1~iAeK&amE^m}vFFwAP+yaC$>FXa1LC5-zeiY6K1X(#4ITW0P;kqC8M5NjH}rj<r{xC4K}IaHjsIhh>HNl+WYDV+9y-o!llq~r|zKJXz{okyN*$uxa=U6V9plu)C1QPYPMyX?{63AGD!gf%VO!@+12PSu$DL@nxXJ{LdWKcVsN9<UU;pw++F+%{332b8Y|=aK*>d6G;J3rwc86Cd<rHk9HeNamnIc>PLab`E(G%BhR_bmiTH3?unTWb-jm%oSW}D7ztMo%#_PlQ5TJkhYUiURcC`;0#r&F?0<X-w(4Gp=c6qBL^Qf$+Zddh7(XvdjP?QQfCs$VP6ba{Vjf-paK6=_z+1-Lqa=x5GlOzNv%JLCxqpJp^V(H3$(X`>6E10k+6jJND|v{0>O0skV0fUEp<eAK?GkPK-0rQl=e-+te$9GeUG6<PJaiVe~Jl*;n#=3>DOp`Maw7IWxpaDC;=Xd0{Al`tP+guhq#q7`!RwDQM*Z!J}$z0Ae{$SZhZu2`-I`z5WDt$%zPGHnGMAGWA7<At7(s^@aQDWb(d1NlC&&jI!SfV1EgS+wIe~(PNEC&i8dncw_bB+8c!#z&)|K{+9vxvy_&FTP-=zz4p8TCeWDlqu1avb_@PUqy03{ple3iA_hRAHO3nwXYr(ZmcHl>S-_okr2_=h9vD`8IC$w)TVQWSnD83R6#H0K)#9zeoV1YG#qs!^hg`0=jg~jyj!Kv>Vx!b~<>lI9Evx+?B27b6so+P=Gk?np$%1`^nsbQ%8lx(1UXT5T*vW`qXeEKg8)Zsyd@{OJFrDQb}5QHPn0F<&otS(*gAxe3GDgGQ2AkV_JrR7Q^g)Qw3I-Rapq{;vquv*0cDdVG|)<AoGI7MgmyQTvvi9q<u&Gj`D)$9)YeWh&2Enksn-sgPfea|RhGz`qz;!^qg>RNSUX`xzdAC7Tqe6!{WQ}<2yH-cL)6=AwCpH>^^8KY%cbuvSmlmK4u5B9p-0lJ-3-rOh#s~GALZ1<Yoy<oiChRYJTOS4Lo&74612@I+Y?lpJXql+M>^al6gUK|&LDr#`Y0V{4qb12od51ZSv7Z-`5nAspiCTr0hW`58%ka_}hQT9GaBStLFUePckyw$5o!A#E{a>l^XV7oh*v%g-lzrJLBZT8@6<k&i-ZVGOW!f29dcyxZ#u-OQOKp$eHEydHMlng?KR7#7=Dq#nxNmJ7zW-pZe^<+211Dc7up|CgHgy-i;bSU*KP4dcV4#y+ZtrmRi7v*TODW@Dkv|}$Ogz{&q*^XSDqxdRxjwwhj{R2lTpzmID7{+O&E(f*oM7xh+&+|rxp|x*6kN=Z7#W3-l(J?d6oH%1~fJB1|l6DMA0OhV+uRw|W!#Lw^rt)^rt-2}2j9V(z17Jvc?J1j}6l><olwr#)MRRKvgI{u6Fjvuf(R`3mL2ll^tVz9E*|=S)tOnl>%BzdPw==57XRtu+!+qi9G%7G?*Y=><Yma~k09_d8OfKmb<8Md2AI_7_;83NegTI*>2$3md9gpF#70V#q8iwE2oeYI4K^=LANP?`Of5NG=0BmRU+)wN?fk`xEs1E2(V}@><?Am<x;lmKJTt(cI_HK0zp%^V6_MC3OeoYQ-r$etlq;ahX1h>$)0Efw84Qk<6pivOiMsQ0J8tA7W9HW;sr8WQf?0<uwK0kXejA`Fe7}+Eb$v^eHF|eN4aFn?;ckT{r(sIOLl!wA^iajxni#wW!qW95(&sHQI7J+0=)*|7gVo?pP&Xc2RrH7fmfZC03lpI<7#N9~e7C^g`<;BO|djy2Y;@?SD&-5g)tB`2y|BFQwYqtv0UzK=ra$=ddj3PQcwLI{<|2QbtDnP4@*un}<(s&-j|B7~jsaKqOr54@90GRdrA{6Kz<!ch}yC^2Gm_x%aPB0BrMk=6ej8F>gOmd^0g-WlHc#>`qkOD4>(Tz-t1XnaALiaa<wSMnD+Hp`*cDF4yb+Z?I_W*N8pz@*wLIF^G)U3k&kNy~Ou#dCcCy1l0iklpBL~;jma`HR8I<wu}?^3|e_F%6)56qlym+<&U#SDCS;G`2ok}c3oL<-KCro?2_v#{#LIFnSk3Q$mnSN`f4V>p7-&(7XI`<t`(#ka@!1pfW>^zpIKKEfM6B31mu0LvUhc}kQ;`Cfh;oPJE1Bv4I>0L||mpMDO1{`a%*pM76wH7PI5vCyIZP<;K%V`%B|=@+%L4^JP}!x)Dg<=dass3t5nhBaH{H6~IwKiY5hlaboc@FddU66n`iV>DXruPL;8s722~j8)t|>}cSg$S)k*s;PSF?5#No@j+COn}G3Lw2k%Q6wVEw?BKT5Ys*Uu8>kx-gpcfnW{eD!6FthMfg<1o2b`XDHG`fw>iZaak+N81De1Og#}C^g&S830ZRTWiedYzEJvGTaGcB5;!zTY)Wk?NCM>gEZtCuG1)lSj@s<++{N%uUe1r?m0HS?ap7oPKM8<n@A33i#e^|WCoN4M;P<r$ldz{ElYr>AzSYsb<1Hm@DmlB21nw2>&n$#wt{-o@Z8zU9wvPv)8_O@x~9pQZfD2jjL-A{y43IpgY>84qSm#g|r`p$4uGV?7`=lW^*_x6FDEqDinXdAV+~H?hbg<2h3X)<{^-3R-?ro{oxD6kjfNOBrhrs!<A4hjcPI#KmlGqhx5tDH|CaVR_XIon-sZKGVs`RK4Y>F+D0|f0?{Wu+wNRm{1Rm%>_mS`7%{d%jH&jQ%`jUG^JLuJG0#hbBO5)Ykg-((;<%(DNfWmQUqlbqn{NVUf>)@?foXQgaL&?s{<GyWWQhGT_qUIrJw`NAr@epqSM)*NFhbHy3L*bU^MP-k5IcEh;+jQ9kf0t?w4R2)*S;lPz@fCcS|dP>lWl~-Jmo8EW)L3uM09ADN}q=c$P$Q%>uSjXoX%B&Ns@@yyP5GWhAMiv*WR1WK?b28lrgxitZM4a?3@&JR%H%R>eGMmob$H+44UDKJ-VaL?ZLTsxJ)wp9A-M_O1xg2F^4N@LwwMDJF|W>pSLAr5uNtGWJmbZ+=bhB0K%a+y{$mE5wc9rY>Kvq!8KD*efj?BmvjVKu}7Omd0`NBG{a0^g1<x+>0u^c*hOjGkCQgc`)sya?N2yxj(>R40Yc$(xtCsS+UZWDBDSrAYvXL>bVqNG?4H_(*rU)bNfIrWkn8_xE2gYK2nj+w}L&G3l-eC4;;LtyTXmwwJ=$e$1l9F6La}k(4mawA~#@z*39e4wZJMy(McFRkImKLpA&5O2){kQ-R|`$f#~xkrmYBm%pH5a+rl5Wx2`m|+nrhQqeQfYM@J|86W?9%LrqXE_%FD1rIbDzryrmF9qjVoaZVl<5VOw9@9?ah{U9T?AZK2NSHC8i@e}%$9A?n7<qMgq7^O!_%#0G`qvFIFKt@pvl!++cj0gQOJnbL!dgjc0tJ9pFed*FSvu9>-Fy22Hk8o=08FPgM`agdU(}n+@{`~aUr@uV?htr>({^RK{PX7n||DWOeF9Q5AIQ`A(Z%=;$Wqx`3%U}tn_ot`-H+=g)pww?;e)nJD|9^k_OZfIL@c9?;+dtt`sO2Z8{}G)23ab4HRP!6C;=jTFe+A$EJN*3*^U8r2Q51h4*Mu^$zDF}nPIeDT-2Oza1Qd7|iP-5MN+{I}7bo;U!WTZ^f+*dSkd7Wsta-&8(wH*)4*@wlVWIcW-VOfq-{Hw)$}asml>1ubnxrl919GhT;6J}GZmU~Y5Dt+~8Y)XID=9n_nGLR%m&>bsXE+7c*UF1k&+m%BU0ZsylB(9{)@_xsX7g95auzOMpQ}_<<ec>`PVLut1kmA&FPAS#Jp!AZAUiPk?dg|sz-pDx>E40k!3X0L`17IM`snBN`0Q_BQa`3^<<{lm>`TU6A#Xbz2U+^E5EHnRN>>+G)=H&UN~PJ^*-)qCWwZ`HMCeY2UTk!JKcRpB5YPerntaaSEXPy;mh^AMtI$X2DcpR#o_N4N<>BcRzi=%+!X|$Uqk8x3LzoV@N)eajW0_2}Hj5Ckh)X*fqfI`XZgej>qnp#i`yax0RJh=gYaZ-Ybe{Vm4uZDo7tn{_QPT5|cnC1}NO)juI3eU~2g7(IXdMoV780il*XuJuv%Zp}#J3;IcM-}(5JFFT^bYqFD`9=v=W;68WbXyDp*uJh?iAi@6(hv<Mkx!si7zBx_oWs`(U^iB2swyM(kTma=u=V+daXpiBd&i>Ie6k%@9xC<btBtU7zur^eoAtCMmOZGeL2k1Fn9Mx43>nc07k|*T}R0n9VN;9P9HL=gV(z~xc-B%h=A<{M$XaJ4{T4d(^KeXJAtE*Rdkh9qC{0;iwRqo(6iDH+WYkTl>hF%Fc!Vbb9;vMz_T|7xdA1RJ{~}8@A**75WjKM+yS<Fr%8M7poK|Japz&=FPfOwuOt0l6~1!h-*m0Q-b|c!YQ1u+vbtHxXsO+EW+DxHbPOLq6I{Y$yh(Wc$8^Wg9f!K;WAji>H6v3D@Y?1o;bCD|4c|qTH*Z!z03j|<jWKU*=y%Q7JbH}(weaoIcI)_$fAfQ5xCGuM%jV-_qQ0ziC$m8G7wnRw@tJ@g6qhph+|vzJjL*SM%pD_h*~B%&-{%t8?iqa$?MpcZKKRR}u5OPu(m}$DLuj@k1t|E80SNaRhKe#r)s|8=`lum2iQ+(*yh==_v`-&LxvPEyN3#sl%F;E#Vq`cUdZg7kmR1d(Tkr*6r*dHum&{BYLxMdL{511fxvY1Vssflz#im^pCLi%Npr+q7n0`wV_^3o25M73!lSDv7N|gNC?^4gBP<tM^A!aZ|PI80y5r;1!KR4dD$O7R1!&E7dw-<ruI||>r{POJFOS7|cuto?N-6vCy{Phw-K^@}e>e5CsL{vq%5<aT>jKUXmm&4x_Pm5hYHm3`Jh@x+s=b3Bpy?yq<u|3uFCepyag<n6!@^k}yiCTwa{SgznpFRc>VRU<SbJ^&%ATa%A|F}Qsx6!iq5UUd`{+B}D`VzZmK6*&Dz<2Qhts)lVB4_U%TMarcfOoG*nxX>rh>o%LKJ5r7*30L3=SwBeZi}3xPAHMsWwug<yC;lYAbm3*w`j__i@y<2xW^ZgrGJ;@9y~LFxPrc<Ju6sCJUk#LeM+-0R6xYx!py|M*^h9!F^K5{mZj8Ui9#$IJQO05JW!7$Q*9h5Yx9e2Uz`gZ{SV1AEn$UX1D;?ihC<Vy0~803FGl3kw~wLUV<4uZaci(UAb^eWusOnyC()!y-(1DZk37r8N#g`VF!e^7XJ-{g2V34uxGMEy`@3p&UXTy7ldfWM+D`S<aG@pP5MuROL#rTNXmBb^X(3GS8tgCDWX0W?dAsB0Xd8(}e@A+25P*P-JAis(U_jH(Y+|tWDAtB8a=e8`Z4NxDUntblkI05aSb1tQC{#R0PuEHafV;t71Kp_j06nh?mzag<l8Dgk@3hIfLEFTgS3Dxdee{UrD~s0SKd_lUW-v%0VrHG5Slwl;H&A*Xtte2EwNNB{%58ik3Vw!%@9ab8eDFdkH^T7t(#8!uH0YmL2rs~=3&7zEAJq`uotcUMZWmq93x|=`tF~GZJP#JuHdi+aFGfLMI9)<z>}ctl`d&@G9S{zGaP_Te>JHY{7c1*nxF5hpwUA|o;vICM<~L~@o{nNi#kiLe^4}4Dk6MGm7h0y{<9835JtEkt)F9aP2iSF<VUMG~hibGED#JJ;K(uHRKd2qnt*P)L>BCKhTDqim*Whv$i>q-iiwuq^ue|uK%_7$`y{JV4CED6QqhPwfUReBVR7d_bT$g_>)L@Tc_u_W`5&Zw}P|E{bmA2%U#7R9OUl>xI1^@Z)To3M4OS@$mPV>Atn>}*Q5kr#9hdU-cAz}<sNiUq_lM5{hH-1nH`-mOoPbRG1ik?g6CQ*I4byV7?m8(nFH`g|+7_%=iL1TnzRn>BEm~O>PI5i-zTDU9J<63F%l4~HqZG(1@xRK~`DH4kogpeGU?IYJ}A!f~mF!^?vP3WQ&Ch%0mLnFDkbRb??S}nj%je@%|Qx7<A8%rQS2J~xRTqiC)2`}J4fG#K^G8bP8GldpO7UUGwT5Nl1Nr7-m%Ag`+O%f5NH_V~$Qso@IstSHLb(cz0U~*WLgJ%z@OsF}IQnuSNeUvP6_Il;5D(^hQy@G)3Wo_(eb>gn{;lzON@GiP4`Fu~&)zjzZIl(29CKCCC!}GW;Q9tYu(+RnNCY|QxH3He+s9;JiPWjDKsDnBWqZ2O@&|xejU56sr>>r(4!{f%oyTsYer6r0{v(uJhMYc47u?3==6U2~dA~=2C{@)0$HM>0!6oWx0Si8weAq7g@hh4CX8a4Egq4cM~Rkz7dFFO(3x*Sl5YOvMp-zjE<VY*j#<xK)#M#78n0+mJK0ZL;bf6#Q$Ksq8bPlkG9v!YY}9tmGTq7`9VPKz%zK`THw9CSMZLjLk>-MgBw{b}Eda4*WIJ?EHrQk%EI4zqD%G*%x`KO?>*{_SCi{mu+D1_$Gv0aYhHQEl{jQe~Ii%6!%-86jCa3MrkZsdkXQNXB4Ze9;`vNt_>|ZU3iQFjW|KD?CQo2z@)Lr^!-5w2v;-xN`}q2vvIgiu<#tljbN4#eI|{M$_g4AJ2IA%2p!lNT=F?f$}pDQw6Y#sMcF{w9gV6lt$FuiLfYsC|($`MB<J79ftqc;QvL@Zwcm&Kglk*5f|QyYT@<E`WPg-${DjNUtr-ue@w;Caj8)EbbGRc>Dt|#L1rfnkI>78Lue>Y&sMW1z+jBvZo!OXZSypMhdRmVatjQnK8({;A`v})qrDR*t1-`~tTfelST~5kWCu@cH7yliPWSoZG*47cN+cr{$dMv7&dVbImPIE@#<`RwPP}PQGW$7Q5~hvYYYvlG%^CYsCuo|SlW858_;R(9(}_P#il)rwT;vk}*_<MG6H)?MEVDHioWx5x7<@y@icm!57tXBLfidw{OE4Yhs@4QNS%eL63wF`s(lv>5v3AYgU?$^~x6d+C>zK!hn3WrA>q~z{PX4~YHIiD?qzm8O>>QnVqvE_!>Qj8rcM)l?ac_NAF3${0n-6X2a%d;D^1tcDcf!kQqOfFMUiD}`)3ldnn3Q!-a5Dl5)VM#i5BI`C<>rQoFwGqD&GHh*t7wq|H<!=*nt(~3wjUG{xV&^7Jqa5qPDwG!ZP5dR@+{_|^5(+w<{~F**FfJ0caH+HF(*@-9WRmD@#p4M_9$@ef95^;7(Ix-_LwQ9weJhz3B_t)Ht4~pFLN!A%4d+=&{BrWN2TEG`@#jwP``<AG;}he9tt-d^sPhRxnF-X!29?!p|#T<J{XV2epwGugZDXk7Q91FZC_Dg{P)9v`u;h2fRRn}?0x6}xjR{^K6XT7&|7p;l7C!;m`M^5R%57XnQU<7PXvG4w2WX>kI$E;e>c(D!nts>8QW$HdKnmr>sP$o@DOV#GL-UN$B<zeaGbbq=`LCWZs~XRV?)pSm~0APP@1gEC)y1=jM2A7M4I{|zo$3ec+aQwD)d4W(L*zof1$j(Dga8iF*EvLCzS>7?$Aq*4o2-(Rv<3)n$+&Y;%>XyD$HGprqjLWx?T~io)`gCQggZGy8A1$p1IMt&m};#nk|fM$_;UY=|B^vaLf_u5gUr9Mfl=gce|ar@tmHC?4suwMb_4pLurmppkDYY6rU3!_pr$XaV>e*hyYz!*$Z>p-j+Ef3AU4&S{8rK3gF^I=h8_WYDsZiI@x(Z>|)|Al_MxZ`5N`1o-HJb*`o%g(oo9DMN&a;<ugJPLnqGxFNs6RfS43}I8LARsl`?Q22Ba1{v9g>KmW$f>_VPIh-b~4*)k=%@lB-#IDh`!KeO1jUfEb*tD?IBW!t4t(f?J2w)8ofEU*~|W=eq#<uW?-Hf|)!Rq|&KLL_k%`XB<+C<C#UR&H)?u!q{Sm6gsTU*ObDk}r5_vu>d&@9W$%>X3*We?fiilR1!LO3jT*t%WI+%qaJOXU!PhA~|uk`24lxvM*Y8dc($f`5m?wurVSRyU4_fMB~7v+1dKkGRaFyCb7f<FQ<<Y4aBE^X}XdCr^sGfEG@{wDX}F}Ci#taHRjldKzLzF8ALdBv4T&ECKTL3Ar=pXF!31Qd{;RCqvsfjHzHt=K*eVE2f+H=H^i<Vo&A)#;DEq@zhVjK*~fW4Nra-kg`P5AG3{;gw84w1m?EFReZkoE6C#eRDe;wv{1Hj+d-6e)7pUaEWIr~tkw}VVc|Tj8&mNpV3E3l4ccr4TQ^PYOu~nP62<HBi*=0#sHI+fBf}#eo8%1cQxTp}ScLPa^L!v^0>=B7KH82m^=K`Vh7C4?t!yS+)F+yW(0K}zrunk3mPIt83MDr`_RD7eS+fr^HbMWYwaL6_+K?$QI$-e~j)_1pLH0{U*mk|($79finNYnu|J@4#}=YfDI3-fMw1gqV}K^lnM=qH1+{fN9Qy2Cx(689NVb>y9eY*%(MBQmHpwL_y58NwwR9|Z^dC<wR0{6Ou?MDKrUMwi+qmq%tKHhxP*Vt<rH+@C0Fwln*3tG)H|);XePgVX;>K8;@oOH0AUVChydIQ<z#+Ht=gQBdGxx)#3VF!awjfbR4Ur(en5e@a(|9gYS!*OQIna%+3`((IM?Ib(m*X!I22_7Pp6@;*BIaU6&|;1qy>4}X37tJB}2+AYQ)ypM3D=vMooh&}j3YWE(x7xps_?d@E8<>kwr?atGM_RfBjLwk#}v-kl$%@OP}xLIDUEC+vl=jVYGaDwY|>u~@g)J8Q9Ft-A^r7c%(RhFxI<I{Ty;^a+Ms@oZ*_73_q5q8Q8L|4-JvPsAy)Uk&3<6ajeJ@<11VJzT>t?qEVi-K<p2n&7CMfTgn#vDglZoxdan#23Rv2M4AsJx@b1$NKfE`g)@e5S})-{4&CrgLeB)c{TNjv&dv4QEUUOI}<twVAp4e9))VoHUuP&iz?r$Y84cU)F#z0^0UgV`Gi~h8H?cbG)APugvlbtxoT9&}(<b(|H_-f$4E4Ngs(Zd~7U~3Fn`aeBLOpUMJ{xx39)RDB%mWiQ+;@Ix)kHkl+H9H#=ike)^(Yjr)WCUKi+le~0kR=vPk~1c{ZfwpzjIgFjXfr#Wxg8C!j8Wa6(O<RGbjqxs;CGPwiV43sfsFOz>`sS3BxTPj^$hUVVNW5ajseWd~LrLW^z3EiT=5R(pOJ`e?zCWZv2?OpUx%`Sw;KdX>Bp8UK4%VB1+#YwVE4|?QJWLDVS&CjH#tIy4#m!f&z35oqv5KkC%<m<M%!RpEGZqe+MAE#6JV4F6Kd8#SBJsq!5s6OQ*JQ=tX<`!!Ayqm&q&FY2k-CStbPOrIx3-m0f-*43*n+7#k!rpIVXW?J}%rpG7NFH4a$3t?YL#zW7!U@lOl(Eh@2mVan9#Y&KHOvgFHR@lT9Ts?!Jtx^dQnF=}upxOW=-r>L;{r7#{n>VR{}lIzHSwNKobSxlOM>9Q!pRS6?GR(M-Ry@7-dTi4iRbt@%9yc<1Z=8O**&34BBvwMYs>x@C+*60k3>01hE-<8H_9ZzF@8}Tmhol`I?Zkm?lG?`ag@5$RCh?f-W)fE-O(NB$M<;CyAu-}^Eirm!%&5MvzC5n%vN^y+IKM)1W3C|U*6v@vJR?H%jj&J-HYSwKE#s;SdVK64ldUNl>zIhJY~xV`f`+4Di}|*@z%|X;Njr0S~ts-Qjc5hPV=BQPOwf@1-~V;$gEvM_o(Vd`RWq#g(g`rc0)?v$7!y^(snEWE_-rzSekDj$=Z>24@Ck*oW%r6le?4SjGc8(0+rTjp~6C+0K$wj)EJvud%}cwhV80HYJ5d+zvPfF9%<(IWIk90fp0zF%e@th2Z&#W$*7{N5C`mIL$!eXH)dIA+=B~gaIY|qRhnei*8d<;k0#5K8HcuhPUhR^!Tg{WZXfL7GQ$yjpe|jbE_m-?kL2`1bYX5@zOmJ8(Hs8hHTV@(=K~{QIo3%_;KqD=U~tokM?*V(oE$(Yq$Dr&IaZyNzvAZ;U<kIRYy)0G-<wn$AbF))z~8ujdpv`KaWWPRLepJmYYb-3b${!14^v^hm%&f$fhe58P+*TR&pG$Wbq=)0tVb%x=j)I8&iVUe2*#M5JRxBf?QQDFlOUg#k==?=ti_SRi2QTu9}jS+B>1B8pn2FGC3EQsGp?n5pf-b+)}p`>C}7KQ%VGbEqKO&&thx~u`;>dVD2%X?SsJ=wl!k8+#@Nk9(UY@7B1{Ntb(q+5%#4@ht)m(lkzXrsCc_X_if_L?dJzFJEW~+6FnRx2{(3z3_S;*>sEXph_!6QdoV|Ojf$M+A88c76I3|0a0^35QdJN<J+hc{J@-^!#QE?m{pMF9h8^Z8bJAHWe(>hi}fEb8$FA6Bcd{kVjhiC6IJ{T%UkHvs4DBItovmb`m=qqaQFzCN@(eSe{^B+;*J0?4M*TTz(wvGuZ>MMed699^$R*FaX0@bcjXkR*Lb8N89(aGmS$rfTFxG|+xzchPkHWh5Au)`d@g8a;3^BD7*C=~8Vg|`Od-GrK=FgJMDIj@P0pG6z#?YCP;mw|}F=hIKW{r20%Z{x?>%pc$R`#-+(^HLntUkqcY3hHlnI!OUO*7ooi6`~SBD&U}BBh}U(-#dl5NSJG|Y-x|r8A##e%Ff_NR!0Ql-7$aQ)>H+~IXl?$WM;4{fDIqU&{y>0Li1HSnQMR3MGu3e{^Im2Ib446Sgk+h%t1HG%O|o^SfRjMZNHSlBBPV#)W$y|tRL2N#aELP(}+85*d76+T4)j$dAx^A>s%4|*FN&DtH9%SntR>e{UqF6m6j$ngjuMsIk+UGz~sskqmAx<uiG9b_z~M0>~;INNfY=SYgZvcQGu_t=NqCU{6}_x|D<E)XlEe$+?6o30WdG`F?l9`L1z%*T;4gJI?MHO_G8{lr9dU{qqz-VuT*X-r=FEcxw^StSs_FmIk~A6q3+#>NlVf@v7kvFuKA_e6C~8X7w9)1GvPTFFbM_@BNl$;L=^m*uw37Zj@=hUDe)Y?y?ny-o}ZXy(Y_ajF_HuiqyeQ|p=CJOD)}a7y?nb-W)BD0TX=}n)QF!OxAC8wmG#E$we{C4>z?p!Ju}87#$EC@T3%kSlowHUTVDab<F9C6TWxbaAoXFpuwagy=4-byu~{q>noU%Eb;3k>oW=Pdh1i|-N)>TYH$t`DS*#<d(d(D+pO+}|BdiN!ww&oO^R(K;fafAyoSieQRgDqKc*F{l)t!6fc1c1yBLN9l;1A9+Ci;afzIiow4u_%39MD%}E6huqz5oS02G-&|^!ZmGkRSLQIRd7Rc-FtXL_cAIZsH=|JC?^@kVo8`S##)%<1s1BT$;idxqIv+JjW*NPRMYY(c}~1oQ0ggaajj02XbR-O%S{ez87=W6M-ByS2u9Km*HfDex5VcOZ#HTzb@7*x0Wim{ld186HL}8JpETY#^u+i|8)AB(|-$6IgkR)ToRh=rP<k+gVRroWCGSVqXT%0J_-R8eMJ`)PtShLXD251suuD-M~A6w+7Y608I$s%OtZMQiBTgk^|bFd)+~xO%~<mTVf|w)iUAboGe<@cz{be*b&R8%vq!=Du%!h)x6764#6kQ<?tGZ;4}<j8!!8QWO(Xpl=%S&q*~1a2h%p^Zf4^aG+g-Qrjg1>?o7M8_;-9bonf-Qml6BCn5m3)bTWHeObM7MZpakTNgOv4Q;FGj~txxw{?-YYkqwWjxZgyGRla=+G>r2&2+I5M_N{6JXcuOh!!qq|08XRo(kQ0Ylib3Q*Y_~Y=#}I!<52@j>O_@G`5Dj{FZNH`5l-A8w%T8r|eQrGvIaxsBYPE+}VmXyb1SO}@K1e@Bv(k!@lp4Lp1H?BP1%rO?K4k$K40oFS?gJvS&0Zj3UD0-d$>;XbB_F;JB*}PS=E&~0&2)1@ZEhu@WVNT6$T5|T()Qzkkr0BTKc|p$!J$V%LuB#Fa^pz*RbUgWI<iuWO=}>VPJtvnD~vA2VQNt8YquLKo68$ZH)Y6xStWruscb#T(N3v0G|+1E(`3|xRO*eBtg362i>sbwRPW7uQ#ZHRdOG(ZC7)rGwBa61-h*~4*t#F|2Ix|#v)$<X-P*I8h(riG{%(7SF)6_=xtS1nIRfFphCYCo1!Rt~awK%)u13fLJQZ*h&3cgnao2|Jy}?~di!&Oxn=ORmL6sKf#5w5S=@0Jp13G&nf<_E>4%(v;6xTf$AXLVj&aF3yY&batOcZtRRJv|T!i&ZMC!QxElYeAi|9OfR<6qn;AP3$IX)Y`Z+am5qC6f|sWeulf<06a^cc1r<wpV##!d$FWk<=<dD7yVxsElSZ^9mhvtXK9*!%h=_ipS?@U*A7DFPy*$n{%5dKEDs7vd|s3hlO-2#@*&3RF<7|n&kG|hkXD`*c`!dhs{<SO}b5ECkW6RB*Dduxgdd9exp%x4{%gIK01eY)1sM2ew9slBmvg=WUM7p<)izheVbe1EfCvtFH!}ZWE$Ybj&)sF?1w4ul^3ahQ-!@}5U*`*GxU}qn1$yO*#o;?<M8^z{rH2GV35z9{y_!a(NRkGTJE`$5L$st+3t?$7Xe6+N@Et9ODaTF5wow$Qm+|F5v6`#%aao)N=!Y|7%L~ramX_l-XwW;3NHP?DDHRNgex-T-9IrtxfAa&&0-BQ)M4Vd(^n2S5nn|3BYn=Nm29;2?A^&8So8UWR_7pw&Znp>wL3A-HB^<zMru?Cy~c<r_4#0MCpj{RZXWGSk|U<bC(nTFjt)AV?qL#=qc1SLgq}(J7v8pH37`H&y9Z~z;iw4Ydqgx^4Pd!dXoW+=o=fI(Zyc^DboI^ThnaLP@VHWRc5sL0jGy`-$gqAt0rZeQxrWbz2D|gedLnDmW8aL{OQO^C??vDK*SuP42*rE5ZQy*}a>9I=byJ9Gu5+87NiMmRd<W>fc#Ci_lcd8dc&I~UQm@oKw~DXgygH~gxIEmt4x5;FQJ?otZ_peUe6xo^>KV6h`@8UJ?AaNb90*`=gI<VmeYq#dhBS3cil&pJy34_(9J_&&#=v9k-%aR~>btQba4={+2p5tyk2^7kkui~$0e!Jeh1oo(;by6<3LUM-f~~W*c1*H&!!*m#b4&b>#pOqwe?r99w%drPYJSyz1aCWKw}fvA#w;f$W*VJa_wC%X>y4<_Cx1dy(~}4?zTWGO+Jx^<C#}&wY{Rx~4$XSyR*ozq7R`$B1G$Vb-;zkSMaB0s=K?$|VY!;yL9s0^w}R=3QQg|S2jW+|*TM)hcE*~geE0GyF@-_YTm(dLln#WnES$fwXz|6IL<soj6pKQ;+;Azk_c#Wxn0s8I6sqvmIGyCmNI@m`XdI^Ys19|ZEs1^(zWtc~-P{lqyW`y|<jb77;9+bnron{9lquSFf;N2ca;CwZCu%@4An8;c+cHUqFYdEs{ffT9ux3>tFK}76(8nYR2V38PSuyn_0te1~(`6TXgTWmjMdofO>gW%5?SYZAy5S75o_Hfs@&4v^idOklZPo-$tV^ztG2oT8#maIcKj50^wnKnhalc`8ZA9(~g@QFN!d|o!y<q<?DoI+=3)b&ejq+lV(n9oXv&ed_hNV;j2e?42F5-=tHOR?hQG30fX3ycC?{!CG^zI+;!i4T(PP>VzXww;9?X0vkgy^+IRq07e%jsTy*{eI@<_HyEDsL^tUy01qqyh`xD3=166_y)2nOE_^2|1#sSfI#CXvZqJ1o6GR+mYwo|H!aNMlbk_xEqx@Nxc%0snuJA73lVN4!Wb=faz>cjh(V<^IY|sTR?a(3U$r?aBv7@1+(N5MFL5p7mS*yx(|_94(?N4NlU>eyXkRvw$d)!eP^M%j|tyfa7ke@_zp(WW7M>K_L_}d8H$G?X7yNKx~Y+C2w#fTMq3A?ZofSmQG_QVVbPtWGGa>EX1(&p<`OKSGG~;Rm)~mKDlab~KHPd`VMBwugcd8v2b7dln1CaDopY8MSx#DLy>@x%)KA++YVPZixmPeHdt)2z1wtc7dgK>^qYynm=rYzzA8Vj!PtJMOla-sxm5s`3g$zU1G3M$`0L}{VN0P<~NIumi&lwl}o0y`$qg>cku4mCU9P3sW?NH>}GP<aTr_TW}T)_VxsgA%qd=3bGUeAXy3pXm|<&7KJ$HD5I)}sw=0t?H<I93kGpN0q-id)AXZn!%b?RUqB^R^GSV+X?wQZ7em@)W`5<b`2={n4?=HT*;NXhS0@c}$-E;q>!kfjjw#l1D#cmmx$G{POf;%o0ha)5qu)fewgn-%V#k7mmqQ#L*cnBj|%d%wh#dOI$t9d_~4ika!W!PDlYtv7|jL+ANN%)FaOpr;Jkl3vgWzzzUg%rn_nJF3jR6!rgYWH{K1SI*KjB@d3&q1?%8)3Bj(oyRVqq571JokU9C#rf>2ib3X6{yOl_w;nYbe-zqOHBm5>47Vd6}Y>2{3tJl`c)s6Md1-k#Qlb@9)9d4B#!jRLqJURh{;)a;2i_N_))Y2e+ht3(VOo_=658)&Zwg@bQzXlqPlDULbCxCbe*YaR{x4nI56l}G7gL|Prn1;ncGmu6mSF$&X;DL$IXNZzQm*GrS+<E)?%HekJpoMmqy{+!f!Qf!Tc0g9zJV8S;7tHfsgFXY12w-0dnM!K)c}JzxxcXM(Mq*~U!Sl2mZy$^YoeqgdT+PDf#@e-OR6HX~v!~+PZaq2j0V2g*xBW^+y&~=GP8V3oR5CAFBURVwlrr;<;nhOMxWX_cV;eSRlE#S&rKPQ;6UChjTr+0&C+fX=?E=qi^n!~s5KjtLs{32+$Nl?pe=pJeIy(t?f~}>nAPT7F^posVw{r4wto?)~w1aS5Vgik+;~9@))W=2nsFxJ~Ns<e*mjr5wmNRjth_JCC+ez=Si!XTdV`pIg#bK=;Mo~tG26vc<%9p8){#3B&{ri*A&bi>8x1hQTmNu3u)jYV`!JU#k98y)9wMWwg!BVb@sY!N~zEta39mOkuwbF(0wx*4i@Mo#<B;2OaE=)c>bU-w7aW0c5gt=r?O5yJ!x|YB}p&H!ZN$|NJp-#ebDe@KDi(*BJ26tjc9ZNz+n-!?(dT^HJ5jgEl%Gpp^r3|l)wRKnr^_Djyyq~0^kwEq$m)u*-aQeBSbQEiEsMWL{Ev1mmSaz*Yv{&5HAo{>eQCdo`S199keZS|e6t*X3#}5prKk*V&MJ&EefTsm3%8N~%ge-L4xXzPw!0P_6I~aNg;OUqjLrGgbr|c5BCCMJ?uJoEavrNn~4rl0C+~EZyjUl`p<3ATJr=qj=_7fUsx_8EiN*#*OHeI!QM^NF3f2^{&ek4k})=aTUYUZ%(iTJf>EYA$QJ}--_0_^MuLr8~-niWpr<Yz}Nns1w~5VP7-&vZo*-67342Yijtp{&4@03y$YnjJ=)Iwc$ck5~P@@os~I<{SIlW0(w>&=(7`SjIBUo`m&_lgeDU1f^!oc+<&^%Zc_EP=R>0{yvDvop^@-)?@hFZ}ubLi1n1o3o)xTQ}NgX-jEVjj}lV7;w4%{H6PM4!Yp;zTMMw%wi_6gl~QT4r#s#C3q~dW)Lh>P5M((}Ko-;?5T%A_*TAiEuQ>!RXB_MepcsOt%Y$6qC@+(0iCLJ~t(SZgo7w{y!<TyP)(&|HxA-wSOiJUJ1?C9z>-ekb<{TeJi`~zjG~(W*CdxK~W5}b&!tKdjh{43ju;3d$Ja=|Akj-d=gN*Hm_sr=24&ml|8`^1Luo{+ko0*Zl@c7Y<(T`5NjB06SBeNF`?!e$!4NC-hd!#AoCd`@|*LjD-6P$20Vv@c`o%Iwp7ux6V7N_(`?tD+c8>b8wxW8KsqN-ep*aa%PovM^4Wn!g!%|o1ZW>3P><e7)7S8sSEC}D|6VwAJ*E$+>H;`jEprqUTT2tkF}sED4hhz5uyp%lm)q<l)nEdj~F)qV<A;N17-toL2#5`|9vc`i|M&B<O2A7*jigoSZ8UF__b^mZ`$)8+Ej%5wG3%+uL07baU3b`EfijY$ouf}UsQ-dNaNUk4fJ3C8yX9J4bNgW+h78(h%poPXnqE<X|QhzVpiA9Qd7jZ(g+q`7di4!BIP>XXFE8*90(p6t?h3wR_Mo2Tr;u3Mi(JmLNJtWiv$H@@8nt!8eMR0YOz34kSL`G9rfi)CE&ttcC5Ur;T%j({=otDeUf{q+GuD!Ii}7lD<;z!F0kb|;<|CQAZ#3wvmxURF_#ow|~c<E!b438@iRz)Zvsqur*%LeVZ%3G^laHgOD*i<{!33fTm}g6Q8US6^>%Koro`?Li+1VS)P;cX#?I$}@T)p}M5P_ed?Ts>lJvwdjC4Dhd_%+S+<$p<LZSnAOI@+Vb*J)rou(4Hk^nvKI1Hqm0q8x^%B?5@)cmcB{O;gi-IymGbM3_Ym_DYl{_fH?FR%y<Q1R#!wUev1WEj>2V2GQW##n1yuf|8Z#}_MoQaQ_oVhUS8pBR^o3bS8WwDN#v4`<HGU;c)07h_u}QvQk>NIBOOsBWKKNZp>t5IAZVO${k^<CHRC`h}efVn{Tw1saQvxU&1gwN-KeJK~t8icS+b5<V@bbU$zOl|UaO11k#W?ES=mq0}L98>Ugq&*Su1&`KHGWC)!Slh+EdFeTuxQ~}1g_YZp6$Zxg^QB^?aX22>6n;|O0YR!VV-*JkH=Z_m|%=iwucWHJ@9N@O9d`god3kelqT#}9ZMZKe%*}_yrj{Zozsxgup#uX;(a~ilt@-?ODSW(qjryLz~`#8e7-Z=Kvd%UY;#tRU)Y&dVhRhvR7V3Q*)2Gy3QB$TBITdHzRsGt%~grG7*5>wWU~PJtA}A)I~8&xnnX5kRMx%PZ65)#_-jvrR5+@r7^6`Qs@UmdZlz8qZ{rvzi8s&`cFGe=>n1y7^yV3!Bcs-Ed$6~MI1~pI5)F5SZaqb|b3GO*eW1rRHsx|>UP9^igBOy(S`>K0-*bIsiY-gZ1@u0}4SLsvNTiv&t0!L$P({q>7PbdW#q&H*``XnM&uOQ_$8^8hhr_2T;HZh*tpq%CQzny^TWYo#E3V0@%bgeby2gXT3=9I4M%HJ$PE4~E>mJWA%#60GQ{Y`EtD`~=P@&T3ZZ+8JZmYol?kFId)NXMG2<VIvWO^OpF55#R{ysD93VBv6mN&{_T+mJxxYFcS-0sI23eqTgZB|j5PLmvJzT3-iKVj}v6`xL5S~s~y+hYUMfQ3L)L)sgO35FUQuBlTE1KzL?i*WVO#KwLsJvz*}{O&;)E|L8qz8IeVi%wNi3S%RSLCDE`F~=}z`CtJJCmB&zHw-TvW8e0B?J@8%+<b|Ht|xUbvZ43LWuo0ePjZn85#$Zvnnx7J5v9osKM<)%1xU_O2;%uHYEE?WAMu0rRes?%;HRSNUIGGCB3j6VsC;T1Nun*BvqKCsHhWH2F!n@Vym}OTJAV%!s%zOEl%O24j35|3wBy@LtBY&5Q9WfWnZ}6MMqOKycfjF`&7B<>OTjuaqwWKc=wmfwNg)UD6N(?~Sy#L-F%oqzjOXBgo$&OZ-L!Y7e<%eu^01RV=&Q32&VCT+Z7zx%d-^0R@!i6W%EIe6*Opc{f~As~&-)L7K%V-&HxoCsm(I_JNsb<+I)^@+S()uvv|~>cksMq1D2)cVy`Lat+r^L^*Y*dnV@Bcp_I`2iPOCc<L2x6^5ZyjRL)GApD=Jex>h(s~YegX{5tHCeEJ8usw)tiZh3pUX8%O}q(C!F!so+~g31j|!5)GWIcCx*Hu|SPo2#X@SMi|Xg!)Tk42{*88Y;4UK2^4M~it>6e8n=9g8grP24WUbEQsLhsyYlMfmDFy8`7=qEVf-k=f1*6sa65VCZu8QWZ-%8-cc(q#5k$(;kJJ+*@wFPp)sYuqW-v$^M9`&%9!QbAlcM6iAyC!^-pU0t@-7q+Xu|R!1Km6h?K;Qh%^Po336=`(&o?%KyVg5UMvP&;Yhx~bk%6>2{c&<Bf`{7!$~qn%j5{+gyDQ^8_u=U;CDf89_ZA*2-GN|<QKP<(ZqINWjpSiZlLYNs-`d`ud+Ge?!~QR7{wty!l&(!v2`B4W9N=RlG6b*o`?C+?fY2`AWjIWk%ky)FngUYt5h3nAI(v_jkD<Hw+23KlOoGo6SZ1GcMlxvRF{Ow5D7b?8$R3`(_l(nwwO_t`<y)6ux%8B2#=eEw#elg*kca99{8<cu@R7h-je~`?m6f&C8Nn9^)s4-?r3(BmU_CExR94_Yc^Q5H>96Vve{1WDmGz{MCDNEF9@dvu@T%OXR5#cdC-aMA#1P)ULfE7Ne4%Dr>LuzZK~@m<UsXk1)4lH3)Xe2;YF_a9(MpklL^S0NuCZ`Z<1(euD+m;}wo2!n9loSPxWlT_m4q|K3^Wx{SCt&C;x`u7*Q(Xq<>lA4i(BrCru$Fj76$kuPAk8GYEj4dnV#I*{kq{EDGTVa(daS#qzpdxPfbcPN!R51zI&%|3di=K6AJbF%nk|{fnxkU!#lxLmy}8}pfuN55R<f!m$<o$RacVQQP`wh*ekQrSr5o|t$tFUhno_iFI-jZcVd$N+vETNZ550o6wi6sbCjaexC2mAVaLD+rc%C1^0{wgKkjH(ukXa)j7q!`<@J@u3a3=Q0~e<~wc!fqWy8kD-M#iqw=X^W1o&upVaLM}c#=np$>Bd2U_kS#=SGTXo>}hBcOpH%{F+d%n0L5}jcCw}Rg>C|4m33;+N)K;dCcQ83Omp7d_#r)sWz5ckWg#+g@mkLj9!~7xef>PmYxFikO<Wa<OazeI}Pt##5iZ@0U_V;!Tta%FdhJBg&Q?oM?1^v^#&%CgzDjDSYM(HG6DgJXbh!<3C-ND%%mA3Nv@o@2K%QT8eC^jH=b0diD^2qf<&vrM=C~#_s$GzT|kCH5&#16V2B!-LzJ-gD8DTkQ+C_I_})N;l!81z7BD?I=A&zy%gdoc)XCWc+FnBt4FqoIyA-pdGl=ve<fMcT%CKyBm!uOCMLgmzoHUNG2K3p)VG5JJ1Q^QU%R{(O2KO*`4%LHeBdi<GVh=xyrB`&pB^pz0-16p|9H5=vwL59XUkoYpwut3}^5P;_Yt(k)sqbk9o}zu4?a04Mrk)sh`xXc?<?Dn@m!2VFqMmM&C<?`{@LA6j(grAkIu`Atsf;iI+v6E^1g5k*4qDw#2P8p`Ek`N=`?}TM?lp(ji-=8ywN6JZw5H<Co6BqH%!a|j#Q$lN5z~Z}<^tMe4=2fF+qT=ZK}ka!cpNO1vD$(7l9QCE=4vx+W>y?uSm)dqNcvPM831h-W0uZUj-ZlKEd`NKbnr!WFHoXbJ&K~|5)Y;HJHC{B-jBWwt!G)8)0WXoVdM6i=(Rx`QG1cmH!m6UkV$WE{kNmP#{5%MkMmE3w@%MVi9^yLK?{0AstFRBP|g%b?0+~71;U`R>50so06SE$h<;Lo7>^!J%(;k-Y|5rNSo4Xm?kD81(iP>-yFGfEY{R@X=gv5uJ@=}EBY4S~aP;EdN!)^7|8<P29wn2f5Q&)%TQ&+&TPJl-t-*yPP!-=^LiqCy=VCNq*{B`?#Rm7KX8GwseVQGG?!smlQ#*;OlDrA^@!2i>G_1;Ke9$6!TJ+z;HA3Gg5EM(+xTiic2M?5|EZTZK(GrGr`~s>v(x@-S9AkCTaGX-nSdI2&e1Nbp`|)J5sy1)t(|<Yr*VA9n|Ns8<m#4ou{Vja{$?1QG&;NnizsIQ2`<T*lk&)o+!%}eLrQ+Nl-}!s^^KyWiI|j$cB1)^rxw76PZ6InEKZ=9XkI5V%9=&_|B^E&J`~dG5;|_1!c<UxQSyWN#GbW%AV#B!JhODzP7yR*uKMPh+O$K9mEb#e1W0k7N1My>F6M>%)a^j=Y&nQa-rR(`yY2tVkz`KtLvEm~EWWm3~%ik9%$KE4M{LeAH2z>e=wC*D6AL{Iu5cj^ku)%ch`dsk;{6j#>ywm?l)A1E2{4x}+4+E(1U6JYp<{00wdWb4zl}qF!PrEpNBIfvEWtJv|C@xLR&!J*r98$?Kek`l~p~z?Om<IUcXEF_!Wp^$+-MM`3?xd3}@p_`o?E3^Cf|~Ppaq2uBLGHGS=y@AUw+Ij~TrOX4lvfuq^Xi-U{q!H9f=}7<CGuim>KJP75vj2;)7v{V_g|RWdfHSErNN9;`JN7ZODBg#A4G~$!0X@^f-~rF;qdg|gdY75m}UZ6P~-oc4jE<vA%g-P!qYGF=H>KL%1OaF?b2O(8GEtxR~7N6QQcT4X9hH0K9ngUXi%S-+k%tbJtSM=M`Uaa%p^8v?@vgV6`E=tdB&Hj4c4Tuzm<z|6NJ0>_Xi*gjJhL&hu+1khCrX`3w<K$KOQzm<FF(@Apl>e+wZcQy#DN5Im~wo(7sY31XkeM()vo7Q!>BR5X`V}am1~&v9_?rz8bqAWe{N^mOIzJCPa^k)w(Qh$hP<fk7|32X0P(nI`D@CF3YkdJ57t=Yf&?~P`j1w54(G4?v@FJ;mWQUc!%3)D~D&hgBDK3jkV<!&p3nd>W#*=#r1SwK~}>M<I%W{x4wvA#`z6PWZ5-Lcgy9~RZNErT<_8b;ajFUJDo*PMneNH$Lz)^P`QFd5(9yA#E`<<U4*>cZ&Jvf&}5T5q;yrp8;(K`#>Z~VyG?B|y{qu5HEu8SrbTAst%kAdua#GpmQk^pzw3=XUJDro-Q;l^EjSQ6eF4ujs@V)sJ+(Nz#Tz5wLfbG~SsT!O&AT`B)=0H$Z;>b8A&W3o*H;_Y*4O^3vKlf!Ix={-y5mWRYT|h^Npaq8TWwrkUKZj<)so}Q^eGB}P`JvB#>2=bun+q?7_BjcO*7aliY(s^_?5#Is}<x&qxl<^<we~l$YxNYxV<~*ZnteK5!BTjHHX9I{escWm@zpSk~50Yc_l$8@=N5;G+Vs%A_*h<g@~MCDKJ=g!6>LDv(#%Z3qNn@QD(mD5iIz4gmDvvRxBCvcsEuudO{$?T4bm1uH1>qOPakIG;;*fnXud6DWOP#)*(1L{ha_V<Y2LK3-j0#Xc`~CVDDfw4tAS&+rd`5-B-#Zw*}rpq$$EY=v~w!cSoaxW*-iIf!#Vtfg52O5-hgc?oldU);Y_5jht}Z`!J^3eaD-hQjflt>~;GEHo{%lbp)gSIGGDF!X9|uQ|cP0g4R?Oz=ZkV7b_0kk6!JMFZ5d%l&z5wJ)|y4czw}2h8t^ZAU@nS?~ba2+s<nZvr-^ze6{ouDW%j`3&1AtRurd@{KpN?SiPkptKJb0{yr+k`s0#|9s>jj3Q}oa^F;QyHd~)@!o+$_ddP8-wB3@_`!m=vBhaHk5oMPDHp4%-J|crL`kXa;K%VbfJLo=|Az=T!nxxzJFQQdf^s0X$VTrRdukz|HuYm+D3B5U-Dk(hPg0HWyuH9a3ls6hGfQY#X7rg&&Vs#4EqE&J_KOS1cw#xGa;Q49Yjr@ZKqWkK3w(L5hq%Eyftg?2^c;b_mMe9uxPZ35YwNB@Lkt7g~qNS0@p82D2%t(YS%Q=GDf<}=!z3Y7l6O-eq74lu_t5odw=p=SiE(im-IYfIdEgT<F0(Sx{u3+mdFRpEFgn85AXl6VU^NrvpA|LLw2Rp;&{_cJF)Nc<leHE+p3y4nvhYyojwb5He<Vow(`ibO0qo_D}$_wAaH=-9ZraotOX;E@spK5XN?TnYbYh%o!RA-g`e>=#WZ3z7a3VsXykSYjOftdiA?vO{6Ap`k6UdzFr@SQ*cs!hhsDv(S6;4ZTQ^h_9+wwl{_5Cp`$;hE9{gS}<*;768QammjByoziX-~Th0>eIMYTEI&sfl^NMSSK!SnszK74eEMhuQ|FiiW`Z+t?7Zqj_wqRt`n^10Ja#Fnzwk#3(bJnqwzwGw?$nzbtQ>R#7Uga0-_Eqme(8A<)xc9o#eG<Qb^hdsEkJ)nhxw(oSmEL3<nQ@4-`?D%o3p3hRj~OH6!`BfCHB2rR=xU+#g98>d|huGd5fe{2PrG#ray|DPuv!0T-nOHveeMmKxV;728!sq{!&@5863(P8rcQZfAHes4z}<YXlkAR5mTn)uU4Owx^Upf~N6+Ylc$4Udlf(rU<9)3#PtD3d1}FVM#c8sMvtg?!+^x`h$KGLb}nSr{l3TB9#F`-!?LeOl(f+D6}C|@L@U+DhBD0I#7q9aj=LW`1F}*H~&gxcy^k$L4X}$G>d6rg?5UVtdy**zy`%Wqp$Ebi6;Awhm47>n_ypDfk{~kj~KCx$oEt9<0K;pHv6bN^BpdoA9Df}Eu2v?W@!tvyy<W+yGF9Q-(OJzB{CpsRwQ(O%{eD8WleV4n{a5Q9_@w1jH=9pdOSyfbt?lPAS-CxjsJc@bgiDdN|ZVkPIp2<P^0mLV$v{})Q(<!u^DSaV98?#%$~Uk7D8jf40B$}PMF85X=7o@Z!0v~i2RM5WL8h+Qn}f9LUOj{lidh=dof+`NtQ-R-lp2KE=vD43iHWI?zwPJaEUt<@<ifAAbq<~ewJc(ctKM(r-EpHZ;$k4+08QXOpfl5o8Vp3RsK#)dXl|86|zm8+)F85f}@lCaqXqRh6~(oO~@E^R}dF&>4V{<>j0TeSr=Ye^;p$RY@#XRYk&7iR7Om^C9QJDP#uO9g_EaNU57%Eg?mwNo;s_uj9+-aPB!pp47-}S@to>L=dLer<7QVt4T{`VNufplYI}Qf`s*s9(vo+gB5Ii_@eZd5jiwA4%ie)GHKe%#OK+ssNvQ$FaOiB`k7rTHsGIWhb15x^-xJQ7ClFo)SC>%MAmZp$=8-Vtpx~Eu|EFJ6AkC^R5&SJQS6Nf(#Ls89@=W+=AAg)TFvU9?^XX-r(Z0%4YmC=igv7T9R+RkY7oCE>`O6=TWY>w{SPVJdpsfjSD~_FMp16zXr%3Ve<}>^u(mcKO8vZh|w^K3rs@Mc}egZ3R25ln!yu)E)cdgsWI<GoiVs5ijNeAgdlSZRei30tLEZa*}pb}&}SC*8<^~##g1C_1XoaU5=DqzbrsHCm}r4bN&)Z+!pZ$>B4Iqvc)By5*7z|I!D?K7ndDb-_h+@w5Sq%;akdwVA?AWU@wIMb?Y95<<28qDlS(jOnqF`Pxz*kS`nKn+Qqz>~C&87aXQnoO-e>e%Yg_`OkGxB*gqV`)`Uuq^yFRgr?kQonHTDOpMeVT*jR(W&86mS_7tiQlF>Xap8ER+g${971J8rh&M@b82u8BRHP|Udj`Im*#+^1;Tdqc9^Fq%Bjhz$j{Arp)PWGstnaLrBD%iJc0N_xFDEFRk)QBAhb-HD%6h`#VZm)$JkhE#yLk1vB2s4?d@$2UDCThJ-1Lui&M1Tt}#ZcN6-jkSXp-3gBc3FP-mG)2uEFzqY#BCs$dJp$5<qv8Kqn^gkK198A(FDQzx*V!X6Q75!wb`twrR)*AszR8M~006ery)R3P<9a>AF<to?#t*+6w`+(lFKU^S~05FP+PW+q_J*pC;qBFV44+L7Ea37?rsQ+#qgo(QMIQz&$F|J0`!CTMkh4V_QVL(z`DmdTaH6Avq78zVIibU<j<^B9;Ib&(OqpVEhP&Ax-_W2Bvpt?x~1A7osQ`ck#0&Aefqb8SQ3OpG7I%S=G5>M$gcFBA8j1nLQqFB<QPqa-lQ6u(O}Rjx%ogt+2g@f}lE!^mUq>1_7PQa}*1%2$_4b;3W(yeAgVV6T?LHF%cN7lO`ZB<4`=rOeAs*Ru9Sj{*npz<x`vAzMgOK@BQ~!2?n1S5lYUKyV_~NoqVag7mKzjDy>C{4f2A;6`96twbAIpsKr3bIhm1xlcs*ifP`x*l!m?m0eDHy>M}XU4~wPz68HAa!lJ6-zby8P$Q`seNYl4niD-cFx<E|n4w#R&&-TWcBq3E<&DTf#HeHhBVL-lgh3JT=TaPK>{R^y694|vJlw3osM|x|(^0RBATud&_WY4CtgQ%pc?-KLnF`8i2it;yX3(ougk$&aTX{vV=o+t=Z&oL6c4M1danEhd%TybzwC1lB8!nD->=#($e6n5lo()Qq=!`2Bz{%+$uQW_IC@UwVQ-PQD)~q~p&(q*$Fm?t>$vJIM-Y>h6x1U#u5HB8i$4D?6d%#&TDxSrXq9oXctpRuBK7t@wup8e9Ze2mRu(J;l7L3t$2m$P2fMGcV2=)a9KWNk*j668|K*amt?-Fe6LwNF~KnY{$Fp3KJ^7PBI4~oI*PtM*s`vC?tWMTkW;YgoT)Br{~e1=J55lQJYYUp?PW5Ce}P=<ng&VCHvL<xNH_h;{8XU;x2{SsRKZxqCUfi~~(?~G>j6?INU-<-XF_ICux1OtK&V!*{`(8mwCE5D<<-A)>yNT(kK1q{p3VNxGaQ#atW@Ps_#$PJk54+BbU`yPh&JQQtWSPA|@kVzQcfi-_3K$SSagd$>aP6Wcz?<rD7#bX)b#H}k0*d&xz6W!!fl0;*Q*tf>{+!`9TUjw)B)V>Wu70k-{hKR$Pq@Wk0%ht?TQf8`Fk~Do{>iO#RiIojyWXeF%ZV=JbjZI@2YwRXh<{RX*O3uha6E-jN`thTg#_p4Pv8sk7HjPx!Gcc6LTZtdMnusp2YYzQp`Aau;GnDZSf#Y0whTf`<goK3D8BnHEt?OhxLFp&+*)C$d!b$@UV|8KeW(7Kor;^gE1H+f4ws+31PuP%q!<IC}=E}1GYjJ`%nxW3SVz|7h{?moFw2{t!?bnctWs9d(LQkQyoywI2Q$E3f+7h_Y@TG=MiDaF*hH<qKj9M7CTw)M-w#!K#KV^v?&YI?AeDL#IGNA+4v$R@Xyj5A>sH|=*zr|bC*3Ky=hxeUoSNYHZAL3^uM0#Klum;1{XdZ4a1=qzO9F#IR=yb<phiOHROhcPQXja1K5-~e;WYmb9Gf{y$?v4-8q{j{cMkr9QdxeY!rjsV(XjN}k7OGmXv6w$OgVmCO&CK9>VG4F@b$~#}Bf{B2vr*9OK*KT)j4^>{q#MglbL?Z#ea%?2OTB?9MK$-RJ-(=X9>(OU$>6Ty{)J+Arn~KtJ*R8^Hca**bn4z<cqiC8XyF}egt-uH=T(^(j2<TS3tDXkQ-bvfS_eb)lN}!Pjrm;PT$LOqI-v2sLvuQM-d<m?R87Bm5Kb3fZ`{05u2#sPKF>cauB}#*S@C6KZKJ%L%sQb%?Qvt=-M&-U--S7d$AfXR=d$EGy}=gj7^98Ys4><}*{_*>qS~!~pttNS1xIn4k=5mt*3^78nJp=7PwqA&d;T*sXrGBd*!O`?6j8Ph1wZ-d?1vl)`!R*If8dgM+BZ(!v({k4)1LuvCqtqcjrU>b;8D|r5~gb0Pv+uQdkp-35{9`Go1Q>HdW?BA%|@`9#XjI?bQe@B8}68N?H4$MaFo*u76Y!p9I$Ey5rG*tu(1Zz?b<TJ?_@x(ruPcgY|SLkiq~Xe0ghMs*umvgE6B)7LcNw=Ov2N~iZ+ac@YtpxCIERgeW77-c3BpY3mnEUI2eoftca5dIMdXI!kPXDgfT!lAVAUTYl1a^tz&>kR|RhL6T!*8vi0(%Z?!JZo)b=%<jT(oJ3xtnB3PUP!QUZ#0R`xeGA2d33(=8<pT|@K%9IZo^Z>;@bu8eg;MV2h?4^tN?@P}ZCH7lawy(4=wXZxKO6>Y2M2Wq11^+$$&FQaBe|Gw7Sh0VGfB#PqiBJFS^yjC)JpF9|zyJH`PvO^ph5!F5IQ=E!ar^|nVaZ?MtG|XHeoe3(KY_ykc=`)?^h<gKlVAy@|3I_Nar?hx3<VRP0A<Q|%%6FM9)fU-lJp1WoCbe<=jSA=5~jfC1Oh^m>_dX+MS=Q*`9Ns~9+|83h(z<hgDOEZw>tPW0fSICJ|X!VJMv|)I6DWDGE2V!$+({Zkwqcis7SnR^%=)&3%}JeUTi`M{Di5S1k6`EU!u$K8azivBsL|V)3G%jYQrpPwqq-Cvpq7VV0$(3$8nI&dmlNnL{?MF%S2|UT2$tYIt5iXRAP#euN@(padTOQDT8a~g45<m%jlWOV6#w+8PgUwZxXt1Wzi?A!AAbAaWDh|hS1a~%69t5*wc@Mp6Oju>U<*cO=!C_@v|@ffY^w4;f?qB4b)Mc{Vl`xKVp~=JdQS=|7Q}D<m?B<3@A&om2=1~@0%?5c7mI*x6IUuxIl<Vj<WqnM20Dk9V^#|2lJ5fl8P5~4iJ9bNLewdA}L#PMFQ!8tz1-rdpag$%@wsuOG8e5oP~F~>UL&r16)Sm49jtG*rpr<qNK?8BolxLG=S+ES1B+WEZf~Thm(_jWQxl9vP5i$**U`DfwQLxTzHbH9WjZ!)Q-Sji?VM3R~9!PW`q~Qy{wUaZe8Y-8lUm0g(LSn&Xs|B@UIz+iLWa>oTy*^NGiJ#;VH)B|Crdbug-o-kL@ehy7u11KgNwJf%QMKuQ-On0Aa0iYiVt>D#6-q5D{YzjhqT0-SFB}Q_4s~Y8=)%6J*E2-O?&0*ODnMz?WaTeEB5K>B>ugwywPN%H@|%O!cO?)e28)>U|-@wu{BAK18`TVIDob<sZM>kY%)uYjS`}hX;4<ZoAonhiMa;|7s;Zm?SuHqjHtSE>6a5ofP8xGAVgR5we1%9qu!CYc0Hcr6IarPpAjyd2_GZKNvNJ?M{2x2Fe3faG1eN=4+zIG#DzHcC9{F_c47-bJ!hWj_dZ_Hu_tI<?A5NBHBKgXGv^cU%QRodI&p~i|BpMmR8pql-}IT7XZ5?^iedZ8hf!`xmH=nECo3!ztw~`<T1qqq<_UYCi)4MfImC1KIlc3E>FLzS3=gc$!~JoMyVbv<t1V}R1rNVqub|Y`jiPeomx&`5euk_zz}5>eO%>8Vz{&<HVqm#+Rg1<Qdvr!CuvPb1Wt_mEYl9}mFigw^f57apNkYKUUtt(%#srEo+gjX%P-Hq(tP>yx%0^U+C+&weHaIHRenOIOGNu)Bj|fGe53Lx33O5B!6C#wl=QR(`XLYWv3VOlTXDbD+S>Zo+)J}hnH1(_6!aG>w`P{t7GAF`24w6Bu9jC{pAR<HH!A_dwUJvHCXZQMS`Z0i5GbmOxd?-`&5eb%mCDT3aurHeE6WusV+gURh#%eQ!G7)+_q+S;UKglWcaOByH^uji;z&mZBUO5Vj?V_>=D;l9?V*`irm`sgQ0G@F8|zC8>MPp9<Yx}km*18!t<6RmDU5m-=Rkw|nSp5I;eFTgjGYBJZfh{WuzS(V=J+apUvJY{aU;$au4<s!#C8#aW7ysw2TN2B;Y^gQ(vCl2Mt^$Uo!#-y)}HLo^7X4L8GXS==jn}8-sul+oTSzvp{=9T2|S?^H;#kK=qd0VON&ITZd|_#|5gxIdud@6wYHSsq=Tp;<&CRrYt;<_hHlJ8<cZ~@SC*Hq-`IeT@+)?iP3e!RCwC9KZu-@Tu3<Ov5!FQ9RKFP6DN}+&M`u<|j<EE}yxJAXkvRx>u#C$EolK7GuAXFjurqpIf#Pxq$Sjsoyz!lOdtcvXUT8CTssQ1sigO^z;oR6wkcmw3Bji(H7jTpj2P2vmbDP*P$8so<+9=e??CdOX!;S|H%BR!a<`LC8^|*~Yc9Z}w(niTGvix~nEr3wbSXVJN&Ap97flo=kZ7WOFs>dRZ3$u4vXmxz*iNnLB)v0l9brb`8_X}3P)7=$#c!gRYlW2G18UFkKQupr5ZDi@4;HzX;MRX#?BnVJal?qf=3`9Z_T)Z_ws#HcHGLk@ogos<=LV{A$G2I@!H^vUz<5<Vc?s&%I{b&E!*o|^e*SOm4E=SC~0QCy<B>P>?Ie9YkWCB#F+McqT0um?B{e0)U{=SwA{|<v87C?Iy%g(rJrksOPhtiD(lkJAx)%_z8j@_qpUi9}q{e3`xANsbF=Y+jj!2Bc3Anj86=G=p?{n1%@TtLXc)2VpuSRV?D;Y?C>ayLYiE#+VVvWnamY^z>-YC;w|gNe<A$69rY$aJ|rJ;%_<U^eM35S0OHn78p7u3(Q~IYLSe<@-=4#)bL5wPB56TLzf&0h2bu-!se;!|CUZz2#e-xxhCPTgP5I52ok+-f5BZ7ZzwBl+h0l9SBDYk<id-R?4MiFeIWW6t}P<M{Kk4b%AjiDWNIbiH0KAB+`T%qk1#m=Sr6Hj0*AG#aZ6?`y!p>V!>)2SMQLy>IKyY%K_JHp}APNv9?++d*-{S2o}(h>3TE;>|!AUpA0F7U0W$pWp06;lfAt!fU({^+1qcJ6ssaE2kI3Y&omfO2*?AGC&$H4DB8J#z~KC6QF|!YBZSfd!=Sp;FzmR<o)4yma3wB;Xm50dWy5g|f(sDW1p`)S^iW9#m;HPePVn@=#GGAw_8{!^0+2s~P6rPXPS|L|=g4i$aQ9C-4UDpx6jn~tdDQEig@vF$J`Y58T^>!-re}8A<4hly#wYv>8C(rrdV1~saF#nLw_o<j?K<(he4GZp&G>H+EPS9q6YpWERB{>2m15>XQi<UH$elrCWU{C3IS|`tlc0mqc*L^whG&X@EU31zB{>3oAc8b9$AB*aSKsxAK-H$P*WvMHNUsvxPk<9Rm{PIy;31-a`K-d7QUzZ5QAQK}O<+6D=KWwIfcd=5V#!5p)oYlZ(RhscQ8|l7#-wG>nPNn<CoaYK7ovH|nXV3S+2^s}=}FFRNX^ni7luxgix=Vpw?CRr{qB57*B4NNy#}z^m*M0Tgefrr8CB2DCt=X>BmEkC6P|2;w)4b4etuM|?w}K_n!iw_(5C5g$#>SrG`ZM2%)UD{5@mg26}z52u`vmzF!U8g%$ruY@qFG`8G9{b25_J9l6+*?j(Gj}{a|p~3B3ETxo~r~&wKrj+@jgL(;E7(MyJ!fS>+|_&91%m@@iW&8<<d&yS|8}V67fi_xxvYJ9s*3x4bSe>k84U$|hxS5uZ!(L^m-a9KBL1Zn_aU!><qZN{aCxA3jV}N^}Wo(-yK)-hAlFMXQvly4cSjYd>$^PgEEk+C;6Jn<ZBqv&w^Ho%c)8j~^v!jCO0XpASoU{oE{-;{CjdU-H24qBkc&mxya#6YqpMQb~Dq?8+Z+ej?Zaqn!70HgR9hLSQj~T_B+^TW(D3D)P|t@)!>k(4(Ey<N__!t8dZEMHo;4d%*ITH2*11l)0po#}o`pgB0hQ<g~z8oE(NxFnJt?1qa3R2$Gi+=!AO$;?tBIJ3}rYL&+n>mhd3nuOFviN(3ds7hURBG_6^D8-lbW_o3D`y`bmGHR}hQ6HSDu#!~i@v-=iWHVGh_qZRP0AK@tWIaz8yFkEGMu#4HWH!fXcb7@gk*lL-pi7z;FSRlkW$+b@`ec4OM@i4?@m&d1Bnfh<~!Hu1k?i01$U%j@q&E@q?!Zt8N7}k9S_X6ys@#qS8!f|<HeBT8^cJDlu6mQbQyP#CMSBSm5UN&zmeqp)L|2?NVzC&4*3i|%?t(x4ZUw(l9cZ|h24qw&<GCSs^Z(n1S7rC~JpWw#t_ooDDYt2R$1pxN@7WcZy^(CkfPy<#_Daa=sjMhnwJnc~G<xZ4wCFpGF3I%ki8X&z8bC30|JjI<5E&*T?NP!_zwFq*WPB{SgAL}fIO3*?YW1WYSkh2L4!`=+OS?Dr79AOkCO0w+t0$hqK@SP}|z_leg-lB1s+=sXxR=d=tQdZq}nIn|BIpA)XW{gatTxdBaNDZ7fIhsCEz1BeDT-}Y?z~Y_Z<S^61wL5mNlHIu0@0M$q7jA2;KOM`bSNhW{cgclwEpD4}`l`8>L6@mJNW(H=!)OiuIbmA=qPBf}aFkLwP_M3VLlpF?uACniu0Aa}?v-0gc5*WDuGh!L^>||sg3zw;5rO&!A)E@Ko`d%gpV6`>gQ;F-`qd;V3x)38+ce@h+E*cN;;`e#+Xy)BS8vAUqEqjR^~+c6kMn7Xw*XT<LkoJ$J}-g}_!!?8DBuH|Gw%Z}2&NZqFd2wzoOl|I+|dm`^pP=2?jLz+D-1^u>(cNN_-h1+z~3Vnge`K+5N|PdQH!0plN*)ps|=<oxg|31Has^Jp=)9#zWksC_g;S8aGE^rMB&U_C5mXKHZx)yWw`qOuuPl^5|_=Ub+aqHd^WBhw1cY{p_g$o2GX?Iyq3u1`bbXg3O{h~-GiYev8jNvPI*FCH__{N66T{Cl1&#r!o@wp+(M-Pq_8gA1bakwS^pBYF&vCIyi>HAflTgUiYAvmkG7)6AGY{ug!JLjNBJ0WN%^sydQ+V^^DPhlir0AlTONn;h^lZEpmz92Jh_Ion4_1=wR(UXh}ue~!r=niXqh5F*(+m*PN#73%|f4;Hk(WhNKAmv5k#rz(pPg#ysQezpMhe<D^;dr<(>!cXE@qzAD$4q<61rF1K9&)b~83s6sFfhxTE1dLj?1ADC60I0HHkt?&iy0hwNr>e~YuF=Y=4EDw<iJkoE}|A(iEQ#WYSsWIz)Z;@~1o<_(A;{A0Cb>Oim$Oj~)h36yNOv9W2Q^7^^JS7*IG)Ak8EgB3PQ<&wKu+JvhKYABaXRXi*Er7}q(a71x#5QYZ0mj38$h{va2p!`vIl%H+NqAavvF`k3M(7Cs102H@56jQ<5zzfMrQ*^V;<={MqXW?X8fVNz0ujCF>SaxJv3kJ)~*#`0IdhGi6e_lfJcX5T-Nv-mz)Y85_Dpgu;d_WnRre=a(8KyWjnJrh_I&NIFC%Hb`tfF%}W1ZchTX_t;5o%jl9~C85K-0NzuR-;Ca)|WSH)dPl%F@{u$g)6fBQG7DMX3vAS!FsHU4&bSSkVZ6&SFksnw880Gkz(5JQi^{B9ok=*u!dp6oFJF$m3dPlU`?P0X?#!xKZuYlZVR5e#2`PW(m%~o#_mQj#9<h93;kCBe6ax@F$?v;+ekVtz$eJv9K6IBD-S_?IOTpiaku1zbB!(^$jGt+LgW&c{UDd^N-$VbY}ySkduiu^dpHil-pH&gx%4qQ30SBKX1G0^;nea@q8F!YM&rV$EJl6<K{Sp<9gvD><+Hb$ReqQ1xssxgi;+Qbb}*<;xfng@MWQG3HGGIsC?4k#vGB}bDEE=@SkOA2C|I7bM$bWCAO#7kg>h(c@oi=vsh~UizfjKv%Efpxk5V-o_8**S!vrRwjElBvFkbb9ujrg!*W0qw>n+H%o8|hYil+^HxGWqFrh64NDieh4Zjk^cq`Ln25c8a2Pb3(4)$~}!fTI>5mrD<Z=h%P&H{%QmzEI!@dzx-h{7u0*s+u<-`q`N&=6P3%iI^zZf&Oj=>8-fZ0ZVS<+<CrlxkL<M}2Bymz^-BFK=_&R^m|SQi^9T^aQ@WqX4!-tI~b(2g@iUSS6W9Sj?c!(~^cx=I1VYkdZ^Qu$Qmc3hZ}Q2~68f(m%8)pPbKn27E}LS_*BNV5=`zu0J8{;oJBr7loLZ8TNLp(LboJ45sjGg^dV&YN3nwg+!YOdVNe5_XgJH%MW@TTxaY2&H0V>hG+JsV!a_x5e#$kiS73N8{4()&#OoK_L6?mt<k@(sB4OwwJE$hyUna>FTLi3&NzXR%B_Znx-f}BL{2m|WTLYno>x%2P?<Hd$?Rp$lyZhy!rZykY1^cNosm0-+E-g9Hw8xG5^b7#>J_-zraS;GR|Dphzv+h|v&j#wr@h%!Aki{xMj#Ws5<`f~MbH??;Oe)13SvdWe~2Rx;1Vemg1vfUuX?=wTwh%V17fZRxv|YPV#sEx+k-ivnEEy~U#oMYBqQYSUEEZ6U~iGViT8&um6DTxTSYvFk+Y-uv0ZJNoMgoH^g>lpNdgC_6nI;mI1e!JVmJI&LRlEGCaKh;o#sGs3qKu5)<9@`wq~IyAktu%lt3EzyAVmuaGfTBRBXK2Bo;D@yod9678W&$vWN#1al>f}IRY6irwZDtRy{&rZz1Rje5++$O#nZy)LG={p~D&OpO_xC)0-lQ(~OITnMGkZ?D1Cxu@W2hpC0V)_`9|0XSTDfeWfWhwHW;fZHRRDOInD66y%++xj@b)@2H!EJrhjH0l;{EB(3*!xGrHcPUyMCA@!2_Znf2wCKt%&%u@*@jP1_C|MkX0nJrS}?sV)K45{ucbo1>5iE5Iz3VW}FN+p%{!HzMn+`#GBO&OL?y*&GFx?3xCLmiU6+D^T?Z|ckBG}RZTyqc6OJ2bW3O;=l>#Cw7vNL&Nt`A~>2YwrA}Ne@>nV;3?+KEVhJ7aM*hEoj4K%3%J~9llykz!`YKdM0D3Ss<VG$TWpjhvdZ&%tG=2%Y;A3mzzLJN%*Z~h)wD3nv|V%|Ksci4JJ2XOK^2}B5o0Sr#SWebRpB8=aop#&q{1fKcd`Bf^%$LmI};E_<el+E*meuws@#n6Dn_cZxZIMC_P_t6_!dSHcLRX-WVoJ2<6?WE3+`e%!`D`umCn0XDYA`%j-1@igrZ^hRReYmis2ZMx^B)i7q&;QhM><&|H4`0naTxI+xGH1amnd;1pktPW!!6JHIh5M1pJ~d#VjnrljlGdM}p_BRM>Ixya--F?38}3`9)79yBMuPS1l?nqu8~{_5PgsOjC{vQv8lXWzvzOLVUF?q_8<xT|nXs`{?|6xf&*7S_0~@_55X+)H160Dt)xq#HL&GuMjPyp*K9W>+lsl7puhPE*P|r!3vWIIP4ttaz`M3HLT|w+BaAQaIm_bS@FR5oGzC!KKbhvMcAFVM48`CnnBWUD#v}KUdroRVW*RLDSActA?1T>^3pI`t4saFrf}N^!86~e{Dq_cv3yG-u~?NySINt07iu3Es?eeikcy)|5Ram!~cG}97d7|aYI3v@A$Qs7#Hjx*7kQW(l4rj-5nB9{hAMiL&3(zQc}0##DkVaNVOK^r0mgd4=tb}$FH%L4R1j8JB!1yMzqQq=2NRTk}5JPjtVTP8Mr<FC)IS?9|75+4a+fDCJNJDhZu3`NfUYFI4)(3(JE+apNs4dOaNkJxQTK>nvP3zN%Ab^{=heubSyDu-NE?0aO#=TF%q+eoXl(N`dQ?wNInr+97Qmhc(043LuJ#?%0OT(r^v1p`Mx{xo=Z708D|*zxWQlHTx~z@KC;ol@${Nd&tmb1hF0|LVAN5=a>vxux&A*{NmjD_e};Yp-%RW=>wII3Ht(&R<S%eSy1ldcBwmNX?=ZUx25pE}z;n-u6Vb<U*j5ZTO0oPi7e$IGisv=b?S}B~We7u4HKH+U^-8FFejrZ&n4EQIGI+(VP_)iX`mo1onZD%CUCuv3!@m>yi8{a9na()C!Y9le;f<<Uoup_%px8G<ASmguv{!6{3kaG)-0^bKg*jt1L4FAVy9))24?Ra^HjW4sPMXS_JCrg0o>zA5>XY57M3rgOf4^qD`ZcM@mjp6FV1ku(JFV7=)UZXq*X`+Lfv+!)FEd9=bCv>^GJPip-pZUJd6c4YIohnye)z2Rg_#}FaJGqgLiRz!mB6Y6cM!SAB-9(wmgJYHi2U+tN#*awO?nySnnI&1GP`;jQ|4`C0#m|mh^Ie-tKN2uxnQI<T=gQB!#2fu3jlFBtQ%nmi;|a4w;0`5rwcdiE*DWlDAWlr*Qa40C>;9aAOXi1iZ`YyigB?qgybi|@S<Y7$RD`&Ph~HlLY8<0fU)c>7NL$HM0q=y&SU|-uSPGJ7?%9b3&fDwA@tk_#ZoDOtFwQwzrK$sTlE(;t8sEzJF<3b)ua8|5kctSj#@uH!MzfzwbLOG`k;@g8N*2rp#>@y22Ch1XR=Xj!7e%r+SdtK`Ur$;YE8lcVrZa`X?s>oqV2SM<LlxGB)b970tp*~e);qrWIbPg$-wD>gA)$0KjPim0%V!vNf=I5*d|qDc)k?QW^oon*LnHuNxV+|K_UR|*;1$-CDB&$UQOWKB)m0)&Sfxcqo-+3yqqyY?==T|$?fCs`@eVaq=7agFN(5zbCVrA3*F3$Nfxv$HV{|7K<1kIKG29{s?<dES8R6Uk+)+GK+~Vu?&e1&$BiOrMH%(Lb{}RjCh<^P@nY<u?NU6rqs82^pRo6li(s#e{#q`&c#+c=A2?XpZ1^k_w?YQ<eq5x`UXI@{74N(FrauU0=cA5Cs%zU70lcr{p}DlmqtjQ+nR#fVs0im{_C3zSGI$BM1l$z0TH^-=GcXO`bQ)J#;s;`hrM?Vi>(PVEXzFC}{v&OBlt0Ru(!5}N5vwewCpvf11(%jY3tleXEJH&`S4CdDqN2Qr<CQQ2($%ZqOl{oX)FY5r3Z755pnfy8u=%i*H;Qyvxgt`012a-2sjh1X5yu)eVbL&Nm_F11Z~Wv0ZaWYGB<XUsnb5i{ov5|*;fmF5rRuEieBr%`h+%@}MXRtKdmhbQ9C#~Ba-w~L$0^<|@kne*{Gt(T>>tCTEr>Z=P_is-CMcbU8bx!_YAs8k$v|kadjspLAJu%I<{Hpi#V7!wcXrfZQLx*nq>EM<nh_q>J2ljS)%Lfc#{8t<$TZ2(I-h#?wfbuNjoKdO2o(Kn5adD4PawlNv1#L1b?83zkT~$v)A~N_#23}w^e<Zb3e_+AH3#s|e9`+HM$fM5NyAx2oPP4=2g>U%%(3}cPexe&ht?jWbxXb(2r_vXsWi9N=`}HkkwPP)2vY?27|qYltzm#Sl}s|4Y~vg@7>1Kt$UGqH<)SmaODG(!0y!awzF8Fi%eN15X>%j~70i~G!zxAi7Kk4}=m85t4_Fa8;Ch?@4p6WG3xo~ecMuQkJQ#<CR1~%z%Q;ri0fuPKC~8w}pcsC@KEnraU_;uvt0FIxR+>nu#ilqGipU_BX~&~Hdld?ig+WSOtLz|1e@dLgLa<FTrcY?2<m-u@dWiGNzv1PK$&u3U#IdioE=_^9W_64<w})AgvX^Sk)k#LWpe8%XU=6*)BCT_?+}e_!2Au8(Ba_b1eJK&{dbobsthmUdw;YBPE13|KhMrfJH!Zvg#ixaG4kpEiR%0+4&CcN%%Ahq1v3V?%!L6$>xUi<@Ah%v%D%{!VDwx1|1=g`|i6z=1mq9#WiR09r52YddDjmT~QM!f0e|5EfiScf0z&PKhNZqyg#qoI!??7MNtBBEu9)nj@G`Vr7o{|C!(7L-Q;VO6MurfopT--oZGr=u4b{juAv|R$+pL)$gsR#uesNZn4GSD87kr@KwQRHOB4!?ri9vXq&of;$c&{>BHZ`{$C!IdzQd4cSW09xtJP@h~5*FU*2MNnhm0aR0RwS5XBEG`)FZVHXG+rSX32;}e@_NvpAo2lg*iIC8n#6z>irUq&$0-GOur3!T;S!uGTR~zDuR4cg7emo4aaOuK&JD+C!1iM?c<E-J#7{m=@uW%O={vnA7cyEZkiB2@raEPHWut~FyH;(@5Q#elSmH}r?3<_xQW9)}^s>OC})RD`0=|4F-_^h^{fy8o^cTM}~+5)H3M>{CFpCrlJwAV4kVQ(27#I_ioE9u?AuANK#I+u8dCfq)iw<pGmG~dYwRJ+P%6X3)WSwfLJ67e>|vtBPEIV<!<h%sTu2M2y*uZm$;+0BK^>!PQbl}V@2-+}Y0>XCqU-L9TAfO$MP+Nm97?7VpI{@MLiHTGZl<gMVX+PS=gx2kI4S?%E2QT6aSq4wi;Ftvm(R-Q3VX$x-&UxqT_v!#w$TS~hi-_?IloMp-V#!!W2X$?l0-Nq7d!FR?A7HWLes`GfWqWC(Ml{S&XQzo6YaqmB^eeO%w3gR6R>f<JfAS96WViDay4uC~W8=|#1WGRF-EAhL2ulf>GFYMHJ(5e-Qi6&>oY*)gGmnjCtW3FN{P=;+9Ss{VYW4gAq7^_KB=ZZ4}4aD?FD|7)HKyO2LlJ!PP(p<@cy#&J|FMF@X_`6Xs6m>L-U*V);plro*^Rr2lg@9b(P@8QkoWTY*9Mj;)3`s*5G)!$@vRIj7rtahl_Ag|11%%A<`F+omh&DDf%)tcMYiX~r%u|6bXE1fd`@ezdx!HJg4*!Eencd4O=$t0~*)~fp^fxobk_pNsPRZTq)-L*A9Y3$a#s2vLjaHHn%(<PUUh*KKNw=WU`q!1h>t?Ctz6Q|@i2Q6c_C(QU1uy0}$x@8rXClq1?0J$TO9Y{mgsYJdt9jYnY{lsa(M>X!LQj~yf{?SIxGAPfSI#}S5+~Q@SFc?95i5Az8y7^YE>z~E(BUtUyh*5%s9#CF#=7W3;{C5aDS1i+0ts<QNE@P7w1^bSMRyR)&Wr7^*Dn;rj$SJ}_e#Z2oDK0klF`ngs~|6>QbEFP>&2o?fTp($4r5z`aBvz<@G9Cuc9gIiP1jOE97CeRzC?SHQwCEd@t;h5Q{HuhUCmOu8IG3dZ$yD&v_c@x2r-ktIZ7=vQK&ak8v{@1!%9BDLTS_F4=vK%FcDV-T%%UvgNZ!IiB?E8iEA+tkAg@csNw5}!YkNAG67Q#M-UI-mR=&NNY~!T7V*#;aT_O;q;pW<>21o1Y~8!0)#u0Tl7Q<iR>76^I?MG!F|f`^NsS&^**YL(Hf=Js3rjnaEPKm@0~RYLj3sh!5$08#+cUWT@St#rVtB@#JPa4J0?*2&iv`sg5Od**w;dMovA|T+XihzvpqA{kx(6h)c7*zW^I#ElMJ|?#c#REuLL-)Cc$+7O&ERWNPrB7cI7da@W??3Eqs4se0Xd7;XIA6?K<t8nt>Qok{@1mkdq`Kg6nt^*a4_ys=n~`a(0CE9OArhmKkMI&zIS$TdFEgBI%v_`yhl|sCB~GJ$Q;!g^_>$K_D@c#`^R<6_xKbh0#l#?6Y;!ua#Tk&A<Sep8ej-G0w{lCYKSn$`l^Ti*&M-`W??9t`*c|(M4yVFZ%xm8UBcSHL=e3hqIU~O8|d2>DYdv-i(Jmu(Y(Ol)kNH+6^5|!y!XFy%UNVzOR+Vha1>K@;>6h%)T+qSz-L-J&X_@6TvAYSf@;meAR=|{BggQHp*2O~SCV-oBVgQ4v<sKi6&*vDNv)}(^d*7w=_7&--gD`nmsOG&q_}NLt);EsaY*L7%RETwpCpnnzFUJdhwJaLS8-v>jfohVI5C8YCsUtr@>Ms{jafaFjEAcjJvKe%hEFgE=`z{oQx&Nu50AXm?{L;LZ0zE2(@mrWr$wPzipvuLYjkhY!O~m_1rOC*WHp07%g$x9;(QONJ0PR3^a(MZ8auf&;-#YA8^YyK7w^U6wz_fVhNMJ{qr!n;q}+K<KwKgSF1?B>>MRz4CgRJc1XIa7ROJ68hXJ~&mPcjf(f{-^-;mWJJ3vjyuYB2<ECOjRX&-$&Qr;0!W=mPpc1y$uF_MI9-~I)LCf$C0`;WJOhk4X1@%)vP`SPD`zr!0d$&8Xo0$DJ>Ga80o$$-|C1vsH>6-h(Ixu>ILD7%Gbv=G*bY}16)^ramuB;RGQr8w_RCzNXKhf)ef`WT7`RQ%7f(7?@yrB<GjMr{>Kl1O~J*IBbS>^1H)*SNHy5=d9s5rXmUv*wl4MNHc_{+hm*yY}tBhmHKJ+ppjLqGG*?7rfbc6BSM22ALY7B(Ui2Od)yN5wR%ME1i$G|J&Ptcl%pA9xQ@<Wya~uY$tgPt=sS3{#<4t$2KSu5Mv1GaQ+&8g1=eLMe(qfl)BPMI}tT9UQxT*)RGixvR;k}J<i>!7YN7W0LbI%Xw;XUScz^6rxspTV)fHyihNQivp4Z1LhN>==1{y%R`G;Ww2JyotMBvx2+`9eAq5mwfJDC^qFLt++Cg%=`{f7N!j~U7Y+F_xHq3(n5sGi~Ez7pnOr40r5Tsq@h65GI=}G9aKrHN?vTJ3`GY)fRXU|DO(^^|#BvNk%)pmN66|NJHk39-c=e>RhS{sgrgrdPoel1Q^h}Y#7VNIc;!NTOi6aoNj>XdxKw0JLxJiM`87ddQ4gVWwH?C1d*#XE$({*Gc=mj{@^<>CKJ4l<MdQ-NlY9s1<>@Z{J}1fF4_QD&@5Rk+iMm=(ij$1zPWrq)j_Jax9a^dQM|zv?kcom~5;q(jy=5}|gl8dme6wf(%d{n_CGkVLCquMkgl`x}Y`|3*u){q|4gqkp30;lH93<6qN-^IJ~H{jEj6e8YEvb^A*#OZm4h#)^wu1Fj1w9=p44zbz)IF#EzY0rsO#Hd6Nf_E_-h1YkTy>-yB*YLAP9i%xHX@QEQhJCWOPxIoaz(S<27Y4p0^>lsz+l+xYh+@ruhHVbyp7YZs}8glhncMF1#XX%&efbFV)sO9SH0_!nY7*Hass|CJeBUC*f9*#+xT-z?kq_5UUB(tmTLO)G=Tq}z`)B2=e=&QAgr&^1Q1<~gxRp>XzyE;bn++IKQ+T)D^t;U*Nga!U;4rIjHqLo3ZQambPSyu~NRDunq>3cn$bxizUY$Is)3#bevCHPw;6b~EEnTUSIh8@k$pdKnOQ6OLxq6q>p2TNi_DU{3t49q5p&#%}0s3S!JLUf9l>JJqui&;TK2z_ItHk1?@#J;?!9W`L{nT#cx66E<84YH&HQ}C0M+Oc-V>^lLY*g)w*)YQf14XOw(F&zmD(3R$bW-ao{7a6e&F*(~QVz!@xeNRf$j?=nHH_3G052kH6sl2j1x@f&k$EC<r%sgq&XWjKjD|6Jhe=Ug}kTUrHXgEL^iM4+FZ*IT6{Rg00-wHt#Y1p4o16Y3x{@h?oZ}8g($PrL#iVrFrr`j1^iJpAW>)^xA{muIiKML-Phd>+xbX$X;-rVpbo_BcA6yzHIvtB%`Ak+Pg;P$`84iTmPhQK<If)l0({`ech_4pPEx1_$`-2OVk>-a7F{d*h>yvnrySBwS%zyCM*{O<%R$Uj;&nMeU6F7FqYju^v(#lSdLpB<Rz!x{WKM1_+!6ZhK>JEd;tlkAyb-fN0e(aM+`>*xVRgg(T-w|@hY%YV83Pq+X1_OGnlzl4AO79RZ<`1e16Foxg%clh+z{PW+!KYv3-{x5jzFK_?j?O(y0{{escKjF*&cl&?Cga5;#3jPXS{9AbZ7f{V#P$~G~Z+TW?@G5^xm?^)c>Bp_|4NM{}6d|qLerLf%e?!s}NmhSGdyno?mo*zGN<kU8L1_VCZFzUnuA{eUnZKi(Hv49db?RGw1xkL4TY<`chZ{n>>EHgDwO!pm*hlJMZCJW;6MFVF?LEBSe+C7q!at&SfBW_i;lWO+tX<_V9-^MAc2t9N=I<PQzQ21=C3pk<dc{HqpKAq^@CP!1HyLy}%e^lto+(g2e&7kRLcWH`0i@%-d?-JoWRWgGPoqNPpoeG`;+qTvcT$6>z1-eCLE?+?J`KT?n?`bXZds^NvXgOPccXEjr8AZ8a0Fru8Ko7270vpK2Y&Tv`+5CE&EGvBxCC^fwvjmnre$n!i>B%fSM)fToqLDyr$8ml##Ag+>`f_atq4MZF4Of2PsPUqmU5)q41RI#lSb``dno&B(79%9JYDxCE<%|OEM6rp`RrUIX`#O2#~ZT;x|+=w4@&p7hM(1spPxM8i8SR)5aYr)2#`!pjt};#7<8}tyHjr<`@54EU^{@jNix0o+1{}CcA;fz9&Z2o(>nR7ihdMfHWO7~ggeIl$2UWw`;F&S<XB92j@W9Exan!Z<Vb?q=9tf+qK@au7UiJnxoh(j`P)@sKSYCS1cr6xhUde6Z+Ma1k!CMSV5ZnfIx)CoOz&w50q%#W{tw2zg8hs#^KFP{(+#L=O1j1~hd8o=VQUBMcn~mHC2~6Ab)I6uEV<1VNqd0^`Q>C*HsWe@1Y!iO8XpY%xVHVA=&@gWQOEc|M2#4x-Zq~!6hechjld&7Jw8=VuIQl}&4)budruImdh_1B%~F<6YARSYxo;4`WVgOoKQ^(~^GV;c&u6pow6d`Q(_B1*;m%KiQy#!fk4HGe)W4(J;ep*+)8?NiZ-Z62ff2%#!+@SqmDUkPeDZ!nOa>4KLrfesf_0e!VV$k_d%&Yw=;#FhoE+`aqMi1J!Q|Sa#)^ih4U8pf)EY?TY5rkA?s*t=&?DoGU2UUQ9_R{C8qz|*EwwR%oj+TLUWdWJzH#{ji=?yOAOz;dE8H(ptha;Od|*(=AWDW=D<FO@(o*=)X`jhi!N~-Iw@HNH{V3iRo#su1n5=Lqt&3!8uoe51bTtp7GrbeUagO%FQSJETXdkH2?=?ii-`}qvX5sRkdkH)nXX|V<OB!%u3o#m_jTW{8(;1crksG)=wP`B5OOkE0&*#I7*tB3^lP{*DKnGzNPN}#CsH!FTOyqwst~JsS(@W0M`l0;yKe=~0#lXsd7!E6!f1G3MSDcSK2%O7<+Ul~vnmb0J!#me;greKRHQ!0O_(;Tgq_>2lvi<V&%(x8$8^Uj-I<tMW?F;rH2h<~e(;d}zfiqlo*O^Ux?aCcSD=tY9b)2E}x+AX?m3vx)*%#i{aN0uIQDx?9N9MO|w}4sD>mwxVgkES?fB+gzfH6iXqZ39Dgv$rO5o5$RDi}Niha6Koa&dG6#N$j*)vK@=M2O_B`TgGbDQ+3Ai%8JxjN_<WdWS@bhUB0~d?p1M6N50+ID(|6F;acCa?e)eUN~i7t;9CgBC6S&(XZ}3sXse8IBD3KsW%SLAOrG{$cl`2)>m^Bk|^gE^zyPdnok8dm4sTA8Ivh&9j;u`;Z2J<^!;^_D7@%C`ZX;h>Z5)Kg)X>ZJ4zAQ8x@}*x?=sH(4}9{Yw&vrWBFvz2qR|zk7EOefRHoTq2-JMS8;c7aiu=oWD-HuIFLa^`+-S>iPhAZ*?Xt~to&Y89!t}}D2>tGr`@p2C3OSFZZzp+BZl4~JzHLiww!H8X`*G^MW<5e6y*w#okwX@Z0E-v5jBZFh;;ApOpFpAo;yFz-(gU>SW2oq=6{BDG5jI^q{AQP+}?0I;PPVLOU`!<-yD|~s#WYvMq^(Y)iRk(lF#N7PdCLy6~>Grcdr=wg93~@46i7aDAKgig+o>@Xw=EW#-72&*zrae#X$&@jm6#3(4#)M=c6mn9wD?7TI+NaC-*Yw_mG=`-_adsgkCv$9FReeMdQaV8h&$H-kT+KB(3hM*raH*zefVHv_7mNCs)&pRm4OutIVW5qZHv{+KWC-I_<@N*hSZ-{<SiAu2j3~Qbx5CbPBjzy}CLttJ>4i?0n@4BW=<S=A1!EolMCTT98>cLlv-Z20>iz)So`Z*ok`e+5Q0rH|K!)2SAjEGf&ozJ1u|C<*748u^XfruD=)##<OJ!0{MdQ(7D4utnSx#sW;l!7;SUk;ldRzqsu$5&5RfTAA0U$;@sop*mpkXQ4>w2l*LdCrgP#K>Eqg)Vqmjt@07|u-#Y%mkqm<8TuvwZ3Jc3#CA<l%;x}tAfx0*RCnqBH(@w2Xf3}YeJgs7sv)?#5q#%{;=hgjZ=F2#a9v8b!r2rr%5~+2h!4?C|&^aY)s^qE1h0jXj6@%x>_Yz%dFJVdubG~|1M_q&h*Fr|Decrod(>J00;D<c;YK8R4rd|_u2Np{Vh^dTLE;S1&JMik_-#ZC!+AO?x2`=HpNlsbux9`wp)E9AQ*<Z=ut-Yw}KBZ!<ZepHWWFY+&rYgAonq61e<pk|pXzuz)GIaj^?YD3Lm|a)?iR_)<2t(G_WX#*N{^uY6C7bp5(?jd_@6hPVhPH3vCA5A037dhFhBmVO{XLqJ-~Iy1{@Owd>o3Sw`<s|!4Vh#A`~O2m+n<x?=ASE*>+L_j{X?qykEyw@sk)zwasA_e|4rP`rg7je4t7uW(ss0_who%l0I?LCWEG1!^*<;-x?lR}zH|nlf~8GAt&7H{U(`Wb^s6U37@X|pG<4t8(1TAN-Tx%s(Eab#(8FS>lx`^L5~a1ceR2dF@)*u$ZTsX{;ne9s<TnX>pJ~FIcBj{s%n-Wz`Ij&hxF0gg7h^qK7c&uShMZvp3M8J|<1eyoitm(vK?tB}qbYTqSwb5_KKir3ADWGys9pvhlC;ZY@|y~a9_EgK@-P`FP`G-(BRp;tHPn|JUqe<Fe9v(v<imi(9*I=WEx35m4FidEE&$snA?CEC+o(V}l;jj8qkv=b&pX%rFj=p8Cj^4MIYrJ!G@^?%tME3)nw!nU7HH8PUa4TQ+w1p%?I|gsIvaTGj~=Ynu-a-F4HmJL^~|3h=x!J}ttc>e#ku8GiJWT*=*52Rb3qx9yF!#0!_~OF#)q=LquSoV3q@^m>Rh2-F>kW!Ms=^o34jnRi5Wm<69rdjI~md7C=lgN2Fpb3;ZBMICfNE5cv(|cqfAx(DvdpiH5Dh_iI?&lWVdi55$3Bz_{=9}zDoo1-Uzx}ac{O@KY1y5efoi!A+#p~q1g}_-y>d;{BrqK<9Or2A1nCK#BPDCQt3G$y&X&z2%PZB_QC#keYZ|&(vz7JOl&Q$L|lo+nT->wGAY+K4o;4?h5g8|wY&Dw=xV(mUWR=gL0`>Nssk|5^C8Rc)_Bqr*+63D4gB?%C3t-Ajbs_$-EZ&yNA1(i&ZF=SetXvK|4UrkKUwv9-P*9~FN)|P29Mx>?3DE_x^1}@Fou8k_TRq!bKn$z8#(W#{q@k_`VWNz*sqDn{3ZMSz5Ua79Mz+bKMo#s9^BCz@%C?E4F6hqheP|)=}q~vkyqM(h5mg<zHZW${MXj$Xf#7C$}xJCkq_LT6Q}#FYc-x%58tzk;iK-y?T_1!y5G;mu=EK!7lKH#cd)+>C&<DaB}4;Ui*e!BpI{aW>&d~vapU-?dgxlu4|ex9p6(pER&}@LS`>U#Wx^31hR7`yGl?`X8WMd{^Cy=!!OzmWC>DN&NmL|8^X|cc`n-K`bfms$ol?)B_xlZ0N2L;vJ*!34)b|b}2xyF27N2nm)t9H*^V)u+eq4Wnd70%i3dQrks8tPZp=aCqs6X(7&a3%!Rv0JQLQL(YA*hT6ZpzkZ(h2#O$r<4y!SC>NJ|omk)M}Ao44*n%llc(sQ(mc<cKe?M!wdISJToj+ZbZ-#6(RDT_$=UpO58DQu=xsTrfPj18-TkQ`v$*w;x%LBEfk4Q&FT5OOw66<rJOM@>^OmfSJ)}QgR?Mt*)%<`JZ!y*ueMa>I6rc(;&UYH+Q7Om*2?cXNeHjt6Ms~A?V=<y46j_GKMY@#=;@{Ry4j2+zfLG7MHMYJVW5UDUIDTFAgY{+j7XGb?=s4W6stNW6qvyexjyHyLi<$YMW9ilZAvqeCoxfJwQ&<P3ujH}d~5xziTxMqZR#cUksE3`TSM&~>Vct9!m%D*<ppR~hwI%K3jQszOt9uqJEx!kwzV6~`!gS&G9>v{uQT=FTM_CD&tPj6I5J#Jf?ME)(;g)o0I6r{6*^6*s6{;>i0Lx)C-o>91l03%z%Q)GZI+6q&4+HO_{ql)TWN@RbQH3-gGqm6F@n-MC*ZKgY{6bD17kCpaEuIsyI5ndWOe3<KpowI7=+k{73{q>p}Bu(9noZ2ym17-gnn!5-F|O8wSqBRAPdBJ53Rrw>=9mpgkiO1ZP!IkA}GqoA!@pqgz7yT^89lO?n1N?*2{=2-Ev$q!5P-U6)rX=eJ%Eevgjcc1~hT6<HXm-1in1>3gtBv5hQZBs>S&f75oL-b&fRC0HL5xt(CKY;8{o_6cT#Xwc9<vKe|%LLOsGx!KVn>radm3Q6KHG-|o*b2b7P)!sU=JjUCRHcou1&<Q`@%@&IT+q#ZT`BHC)5{P_@LaRl)oaa;WcRnX@fbC6$sI_hIQqzlIvU8{x!EUoqrFr+efz1SSs6m1=Cb8tLTT<mxgBV`PZZ&#me@6?_?0|D&bW}I+VnwzD~&DOp0M@tPIwl5*E8TAF<Etcq2+wTovyJL8f{He@8>3U!vcA8H3x%+Oa-gI^DC!e@F3~V}qFXqRJoP-qAHau@?JFErgr9`OP<xMI>zehWw<#~?`@l+@K^<y6;7zCuM?<;4gYJK<QsFs#$sXTIxQKDZd&DxFY;p`l!N;_KYjfm!eur)g$;LlrVUg*y|m<dkgC_@`v7hde5S$udlHA|<<Lxg3DIbAQ-{DX`<vrXM$0o5Y{YIzSzSNCBp%oe~qI+W;pww04H7omDY&Rmf%my%A(5vlw9&WR(986!WmstoQ$K)vDALaZ-qIvcgmgDEW`<`T7@)Sez3)x<dkkpV9oSf4HNGv+73jTs9+hvVGtjRWXKFEqJZolVAGl&}$Jb4_HEPpB+|v$H}Rjmaz6@dr4_Fc{iST(jwz{N_5L2Td2pz01)|<d-Dttkh|<2A7lOc*%B_)(-@LeDSB8G?>#gp6t|X4dkn)>_6^#u-axLCZ1L<qPg#2EQkCdmf?}am&^pjcqL<QhCt5isrbNrRp2UkLadx|yqlHva?91emn*F-S3jzM%>xF{-Ydfr?A^DW7WsX{C#)JLn}Z49clCgDG)<HD8IZ`Xd7#ykm&?a#@0D!5>o5{IU{fa0ex|Q`tU`*LiBd<|xwB!dc^ISpdHwkL!O5|2ubrjekC<BZ(0_JxaB@gnM1NOKS%!m|kO(OADK=iobi{{};(+oJpywyNs6ATZLXMj2*v*FTD5-`nE+rmdHURA^TRYDV5Ws5leEmmP-VcNrB(MQvd8T1FJ|PL>xNI00f5Q<jt0+<<(%Wk`*>CMNeAlK-O%uw6bZuAYu*Duxs)i*kNFWoD(7wiQZiz>7rkK9`cp1Bscw690dD1;F@w1p`&pW?!&2m{hk3QY`{fILVF(*hJg)0TAa2jwboli?nj_Z=rNhy>6y?d~~+<W)1N|OUPVz+tYCQ2d$G;m~4mXt@Kxeu#z3QVgn!f@=_ut}yf+k92vmcTwbIS!@3buch57BuSu@&ZjnE0@lv4=XX%+`Hg;E%T2{oLwn%RB#OTP{}K{2IqKLjNi`^oiz*6i#RQ;Z!mKV^ODrR01rco-y}=2;*3d^<?np5WWRmhW{4y^nCWx4r1tMQ!?=%4yH4Ncd2^Uh&HljHaX4TW6jA2KPtNz*Zb)Ym5MYw>q*0FWr{qlFvnELRsDNfu@h})}@zbyTZfD}3o~!2>dFp{ZCDs$)(ss8&Oo)I39WFSP`NS(`rlU9Y#pOjga>#2r_f+&2UbDxqT9lu<L-|?=$Bg=i7g;ZHOST-(i&|=BG7!B*&ORD@QT+?oq%=W1P>y@aJ<(B5Dv27!BE^jb*>w@m8LL8DQC+QW+7d!2*4+nq0$G~`*WZtgcaJK~g|%sR=?Ra5Kg;eok4wc;Mj~fh%F!=setqxo=-@?duZ9-?L_@62dR#)_z-LF*9kN-Wzk$q?E&$TD5#ar34_!qa6WAQ86?_VKi&>0ak(8d!(2dkT(jLZ)bF&Gidf$@TOBP+X>s!`y7_6nw0@}$iIK4lb@a8k)yQRC4KzG{*ZaN%N_D!K}*<hJ~0(|58PrZ%Zg9F;AO`5h?1P2CPYLdf`sf1ut5h5WbBPqebjWqj0b7AT9CLe!IsXvGcC$J=9Jb<&ccLMTh{SXQte{q<EO_aBvR(C|$6J-+=KLa-G_CMWzBamhU>Imb;2yBzvZ*Tve&}iQN5g9FiE0JX0{t>Aab>KS_nhhC<m0#x15Nh)6&n*Hr`ER$sBFK`xYHj0auTd+S!0-e#R?1BxAKXrPp-Et^irK95K0VOJ<Cw#kNxL;3OYn0)>~?|X#B*wm$HnnzJnxg?K%`<xr`U>ovf4mx3l!Y#l`>ljJ3eZmu(5ICro9JGqV6Q2Df(@EL*8>dV+JKsl9*}gD%723Yn9x*H$fu6S4BE_`KC&S6D768gq<>h=WEAt@~a(sg_y)ZPzsJaoe#ux!|&PxL4o<e(9@{Am|gXnoNe)DLbC<mhPtQcn`nnsBBmk91>#aPZG9N3EmBTLHy4&HMx863&Cr3&aNQ&die7Id9=psACA%{id*WEhI4pP#-4>S}CpALzaWZstsRv{t)GOUp%orJRP73AE3sY8rpNjCyI{us3vJTm&OjY5o%%}^0N>zprvuhg~s^KIIsfvc#wdi{xqi=eVRMq;A##W597@Pin%WbfiaI}-{nwHEpPQIa%y!mhwb8falAmV4(p{Z-iKDE-(UMBStPs{4v&tJV3C*l<8IMMo4%4)T!4~!j01q88)`K#~C{Pf3iO|u;cc{+p>AR~brC`#9kydt4L=%@$X#=G6Zj(KmyflCaM$99w9Qp!!c%lm7HQI0{QqS>CWRp?D{HXNZDW@^o0NR&^~!juujG$0Nz2X$3<qiA=yUa&(01R%8vX!(a4*3U<6=7AJ)rlavN9IZSh&zj~efmV+b8$*~+CZFKWB*5#TP@H*frC<$WS#z7QoVii5+=PJJ^+;SpHm-<9LzH$4&q~OKAD<l68f$2iCuO>NRAf_5!9n~ZQ);K4i}uu*t5vs~CH~H?C^u}k+w85CE3F)r`!Iw#(Di&4zn*xyOz}>B9o}rxY$6Z^ybos$VScU)F^(+Fe3`w35_&4G-g*46guO$jD1L&CwTi>}u=mv*I9OLa#pa`@@Y<5zPdvRkbV(^_HK{~((=p7%AQ;-&oz=62O`=58nEkj^{5VGmpj-|kHBzKE(lbD7`lCb^=v?KI%xK|P?FvDK$nQ^}o~x0{Na>R-DDu1Tth;L9{zR&nCNX6acD^Q@J>x=!8de4!sSkk@0fA@6q-4}3a)F7TGz9R=5R=RkIdB<GD40)!z6A^=5Q%AmLd3JlxDxUgj{}Di$mB5@8YHy$^pqVcDcrg`TtUXR0-7OPRTKgfW@`^AYxWjh-`Zng0gu=AayD^a&O#TnpSixr$p;JY5c8v{SEG!?+CxtAmvlBFr$U*NN`WTT_b~9GO2AZ<D%*hP_xM36W5~obhLm=V94E~_Y`W)&bfJ<R87c=U7KIYcFFcroVvqDu+?Gm3e(hyvc~gZ~0joKM`|}*{bI+7o>6#rKiegt57N+=Kg3TKB>4uuqw>f^swZj>l|0qsXTOu`bVgclxU`~lF^A(3?Ob^)&(ySKBW%vHa?#JakMfPiNA?NaS6MEmuazS*8<8U&V^I<J;*=WY|Ks30&lD1_hxk^ql2%zg~S+W|Ce%9}#ffok{FSveQ>K5d*dc!tG*D6K36q9*$n5TT&@wF+^%1vjTs8~)jEUx%WdhN;(s)VHpNt%_^&<A2Tf_*(Bx3pM;H%83Ar)ZvAC&z4bw6~Dbo=(FFBK$G8CQ^BAS!2AG$-{Fhb2fCySrcSN^mhu_Z4w=c;b)`A%(63}YXb}Rv$79F_1Px<y-$B1_y}&@tUSot{j&2w6DGU~mvRyEm542PT;5#57KEKMCTZdo-ThV8hZ23iH=K?UJq-gB%4NsNXJ)9{<xEM?ZqEmE@=XNZVQUgx`6LkJoq2XTSH9-l(t7UU?~mN`o<}8{7nO^(%~l>>%=99f^V4v46^6L`p#~SuMt_dJlb#Uja!nOj6Fa6pE<aoW7jV(Xb_J5h^g_}8#ksfWt1o?~Ui}d~=a1*ToNf%;Ui=&<2>iLTo_NxC@`#+aVrw4Mo`(*}i>cDXU$`)hYv(;@W3z};>1uVkjKU%(EN^?sdUMtrPj@(p1|G0uU7>l|ysu@UmNm6xPGk=XJTMi=n-SWR?x5G{P_}`#I~&b{zB?T1H7r3;k&!b$Bz2em#F+8#BI1pq<9>g1Hf;w;i^VW(_~P>{V(L5^(K>(E^12U)%bSywof&UIf_?I(@QGOaICfw}eP(#=)vz!slo70l2~{b+e4`qr12g6TOQ&d$c#$|eFK=20<CzlXW3fybjDEEv?<)k|6+HiGKeR#to?oHJHa0@x1ruBby*`zgO~R1D@?5;R&cl9Z9d9F)$hGSy&C~G|Skv=h9RGm=uoO>EdQQ!jwn_@QL=D8UZqdBcC`8Nip&G@C6Ax!NUUg-O4=xBeD@!(!bn~O6+5^REEemrb{XBp6gk+xMCkF=&^q$_WKYM=s?8#oH^pnkB6Cnq|WkIQp<MPZzNlNZS%18J~muoTvnh4JNB2=;swV{bOD&`Je-=ZHq`pY6k1>eoM_;CAqxQB|z@CkEcK{vxmA<D#_(I4TPi|-?7@7Na_9`0JUuRsJ0!7sR|OL8irI{v3wYUMVn1-J2&e27^tL%j?=PoF@ZB>I8EJZOK@Y}Rs-X60sm<qkzoph#`SbxOR%9P+Og?VV`9@S4<<i?5;~0oqMDY@~RCsa?hzCy)9Zu?Q9S6$rA~skXEEdfaajYmZj_V8}+wQCEXK$s_*DE&w7#%)+yBCGQGYv-K>H<xR{2nWRm$8d*1kTB=kG2K|Dmi{cUViP~gLeB4<f6%kK&0go)~_l0&mS=G&!H5!Y{hRk>zK0oUL9nMl{A0zSMRC=?k(fU<z4bsj6we+lHs#thUp>t<&+Uxgb*Vcw5{B(2$xN8kD<Wp#<wA<+^M(C$Qc+-TEVoWdwCzx>NpW-+ehJAUZ9O9RQ-A2NqgivxmJYIII)u|RjwXQ6$B9T}zky8V+v35hDHpH1LphD8gSD+IMpP@uI+Kv$SeuCNCzqdLR&LApgd+29~EVannE+ft;!=qg(;a9R~_b~l$DHf+zGF6J29IYgVs*l8j7+RO+gQN1^+c(Cn8HU9$1ys}2Xh?f-Iq%o1N5l*?2vNSi-#D)Bq9JEXW(tY<e7IakB05?!Eyh%DXUrHpE0fqxQacIl_~)U~VY1uxnq7JqGcH&Oy9Yw-pUI>TRA4qSMG|ybvxo`9h1Wv20-|WQRfHKs*`LKDSHxUr;o@N<-HIBbQb~0&xY4*65f8V@66_zq(ERO#7uBPB^|;o!%fzn&yBu@<jh}_qn(Au2-Ybt~cC6fd%GBaSgS~>OO}NwGqm%_F+0<2rGKSD6rtq_Jw2cPo6tzJ{Dw<T)&8wn%iKF7U)+YZ0d#Rw}z+n{-Z}j*J|KLdjbJ{Xg8M?xg^N5p2ZzNiOL>W9uGOmDPL==J^>%EH1PA2#xNlL_P+G6T23YsDc8K=gOd4@ap_(6$`WX3){-W0zX8-DzVe>67u_`yos9|+FL)2H?AI$D;u5BB$K+Xxv4i$OH)E~nu`YX>%x6_HdlrHv-1a4DZ&TRb{4R`!6wRM#|=?RDmG^Iry&9&*uSR1#NuCq!lqDp(Zn6eXE7$;5ddj?TLh+iK~-N>Ag0>?SA%;R$o@p-yphRvt@slb#x}|9Pz5Vtt{GmA3Fs5w6zZ6#6hI2p_RnL2ZbJqH7Dm+8T+dY@0OO&Zp&Y{ZZv5l8i-aW4_Ep%1!SdxU4kd=~tgL4vwC{272;^|Fr%Rvu!?o3R~(<WL0*x_d4vn!^*v^m{J+xShH6Io5INm)`L|SCRlx80^`CU4v5F<=%9aA90apTZ{g$9g5Is9iOiji`W`IbCuxd;?><Jd5_>;Ka8m)zSzz;BkxnL6TZR<a4u&1}QIl1#8RtDRzb_emQ`OkYo(M+rQYG_5R2A?n`3S>lc?p|EmTQ@An`WGIY*F(s)kJ@<1GP;)FzB+3j@NWA-ia6NQwAtFBazo75=)1;F>`?N!h1<<zIZp`u}S|@{4I$W1~>22uwJH?iVrTlwKwn2dUgCpjy67}mIZ=Ic-~PWi)b=)`hD$*9yb$wmBeb~#*L28X6OeYA=(xAxf`@+D4T?ZXkMaAShO%jfdn|w8!<vz)I^}nN;dW>kYZxt+FiJU?1(Ee%B>ZDLh*s}hUG^D3qfZ{80Aelhi4ueN#GN3W0w@s1>7Nz9NiMd$Eha}J`B4S?OJg>-gl7q>~Vna(tLN#+8AV7yWJkeX-bt2^t(uJPVtsSnN4^{`^iO&%BwQ`j=3$N@Qbx06i3vMvK8o@rMar1AIB-+cQV*cTXk8L%iuH3jrC$aInq86*Nr@_ZRzeg0T8ZsFJKQHkUK4@vof`y?ty6L1yOLa2%2QaMOdwftYISO*~gzB_@7t5Kqix{**^Ln3iKz57E;SgLxua#Nc~o_1(@mX)A}*x2|FZjN~6lHqU+o?OQlw32bU1)nEhb6?m+jWB;QH1oq*3*%w5uGH{f!S<K>T%j+d&8I;%hO%2^kFrcHqs>8rrY(mSQ!h;(l9cK#|bGqWh>lRcL2Re{M@3>Pzei++I{h-|>N0O5~Cff+y=(v81at`3UnZ}j8`y7KpF8ooSgEM87z%Q&uYH&QPcm9_DAk~-V?uPdu;74>*pvlNZiFkchZ-ndpU@AS~|B$&-`MWS+ua%#sA?Q2m)99tFx4xO?J&+ivE9ardW+tP7`k)Ssy{?yX_j)hZUyC3uhMZJan(QrMOoc3TK(8jZmD!}v&0<IGVCH2DSClj278@(Yq#-NCLKAKM{vsb$foE@X1jM^8)q#jjRn=w~qXM{*(92?~FmlUH-6DQPXJ_Zsr^)D<%pu;+SFn}1uN^={mOOfsUI97sgciBm)YcuEt`CJfxZ$2-T3^@{i7Pd6mjNM8r3Y#<@h2@v=o)OzDj{EoQBOhJeWyPLp0Z6@x*xb057*zl(8$a?;%kzVi22#2FiZ-z#u@!vY3KZ1A4`@ed-+}(Yl`yA09BU7zD$#NBE>3qN8JJgg&o;fpt8V(W<#y)q?0s08bFIec3#;q_(yfO*XQfd%GtCVhe$R%H@v8Z9N;RrrNSn+VZqdy{s+AWNsf~$iKNl(0XIfLU^bN-}>?U=w<wbodz{9h0%ej|$wAsS>FqIF}<wGiOzT0J+apif0qREUD8_eIy(2v}q>*N1<cwtKyDq2Zpk7X}rvB&r)W$#}TJ=8&ZxD4}*nRfR{*!AFwfd!ZrBF|v)>V)Fr;BnVJ?_ti!C_J?P23z{_14;c3PoC`7w>cVSxBBeM4_Y^wZDMjaLz^@S@4c}kBC#oCAZ-qhTh6mI8a9im7$yLeLw<V#-!@$v;Q<*S66TnqvQX^O&KLg(lOcc1Y(@i{jOtp~-V_#djzmBQ@yx2UJVuAR2S?T8gQG8eE%={;_p2w3>h7Iw=-SW(uYT^V?nI@bSts165(}7O!;p1JGGBUiE3#g1Qt#cwd6!_=hAo1AxkNHI=$f<s$G=ysi=NLZ)k~gKOwe#5A|vQK*&Qj@8)ffB!~si&ns(5gCYWK^m&I`kzzGr-EdB}{S-1d(3<!Wrw;KT@jQSY>i!uMQofl~P-$4v=%%xMp54HV9Z4ah~JO*4Svm@ZOUFI#U!`ji_$uXTef*{{G+&SE+Z`UMZxz#wXKB@1*8?m<feM@uM!2GUV7{+Xb#{1XdP)f#VvY@+6bD_p86Ee4}b*LohOuqp+E<Xku9)G%ffSG{IznPp4R0(FVa1zpQ!%U<f*0*aJ?QGXGeo!-*@w*9LU6s&4+2cMFnrk1PS3mCKXFAe;Qb413JY)^oaj)9BL~~Kc(mnXPl(s?&>VAmMF%M17F$tbNWH&L4s4fiLg?oCIWRX};H&9MwTYM?v6L~#ufjB+oNWw+7GFZF^LU(=V8SqzW3J2S9%t|_fP29rd_ToE!oh@Qm=ulWQIV>EBjXpcgDF)+nP1;e;<s1yX*(uvB2-3LZz|i#Cut!DI&H0(NTzZAsVjb&n?zKl-r@er@+`99ApZ(QD2e>XG+_?Jw8ou(c+9MO^g_K)NFiy3z1{bwVO{c=({WqvXy8yW}a<RwJH%nwl7KMtEaq&sDeW_gLJ|$BE*3L9LYAsXSbLryf$Z>A!+SUE)?iZXllhp%u_7<qU1ilS;w_2^S%0?j9#%LofR@w-^%V}IJ-f12&UpbM6-jq|70}+N-j^;yDdz1H*@JrKlEdJAjBVV6DBd;*-d%nRL6lc<v;@>!&z`v(-Ie6r<=8uEfxvwAsOjV~HH?gs}<7mzr$n$VI83mno0L<oWR7h5a_||BI;tEAwXr942F07Z)^5F7e%GsUll&Wu0oG<^ddi>m1QRO+C^yM@5B{s>7Om!D$r=!tymj1mn@WgM$ojtPffs7@mows4u(F(5gxn+|D!n{#!u!4??^5_hRZ}*A$=$*28BzxwC3oSGd_n#FBnCU>29T$f!u1bWWu0Lt`pC287{7t^JdCKYFs#$Rf@0LH}C#)72E5fIu?Ml_;OuBO0uB<pWD&{3N8SiN>cK=!DLV2^BeAUqX5@-=^3*~e<8ujn6rR0W;LNE$g15N`Q#~Ls<FX$NA-+wOUXnyw-x+0B@`5_XOwbytpQsFEa)Rluj-kH3vLR!u15GQ#_@*25>XA{{Zd37Xx6|<aAIul%F(7DtVDefZ4RF20+m(2ltU7P{484ga-E@+CZL^LZ_Bn%`7j@`Pa?{LMeYC+Iq-pmt+inRHlFdn;t@5>#UTwqf47Q{D0c)rv{>!@<i%7id3+NG}wA`Um8!ho~ihexZq6K89-)`zo|`YkN8TGxf$lD(aFPWJZdz>BO@7wMSvNhral+B<l5R6TrN-}X6uz!#ZyB@sWgYRHafTw(j*_<8Lpd&^?aFt`mE$yp#mmBYeIm-C15Au#0`T9yb0P8B(IkcL?JU~rwe>4{pcm^TXgJeVAX1?|<+lCVqAOA3HFdHNLCMFbH)uKHEDS@#iG5@;O3F50O)IKPM4R4gq_8yQ~;AD(jlo4(K^u!oKn2JLfVsylLN7?&}X`Q~$)T!|ft1GIReG3;Szed=v)0eR;9CL#h_X)Mq>Va<o_bBq@57;;$oE`wC6tVWsm!?6iY0NO1DYsHL48jT^go-J2yhi*$uVv`9h4R1S-^WwOt=ksp2A5zjD9lQWvG`jBzjEN391WcvWP1ibY8Muw;>TSX9qmrTl_hn954I{yLh@hoh!YH)~64?WZe=3(quft{%rRN3tj-k@odLkTXw$Ayxvc+#D%q|IgOUgri>cwvoxYQ6Rp2nUm{-L!);eBWgMPQe35wvF#-i6w;BAqxTfD*r<PPfO9QpAyQ?@4=>*s8w!@+DarlE5fQOq67ojj@Dx#xYqtk0F`7e3>5yM%ke-Mv<yWEB9=czn~gtDIUtp1RRQ)ux(*FM=yn)vI-SE4cZrY6o(xTF<dEhtuQon|4DWGGt?#S@92Y|*czTa!ov?ebLODfMgd+4x<;F2ih+X}dxd>iC{B3T(K?#i9N4wQ_J~Tq^EQ$9LtHiIu?_^ARL?}fq&7p_m|J(>->E+p>5J@Cpah9pl(Um|;_GlSnif*rztYT|Vz^jTP}hbzTcZnK49F`bh<q?GBT5j<Nbjt~H$pug|E{Sn1^B0YF4@VuSP);n^atL{grn#pGwgNhqIv2P;RQ%5AyB*Gs#xldi#5~ESQ-^OHOXF7zay7+ueSH3cI59>LGF5)EpRcpSLXe&qu+Y6=e%ZX+;Eft&Rkokhne$YBB+?{B$tlk%@#DdE=;jE23hqQD?|)IVbPBDd7ayI^5;#f?6HB)9Utu0j_^BL`SdA_>@zGO=7tqZ&L|dH%qx@Qt}XZcy9E@}j<PFyr|qRl3OEWG8rIM##0S?VPRFj~h(Al~&ks|+OOUdxP52P#6{?z4MK&xpY`1<W@J{WDfVf@K!cZPa1kVtMiYCdf@cc$$(W~2V({dT^3#5{j7&V0<;6rvdRa4wr^SJN#)G(J|<L8g@LkzP;h+Ru)!7ZiWo<`I?v|gcHfKIg|yzhDwOXk6V^6-8!zm*IjFs*pcO=6*R_~Jc^IaBo_mU01wTC@*1Tg$6KUrJHaV|7ub)<71y)5NREHz}G<_7T4H*?#@^V|GQ9{-uWnkQA}r<~HaSRUOJC&gPemM%Q;FGDyRjVC2G)Er4hsrphIHfFl7A2_Hq8*@y;{N4Vk8DZdGk5K%vbS%)m*Cf2<2%Dxo;;^&NExtGmTCRDF%K>G16^Yt!&RBC0-aD?3;STF(Gut8U%p(59&v;=nAG8`?|k35dz(m6S$CrYq|f~_zp8<XX_T@61g;u!cK#m>CW%qSkh%u{2vt8op|yF=$2(x}Bh%ucgEuc1K&;}JzKOdBI-$Kf#VbS|S71-H~V*-PDpIjj*=VuQ-#d}K`0BlgguDQb%Vb%lb51jbJvW*ly;R;2pv6dU2Bhe~!;m%|{t(ND=60$3IU?5<-JKHe8Ym$#BSD`z)O4r@odwd&D6@_+A>tIOr)40E)Et;=NR(p5!#U1S#Vb=+TA(5zNcoM5@==J9#Pt(;Z)-I+m^!zltnTsTfP1a^9EdpNfk()>vd=al7x^#i0)i^9_JdJc391XAj>5~-vKFc1rBPi4`iB0wx_HPxa=kEsSzyM9Tv@*Kos;xy_ayGH=|9pRR2`kmAO;=*3iX0(;f8=uyh-CFU^l7;93TV<*upr(lJ!n=*$ji}2ilsl~ol+j0$!lg83N38Z&k(Qh-ioo>P86KHxq09+})=#nKpUPaw>~2WTm>6d>jU`3K6z54~*KC7KOdGmZ{v@U2#%_A4og}jE0PFtQ{=w(_m<1w=fz8yIqq(4oeqa4yHXIF6<5dtXcvB|=K8{dZ`PXE`ie^9f)KgyXDmkn$9yW{$FHDbxb3wskg@i5?osg?~RZ<<wI=c*d^(s!Uu!T&L)j`Jtv}fQVp8+pdD7kVh>Vva22AL3h3A+Lleh!fdK)Xisq7`{yVO=2DVSCM2g!@eQfbqt`3SnQTEV&6}QPaCl-1%=d8v7TdM}EjtLvKQ#&-$a&pzp))-ZbQkZ*oS8cu}OePVZKtjCffSdN(eJz2l&92KP$f!Zu%tEhT$H^L5&ay2H4YrbW!eN6mASeZO~zfkUBgBX}wN0|Zd!d-HokpPu=oho`KRa!&FZpuhl(6OV$K#e%J;*HA5;P{X`hhKMbwhDhihitGe5y)kqybqEyGz{Iq=|Cu(w`v?0=n1^@RJ4R>d?}u2s{9b*}+#*%kAARyrt4ZNq(~iel+)}Z$`4Ab!^WO5JpM3mqwbhL+-_VaT5bA4<_Y$ls)8^FjeQ50=gT9Vl`IliV7_?};eD;Jqx>)8#zx;Vf$r!ItnZz#N+6!CBs<(p~RbWp!XN3p~;g}!vfeI{{gJ<SitS_HWg1$vMGz&H)Y^MUoO(3S=0<$ppX8miY_`OUo*~SyZ*iP7F@?9@ib-|3Vz|s9O?T#q`Rgt(qcqpBnTB|u}Mn-!eMV#F}$GTzm-sFC<)1tnjOFS17CXA>D(gC^B5=J25d1vtM)I%9^?LdmqB(8mC5te9c>F(Bm-e0}aJF|+e7j|bjJe!@1TW<4S;@Xb}kH`3xdvVdq#XZ*}K1J1^mR9ITns8=Ew!JS|DiP~OC!?ayMFjmF5M=nY`RY|9>-eYQAxy|!n$ZK~<OIPu#3$DVW}1L02?7AYEYRJm0n$Qs9|5rZ!)pEL^Lj(LVcw<Xp#q$xd_HPj9am2$fwB2^1<yy8QWC@?BR$FUq>bEU8I~&V7#loh*;7`dwf6GED<G?i!%lDDd7I=LKo3+9fp>p}(}hzc&euh=QZAKRD{N*uCuBXr*U8gUeJQtAJ3Xbx-m6!U-GurN&n_+!&tARy5j^}~>V#<s;=j8iCKQmk1e;5x`9yfCb|jUDwQz}U90DU5n@QA2eb2+WxXYQ$i&@p#6T;EZEC|UXymBlnGeOT#hslZth32ojHR|a6=&D5?U6XKKW&Oj=3y<O{{)$WUQ$<0EGE@;!+zIb}bw{oHAKx|SluT+Tv|fY}zmnsW?+z9Trz5WGogI!&Lm*6{l&xiQI&;pe^_1ymkc#IK9<y)GDmpUnK>0F}d%r<_?Ref<OF2!OH+!ZhwVUUb?Pj?vJ2wW8ZMWOi98+^aC=lDH+s$zi4f>=$!@yGpM8oqZ&}y@A3}N3y9GGd3)(&gcW89`$$_6+#UWRuxY#y;)x?MM>UAuR`yar8Yfe^4SFI_?ftgPAXwxphI{S*A}k@F*OurNJ$iox(&+UP{4nG9wiF-h_Y5zS|3HV_3WV#4Kn@3^UPfJU}d!Zvd&%LMjB4D>t6?84r2mM7>g#eUj9K)kT$)%|BR3P%)i0F-T>ea$J`uGl2dt<nvK(Pr&xjFQpyd3><Z5wm?!5OX7xc_s(5*BOY0Y!=|P($%>M3xp`3JzL>sv_)i(N>5(F2Lg6NJYhm;x)TXeBB8j$_t*Iq(@w!P@yNB_r?(5Q-Bcf9&ATh;dgIQ)=|+E-J1?1bB9_%t-%Cg`<fJ9fceZ(P=&P%Ys_u3jS4Z}skFMSB`TfxqN<6`=DZjN?TnuI_7<BNR^WNFHRt0@E*P-i_xP0^=@kS}DK_If>j945oG<9Oq=6Px6`Mm>R3Dd`xP%wNAX(H3qnF3yn@FC#B=@XiDECFgHE*X?z0?GoJGR1!}D3!%fjw4K@+Z*&=hYW&7Tv@yi6jq$z&)~8e0U4jN=c!QbE?4?H%?gGYCLv5I^OZ~<oabYXcoaG)(Vbn)eRnZS1`24+zcXv5%r|OcL{iUq*AEWO**s<N6^lZxRb$Pfz%j&?!#%N>X|5Uq)CsS2DiHfI!h6r*iY}mztmKD!Z4&H11<|V~Ml6;dy2Vc(y6Jg&QYTD)6Lkh=eHsqmF{SJRdVrg#yy=RsO9CpQbaj_;wLeWn78|!%c#E0JKNy*-#XWTCY+a8Hu2hR<HX(_=@`IZYk}2=l@k7Br;N2jHB2pryjo#PeRV)iK!E_XUAq3x<ZY$>_u7ykBOzc&UD1H1YZouUIu^{(f+CQs!a7b92A9}WF9(1C^awK2BJ=6fKvJ7{2O1K_3dyAMy0uDGe)7geG;V8&z3R9%ocrqgN@`BwsMzII;pt|<$-@g6%?bq<XZ{Pk5|K9$??bp`rx7OQ#bNlt}Kfsgk-u~;`-@g5m+i$GfU&4>Sfycjk`zIDW{Tkl*+1p>hn@|ImNhHZL^I^4pf*4GaSxW9eCdcRv={qQa4Ab6|>=baYQzK1hWAEUz8k`iY3jZ`K$waB-o6LD6E+PXvKET>8-~w5iPaG*V7K{=4>d8x>@`=W@2`9A^djJqHJ9|gEx?;wa%s^75Lxw}YGa81e;2`WucrAyJVS6%1CuYww!m_uV-<97mG=_jC>h$9GvW_5*4>nu*7FOZ!sfKd!qIOi>MSVrWFL-6|UqpJ&_5OtcTC}D2iC+}&LT8=K>}{zAGQ5K0i*EGcPHwHJxVFc-_U&Kaes}w;+i&31eslY43%>l_+kgA^&)@#+_P6lhx3}L_tT)lCHydx%%QwkRC0k_T@t+eFe*5;%E%@@?+do43zz<V&0U4nsv|;tupsk-HS$g|tK#8a|3w}Y;^mFLKcelTWAK|awh1lLqBuGtpcsV8<x4f4kC&R2JQ^?T9!=g1D385M9C8ShS!$4dzUY2Z(6N8Y;C@gQ`%&~3P^TsBeE;8R%9w2#@#NBW*>&k|9%(u40kgE}<aI>O2-nnFY!f;oLY&}CX<{8JrO4-6f%--Fa0JBl)M^MV!slABdRv)007jXkq##vO6wOU4jYgYzX6NUh4=?Nd6{B@v%<krv*jj)8&9e?Mfwwp%k6Ha#0b(erEv2uKvx+EHo0j7R3y22|cU6>M#7#=fGM)+%%&oSHA3{^VlIAl~23s|5t6nTMyLW{0_N=g%un1My(-~CQ_>7%9yGg=_tO}65ZU&zBGn4eC=nWs$)jmC5%=w;-^hJ%A%s_)K^^-{c$vJ8JSfPg3J7FVKtO?-{lOr9)S`x5bgdF>31L>S}Er1yMdSHS_b9JBb#^ChtPv#0g7Ckl>xm!p|a>7_-6r@VxEy52ILUN4(7kZEOv5!)3ALpG*Q>Cb0MT^O95b?3vjXH!yd+li;bF@aIzmh<#U$pF^IaxzQvYh|^86BD1`RFS-GnbwAKJ8ojl=z@*uJXf%K>S|FGcl2$xhUSLRq*G|K2X9-bQ{vk^L)VYpu4K|9#TRCU7btE~Z!6*%qeG<ejmiFJDfCoMjEa6<mXVhVKg!gFDo9>nJ%Q%?PihGM#n;0B{Num05(kXD!Y2dDkYHCx_mLam;>$b57nrrILT1cbHc7kp?%}N8IQbVs*3Fn$A}4iWn2IJzx2a6i{>txmCjRNU-7*F>F|G|9ziI4N(W=luZp0Rd*4@>ToUDLsW2_ztJ&B^&36Rya4xgTbIQ41{zhxY!l`0&|<UbVm)9v@hF^wf7H>;XnH1zG0gq$|6gGctoRjyuPMHE=oht;FwdUe+TQ_V~d=lC!-3DU#THoyD;8~ySF+3L-fh&%}TmhvXuLMF))=PV-=bUot7P)klc*BpfDUAQ2Izz$tbEd<(vJ%Kqz_DaZlML9-L!r2hkhAndRl*vg@>0Zp`jgp8yDK?>*d8x9ooU{lDYTYvB>QJd2+;f-qwBw#l$f7N-(C8E>0_mC#gyad#hVIuM)cPK;<K(rwinGo8x|(6V^jTIHYtEf}x4SlH4GlN$GAm>1Vw6zYE9p4n8qVk%qfaYcOnW6i8qccNQqHxqeXs`vq~6FDxa0tHt`n`2%<Due^RCy4R>co-o#@rCbe*j9=8}6OdycSpuM5O)I79!<$>=gfcz@`DxnX4=E}qQ_C6n-Ae2vY<;vHiNIrd^7$`nfvo$PB*C_}REJN})e)#OvdMHlq@h2{60hy)KU85uj7dhv@b-9qwO%kC;0wBg#bQw<}EJz6zaXbp9l-?KFVM@ZDo+O8+>b=KA3{Ty@8H<XHJuGnIqsd%|aI~7+eel_ofm~;lm8zKAaNgFC0^uZBlwwr1ljsy(A_!H~l(p*3Kjttl*E%BDtP1tffhl!uz2|P#reEc~)p-qg9wBXMTHuna<eb5g3eM+_YL4|jBXLKdH^FgnJ4^P7f4^PWsS3Jb{%MMUY`033JKVrO0c+eEQ5Bz7{{>R&|Z~yD<x7O`Hz5T=6zq|bn+il+dG5q^$*LwR4c=}7S=lqhaGvD0)=Ix)NzL(UZe`?+S1HOss(SLaRGpy!Y3ro?jtUtBh{vkd21=Ra(ap~aNaMz15gp06Ez6I==0>2LD{k}fvcIV-vkHc^?d(a@zjOVi{j=GgG4OW@5hrGC7EN$Y4quSwa^^5gw*oJVW)%K4qEnAPZTYYA2ScgXkFTb!Fz?ljt3V8PPgwk|vSNE$&UznNAr@(@+a&7>`7d9wRWa~-o=>fb)n!0c}73dMbwvNV>b@Q?}nop^FC2M*Gv}be$n|nVHXgG=da?|r(7p1u<DM^8n<hsaWfmnAYV{@kN7_-LuK)@#x;Akl~M)D^5@&o+8qwsXz>vwcaM@keVvz$}Fpry>0)_h7iW-MG-tMpOPG#U8kfLK9mhA%J(ps1I)I4hW0v-R=(w2#O}ltn2xD_T_rvlNv&4-%nEyvVHI>`kq907FE4)zRcCm~;flB(#TEsjvk1#=&%UPsdDUr<q<iVJDnTt>7%Q9(xwr^(591f?e14YmEl^0X2%S&J<6Gd(jHu1BYLuDIDD}Q~0YV!+!4~<Ox9Y??j-oFvct5M!E1KjAPKF1Y>ey?Wxtp3fHhtXLA@?(X@P~sqrXpC`I4Lc7eqA7I~Uuza_Y#E*n{Kmr*iil!YYX#nee#ScPY<urAZ4G-h{`F<IdZJ<LGlVIif!9oL{1`J|~;VBm4Tad2|9jUd}FKHFS1_C(aynU86Eg`Gk|Oo{e_fVvg_HL6|F(9X#r5r>-pS?!Bd8i|V`5b6l3IKPE8bkwDU)dHt{d6!HRNH8>K1f|OkGdX+J=-<TYdb<X$l)C`Zdpz?Ujk4Kor4ry7L?N0A>c2;?6_dj%UY&=qNU|yE-DIe%52x1CDpc$@SFGmF#W|G_#dKl{#@yC;G(FYs_o@u%-#ol^?6uJWi>A;M6oMxK%gPD(G1OTEp;|1jW87)n<|ITBx*!GUH|ecYu?6k!^VAufK&JEl%;RH<USF!VZdWZc$O5AYEVFSNat+O7LoJVgC~ArTi3Ne+&+E8t<z;2=l&Apq-dV`Y<Bo<2HAAvh_k48a+5NCPBmF(4C-dw99BCgmGEh8QU)W51aBf=QgVW?h1OYul{q5?}4mvNPQIXxdQ}|R73^a}+Hn+)cXCvrirrgma==<d6cI{)ZG^uV2<~zJG0L}s04EpOp)jNTJCWQO8Vx6LU76$sjHF@n?UB0~-N}T*j@OspllcMQjxrQCl+$@zJxbWu@{`my{Y?km($`f>So%A@@P;Z(xh7<W!yLAtz`I;ga(_0vxFVzXz&o+Im3@R-b-Eyhu3b*0=_wN_o2M-<;Q_dN*sw?#_uFR@5akMR0Z?a^l9yY2h(c0qG^D?=%vfFiGvdkNj{k^z~qNtODmr5<zqBvUnWWSD1bhJTXZ{mG<aP&EBQh%@BK&@+Xck=3c(5Q5D<R_#;5Fb+{8_ppP-iZd^^2%X(GhTJjMZ}QYv2p`>!=*}A87=eUzVyTPcWc$pQobm7gJ?s=?=(>QkQye)%(%DX!`w_TT{12wcbr&%-Y6wdzhRu4iE$oQ8^_TokqbI_(x@H1NE05A{QI$$r3W-#{<PA7O{y``cQ`b?Dcn`LL#L*k)X3t4A~wZ;QQbXZ#;*2qd-r4~JtWrj(3{7#y~Bee^q3(5PHKBx`-vcLbXmI=%0P<74KIU<@U$l)r-B<%6+}H{##(PTkM?HK>j{fceq>AV&q^XVLuALulD!llv(*U)qq9jcKJT@06ap+?$U1t$2bo_|2h$f>C{2_bgk(>I@Q958wX<)1M#9=8v^NjydPPRpwIgVE45{eBpy=0eIPuG+QVA1xm{<zsXDL7wU?t&yYY$8c)UlV&!Yf?soUMWQF=|@<mYJ(fjBzFA3WWCy6!#3{t&HywH5}P$({ofAnA{3+E#ZH0wxT(92dxy>W_-Ds=zEK5)9$(Ti|j_3BNO?PQY=}b;^iFKs>4gv?4)*glUEx!`m5PF@|9;uoZ6_hhN-+)h>E0l3v!{Yjk_<IwWQLuO{cY9zEfXu;N~BTBG{pTU}?Q!*Inm-TY}xmW@w>K!Of~jv1ihl^97L7CnxpY9m&aZFpim>g^KdC_C}vf>E>5XHlLAbUdF{l2_8uc+sLrWNj-&0hk?rQ0_$)E`D-+_Fz8ehj@wL!w3xAag8q!-B23)DL?vZ$^J(u|St;?Hj5aV}E29a@^YXn~$V&FErFvHbb!B%{-uGD8RV>?kZmxHzN99~s!-%I|j~>6j4D1r5J*Vx7N;LmNbU;%wzkAdQ`SjrC<{%u^&kSECyL}$EFW??aXkYzte?Ilg@>Y{`!{MLuTW&IVn<6V2vx{E~P)8t=z-z&ziQ3k+V&;ld;gkxX3td`AXnRJ!Zq^(0hG$#s=|SNdh+$!Mdka`W(}jx2n}gA1sB(7!Cpn&s7Cp>gKBipYc*#+Mb!?qOF9!$!h9>fUI18hQ%t06+<wc9>)ByP31>sNwXowvZiVW04MG$zwC*%eCd!PP3puZ1&+i{H9SG<VGfb5CmA|wRvIDIz9$x(k$K@xTf-L4B0=k<fZX(zDSmD!}(j)D-$l;}>7|D1*lmyrb;9d>{*4~9bNk{#d$gT7{fH11_ike3QBZK6nX?RHyH%wjH*kah!p`mjXozXnr)b+s79T0DWP)qq$nz(uzYj*e=(RRn=0OOd~Qa(wXgsl5zYD4G$Rx2@Aj7+hrHE2zR1v5=B7bTeU{Fr9*!!@R;&IaQg_U)+_`C0kYc#fX|(Sty)Kuaem$qO0-}p$^>P@}SKRHIO6DKf^8Zz6g2d_Q|2}ikS0C_z@lR1swxsn=k7eAFCzAK26u<bmvlSfxeVU^;fSXT(tXhkn%N*0-%ING@cd27KEutT@sqh9>EuhP<xCG5P|KfOJU{cHP%h&%Pydq;9p)__;}6&0kFU#uxx2IiAN=UhiTQvb(YEPmf`<+d^L)JLAQfpFu7LBVLcS^L#t4BEMfLyB?}_sOh%n~JDghg$TZx)zDI^k_Su_S@MsPbDj`-GuQ3+5QrL8a?VNII9@WtsLTJ(8{eE9+I1m=wnpvNE7TN?u!J0AbTR52yr+Te);-Mdn>Rv4}P1CZY4H-#m`y8#EXW?)T1d$ECq+GOEtz)5k+e~-j6uK$QJ#(}Zk9|pK=Xn-=_ORwL7i7Ss%VvX$QHm011+r`;5wQL&dLD-^5Dx`nL9@KHXOZ__fdNxo8oE1~jrv}x=mz+2Ib1K7QhutqDLG9VXT;<8lWFl-O)DRA-aalUQmk<S`eL>fM-5dn6I!z)VUuF}YW~5IA>Ce!aRdu+_Pa^ZwSPYKM6VQ{0jD7q>GUR0fIDjZJ*kVF0jVVvh2FH?A5B5RKA*Z!hZ}wchXBs=kS=zfW&FeQjDH-n61j=zUXPQA&o+wtXqEm0(YA!^jsUY;=`=3Huu5<-=;1o_(Ahd0jXK<Q!ouU!EL7BBUQVL#1dFJ66KiU*H<{=jK9NpjMdQ9d#f<4<5bR&)xMB`m)!+}LOOTvOUa+=Hf!V-t3&0mlyk2{GY&#kHh4zRP%W!|i*rd3kA;vU0%r!wFa$S%s+%kgHsq3AXTVHuGObeRE%_*LTK}T}|NEk!}vYt`XiZjC+;k-C3%oaH5cy|&-1x67P^Eg6LChCxgQ(-$ldkBeFqgmFGdeB~Qbh2;XtfG-&ZA?rm>Q3=H)6(<{3ty=O%w%4mxfw;tr#&c%&Qg)_Ra`bA`UdaKUiX^N6J^|&xc*43S~QWuEEA$IFXXFwTx}9FD>s`j_(zxat7Doe@pi^M@vp`#KQI~Hd0&+9DQ&p;%#_D)jYaF?%7%^O!5v$ik5St!T#BR_hxivyHE!CEqU4}AC0gZ+L+ZbR9(eet*nvx8TF^Vr%2q#^wtGF{UXWWiP&=l;VyybW^TJTM+m-BGq?1ll9+H+v+@>F6=GcR3r?!7gBrl^PO2=27PQwZEQX(N=+$Bm>w^wiMRgbryBm1iH;%a7^l}E;O3D?mH(}08xiB6*>&jt&=mYZ-{RbtCzRwRAL#&|vP?H+a1mCXnSO7TyhB(DA(JQq2MC;LR9F@cP@!)P|MGYHeAjCM=l7v$Z6j7k_9;YuT|=G3FHzwV6-lGsOHPW(VK{Fs(1n86hIY}XfEFggdw-=b7TgJ4_`y)acg2&NZa`YnoolCS+(YhB`7n%sA^`H{B>B|@gWNzhIv{H0iO!@HlBl-JMmY6A|Ayj<l8FxW|As<gsV6KGBfD@Jb>lOrR2<##lv<EY%O$ps_UiHP#w;q6Q(j=SDAu4vQAYQuc#u$+?p?m}$Gj7x`pG?7u^2{7{)lJS((U3~}AY`%HK*%wV4sStN+84=?5GigxN$!m`|<~TJH-4Kl}B}GfG$Dz+|68g^fC8aT(3L|Zk{99yk3-!x7^5=~Pr;6%}VyZj(T?4+*-ZDp3%bqX2#gqGB)tg*3ki5#Zjbwb0Z)YzGQ_?1YL}+bjBL0Z3aY1}`YKOJ`ovbUOCsndhmBMDK)mpjBt)CPq*h<{+pAvc-Mm20#8^^KhKQW~y7GP!l8mXxhY>lV5|4IhSBb7KO`<QpZ&N>54JCq%u*c5m_rJ383MFkdA%Trpq2c?ohc8DvGqym_0aWac1L4M-37~U&<esJ{JhCS$)?n`ndV#12JyFoC#n&76Kg^O81+wbv;4ci@qA5nL(=?YLr9Iyig+aR?J&pdlR>#jdC@+YJVRW4fGk}Y+f$)#GeVb3Hd*F>&xTgD%+IK_9J+N}P^0t!{7tnX4^mxK_;A)-?Ob`7EZWI1t7x~@r2wJmg9o3hOjsEmmEGqs%+T&Q(h#Q8CNSM>ZQD|)_x&_dnb42`1Pylah4UooIC8APXOYr&t%$rx43I5j#N(fSlIX*tM4H;O(%m_k|&1{&IG)PKL0MjiScDS)yRJ>+Sj)G=GvKBk)T@LLwPd4k$aR1Q6UcRkJ{2+sn3HLawv<+LK>*c)~H`LGQ%8dVuMBJ78Oa!!M8==}lySj>E1a@52sNC$mnI!qpN@NTyUOaWC%{-n@C2D}ga!W&S_#7G|1tdFAoAuEifkW@&vu@sJClf=v&IoTZXzu3-t&&l_MdJE1stEljs$Z;knT8@o@Wuoa=Epb}n_56u)^7EsI+)Y;F!bXg;0^@J8YSnYCB(1NOwX?MfB&<@<AxIyPQSvZ*T7Rh|T>{WP*~@_@1)(X^tutTna7<9tSY14lDN+PB;m2hGO;IiCQ&WB7;Pou`gV{OI#`96XL-qoNwXEKDo8^uB?&ik*mZH<@H|JB35SdnEtX_&JM0KNq3yB+r^bydT`l(rJJ=VUJO@~EzzQ(7BXq$0z@+lo9HbJEWjJE^K{|v~!^c9Udk|8uxFjnOGdp}D${{NeM_wF{XD^K+Q`4rX8Bn3!Ch~Pt(!A7Pb5|q&9%LFLLp*YnbP#_V30E_}DQVQ3yo!H5Yd*Z~gSJLUc-08V}*Xn!k$PdNxL-JbF|KCM^g?y5`ALpDpr|J{{O6p#>XJUa>r_THAv!B1+LeFL0iop$JzmuV+i&Rf~&I<fo5~uw92yYE_7yUcrf5!XJ7f}BhH^jgl0-kTS4-e3d7AthK+f-s47?+?&Oe~_&@e$V#G`^eU6zibI*w1L?NPKb&8b2Ami}JUQ9lXfpNli>qVkTf7*wb|?I`j@FQCZLs4>hQKgKpTa2$SnG?gQPnj*FabV3%7U!6w>TjPr`NGLtZbEe>Y@ay%t4RV08U&j=$XSZhCBaNx^c;}G4?;HzdYVCv95=p(o!VfMI*(a<<=mTTqB^&922vSlpsd75k-HL=12U=^(#p*EgG3kFB}E$8NJZXR71*owa!I3o2XP7M5WcR%PlnlB^nC#STwr1|tsp)bsd6RCAVirnraPEdEKv8Ct+NrC$zCy3O0^<8!x(;nr>`Gz?EQNENSwm55GY#*OIT|*Q2`6S!pwZ?fgR#y#{+S5BVnec)WtpyIkYN{GxJ0_;zD5|#>TkmS4jY{DbCSx^ISiprB<yE8YumGcI+;Ms>>D)tM`Ft^Y&h80D^#`4n=hU4Ryy-Z5(4i!B7f1x92+vesjDjsB38#=DO*S^mG82f1NJ@}E#B3qp1M+tgpeNWK!lp3DPE(+{z*FGNyC`l8nNEO&-75Q))s4;dTjkaATFpR4Q#}2UrLS(V$4PTVMWOPn)MuI81t%X`EYZ<gPomrELqm;IyIXIfCUR)vUC{mwQ&C+soHkjbc2)}lN(JZ4bARZRR?6fqB%iO8E%+@z)Y}&&^>xLHSlFIYVPP~1Q8to~t<v#gd1WO(O_)V1Ks`z48b5EM%tmFSyi!>!M=@*FsArxd<IRL~mS@V+L@bGRQ%o<%ilqVGnTW&)cOv0uqE?U<o$3VjI|wTw)DFmlp*xjy=VFmxN6IsNx^9Vho>2L2v1*!?1i}l9dLN%)Jt7a4Rxl>iZZ~ac#4&Zu<hp@$$ol_qM-ibuM|<dV_7p+B@ju2<1X~*CC-O8-BEmlet0xg!x_}s2sI&>pPUP`{mIs*?NDH~4W@6DPgOrp&+xeXf`C(AiV+>>ZIG(Q`27ffPj9J>OpMKEFTS9>Hc&YvwIPIS3O1iC8)u*Aijdv;#ji)&jT+ld95RsZ`oVRym8K=L;7{*80uc`j*cP?%(=MS^vY~9jqtxVf!tya?+Kiq5U<E_=(;{$dtYy0;i)?|^5^EsTFQTAlfN{U69-;T9MC(n=k@)-^gOI^y>k^i-wA8~Y24cI5svCLEIuq~YA<FC>x>D^(wGWUv^Mcu?a?VaQkcI6a9va}-KM;E@Nv6xj7n*W98THLV&0(C0lwiNz-Wy5p}y*sGgKn_EHDkeG&3%h>oYx$VbnN?~>q;`IK>Li~fSJ!MJFCbony*rvo)*4DC4kq?5$$^Zu;ZAH9HRe8^wnkYsC&*kgZ_ivalK;{yZmDL#v<h^4pk{xa)b#fh{Yv{J1$5R{aXRd1VsuUl{%EXV42LxK_SuYy(pGpkXz%T#+zopYXjMmY=vRbwQ5Qthx|w1X)QPm<kvYdo+}OHpp!7$X)$j%ciBJAbVLMIaT6+VLkXDq4J<h5Gsp38VWlDZ-qyY1EjSL<a^&!j!qOirlm{`WWRo<i=pYeKiEJU9~_AaIBly#=PN8FQ_*VbQMi&u!65?HOngT7)7;2e*+4~-hs*<V^;+(NBD#7!-gYGs#UqX(HWqjs~1!bd9mOjZD+sE^Wzg`eq}&DNKfD~lCMe8yW9dhEjn-df$D0j#gByk`1rQ%V@*Kje5((Sb$=y2V^J>W6N2CkX;QViP7KZ|WWL1sw5>e&|vr)1F0JsMQoUwy|XRT~$}NQCEuN`acGFm=Aii&5QT3T4B>4RqyFSLK-J^m#M;Ynmv+(X{}XwJZ7v7>LFoX2BAAHZJ@3fr;J>S9Cn8Av11`;KR<f_uO6PiasE>+E7((^=FT){cW39WH7|&oduzUcX$v262RM7!H)l^AMAF3{)NAbK*9?J+ea8_%w=tF3BRwnPBg{4iufK#}Z%Z)UPpS0hI5>QWiN_vO8Zt^ph7~<$B;DPsjk!kirN*-%>2A&vlI}eI_w0{nAD;c6@W0=k{rc>GIcI-3`@M7aOZW{x&i)CW|LW}LXTNpMe(jw727dhuJoyNozkl{8=j^vo<O6u{kMO^L@SL+>U~#PCkMRA!!2f;+|L<SV{-;Bg{07SXKU4uW^$$jLW4T(J!j(Gv9Cj7#BTiAt(8LcNOjU{NMLP^ioc|2oy@{*&89bszeMnzpy1O57S~y<qM~IZ{z&?HV>|;u7`2e%kJv{pw)8mQF3+w*1o;6QXVcnYdZ@#v%Ub|VYR;s}BEmo?OTufJ9X@pqQbQVd;Nxp$0<_kk&OOHJLZWT1bxNddQq$S(Z`k|CyNfH~QnfpcfdfVMU>Gy{FAQpv`6k;K4H~EE9^~5Bed&P`>SR^W@yi%F4gcAz-oFOH~TPVa#K}oSn_&TOkNUAELZBa*YD(uC_OeF{Q+?>tj9YW526VCNRv5hF{=}+h^e~RQGevThFnbrJX-g$p@He0})ijT#%WFqj1*nE%S1R#Z=9fwrq8``%II7J~mJ^v|?MW98D<jpnCAT3>;p~`a!mEC@^h_;@*s|*_V_-5DBI+&THZ23O!(XwF~`B8k0+|@j#gIJwSBsFaY4cNlHZqY3x&KpHMyjrQ<B#rr2jo|**YMZ5MO>xZk(C`)UxzTr(H3fQyw`$3aug}>>lt%IsneP#2`Aa;wG{bP#3Gp0OK#<%kDU_J+9-RJ1c(`d5DIw<X&OW5&KKnBsctHWMsmHkR`htEbR~RY(BQC+)1Bl1@+v0}!9R7L`TeuXe5xF3k=MfdDF`X-iorRp~YCHPV=0@N$K&daeNV#69iIpI^zN|qgWA7o{09o05(=O)=-1794R4SpW8{sQywk=?h)`5GBIK^mBs2$;#mI&06J}p%?^(~|evT4-UIO8dWv}!DrvuLBHCKQ`;a!sP>GJM)ZYrY*PzFus(QdVha2XWcr-$qDmhLv@dj5WI;hjBcjmI!GVc)bpW&9~7q>;Oe=^9x`TfEol6fe_vucA<Gpg+a-gkk^e{SYeFUOI5OfrDuLId3^oyvt$Q}U51G)lp|8Zq;<xgrX*$&VRqkKv(nPvZY>F(TN(IT?)|jWY^)>Oc*hY)W$ir$!bN<3+U`Ad<kAPBvtK_76r{IqHYe%->AC90Tzw{(X<U@6K6@WoD2JGSqKL#^J#;83IHjgIf9L#NxC}p~3-rg7s)ABZz`sZ{KZOew)nlFK%p6bW=U%MWUwqay6M30tf_{|NN-M8Xq@{!I`q$UjUv@4#@)mQJD>o2$4og;Rr5k03*Lty5)hzU^PvoQ~VJ6EGdCczYdTEg=vLs%mvUI`W=?FN-Kz>>P4mfvvgFCyu-kk+0{~gr3hiHf=Uv!a?q1mnjEenV3h9Lq1DSTy%$Z<Yjj7fiK&QfuYA6Et3Q6eAHyBi*`xfPXDY;g5g%e4}*0(wUT<}vm|Pus3{kkYqHORt*Roy4pYe;L*1@oeUS6@jx^m2V6)l~z2PbYEfz33NJ=K2a)7g|NCC=|O_#UNDekmXT^n*NuJv8f~RH^cQcI7hkRxF^z&Yr`rM=DDqL&*|%&=$k0BtkXKO^PMYDBBvO^q6^+ejt)5gPoug=lv|*$;NloJSlwQ^1&C+Jg6u|nOVQ$daAXKSHq26lmVIWPj-*yZ6nR#HbfQrIH@#N~X=eFRw&7oU{TX-pU?A-b>EKk^ns}r3f`;s3ro)1trE8ggQ3j(-s^L`7Rg}W9UnSOn^U*!0o#y%?F+$&V>vYYF7_uT9Pj!JYngN41_-5wfB2)e`_Qqvxcr3gcbdtbw+YzWFO@O(+u6Dx{!6g|XY8GFMFArE`4ZoI2#o^N$ClbJa>y}gXPv9ex+14R|GvBF}*Fu;=L5s>}W(&m~NId|Vhm)caUW|}u24iargn=c4?VX?yl<CnzdsfY!(*;z|weFKSWbA4q6k3R*TWxvDk+5G|DY98%u1tiqh><&9;<gX?jFZ!eU0E6Ydl7WA4GZr&IrRKBQg<}dh5fIA2EA+EGH@o#g=Gf~;Dciggj856so=o*cW;Tz0?89!E(q8X)je4Kjnzd}5`td%pjG1Z7?ARL=-CcCdbG^>pV(vxm!ZBPOcS7jJcI);F`}YSj=k96Y{y>v5Pj9~vDTn(;%I_Ty=9^}*I5+R@*MTh1mbUcgg!WO!{eM4jR!e2zE56CW|93L15h78}&_4nzQgld-^rQ2iNY&rj!>rg^!}bu)q<i)U;7#5@c@f1zmh;|VNs$!<KT+c00}}c_KL2kdkIKARitj<yA@V)vKO-HI)I5>qku;4@s08(nKX`=N1rjnpp%?Ea!2LDq;ZEn(CKu5q{$fS{B09pZEa-aX>B!oSPN&ziotl;P^^u1L-fw5H(6}S<9V(T2u;huEdG-_Yk~bF}j{QvH+~}vpw!?TkCqLT;{y*B>z|rO$_`BC>`t`xVshhgFEo~f&U3~UF?5?k1**{~o=9jo}V4wa%Xc)dAz8SXfH?S$+b<RG~H@&%qR|2>nrh46AiY#bTz}*%s`Vg%0lc-Ux_R-sJ#_wPQ2=UEf4n8!HvhQFk$t?#zt-Gy_Zi~#KST<b+ZI(5Z+(zy_t(cyGi2|YEZbaQg;Cl725s=5M%>Tn1Z9Y56PMX(uQa>lmo7;uLRDQ=p46UdoemH8e|8)K~O!j9H6~fgN<VLBjS;r_04_}@=j1j8$;oYBg!-`aTM+;3gL>3C%Mdv?;h994ONeZ(^o~E%>`YhTS=l=sr{usLV<@sBq_rol@5lNGiwwx(jLM!0KcYt++0BdZ$#8*ad33$JaWyC_EB(DD{^H%IE*p~Ym5{_i@f_<;fr&=bLQeKbN6b4W+jkzvc^$*%jI0$=T0|w>Qb9Gczqm_^RmoVl5u2ncOv`vaYWcHD`pb;X2Hi+}LvbKw`6YL=q$Gbz`if8`^8Q0#D7nrbJ!HolZ1SUdkCbc8~fCZ5-pj`{e3tgd~(q;GvHp18Oee_ej4Y60YF3COmC(?UAA;m2=O#0ieXq!De`$FHmY-ba8ng>Ao{u?W1pOET+_3e)=ck0;}qj%>Viw-^w*?u>6)K08->|1a@fF<g|EC(RF**2pRU(1`D>zmb)8#7jRI}`tS1j`vODz;y-n2)RH`$cV+GJAjokW*$$`d8oE%K+gj5}~9ZHG)A-&`{}OoWJ%QxY+49DdpZV4IM!=fkMV8;6ITS`3=@M(f$?eRJ7O7e)pJO%3m_^smaQX-bC}h%ude<$-|VbBumNtWb7O3&KkB40;|6|EwD&PIchg)<+d~WN78gqjHg<N?-Xqfjq1|(lQliI$O<yy7MqZFOOoJB6bo={(qc=mcCrOUgpYIBL+==OS(4WBryy}oBYK5G^-ZU?s3f3{z8E8<zr>y9!1c;z<4>e@$RZ86CdsA^o=!_Y5{lGEafga0K#L#2i!YgXc!GBAH-s^puk)`Ra%y0qNHl)-_2@l+71=t!*P<<o>TGrrO4<2XNbY*t)7IVIz+9yDJ?nig-(8TLyy(qp@~^6dHif7DL>*|O9!5Q|%9HH_5N`9ad-MbHFL*D-plG>|obxx47vtqYj+$0L_ZcCF1AH`UxqqkC{D4+dE%?W@l6Vh)LJLlp>_b})=qI{;@+-11k0Et03W!c$_Y6x}bUB%F;VqLJt-YrGv@O4`bsR<S^m6KcW96ll_9*!dC{Tn{egQgkP2XU6T~T9WB**&kQL3)wT8v<PM28hFr}OvF=K#0H=X6Bh0hS2Gf{eJ_J|(^ht~lo}KYbr%w+G_Rdz``;DwT#|zJ9=F@XzSaKMSgn*g4WLqipOZ8_B!>8`_50-v_K|L7r{&{<@}bgp_@ym+%t1$GW?sm<BXX+BVo~<;Eu4pI;fR{4)v?+ap%Yx-*iqPTPbv1kIq^J7{;&$Mw{6{_R)rh+r7V>6L_l!rlmVfe<cJLhk#pzXt&&Bk|`ki4=)6EP9F^CJ5RHK~z#VAR$O4h|`X*G6@CJdkWXrV8VWmf-JIN(u0cjqcrY)jr*4DeF<`u*gVjUrRn_W?emhh(qV9bv5t*2ZcLSwo5<WGc5MU-V|7NNv+5%-Znm^oBUA54jf7tC>2^_Oo0$+6J$Zi!&SqyU{n;N!e<JlMsUAKdqVPz$ynUfuINyu3OsOR1Mjm#zb7Q7J$}7H)@D3y*C3LVPIzEVQQjH#cMq(bi=1705FUIOI-jVJ({Bq>B$xowhrU!h?e(w-Xo%%Yb=_$0GpfnZQvlK|JQu7Ek8BNgvk>n9wcxBYBblDHDVD^f7mvcT+`ids{>v3#1b|TEB1kE3{!^3)~eNUnxU}{Jso1_S<kuk<*0)^1Yu-Op>lL1s$S#s)~UKegm&G2}uQY9m7@jknY8!(XB0zWie__Glut8JFns>|z}t4Kt3qGly#D1yjL;qjJEEqzkvJnqG$oihB;^-d+MbV6OjM?_U0&jgIk$(IW(M?_sjV}8q?8i9&(__X0-Psm7CRT6wr#M}#qyP1LOcQUtcH!o-Zv+E69C&PT5bNhB4713GOYi-XLcCsp`po=-B9M_tF9lPwF{hI9-BuT$lU<=)aI({R+iT#%K{n2XR_(C~Ql2#F##HL#54T<_%V@4uU5O7{b_yEHe2JL<(yIpv3ZYP#MgP^tAtrH(JE%5q=8I>z9HuLZrqq@z_kd5Lh<#$Pe8Vv>bb@s0~H)Am5KWUDzPWL|iICZw~w@!DM0f4_fsuhD)u~d}8Z^t@dEHlOi+C1{v1;VO8RS~*3OB<tAsd~xnjQOERqgA9N9x5Kr%{t$SwV?cf$**)$m4vNFVQ45LWTcUdA4)1AklFWPx+ousKuL-zx%3hUg$2S-Qr(qemA$452=*-&hKfpg@Y`r^6o00K1NbjUcpwrIe}=Z9w}ep>Cl_6ekqcwF;rv~T{=pPqADCR6jszjBn*SZdkvDaJ+i$ole2o*gz~U~cZcXc_RWgS9-X48R=Y4rM(!7Yho2IrE7p+m$i*Nkw_!FX8FGd~{PAR%k-dvsRutepxBcoL!(1HbAaP~X)GSBPG82Vs7Cy8UCI7;ggiKOT3g)Oh5ZG&rF7%wP~I2eBvHhH$2DIJKAUcMH)QL>U&gW97=lM*`)XTM7NiKF5rc7{l+rddu+4G(H(GLx9svZGJMjITN13X7|@XU83n<OiO0?@w@2v~c!O2SxX=o#>=!A8cEleA7)L`Yoe>@ca%AvR>P;B@9XS{5^7D*F;RbxsQeGgYal4vH49whlz-~*~=~0Atb0{DavKZe}H*_V~Y}C(zA9v(l6-v*#$=F+;fa7^T<tSW@0xT(q*kqllDr#aIy=|1$}m6r}cLtTrWDYb}<t*6K|5{^<x<y_9X6(Gg>R2pY=oRd53%MAB7qi#n_8V`%*uI(XpE3wrciA9=?sG33)jvxNUs({Ep0^cXyn%Nb+nTr{E@z-8BhE4Z5bbr;|3$ZMa&bzY)g7Oz&jBlj%oxQGAx?XQD}dHnTIy$-1U)F<MGh2lvxgVs+5;n61<j8E7`9^n1*dEkwsIqfqya3S4A`uxY?o*7pb5m=Q@Rd&M24hcJ$^Ho*E%Xo9zTjPQB;+;(Y>K2CEH$H!)R=*YNZ1Si0_!kYaFj?FHjI&(s<(XtR+Eh}YzX{)?4VuLq}whwg4NULVV-8C9aH3rwA*J+$p4dr|%4dkmi`i4W`dA#I)JN}Y8id}O5LM#52u8qT0$BXWNf6jN^QS7cuPZfPc*l8EN0EJ6I`~qwmj<L;=_aB+{69t6_t@dQ9DLrVL%7eD4d(c{7m=1<Tx7+K2fEt*WtMs;=d$tRiP1vD!B~MRwB@^%bM97Vi<`wT}dKJGf(X|#O&o>=5fs$2#!eu#>n)VJze?LupnjLM}j>mRl3-Zc;MAmD)lWfpN7XK}J@dW?Gt9+?^i$d2$>9|Yo7g8t%?tC;^nor#F8vPhYpJJswD-p6ouYe$kevC1k6|JZDrGN4R9zP7E!1_Bw<u14+4+R<BcJmNQudYl@&lW%cVQn1F9-I9<c0m5I3~wf>=1a6;Q+zy%OP{eFI*Q;@DtWq;ek3OQ3%VCRQN-ZX8#{#e$(83#B09Ei6K=|_@+OACxjUzxy%tnF{uy^ndL0;fgcVFQsWSc>=EtcH5l^#{Uh`&oV;ys!{EaKY)WZGx_=c)COH(uRSDpQOx7lm8to85LL!bt`m~Iz}W(M`UEV!dv)^W4F7a%tGq_a%!e2<90#|^q-j4a1~!Jd284q>!necP>vVK5||0HoV1>xhs7d$?Sy5;Z4hN&b71-u;$5Q>kZgoF9<M`7LqEKOxhgtt)uJU^uG=wihK(c$QZG?N{u2X>FCjSO~Cjf+jv7EB!Zuk9+K}FQ<KChl9*PyPF{otSqpCe|h7V?2LrSW#Z{XO+FDa)ni?v;y8m}!0X?h{odKA4`7;xsRO*ew7Tw>zgH`7uBnBcpmAja(}g@IIK~tuXsr%Y*D(RBR<*!onb~iWpQtoAlN`Mbr`~MFFpS6cjx^oYyg(^^6CX_8v$KEJ6iL$ALif-ec`_+QTQ<@tutHH-zp6PQWKFb>{wc23HC13~y`bYEqW>vwd4h8&1r^?>*wzza<b87XWoj#{<=SRtu{s&`(;Q}qrWpD<(dZ-VW%@;G2dcHLrAm3iAts$6=;C~b$1P^}{B7o7kgYOw80SB-_JSZEj}WriAD~~u1U>mfQhIyDdU`Un|2T34O1Y0+Ld+mYpWnm8en@yfzdQT6^OrY1APEuuK~REy$gBi15UiH{sqPVgE(Y#|{WXJE9>x#EqlD9)wc{;U$gfnBG^}WFu)zJ^$NT*D3n?&dw_n&foutoyBzlkjnh!_uV}o7?82!B|^qO?8I3W29+QZ)9WQuYjTyYwM9^A6c1?d^~nV}s+mKTPOKuMGI>7$s3&V%uX%q4E{O)B@ua1Jv_!TGxjjG~fIc0V+f-LNdDI~Ej<a&;uv`%4p<3%W)4H_+kV2nTTDY~Y*%hs&5e3l6(uPwjSkjXTa_#dB`<ItL4w8J#f_90cc%1F#84TjG1)gxYXw%S#dJqsk*d;h!3GP7E%GQ#9CM9pmd{9ovk3Y&Y`%_NyB+j^>E6fH!NWM?PgI;|2pXd8M*ip+I~DDn(@kT$?4#866>8>b^@)KZ4nm5zIOiCxx5<0*+uF+k*p433J%TIH0E)!p(ANg@83cLMYuRr!AJDH6fh$XJ4nelu^!IxiRnPKD3%64yu1g4^mIDC4mAb9@N}34{F{%sCWvN%Of*@Tx5*;A-h=4l8MH1)jpp6pxznoL!aRY!x(_vwwvr?CD9v#=3zq^%_biV?9Ut2Hyn=Gno5`c)MI$jJ_bGUj!G<nxrIWQv7p;H5s7%7b~P@N>u92JT$^AVS^^=5{Pj3ho0k6U8HO`ax+CL`LVCk+>!UBJ-!pedB3F!8dZ?taCj=ZPy(xqm%W#V2@ZSeGPaiQODt$4zwhslE99_z9ixwX;oZMrYQWi8n7Ow=mgfhct2*6WatdxP^FIN_Qf^%X}Ix>)>#{Y3*9aG1ZkewqLGy38k=EST+`Tf~%K*T^<l)18mS`i1vXlf55hh(j8xXt@;K;a@`I~?XLef_YFj+aLwiAj@LsJr#foykY`d*~%83Rxh~6hSKLBI)n^Fm;^obCh-DO!JBO_>nU|<47n`k~Y5PFh|>DFvI(`s73QjpxVygUO>ALYjwVqm;@U161qD)M4M0ypHO;9m>xvJvGzd(stLn>Af3HoSQF3XA5-xT91^@fRhW4XbQnJY@M+BbqqG^u{6fdD)OckepjD?=Z(`Qk!y)+-B7P4h5^E3ld+<o+#&q_OtXz4D{qYgcvNdudeYew0{)qkykKxOQj4JVjBC_9LQb3yO2+FcPTrgJt{5^$FiBz6*PoTrFc{yL;5RZ{c(}uGolKYXILMhtXT9PM`ZW8km9X8uTbs9s5;vRURqVR~xyZg@~3g$6%CY7+P^a$ndu!)AMu+PS1nN4#z2#74yfy3ZGkQa3jT79pJKPT_ZuhSLgX<b`HlrjMn)ouYVNGY+9*{C-fK^QJL^}PTt@O|Kg^%)YFR`3C)6rQ+;AJ8AD7MD_UB4#Ib!>&==9pO|yh0?K8KcN1TyZT`lm_LH7it_(Kms=g`Z~LHsIJ`(#{~&AtaXP{J0~Q}eC{${fH=-1i3^{^}9(4|`2oMDcXp3z0Vp5yHur?0~278DMAn<z=_xd;KAyO74MQXnzM=WOTkyFI*0CWh7d<v+)oFeRSQ14t3De(>h9Bey02rn?(64r)a>yW1x>UkA=-C)TE<+m$gFdTMByo)YgqP+V-r%9Ogyq%6fh(&Y5VOJ5mix92H9L#D!wc6Z%()x|M08){{^rE65WE2MH`b}qfX_MY)g0)m8U>3K|A(jPZyEjB0wLf{U|716inO+*H^W1|)MsPk5Sb`>#GJ-s0Ql=AMQf_D*bvp=6j<$NUT&`7%dIks97ZIK|?B}=Kg0x0@F-9J!ZXwdzcnM1x4585O!rAu^>V1!WDcMExbA0K~&$xx9^;g$c)=NwND<#k9gNi2eW1(YEsUDc;;VdI;<O5&1I)CBcnsp1c(&mkF&0mBQgi;HF&@O{OcC%(S^v6gy>65<mbLtp@HRk<|o3B*~@4vLQRKXeetys<<^GANdd<y+5FRrhxuf8VJ3apmaD$AG{JwC>i@Cg4#e$y?a0wH=7Wf5^;mZIEQqpc5ho{n3<YPQxY$$otxtngx2`f+bonjM(ca<$4U@H=5H!_E9qUkBVQK3mVorA{u2p0dC*j2yVB^za;y36#wP3RThtJ-)~s#c$|dd%*A|J`Mge!JTO%S%eu~S-*kJNQuio`(@r|RvJdSSXxtyZyXF?m(lPWoKfg=m9$T<nVtDHi!OrChMxN@hH)pBDyt}lu178XNzg7R>)Qkmw_xTl=A8!nW208y<XW;``k{s`o>Hx-2@Y6A^``pDim!tUh*`nM3_l`@Nb&Irab!^;y&5U*#)&g*V%We~M4i*biW=J_+FRm!;)o*JWy_^S5cq}uMQ$=B$&LE5T3#uWzSF<{nt#)?D9U{ze<E1`e!xmoJ_sKW8Q^3VK-*Sv5~VWIeC->gzQ8!B$uE9G%7U+`n?U<~J_4KxL2TglHJGkNnl1O0(&%NioOB$yZ<SsOBa)0j*Z%Lg--ypHICA)S7<fEX{I@JtQr(ZKN#sAtP3SGY*@T8n`{?GpuVbKL&A!-SZwcUoui(4+h{)e4ZHSvl|A-HqO~m{nc97OLZveIaK989XLHT=&D_e}Z96M2?87H4;p7;Y+V55ZETR&O}yiOnTj0t$A_xv04QOO(Sbu?<erk9QJVn%sMHYE))qWNIIjq<8+3re{sD7tLzp-OzAx%K(F>P7qZ)yf*&=v>7g;_E9>Aw2hQ*I{1u8w~dC%jMUqaf*eylE=P4DR^-H6TCmFFZ*!&)V7%A_$BMI{G?3`%Ho}io)T(CWa-FJvY>u)NwHPc$@G`Xx8Nj~Z~4N$%sZLai&lUvZUy<)<YCz7jgJ#zG8QPohX(QIudJ`T1^AE0+<+$S_~btO84=QQImsY>6cJB9+Cpbv!m(BMcF}1bmCopx32q`bPJ_Ewv`<Tf;_Hrd$OUXIy{6-nFPBz<K+wYcCXtlk5z?y7&GHH>P-z-cL(ScSZhM-9ZF(5HqGAYH9=b3*LQuo8Ct($sqQ%Mvd|G>Lqii;g=wSu;k4HWuFa_w)5rGbR?sD13tCsQ2&i@{n8Q=*@H`dmxHQGMQc=qY>E4&p9@IDx~RvE3pWAcV$tswo28zO>m#*0l2JIQcqI8MeuI8l?Yr80TK{Z;7PcwThftxuXtOK)su0Z*E}iRs~eV<L0`<&7)gJ&8Vt+m4umSe`k!bm*e~#x30OE?qi0W$15@yoH%6$i&Y(F=8!To%oLW2Iq;03-sRbZCZkTd}GZ^mjB!oUgF=;bg6Gp`Tir~wlEP$qVQ(f#v=aQ@Ra>~raa1X2>$YtWtjLRU7svPEugZ|hjiF(l~$^wy99gG@X*C`YE9c<X+Cn}1+hRl=4xMSPVIcQeA9IpXVGw@GJ47%h!DPrB*Z@P$uTji{xhM%exyq~@o#Wt>4zR4gm*|KUUIRL>{F^MZ0uXuypmDn-hDEPHk#k2Qc8KoO&@H%?N}<vBt7z@phoSpC@|Hs(ipi>&;yDrWh#qhefI&aGxnWDZ-T?t%x^~dTfioMO!3922YSn=rf?H{hHL;P0@+$Zsq=bi1slXA<&QoSa^R;(ImPLv4>s3JOV#+53(pglo(IgJek0DsS5ar7w2}>$E8i<8x~$)&sIc<2EM!Fdu!a(*bFj$s+?OXD@JclClPJ0nUir7?rf0=aUIl`)T;43NB?SPS1kFuHNL7Mb(|PBc;Y(GO;)bEpQq{H+CZz<P;eRMG{IM8T)Soo{4_5T4sciCX`QNBITZR<tFf}J7pJUr`IjIhfX}Yz(vbAcRDotcl>>2*h#8_H(OwWz15czDtp*~HEOS(IN>&idwgvX5GPq~c~@Dty1*1N$JrhG+}Z%{v=%&#JmN6uO6IfN^eb%uk(ZX?nicYDskVQ1K$+H3FCcTZ$8kVd_+AH-5pk=>!!54!!6<4%r>F&eM)Dt2+b+c{Z;hvr9jcZ;uzUSxPAXaeGN><grkEKV<k`vJO!Q+WCI&cb1cS;EZ(5;EuqQChqvVPEh>$jjUUgo2?=nAQSD-^KKmWYKT7vc=+b0n;S4yN7`-Z}BnatS+KwHG1-rzdmefki9X`5%T~eS@TIP>H-gupPsum|Khv_Go`VA*u4Wa)sHiI?*JALX06@Hh;CdFEoD>EvD4Q$P>xdK%53ALY~HOF%vg~MT@n@BOE!zm7Q=t>HwFIZviL|wg()DF?QTEU?zX%s{&&ZNf4lXr9h}FZrNl)lQYs3}EWLBN<@)!jXP5JZ>6zxKOJ2`Bpzr3{rGiV(+VZAr`LtE_GHVO&70><NO7(ld7i^F!qRdO<ua)40x|h`_S=v~+db1f-lQIfe{=2)m2HF06Dy)67a5v1g20?Hy@cZ>HX3{9lxcHE_@liXt3qN~zZw%Va6-@HMfvNVEFq?g`(+-E3rc9ucO%Hkd0S0*Cumb@N7htln(95T73tE!;LBA6eo7+%SOaDUj;HBwo`iFPh&EbAn+$L0-;UIIA%|?NDT<gw=uX*3kdh&B<KAXKf<Kdx#J=6~d2xpB!RNRnvZ5rR>*5C)cG4ivcx)_RC^P3nhfH~{J?Y#iuCFyA=7!HF$<_FoG9MwsudhVKcH>J+~=D^4c6JLM^4jrd_#C#)VEE*&+b2s$Jk)BYX8d;!x@HE$IL&1$Y_OQDQzcX&w>tK$y*4`q=@#SBfy*fYZW=A)cLwwodGW;ha;{TTE|6Fgk*J%>1^(b~QKRvFddPkVGizWFha}*4QXq+#)3_tFAcL(*pKManCf{w$i$?Py#OX{JLg`oV1O)ip^8j+nE^oU2)1xfm-O%Kx@rmmzBK?=<kM#t8gxsi^%44zjH&8!}EFPkkK6@P$kz0Jdeeh4L)TKV;`(Qb=g!&P2~ixzi4c64(=C*B}Y-eIs846r4Nl6{&wTAd!H_rpr)_fu4p0dc#An7u<d{~kJNpf7q({-CK~S}{S*M|e3yg=-~;SA+km*Ps$y;EQ&S29*_j&CtPV3tR0j>?7#UM0Lh%#MPquj&uyOj9YX)O7`_PE{*v13-0X~E+37q8@Hr90Y{^0nk)rhj5th(!yX2q!YP5vx>ansrF`LjlK3J0MUs`xp5EtB=gaw1cf|G0oLH1<lx^^Cfp{;Mins-JA9^mb4IJOn%zBfAsf(O+%y_th0rA|{VW;DZa7eArmI1kEUY@`|UM}WiS*4g&pqJmB7tF!UQq`}mulvho6m?L?PFC$WX=OSSzx&B|hq(GLL2M#jpj2?$zLzt%oiKN!z5XBhYcky{(TVhUUrVP-82~~_63N$7ld^U!TrnXQYHL?8`!U#-Yq;a#Z(2e6Nz+Wf#A49z#6s-E8-*Fd#A5MeV4TzoWdh{j4-Yb`32Qg;xM_WpxNsIFP1d@^s;Oc!f{ye!S+V-j6Th~(wT4XMQW`X1)IA+gQyRw^kO?hEp9r5za)<mF!)F*!>T|Z~pbO;rd!(B9g7N@Q&*U)erL-%5;S67I9vJ?#n7>Dpxv=LVz1!-_tj`8ZRK8cYR%+E8N`&qL3nt9)R@Rov-z#eJBG%yXM$a0V^!5fp7-m?ialJhJKh(y{yqad>X~MgD{jd$Q(jVa6<5J$ug6m=RIgKa)C+x|P!Xr%ItnOIug9;{eB<)8kO*i>yvf=Il!?BOmW7V9FF8kCeOJ}a;=kwR*W-gGj)R9VMr*g}=0{_W5<s<a}Zv>2SV&a(?0G3j1?#Lwa-;mz-ODCzTK#KBoW4bz%zxv|Ux#^3hVf_O;n%g@pLVmr(8{u1x{;5E;zQ&Qs>3GiRSQ{^1y*B$&WBwV_vCbgW!)AG7bA4%R5q<WZ@~z4eYL`3*Zv$tewE4<bx#pnS54Sb_SX*~CHp{DIF?DWKYB#s8JFjk5YUOCNmTxVIBN~Va&@R>*CAKbq4~;c3|Eb;PLC*&U8aV&DBDDzcn`Ue<KagI_{_Ay%l|gSP0+b>s2?U%_C_4Rm8{I?g3id>A!=NG3$B~itB-LW!LILzdZ9q2<5T~cnskaZpn6+Q8pJQwM{a)B_4`FyJSAfOduV)X{L#BLk9UkqFV6-{!Hv@Fyg;{NM4x2DVK#hFHUA8z3=-(dAD?D*dK`M}+Eec&MJHV9^Yy5g6U3m;e*$dg<t#@$MWNDjj8v8&ZxPKQetKMB%HdP%AAG8L&dja829kqLhA)h+#_VVWX_seUj;=EN^->UkH>sxCz?*7pnOi{ft^zQ^G8vfXL%`to{vIWDYA?IZn5{6Owoc}$XP-9Gen4b0PgT{XQC@^>bM5<z_A##lMkB~hM<kw@hy0qG`JzDjHcIU*$;SWzzw7BsPwdCvd<_`{=V&_h#lI{)m>Rkcc?Njst@70hu+tWIwgbEwnr)bw;5c)^+Nkgi5iFQ<($i8r-<+OKs73rb)xiiArN|%20xm4f@I0VlOslp7<TUDSc_!KEMh9Du&7PHx62HB3U@oceSB-kV12r(Do-8P*w$qM<LVmbCPTo*Ss(SdQL4Cik%YbUJD;7D$b02v&yz@}B!itFd2>#fx^Yi)?)c#n|!YpJ(w_SxtYY(P7*VTvi%eqm@(xiPb9tWXC7DC~CB^qM0m4N19^HdXTrk+-zb9%EEBzZ&-XcjTv0Vxm;jYI;Qm#{Z<|RaHd<T^FFu1TQ(VypAxmTI(U8L<F|eFk0<KR|5ktW^9f26w39fMpoq&t~iu>H|U_Z_ZphuRiY%?)JN?Q8{-n>gT=)R(t}R?dB4AZ5~4<tNN|c5H8T+>K3ucy7GYtjNQQ~Mr=vt{iX&0E4n}%UCqKlVPT)hG{1B#aNMEP9QZquHMmv0rdBV_Kyt7&zP7t2%sqQxFgTaXpi*dI;X!<b0a#|zzYok4!%;+`@b!RZ39bq31h<^Z^74C)ob{~BQ;bfTGTBLmf@_7_=DLrS*AzKtspqJqYFKkgZfRVNZFSmQ*59{F_yO*xS!3=|rU`e64L<>!L=ZfhySjB_-aL_)sdz$M#1BO*>{AN3pT(9B4jYVMZ`zX|4lAf;V4X%t5^p4gg?#FF89->XY@ktjWD5TBtI(%K^8B~R0e%D3=SXc>N@ef3en195VU4aj@T{zwWyWWYu<zaxp)(6TNUf3O7<DEF>26nj-f0N>q$RG#W*R<p5ipJQM5iW_wR3Xa;%RX#(8^bR02z081o29Eo(7<7V?ZhO)>Q|}jHso$s@`KvNF+XupPKr=UB1-swi-Md8B~je&G;R!3EEB=A0ef2S^!ALP9C-wEt~-s<mSA==+mhU4W;l+Nh3aWe=K{1&WM&ba5~#1}KcK~AL`MuLF6`3=3N*UIyU{(4|Mv9`Bz>-BslA|wOM5~Ypi+FUWm!p-<e9SjTSphFh(KonUU#dOunbD9tJ{62>7w;9R+);}yrUT_#TQs9+NOxStdYtJy+h1xS^{10>A`T$(Bdu)RIS&gt6UYM?j&8N^^PIS%==Z_{e}ZI;x=GPNj_Sk-FaGNv?oc$k7DeE54A1zI5|>VZXXFwf;S>>w74s>D5#~R*8Olz8i=jRTF4WaS<yid_<A0vq;IfsQQz(Pck3sxhm4(XaTbmQ0-Ai`H>`LO1tgPjc(yWoctfJHE=;;PyIPcEj3G{Cff1ucn_VNNpXsG6u4>vuB6V)6Z*uF9b7ad@;*y)zT$9yFGFr5IBBOcS7vLuepW{<q-&X;NzXCEEIx0yeqR_nS=4R!q)WX}2vRvufz!a|~5Rpas$Z3fl2t;PP*HWurEZ<tXLJl{V-CSQFHD(w1(_zmSk20;kr(4odGDPtgae&=>;jkIEi4gX|NwNQWPUwP{(Osj}x0_?QbWSR1F?}JKkJvmzoymGFSLj+v@v~Z~u9j+xH=X;~`O|p&(R52&R-$cl4I-l@qVuxJQkxPwh%@t7-GTr@h3b-~2e+%4A4Rua;hX~x(OHLQRyyp!UOO4$d1>_t=MfR=5xn<apj$3E`B7NB@4`fdMDt9fS}(=`LSNlN^jp2At6FqV0BxFsa3fR;luvyLVWGafI3#PV0bL7=2VC3_G@cYFJU51SZQ<jAFGVw?!Osf`U~#LqzP!vl{bbcppGfRL&WG6r^58>a601W{p0FP@NH0d08DGGm5)<h1;~Tgd`#vf5;YkgShZu`3E>edsAqT`jPsE2`A&>;ptvVo*H1-j>Z*9G%VGzjr;IkNtk$X@-R$K>c4A>0Fr0NTSgO!UBiGxH-6N{k)9JrwN>xUu2grKk@1RbDTH%nV^<J>C8KZKL+hX=ho0dVPr$yPua+OMvzzg+ea=XkRg`-Hc22$y>u=|<D<1V^YeaF?d@dL;VElr}-@FV|MrJqK+LP6&ICz(gDnH;BwOoyh--;3fo|->^4jfJHSrz^-)n+?4fKY%B7TZOql+79y}rc|*zh<$Zy3DU(&mH#gTeE8mBUFfq-@i!t5-UQDn{p=+>h*ucWmVzH}~D;u@=mQxBPCM!65<y#IPf&l$rx>Ma?kKtZ)Y+cW3wGsRVNC3_bUNUzBl<r{OaECjb?h#!ji3ki2Z4*ZPjB2^d&D4OuA(-4N4#LqR00^9k6LKC~(6MvzctQ2hDTfse<YL3!(2`!p8`V=A$(^sWV=i*TLxPWJcc;V-pe@9xj+)L)9zhM5Ax`0hh6FfBsdx>=B*T@-2q3Q?*2#g<bDBZe7_|E|4TN!1Zz1dh@oks+n^t>BJCjmtcAULK%swy-5~E~ZRp#?^3nn117kh8eO0%eqShC5_XAzn0Un~&E#9t_?OSr5Q5==SXALMqCxmjLW@|mYh+z<)=-mT$e{N<hZaXQ!%<*?rT3KB@zOmHiTJqYYiqz|JBO>971N1+18+ytJpxmtz)_9ILewV&X14Bu0&`U1gxI^wt!UrVSS=`fit!>IU#t`qRSTi>er>&rgS%8fFuaj90jva~t1R9U(Lr>Q%fq6sIb^?i1G2+$)wvGlNW4|mahtG<i&)&bBYq;7($un8KZ_?6Oi;7=Df(f>SFMU-L4*Y9XTQvwmbafq}B$r^fcjIfpJc-2emi(9K@;P02DlnfNUoItQ1QJVy_wowr_;jk-@;JZ>hhZ4q|3-}Mksi(C0!VwD4t_U(zWX9cgVF7_z<Rye3<y$a(pA<(fg`BfmWw}zTRn~6cqe>OHl**#7#0xZsuT^WCTO<-ek*zhjNXd%jM`BfJvs_B7p>!pX7A=#-JvA~2ANATm%YnaxTZ+IBiC7`@Ob2NZ@4Mda4`6JeGweBkTU}pUaP9;_pXn6x|3JBR@kU^9o@PKUlQ<K(1oOIWr64=S?fb687KP>8c9Ew<36$XVgw-OXH}PjXrszznVyKO`J@Tl71FTBc08iwHrxBAHZu6HrJ23@FbgZdy;{ntv)9rBAqQ6AlAzLwtI%={VUN960lR>e|6;tEG9p;-$mqVvuSo*uJ2M>Fcq{=-Uwx+JRS%l$f#l%15L!1$HdM%qPahr&8aVEv5I4Lwu)J?4hI99+mFP3u*rJ4{^!~H|J_9#$fCPO7XZlW_i3%qk<pHxCQR3v5>(z;_QYaIhRy}(Dp>uXdIaVurmA%`xY5U3Q|Ff;C@+z)y{4|m<H9&hvg%7*4*Cl6ZgUc2wc)*Bw7NSXV7yH7$ed`Szd%h0M!Wy4=8FR!4Rp(;R3Zt<!IENkyB)N*x>z2|Ww+$$t~Iw{5}8l=TXJv`hkx|A^{B*4-Z%DJ#J$S1EDy<*5xF(0;q@)Z}fcrwWr@8!t5Fw^p&?aS^J_i`&23!@Rb0GNW9DE!w-CQ<b4#&vh+(#)kxv(rdo#X}M}eTw5m7lv|q$@JAsTMg?(ir(DIA<K!$lCx5^qP4iVx#%y$eU<bG6{HG>h;qA}nJad~sHSF6B~xvrQQ^@Z(^{ir4)CdU=;k1D?GRQDW9#sL_h|7*p#i5tn~}}(D_fP#GH*Rp(x7Kpe0q2xazqFxo*FCN)4Cf|HMM)W>!@0(tY<K_H%2gKvuPS8&at8mgC6*#&a<uWx137E<90+!58%-1Lx=P^%&!HG@H=Wshe#II=@mLbgTNvNFWkSAXBoA9@;FB~xn&lhsbVyj?q&RNn{`<`sdEM&Q7&+8HML2grsNv(f|~VOoa^_Cu`EKCGWzmE4`^XI^t6y3+tY6cG)*`on1cOaw^=WU&v<?3F6F0Z_&-^1*L6n)`RwNo`^ed5sM>7m1nujokBUVeywC*hB&G^AW>POXda5Mjt0gZ<-`UZ!#x3yba3ohztD0P)N}G!}E4Px)>*?n)wXxKb_`FV|HY@s>+$2X+%aOcWB=e#ST4wKcY$XaA_DI_lMG!|H4+kfOL>vOCLW=j}XoYrs9=Vz_NsP={Xn`0yO^ZgL22C1!H>V<HLZY!G#4K)BYGfW`&-gSFTW<^xQLBi`|3ow!ai|ES>*#G*7L;P)?rw+O9SW(VHNwPE_@jt{$;b3-hGP^%>-?%C2kQ|Gid|s2C_Kp{_x%N2m-s_#KsJA`+c)r2{crRO#Eb++fPj|p5sGIE`ou`KGVI#T5>%m9UamoTbpwP`T7ZIs(Oiz3Lk&_!FT7p2wxctQEd;skoD}Zn8l7Gk=!6j+^VyrxU*D>2Y}L@Kk)0b`95Ye8PazW~UsUi;o$Ct@a-#Q>T$6aqtX%QF7XNU%;E1CvBH{V1NzAf_Nky9wx*9>&=#z>dz+fen6AilE+ZABZJx+9kxpyf|1da$VRU}3r#lEstPH7<W02|Rtlu1WR_!~7vb&PB&qczj+l4?ye0sOW0Rx>U_{hbchzq}4odZ}9T*Eg5Sn=oUyN-LG6G0g<6R=|3}2zMK6Ef%#x+Q_Cx8%0!SgoUHBhI)fawN_rMx%Q|XvBENN=jTAw-bUaUPcbMMYGR4mplq2OvvK~e`_ut6S7&xx^Q~s%qQNHb6A%eQkvt|yldmF>dzJD4BT)nphsa3uH6qMEI{Q3IHHi5qzcx`;F<F5Cp{KkKl)n=(w$C0t=a`hMFU>XP^0UtxlQN4jDYxcx)6<lP^rN$%BeV?si~sv4`0xLM|9*J(k7xha+3(K&_3R&<vtPrD{|PTXkPm)?e|Qd+$J|=Kgd!hO=n7Qzb13!evwwyv;I$FT_6tV#x1?ZZl>N7ioQ}>FvSoF7S3QaV1L4oVJ?x*Lo*V8(%8~v_+6s#V>S~2y+)<a-uKVar&#J}i*XEs~<NE&U%JeKfFRv_1zQ_07G?)}P_^buH7x=JYeW_!W0+7#oTahF@c1xlmbinBUW^v_*zPi4&Mdtg(m8~Tdj5gO-p?5JEnU}fP-DOHdJW`mshm3SHkct4xKzVg8w|i|KA2x$bd;)T$EeeRc5=Eri$g79GFdrvU0gEK!w=X!MFT~l3!)_D38wRlb+XsPkRZ8$5NtTO*=K5Xr1rppg?2!Fl7Xc@P%hN%<Ti*)?Id(C_r#^gSsVUxmrj7B~nGl2x2BLt!S@enEFtp~gv1o26OYz)0aXzsT^oLHF{()&Nj2pKc$L<LRvlQ>2CfTxnFO*%2)n=KogW@2_kw}>txc~L`?eNmK@IRU9?fleBJ3q{CPfyM6{BRTgn%nu|Og@V*|M%xEHelzm4hm)$w-0ipGtK0)JH?FXTkKOJhqAB3K^=Cnjo)O{pRU;G1OgP=k#`7ALJ`@&D!RCh-4VDu*m2sx1hIi%b0aAdidLei{Z_CwT$ktC&cbCVG)SDfte3Z8_`s<}*+*-`bN^bNC3ffeE^2(>-mpn!k^hEIz#uWfTJO3z_4242343i2L2ynL?`yV_eSW)4>!77pJOI(|qBRt9)7IL{YwNGBS(JzxZ&@@qY7PiS2L824@^meOG)D<6wJOMTto^;6-^mT&n78{G_xA0s3-Z3PJ@i-h<0G3Y(04I@fY3Me=B91>6PyD3=XMARDk``JD{4vdct)O1_|7fBw0W+aPGLAQ`-x-ICyv&MRH610rfQRlJhYW{ha!ilR#R4a^s*OAD2bh0kaL;V1pCbkpRCv>KDCQ;<dlVbKg)WC?M8&mqmv~fLJ<Wj3JSE75^L^8v_OTEATN&5l(lmFWH(xSe4I(<ZHV1<p^nhTm0#-UBJQ`G>3m{yVvAk)4)#Zvk<hT6dN@Rd4hW*MN7>wN4^7R+)z|f~V2P5}FrEoYSXzN06<9-6KIb0HwVTjU5M5mM0>Q?*LWm0ES17hkOHT`;dI}4a=f<G!4yqndsy?cBq9U*@85ID^LSuFy4_Uj1YqYtwR;#R*F$QP3a)YG9@+({D(z<%RytJfj)bSM)^Q@Fr{T@&O4g(Bbo5DgdaWWfG#!3bG)G8wqUkl3>^xRqAT0!@nY8ja=f6k{!r)my3qVBGn(iq!U#681xCzX!1^_R<QG_yG0^jG$~`I&UxBYz+wF1c3XO$o#)ldw&C($#>z^H<8HmwgH-#mg=}u^b}k&}CQ#Ar>WDvI&~SlJCv*cu5MEu*RT=z$ouYA|`a?gzF|z36HJ5dxry74M-X#+5(nD?XMIfr3@Ju_^X^L-2!PO<y-2%q-4~x=SmsGk9nPnbsa}{g-@Uy65c$?S<Kl<ukg(n*My+yW16sE1Y??z{UWx{m4wu<tgdbeS7>=)p+z{x<?HL~FUM7@!j@~U0Ph50wdNDU8(G8P;xePG=h>u9-U<lAi{){JHtDmn@!HyTPuv|G1}^*_i>X3sLTzY%egpk$r1H?M2AvjW=^WN6huW=q2ZO={MPs$Uq#rVMnq-XNh=@$}hX<}*SE;5xQdTbZ5LjRsU{id9Q{FDw`+{?y#vJ!`W~-Urj47hy^7vuBMQB_JqGsT3p|v7g5G!yWrM^4aLiBw-E8@(_`{PJ@)1Qv6G(%`@9-+5|!7!^D*iO{U_3n(WPlsg_HQBJ)ET16<<*Sc4C;RxX#^bkL3=mB;pJYi#LbHwf%AD>5I-IA2$3@c*>lgNT|7PrG@W{5M8gTFlmvGx%EU&EOr~Qq}MtP;OhDXO5Rg7DFv$BfrS;D7#X*w?t8-_J5Z7sgMbe+R=F~}DKTF?(nIW)r`90SSK8=;U`hwT*GoT$+g8(yJ~9`jP8cbSM6B)_r6%1Wgq{Kn{zR@OGQuz}KwYFmcj$s*WE`I|#_<zM6iPkw5qpBcR*m3OSIT6q&jRg+H>00sIp5v;F2iUqREdQGE<PInR?nEi}`2gRyIS;#%5SVsr>=#Lrwk6HbXxkNT#>J2Tq+9dcydGg7pwO7|UnoR{^!?1UB&FU?|DUxfz-Tht%4O&eMRdrOLoWnkIoSm=kw1T_neR>ph+mr+^!fMbIq{@dQ3b#WyJP-xk)oD&5kE53f<#F(<Gcvh6{GR3Cb9!&}1mWMECK93f@Oh@I9R8_rUrr+fr$}qhQ7|!*-Vm2aQ}k7pN9f^XHu5OnO4j+#SP((caMZ;d$QU48WssI;jqYlYx5J+Bb<qqwB&l1vs+EamU2l;}rL9ahnxk0dTD^#;c7SmJ>f>S}Zjxpz`Bd1-aOU#a?<C(#S4NuJ&GnTP^!-4&$<<{yiY=FD14WXolYJ6mwi)q2cZ#`6$uEVtnCl*P+pix6nP;d;^G22!F@7YQvk7gYiX<RW6xB*Fys$l##wttA$Xd1a>h}B=h9di~*Ne(ZO*$)yYLjI|8}e)bGNRe~Z2m_!NZM5BoI8J2WV*znnVrkhvUOup^U5L+wd*VEi!XzaWayQ?<=mJrkgX@mhr}RoD&vw&>dA>>9!moe<K1zd)rV&vV_srJfTJ|moH*nWp<SYt=Vt!O&Dn%C@9d%8kNqNcB$n#U(KDc<^^d%{$&`d17;l`vN8NZLlV6{|Da=i#{g8qD<;Y^m+ze7k-tc6(RZ*i9hFTx`Kt=8dQRlnUF&Z~&zNLcBRTl|6iNEGYp5QU3Te0f2qav_0xzCwrxPO{8x187)?$dM)8hI6U66g!Dvi=%6`LoiiOo|o38;tCy^n3^6qwjb8?0NTAum2WsDbRE7=J?rEyvZ%N*ip5|P<PL{gOWeORs@zF_Rpq7)IhjU5pz;xvZyt)SzZB3SN4JBTYnV>w<85Qxb)YT$`tN^8YVY-u5=+e=#Ee(PLaooE3|$<(X?p^lfy5H;Ya@OlFc@V7wImo*CM?#E*2qvDB7vSkx_H1v|G|0OYkUh&$91Mn~P34k7ILNNSaAAG<81p03DH~yuFF24HlPN_ZCp8C?rg4!r_^7RB##xCaM5JN2D$esrzVsz-Ui&xF$n=7zB<)VRPer$*7)^{IHCAfZ(9t=IrSa<_U$^c@!QgSrn%}7zPMpAbJvwAkrDwZxyKz7fQ~-VOOCtu@c6Xutb)~XkkcJ)OL7Z)br89zPqpR<_Hyz&VBVJoPn5t->;vb%OHAYANPYnoAR+2SPLuQa$%BE^@XgO(oU3R-6aZg(C>+&sM?`eZK3z;9UQcWiCFE_n)YGxF*;eQ1Y`C<fqW7i8`eKdb;BN>CJyKph&yIPxT~VR$<Q9E?hS%Q(C!aIOQjlmPZUBf3_eIZE*3Lx>$Zqb#`ydJg5P0228u|A4tMJGq<$=(oLWQkW2vf1XNevaIptXvF`Q6ommScxh<@QFZmU~!(g*6+Bw<c`6*+ftr8<CC!rj%{ZwiQv!WzYQlD=_IS(fN^hXa@k8RCy3nzc$hs$imtB*5Uf>J+M<Hf>gTZOax?h<6dti7$_{^-7i+gpsvP44a2=7q~JR7lCRA&2V(RacIBP0qgiCBWpYg2@-ObFzX0(*N(7j>8Nx;wiOrt$lEp|_3Uw@P8`$$cm4#Bx|7ZJ7$DP;*rk^0_UEI|Wgp{sy_0opa>U(<>m#t=7|QR&A1hQajM2B@2~S*;27lHJ8tqWw6WdT{C+{C44r8fqqupsoV77Mr;0Y^6aNh`G*iw5OX7~d2VvW&m5x~RjjKRk|4T3mj3X(X&Vh5%`3149CBaC6VDTeh}CzD{qDHTO7C;zU7M4sLV*jb`8j{;?$Y%&xouR!S)^N>gK=qpg(>!r;c7cc?^Y=P#E2>B+cV0E?0>=FM_LFY~k07SrkNOo*8+>ew>C~`(@#5f-OWRorftncpubJo@HC&-fC%6bCPA$rkjag_9)*P>We6MaJ2<ZS(y+<he4uo%dqCrof$gcD9>5%8ICuGm^LKTnn4RHU)uMO@WbOe(&UE)}Qf??`bMfmthSO~Nh{Sz<<;M6xpaXvB!LDM}70W&S3sPRXN%?K{!fq#nRARV#PWRVX{oW$T1xw#gSQMSs9-PGk@8SEv2eIi*AaI{3;K(G^y$tSwhIS5ZBZ9o?F#k`Rq3Yhc2(qq%;Kt65iVh<cY1Zc4c-y;8!Pp^5<=Sl39shh=3@-WEm#otMVFFZzfyoR?(u!qQd%N*8<LNlX*9kK%6b3~gfDHVQu@Z1nnsg>t=$FyuAH8I(8Tt+IbJ@86vD*Vk5Fb5ltITztcNyQ3t9bb!jqh?;~$@<oKZ4l+4e4H6#mDUivW=YT9G=Vmh0?8%0dK-L!bMmnhWWc4Z#G=Y6a`81+%@V&h7n3~|ru`u0}R3e6DNA14gkZdU46K<dcL>meRIy(1c^*k*=Tbu@9c(C7TAGC*3MX+rnZtjhFUsLT%b-QM|aD=uscI~tAV&0WEm>u5$F?1A>fuqA|Zh)jm$U12sN~E0#L`NA|7B|<c)mKX^FDn28>gA-yozXqMk(Z_skbmKbC^ThwT-{m*YK~Ohb)MCZGrAYj!z6BMQ4z$>^RdsQ_rj(*h(G{s=zokm-aKsFY3}k#VhvXVfs!My8|iQpea1-S;~>brn1`kUl{S3wEWJMLGB$Zrqf#JI1PUf|YmMQ_b;PDuD@(Q$wwLF4VTv0I2qm85i!pwzY(Fd8usMK98sUd;L$9~O)=x}B2+o{j%}}A4Pxdk0TyZ?+3AJq!s2RA0!w$|Y`EQI+fT7BBgu+WH+s5WLnMBK`6`!cl1hidJ!8%E^kv4XcM#tw%nMk3?MMt$L!cagGWNaT02jif*8sKfh2UI~9W!Nv4P#h4R=XA8j(JjCqo*Q)PeN?}P$fY!evVxol4}fbKqQH>JP62`2tl4%NX>x&D$jm72s)AC5hl3vSVPwG5Y<Y0ndkt1Lg#?P`IcsGg?GAIr4h*@Cj=W(mZhw+2Qi>09<W(vRRau)qGT}k_-Fo+;{>j3(pBlv4!;I?B2;ctfwlawAP|Ss464OFo60z`BfurbZEIz*7Vz<F+B(TAyE`n%TiqH^0OiQ8F`1md7MtgYkaF@)A1aKD;#$A_zKszVSVHbwm*e4g|-2efX98?bI7S4JDgFEpc<xE?pNb?FDJ|oHadu_}C*FoH*gWfRkNLA$R4&d8;PXt1v63J@HL{lpaq<G&w07j3X_FK1KKy##X-<;{wf^&bPKHL}0Wa*@x-hLsmrCrgcTaZ!1SLZ~WAbn5$%kqNLnao5n!+1=o<M1*4oAi&?MpO{JCuAJh8``T_#LMmviuHaQ{V`C<A;ZJNEAHjPK?iM_6p@sB{jfJw^N7cw=w5;A0rzWBwB;#~mdwobv^O)Ihu&j>6hB~SdfR1-tGgpQ>WPOuP4cO^%#2L#3p<IvzG7u5k8L4koN(E?9xP-u()$w(^z^@AQf6C8(I%IKg129=4|=!L+bu@RE>_hXZkD6^e!yIud@SgD;#&*glp3IW1*Yc;wuviAt<X%$zUhrx2~o_FjWuOW;3e^kCR@cA0YkTRy|nnUfxCo&EQ#}xwJz!a?qL`wGi@1Zo>n_6zN)Yu#nMLQvU^2*7+zs^4+f2K2C{<q1}s*AZHKc<6AhJkjAVnba3!KHaM{%ReR-W-=|SU&`-;d`?y1K=7{my#Gq`1onb~PC5BolApU;8tdZgq+#UdUMS2_i$Nkb(Vym6C(04}*^PzZWX^p!WsA}`}q?W7;Tvb$NhJ?;jBj19XA!{v$tu>%$c67*)_u!wKCzFkzA2ZJa<49pJIo2;&1G>!xO!NH4%>DXQUT-ewT2&Qm<IP8b)?$poi_d)_ZVzn!u>>Q8=Xc_i&PJ57zrj4-$w>DR7_nX2UFqis5OE(TU&5oIny5KZcv5+EqZ_>q#4_aAa&=b`TAcm~lF1JBAQR2KCkZdW@_^{u03kr^HH=d$gfF=!+ph@W{ZSnxe2V$ic=|M-tVut%FY%j6RMe7NT`B-l|;g-af2%+!>=7ggH>ESD-ND&baN5v4BVj@B&*X`Yn;%4wa_khpH9S$4FVh>vQB;$VjwQnDMyIK48&2O)MyZZfj)qK4a31M|gNL)bNVjhALC&A6ZwwZFXm*GE`saLTbTi=UuiD@O7Cs?nly9DmP(B=F7KFB03ppf|J3T|NRk+=ulnD5l7!4>4a_!;Zs{lR^zmT#<*TAi>+IVcT@!oqP{8I`j<<`bA!Ua{24Z_;^3>Y>N-4-YcI&K!|R-t&v;wa^$?ezI2bEvMu(2lW<W*3eBR*xb<J3r#Y^;shY?T*u|iIUD?h!c^dahFBkO0ZT~!oOZry3Y&2D>qp3Q2S74{smNoDO3DM5iw~VDC>py(Ud3v+K(B-c3;a@<ftd4yo=b){$u9Fp<N^pR1bs|%B%^gHfgCmF7Fga&RDtpHn70G9?oOrRU|kEkG=XP*Ap-pbIy42O9-0%xi>2N8CbqJ85tKCH2@iJDVHpqm<m)K!rhX44@!K!ZrC7Lfg<pr-#digr7J*mQycSO*rsux+#aP{^f(f!nAmnE`s2?Dw!?TMhbtJD49AmMiu|mdmWHLmEsqEQ#rK4_B^!qX+*-pwL@u1_^W<I(ywl?U_yIHGk_{`rWKZ(~(Gf@=fY>Q@*CiujQO-TnoyI?7HtQW(RZLitF(h5j*ZHHwOZnVnHR`Z|LLq_7_AfBr|K4FPNAiIFPTtRJ>{oz4}9_>UB$G-9iE$BT>)Tqi+l~i;L|7bbgGN7L7(00GJ>&^X+QplD2Lj1jQpVrALXR~|A)7E>OS}$SglT|A66ISFOHQdeJdV>%G;>ZGts&xOn{yvc~CT`LVSb6+1<@b?_>MFLF{GaZqod6aSmO>uH$XknUQcqr+_M6LcG7j{UA-}e5@o*nVDqkMN_-VI?kl+l$W>FG~D@I%kd<mr9FsCsUMG764!iwx6?`&>@Ajd^Bk^F`xp{l>MSz4|ofd^W}xLgj;uE27pC|U3bOLc<1gZS+}h;l(|+vRi+7FCJh<YR>}u9Em)q81hvLKluDqI)JfEjr9?%YH40uBgw(bVy9C_^KG+i)u=1r5hmRlo=;^W4*Fg^K0vVlygIBHW$}dS1UF5`B7dCOF<D@5udRsxFuX42WA9gFxnJ@;hAieVkY|7W)pb|N74p2ozic+vFY$xcNcqV)un0G*kmUabWxln8UuFCMP;QzQZkO(OiFE|LHm$t3XW|G4BO(w<1#%89mbdopSVunzwDw03aJS^-7e&3tuSBKIZD><a(-qyM}<UFrxqk{lQ64h?D0Y7;LIh0zk6IAv6{eAQEH=h1NI^Yh&A@(!9b)sM{oSpSh`sBetHNQ)~w{GlvE<m->5)GM1kcJ5|c#AegFJTf|^Z4hdx9&*(V$dODL!Dh}1_CE*1Vf|0xEdp1->wB4p2ha`p)7lJT>MJBzg-#43WTetGuQ`OnUOa{i06$B6*1$>u^h?Ilo&O<&k<vG~%=yyq^ibGpA9<><W~$n0BA4aB>{E}bwnXFwk?hjfUT*=Qa^n52%hg+-ZWIg0H1dhMpnj{r|V+O&v$uOGGp#Gpr0p58sxCog7(a$Mc6>qS8@t{&Us>ESckl;&2@J|ZxD!h-8v+QvhD%qTC%7CU9ceC&8(&@UoVAOiHp4um>l+j8IU6wLtbg5D3UxXxMcJT(UMwfu{%`ImNgFN(qZ2N75O01<f+%=b--tNx7k`h)YgI2fF^EfzWd(b+e!oxh~lUn1_Zu*Q|eh2>ddw;v%wFvX33!v7+;`Ev$gzBW4_%*?muo&|(iKM6B=6OQ@U1i}1E_}}{x`0__*|9JM{*$4FS=L((qqqE<@+ke18dN{$qho=nCERnk5Yln9QP*!alQSL%Vs{8N`^P9uOdt{sF9sWqX6`63Tz`a)DHJgCw?e5jN<k!GS@~7}4@m8TqVF~ynQJ@#nt6rFRBh=vZTKrCwsfS!M{F`_;;1LaiG`+5WH1D@VnKuqM5<$pWfTc_%TIa8dj=-XooDlKmHnc#esKd-!xu}e*0RGVN2^fJ0{*fpo+kYb%!0Og2x*XqrAyH1)!*u?EuK-KgWkjuq4X3}#j+!bq0QzCK-VgVC!%Wh2O~b)e7O2&J9JU%a&;+2>E;%>2HtcFA1-v@hT@XQ_&3#(e8JQ3{(P;ra81G5SQd*}WrqWeGpHO}ju&8bKWp`5XN~3zSPO<3ZJ%Sr$JVn{@D9?dpJ!4;H6Cxdothh&2neFKmtZE!JluUwl*$q<SGx*}!AiQb*Wbeqbgvn(=z{yCA?sR(+*}Eb<F;xv=c*{f~>S?aSz`ofSlIl#}%@&+=)M;rfO`Kw3Zi{*kExPj9C0~zg!#>XmkZCnY$Ng&tJH>Dje+An`khAY%mXb86>DcPYMOT%kf3XWk{!U6JAeOYMC*5KFxQv2K+E7Kw45G+VLC+}-VNrGuhx|7z1d6u_0<HJgGD!kONS$N7xQXX{nf{6OD#>T7tXv0uwjSFrVi2BeH*oaWyn5X+dl_h2j$<V<12^+u4zdKo?Ow*`LD;B+SP1oIFArRN{+~f8)yEBU7oP_x%>%w{ys?aRP~XFT7M;v(DLbT%nQ(sXn8xu$0kQwP{erqYGC;>j>B6kFLG_u2Lbu0Or(wxo3_V%n<=`6f#&?|X1L2yftP(*+O03{+VH8nV*6kPUkl}KMF=3p(7O_Pu8h<GhtoYiyjSzA+G$hJz@&+a|q1+~KAR^Mol*EIUDw|96OA-{LK|xd$F+tDh5*nvdsfq<WUvdHBwr4IO;t_BQkiH?5aLI94d(U}EV+{neU}Z)Kc0xbxBUuJ<7x(UDy1VhCNDc_GE4{mWvFlv8(o$8~zbfWOv5(>n1{RWxNgpBBwRWbCpDXRLP7SP`*x6zU7fT!HdA=mLp%gAFs=z$Q4#QfrTRl`U3A(MUL47n1&{xxP$oSmuF7`KtxQyp1CH*z_h*o)$6{bKN)$n+>QAJKu`ii1s&%ko~h08!ART>uhb8Ie_DCLIUt;rK_=;F@E+#BW?VktPrX4^2n{`X0`AvmHkh203ReW40u@|;S=G)7z}0LckWO&LH;`!&X&FROs7jq)O15BY1*m!&ctwQ5S+=H}B|&ay0^s^v3A<T=Ghjp59F#Rt$GV7ad+9v|JMzMdY0QiDwARSpozgk)_%`4Xnz=$00eZ%&)Q?WX~zWIa^b0gz&A$qsxi+kxI#O3DL&Y|aD!S&|;;;$0%+fj@D|1MBQ8Rh9`op~eBYB9J&fPec+`b^>El#jn3(f*&33!;Bi_Glx7A86r%w3LI5%aM{iI{_bJBg8>A-FA^8nI>A{B^d5Loo0FV~h`LALnS5i`oTyjfIw`Yt@3j<{#u14Uta%ZlmY*Uwf)eKw^V16>MlTwhG202_)+86-h(uueJuQ7Q@24V?rQ|ixvne3$8<RyLCA8O;9bwy#CqbGd15!lTwD}oLw{0j_DJdDu_3&cZ3!+<x&LdedqkJQ4rH$&%b+md%CxE=^{B~EO|BAiN^Ayi)V=_s4G?A}`6X~n^Oo<JIX@o?Lw2XZk1S^>e;ux+EI3wfPCoG%5JN)Xg(DxG_D`pi0>16?KPUrl+^PfP8hi6}UQgosEKM~0U<%jUk#~|VSL}m@l*&Tbu@iq22+aT>*n(2Ew2S~JqYfDIdDaNxDrefl~iC4w7k?qnINjOSImgEU+!`wAAx`0qa?ln9C$@H?{oksNa#hZ+@p>oF(w8myR4*jadf|=?vZGXAbdEXi;>bcW12GR#1#PtSsA>QeMoYz;32!9ti=2+0Bzf`_uz&Swmb3TkbIKYIg*aKT!rdjKJ^ubX$9<dFYxEg&yp@x|4SSw5*alo|2+Ii6zFcLbhfkdQ7J2%vPO_o7$AEFc-39o|3IoRB+_$8lJ;1t&xiwO1H{4~dVGKzt}Sc%7sE>?`{@_Bv?TSSH$o8_xD5pE21&+sD+IxmbD*CF%FhGw~d7SsAlDH+fTt<3p^ON6&(FgQ3=m<6gRM^tZ=H&@LT=KYN&eMaCfsP3v?SuE@8wOEFyFZ!hwe7a)JF7WknM7)stIRXz4YJ16-_;k&nnFdd1G&D2qr@@?fF39B?;^m1iAPfllQUM>lVjh>78JVgFPCZ53Q?R!HlEYrk$OA>w>(i}8@rt9i+svHbIwUX)3Bt&bi(I32m|~^Upc>7}ugX3%K%kg7At^E{AR%`^Q!o0241g#AxE@JJfW(AcgB2MDU0jg}1(o15EU;U|VL~%6!MqFxfrwoXVR&M<;$*H>E49ikJPUe5v(rRY{e8Id2fe+Z8*p*Vcdvir&gK<FAFiZ^%7=I*)mm(JW@mhbMMB<0p$2SH-rQU_k%Qb>L}ru7Ji;@{NVm0p-`#@^?H2BMmo6Qhren{W9+d9voDQMJc!2SfOW2?krj$WOL-@^#3xg43nVgjHT7|1H*2KJT!DX!5aT#GIulfRZF;v)d0(a4@_C<dZghi<Enjb+OGE@(jqwb1LXw>|Y&%l63?cQO?w;%Z}gd?C4J$G&zKWhwnq2lBTtxyfZ#5^>boSVUKd!42o-%wSMKs7XLnH5?<0pX`Sup>xAD~z96$((P4MN9=mG(VpAW6*~Z!^4Vr2$OczHy{ovQyi6!!5Ny(<o%0cVrt)?S~;Mqgu|BStcCE!3(jUWcW|fKM$bXCi}vnN<Y3m&%EqA=hdsX$9%VMGS6Ij;eoz(>X#;Xl!X+mESTOuCHh(H?@9Er{1~ax;y@eT_9d!x`$<W9ML0T4|3|?y>qACp@Pm+VAdxa<;GW?>j%W7&CdhhAPC)FTJL0B*e8SKbqmqwgj;1=W@k@P%v78V?H^b(@qqK`1=BucBwu`V<v1PZC}A_XKOjYPa6Fz$4iWHJHPDTdWbKRH@kueH%rNepT5GRKY=$Ks#?>GEXc`Y6@Pa)7Qk>&<yB_dZr@PEYpKAwdr%ugB-_p1<voK=?8GF`mC;7Xey>-T{R!^-qpFjxbBS+8f+K@tW)sukNOl2I=FZ6NK`kW!Mb#0sind>O;6YyUXoPhabWZuhePpbrJBA9?@TEb+f))bO(i{^={{6F%Y%kk5OM5pI25^$~TZ*xmiL!w!DeAGgSk93ij%qF|$Fm^sYW7_$29O1>*e)`&jjDGmUygJQl9ilPGuG36EW~v_CxPkNCpS>cu|4TZDDXu#|~$fpO|(8s1W~`5oyLAw8L#GjTRKU(dGZTp-<^ALZL~E|PH1H!|)y7fZRPnmQu$qM3LvLVOyRKo4$IPFKj83pr^abVXMS9ock2ur~F>&ajBSGI4*1L^FC;KF@CsqkMhx4{mitp1u?>i*&CS=ma#|4!d>SKIz*)cPzo6LUt^<h5GZ6J|)HoaTYB7fDvsMW@lc!`r?k&BzC0x6=d9^>$#UHSlV;{#^ouw{Q{cxL^W3})kwcOK`?UR;cmuty(7ED8@;}cva(YaFDO_o*z+D`ITQQHnj7)|54ZW<uy{8o0>#4pAQ;lM4ui|@!2P`693(QGlNPN<)=uNvE7%jF7m1o@?uH)OQ*g6q8aOcvY_DuVw~BPBP|HQP)F1Xj*Fy=dn4k8bY;RC>m+AjpnAA=)PVI~Y?U*R6NcRyc8atRKueG<xmE>QXotay5Wz%(eUlsYX0)h&cKB{!N1scEPHOOx@K6mmZl*=Y(3Am1Eme9;Z1R~7nj1i5QVT!c+wgt(iYDnILx^}Dt{%$U;9|akCp=f0|TB=CbnR@S$)kNJH6dY{5LXuQLQ>Jp&IodEdf>5S;Z$;2z=3b`77g^Huooen<kYn8yv>BnJzrgcOC|8wd7#t6yA>s`SHmnft?>2t1%{=1{ZSft(&_MPubI_)|62-iCP(K#GxYySCiNS)9eRsf?SJSQ<jpt^$v{c@d236m^toKK&S|V|Xc()`Thi<M?3yVKe)WECjwX%#iGWEbl&=>)>Q9Kd_rMD@oq>)d@`1CnJTWhG_5}n?RVD~H~1hQRHS3aiZ7-Nosf-(n>M7TrilTUXFND*V?OLA+HVQ~9F{RqT-5p!Xq+qfxUItAvauuv++zE~jFqn)O%+xwR;nJef;tF~0BVwUbi&Ly!aEzy1$iWyZV^rHg*q{=ok`oL}vCZ3Efv@sJa?jNCMT}gb6S|)3-EQ5?-B$FE<Sd9!cWk?pC{~<IMeDEBBSdQkWrzL*M0->w$dwMPk+3{7vV2^GT6Qv|49AK}Y`|Fj#VVA?86JICgLfjr088;8}O+h^cwA*1w+-GDAe1u_foGuZ?Kz2!dAgp{}IOL!B7RJWK$Ng+TF!*pB1WbDPxtJuvZZSO?=)9XIbZc<>By}fKppA)mpQKX5b}4E#EyMP@(^hsyUs+0}`6E~_%-D%W*sp2zFy`@*h}v=}qvu05=qC1Z4|5cPJUKNzE5fAy^2P@a`gHZ0hmAIZFQN_a5Z265#8JgH=bUOhV;)91*kpsu#H2oJ{oc(v8}-5Khr#d)qrOvEqSVr<9(fMX71*lK|D(W&fNgZ@?Sn9fG1x%R3ns2h_9@V85<m`<qT57hHqoqOC?cQlG~^om?+DZzTZY7EL#ttcG#hsmg#BI@rpExSamPZ|2(T*V3@(oWc!2i;ui1*29mB41rTeAD>MiE=vYSJ!8i8)ZgR@rs?+kl$Egcz9%&P_XM!WhY+TDf*`{v6TOjclr!BHhgpk8QLE?8e6(;)Nu0;|KktLhy0i{hQAZ%NlJQ8B7_^vZz=bG@m27DNIIduCRD{WXD#p1_A2MKfoO+wd)*hHsN2%1@9k;$6onlHY*Gk1*#k28Dl0uK9mPmz8)k1fQsLCnGQ?#daQvFM<;i0WV%?J|fsWv_+=P3sX9uQ8!7Tj+*0lF2!DHtD--4Ek1Zgl}4lnbIb*6;EWis`EeiNGw|qS;3Q?YXLcmmp@J4=uG<(m!UU?pr;VHWzd_L58}3s=C+HaadWmT6iPT-s03u@T07{)3W%$op&5t90#(Knp|0sr@XG5iLxBwW`q=-<H0YM2|b1FL#6jVZtlJk^Obu!@%$rka)k&|B>f6yWpAge|Sb%9x=z^rKA<B<!-XKC(y^yM@8@lC>1j_9wax;QaP(`5BZd}CStgtT|gpuHL|O1$ZhNGkaXVKhviL=`6GJ{Pzs3p&(u#(+||$9$lxwF*eaRv;*vR1ILljeuPi<VyoiqNp0zPo}j}+$&-^#kZ~n^eoPa!O07hz(zursAIAvo;O62Z(vc$#7JNh@1VbD0%j$-tjhEQ!>}n%lG??XmMT@NVuN+k`6)ERmu{%^+jzN|Rhw8KYORFW5<}*(5HGVf-aq>KdTvcgR(i^8Wa)!00^m1|NU(`elHxkN5m$)8N=jTK3#3sDJ!y7+I^`PBZx`u~rHxTWw6xJ!%&;`bSuSFpTM8swfT72cIVnNF0dUzc8K;vkX>4rqh2^2{WG7Xs<h6bN1h(=c*y)6yVr42HI_E!zeg6f}w|5=#$bO4Kgy7fTJHTktknMJO=8b&L`9XJ7Dv(6{o-#g7yQgBiVZ?=HSRF@QUePaFJM=t4QOi&tr)F(`4jI`mz#>rEK-o+7nA5Saq6C)cq*y{Mqx+|(JKF{Gl^f2UB3rsWYq_iWT#*+FS{ZmnB~7!WCKqWu**jaycCGS|8!gC$<eGD~j>aS%K4p5n39^m5<o+VumQ#{#vzv=_k{RP%ldPOSgV~U5B&V5At>9^#6phR^ae_Ilvx7L?Q20`}AsLm3w2SSobihkbz!>>(Br-9^6Y)NK{$z@hA*0KPEICAbWs00u3;3fj=Pbi<anJ$Mi8d!Ah0&s*PHlst<Q0vVKvGUdAWIP#b-QI;aN%L25d_Vknaz!}o5-^&)w4tq<q1`2bTL90|0s<X>g4LVP(ma#2P|2jV7FTK&wh9I^Rqvl{R=@zedwS4@$AF1U%>-Jn))X~o%+|@LDP*_q4!u{$!|Gvl`RJE&?WQ+Ts#l>#`z5Xc~5fzlhHUDnBCa}#B4C(HQck07&Q;-`ZL0QIfNAUHDiWBBaflFpW#EaCVYDSBVt85o{BAE=sgWDj2`gy2mgCSzhW1q=9mUuU##BpIBMv>pT9?8M<Q~Fw_1kvGw|?JtnvI^47W8#@CgnM7*RD$WJCo9AXdiwC<6ICI{TWs_ex2K9@K3dFw9}jHamaNNBNlhef~3o8KgdXF)YIKci~AQk2H@2WuP~q&9^B8>oLngN$A2$m$26um-RM2g*kca(k0{|1f&~c6`~dAAq_@XW-Y`f<_Y$Z_k^kn*5f^@EXl;cvPWlG9AycAimUJ$uEP17`1Rk7W>Eevn@h~(hlEr2NNo++S8qChXYg!)=cPy#j}_)4@L^w_Jt(B{V8ZStW;Klq^Kiqy&6@+)=yQb0dlM&CbnWjGYC)><Yn;tb#MC55_`Asc5|)J+S)&O$8=tVjh8w~7JDdkNOb=mXY?X@v_C2D8Cga5<xRLN{mWpz9^Wb~Zj3+<wYpbI_GdvE>I$G_$uo#kOZKhA&7mUio)iw4}U+89Y$T4KxOSyh`kMr8~qu@4E<l(@ZW_uud6g>(s792!py^wI$2L1*!JF(|Vn&);ribwh+>-Am1)TZh|w}&3;?Jf-b!v5Q_e#<#BIxlC1Jf-bUJZQ7=@3nH@Z}*pRw)8dg+`FzP!CNqrm<qzXrSkF$M*pdoo!}_w6t8l&I;iUE98V64l=os1<vkG(j&37@8A4?X{u$5XBpq&O$xWZ+f~u#lDuH1_Mg5Vr;C?c`<t;S!yV!IaRm9$<)g`}}z>nWmAq-bWmrQBGG-Ru=W~<fPGUC?-svYepb<~2!oqn&~HO?DxN?AH_)^ZZpS7wxq=I!-{r<aAUz1|qP0vSUd_2NvK$DB^gCZ5$7DzPfbkYR22#|$QaZOQ;rqF*h2&tJXn?nFtDEMeu^*2T(!eJ_`nszm$zmC9<RCKVzh+JB1kYdwCLTMxxHN~01#dtM4J?2XeJh}jp33hh=&ZWrisc({wEk9$-a9hgb9l`UiEWnLtI8Sd_C8WM=8`Cbl1ihBs}CtXv`+7>q#8Ebn4^2>3Ax}^6DH}<h%mXh>>W2TQmPH&1geEIAMk3=Li*%TXu(Yc$Y%~~1;v)%K~pwaRVLu|mn0a)NvEeQrAN!K(?RnJw3Eg)Rn)YEi7H)T^Xo!eBGBxPptJQh?UNv{ZGNtrIqm5$GUQl6evU8C*GjK+;Dl~*bR8p4^N1xZVj@px{Z+AtHT1ZA@lef;<2YxCHMw27IMb&Ce@jb54wIxY#s%_C2cJ52D)nj(_%%+G&}oHB|wk$0B`;~;@Eo}b2Cgo+N!w}cvC+2iK-371oS?g*Gdmso%3#5m(<x(cpm?(2j<%nu!!qR+|M=?!4qe+%FJ0=^qy5OpSH|6iH8bZKVVo5|WL6q1d7T?$FPu`~#n=;7lO{TS82b0L(-$%+#&;^rJ`vga3RyGV&^iS1+xp=VE^ee(DjGod5qG7Xtc`K*?<YU{9jR!Y}Zq^Y357FW~4Bp|jFlC{_C?R6-SpuQ`^ve=%0@mBOJ`6WNkN%dhyzqqq@1W4P8%7h4n!4_G5QX*oE1(#AQ3EpoBih$(fE1Qt}vz&vP7`24&q0$8*-6e#}BMI!zwBeh{_eL%aXx27Imgx~Ds0z(I!^bA;GZ6dNdPAq~+@N(6rXB$@v>-DLL@5e}u5mp+6_ozf{9OLUndVEAL+MXVO+|yabm{C5lm_A#jCX^C3so+^WhYvov1G<}NMA&1n)CP2_mWLMU*a1O?MZ*|rU2g|wZ#*Bqif`V_I)apPM?rFCn?7kXr<}5yg8Ht<1OmfI}xtGcty7hD!mZupYMlvO(Fn0h_yYivp$f@!H1-SP<skBNUR8XKo{2o1s+6RQnZ0TrN2qni0Tkot3&Fv$F%p}8mWbOgyVRJYC&y=LvyVI5!rnKL~4$>L-QkLXTu`Y!_TQ6Oz6R@VjK?Iep5<G_e9*@qIQunfdgivZlAxc<vRFJ4kjNkHWfU6;Z88<g22=d8sdrbLbDwL^LIifRNR&u1iJ0aif!4cclQqKd-RI&xWx-uo*&M>Ah*Bz0__Vu)r?s@DjmT^G<xE&NK2Fo?mr@*CTRvh<wo3s@q$^GU{A3&xXp9Z&>Xh*f=ny`FXr^W{eyP5H-M+}|LOGq0|?W(^#'

def materialize_own_notebook(require_causal=False):
    assert not require_causal,'CAUSAL_MODULE_EXCLUDED_FROM_V5'
    source=nbformat.reads(zlib.decompress(base64.b85decode(NB_V5_SOURCE_B85.encode())).decode('utf-8'),as_version=4)
    marker="NB_V5_SOURCE_B85 = '__ML03_V5_SELF_ARCHIVE__'"
    found=[];runtime_history=[]
    try:
        raw_history=get_ipython().history_manager.input_hist_raw
        runtime_history=[(i,str(src)) for i,src in enumerate(raw_history) if isinstance(src,str)]
    except Exception:runtime_history=[]
    history_by_key={}
    for pos,src in runtime_history:
        m=re.match(r'^\s*#\s*(0[1-9]|1[0-3]|R[1-4]|21)\s*·',src)
        if m:history_by_key[m.group(1)]=(pos,src)
    cell_runtime_trace=[]
    for cell in source.cells:
        if cell.cell_type!='code':continue
        if marker in cell.source:
            cell.source=cell.source.replace(marker,'NB_V5_SOURCE_B85 = '+repr(NB_V5_SOURCE_B85),1)
        compile(cell.source,'<ML03_V5_EXPORT>','exec')
        source_sha_cell=hashlib.sha256(cell.source.encode('utf-8')).hexdigest()
        found.append(source_sha_cell)
        m=re.match(r'^\s*#\s*(0[1-9]|1[0-3]|R[1-4]|21)\s*·',cell.source)
        key=m.group(1) if m else 'UNKNOWN'
        observation=history_by_key.get(key)
        executed=bool(observation and observation[1].rstrip('\n')==cell.source.rstrip('\n'))
        if executed:cell.execution_count=int(observation[0])
        else:cell.execution_count=None
        cell.outputs=[]
        cell_runtime_trace.append({'cell':key,'history_index':observation[0] if observation else None,
            'history_source_present':bool(observation),'source_matches_history':executed,
            'code_sha256':source_sha_cell})
    ensure(len(found)==len([c for c in source.cells if c.cell_type=='code']),
           'V5_SOURCE_MANIFEST_INCOMPLETE')
    cfg_cell=next(c.source for c in source.cells if c.cell_type=='code' and c.source.startswith('# 01'))
    cfg_nodes=[n for n in ast.parse(cfg_cell).body if isinstance(n,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='CFG' for t in n.targets)]
    cfg_match=len(cfg_nodes)==1 and ast.literal_eval(cfg_nodes[0].value)==CFG
    ensure(cfg_match,'V5_RUNTIME_CONFIG_NOT_EQUAL_EMBEDDED_SOURCE')
    source.metadata.pop('widgets',None)
    nbformat.validate(source)
    output=WORK/'ML03_V5_FINAL_FULL_RESEARCH_4_STAGES.ipynb'
    nbformat.write(source,output)
    if TOKEN and len(TOKEN)>12:
        ensure(TOKEN not in output.read_text(encoding='utf-8'),'GITHUB_TOKEN_LEAK_IN_EXPORT')
    global V5_EXECUTION_TRACE
    V5_EXECUTION_TRACE=pd.DataFrame(cell_runtime_trace)
    report={'version':CFG['VERSION'],'code_cells':len(found),'configuration_matches_runtime':cfg_match,
            'source_parity_verified':True,'source_cell_sha256':found,'notebook_sha256':digest(output),
            'history_matched_cells':int(V5_EXECUTION_TRACE.source_matches_history.sum()),
            'history_present_cells':int(V5_EXECUTION_TRACE.history_source_present.sum()),
            'source_mode':'COMPLETE_IMMUTABLE_EMBEDDED_V5_NOTEBOOK'}
    (WORK/'source_parity_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
    return output,report

IPYNB,SOURCE_PARITY_REPORT=materialize_own_notebook()
mark('SOURCE_PARITY_GATE','PASS','Self-contained V5 full source with nbformat validation and CFG checksum')
mark('PASS_NOTEBOOK','PASS',f'Complete V5 executable notebook: {IPYNB.stat().st_size} bytes')

def json_safe(v):
    if isinstance(v,dict):return {str(k):json_safe(a) for k,a in v.items()}
    if isinstance(v,(tuple,list,set)):return [json_safe(a) for a in v]
    if isinstance(v,np.integer):return int(v)
    if isinstance(v,np.bool_):return bool(v)
    if isinstance(v,np.floating):v=float(v)
    if isinstance(v,float) and not np.isfinite(v):return None
    if v is pd.NA or v is pd.NaT:return None
    if isinstance(v,(datetime.datetime,pd.Timestamp)):return v.isoformat()
    return v

critical=['CELL10_PIPELINE','STAGE1_PHYSICAL_AUDIT','PRIMARY_TARGET_D01_GATE','SQL_DUCKDB_UNIT_CONTRACT','DATA_SMOKE','REAL_STAGE1_SQL_PREFLIGHT','SOURCE_PARITY_GATE','DISCOVERY_EXHAUSTIVENESS','INDEPENDENT_METRIC_RECONCILIATION','PASS_INPUT','ALL_STAGE1_DOWNLOAD','PASS_DATA','PASS_SPLITS','PASS_TRAINING',
   'PASS_VALIDATION','PASS_INTERPRETATION','ADVANCED_ENSEMBLE','ADVANCED_SHAP_QA','ADVANCED_TEMPORAL_CANARY','PASS_OUTPUTS','PASS_NOTEBOOK','TEMPORAL_CANARY',
   'MIGRATION_TARGET_SEMANTICS','FEATURE_G1','FEATURE_G2','FEATURE_G3','FEATURE_G4',
    'PASS_UNIT_AUDIT','PASS_BOUNDARY_AUDIT','TWO_YEAR_CALENDAR_QA']
# Validate the whole ordered canonical design, not a few convenient comparisons.
required_canonical_pairs=['M0 → M1','M1 → M2','M2 → M3','M3 → M4']
if len(ABLA)>0 and {'study_id','confirmatory','comparison'}.issubset(ABLA.columns):
    completed_pairs=set(ABLA.loc[ABLA.study_id.eq('CANONICAL') &
                       ABLA.confirmatory.astype(bool),'comparison'].dropna().tolist())
else:completed_pairs=set()
canonical_confirmatory=set(required_canonical_pairs).issubset(completed_pairs)
_primary_canonical=CANONICAL_YEAR_QA.loc[CANONICAL_YEAR_QA.horizon.eq(1)&CANONICAL_YEAR_QA.protocol.eq('ROLLING_TIME')]
full_calendar=(len(_primary_canonical)==len(CFG['TARGETS']) and
    _primary_canonical.target.nunique()==len(CFG['TARGETS']) and
    _primary_canonical.status.eq('FEASIBLE_COMPLETE').all() and
    _primary_canonical.missing_test_years.fillna('UNKNOWN').eq('').all())
no_fatal_qa=not any(v.get('status') in ('FAIL','PUBLICATION_FAIL')
                    for k,v in STATUS.items() if k!='PASS_GITHUB')

SCIENTIFIC_EXPLICIT_BLOCKERS=[]
# G5: отдельный аудит пригодности, не требуем в первичной H1/H3 программе
# h=5: проверка полноты календаря, но не основной прогнозный горизонт
if int(PANEL.boundary_year_at_risk.sum())>0:
    SCIENTIFIC_EXPLICIT_BLOCKERS.append('BOUNDARY_HISTORY_UNVERIFIED')
if STATUS.get('G1_DOMAIN_COMPLETENESS',{}).get('status')!='PASS':
    SCIENTIFIC_EXPLICIT_BLOCKERS.append('G1_WAGES_EMPLOYMENT_INCOME_NOT_VALIDATED')
if STATUS.get('HISTORICAL_PUBLICATION_DATES',{}).get('status')!='PASS':
    SCIENTIFIC_EXPLICIT_BLOCKERS.append('PUBLICATION_VINTAGE_NOT_VERIFIED')
if STATUS.get('ADVANCED_VS_BEST_SINGLE',{}).get('status')!='PASS':
    SCIENTIFIC_EXPLICIT_BLOCKERS.append('ENSEMBLE_VS_SINGLE_NOT_COMPLETE')
scientific_ready=(CFG['SCIENTIFIC_RELEASE_ALLOWED'] and CFG['MODE']=='FULL' and CFG['VERSION'].endswith('-final') and all(STATUS.get(s,{}).get('status')=='PASS' for s in critical)
  and canonical_confirmatory and full_calendar and no_fatal_qa and not SCIENTIFIC_EXPLICIT_BLOCKERS and len(METRICS)>0 and len(PREDICTIONS)>0)
if not scientific_ready:
    mark('FULL_SCIENCE_GATE','PARTIAL',
       'V5: predictive scientific status conditioned on confirmed vintages and strict model QA, see blockers')
else:mark('FULL_SCIENCE_GATE','PASS','Full empirical hypothesis test, all required artefacts and scientific validation')

MANIFEST=WORK/'run_manifest.json'
def write_manifest(publication=None):
    pub=publication or {'status':'NOT_PUBLISHED'}
    payload={'experiment':'ML03_FACTORS','version':CFG['VERSION'],'mode':CFG['MODE'],
       'source_repo':CFG['REPO'],'source_commit_sha':source_sha,
       'source_handoff_sha256':hashlib.sha256(raw_index).hexdigest(),
       'total_assets_stage1':len(ASSETS),'verified_inputs':source_receipts,
       'all_stage1_downloaded':bool(all(a['asset_id'] in LOCAL_ASSETS for a in ASSETS)),
       'years':{'first':min(YEARS),'last':max(YEARS)},
       'n_municipalities':int(base.entity_id.nunique()),'n_panel_rows':len(PANEL),
       'n_supervised_candidates':len(TASK_PANEL),'n_predictions':len(PREDICTIONS),
       'n_model_evaluations':len(METRICS),'n_paired_contrasts':len(ABLA),
       'n_advanced_superlearner_evaluations':len(ADVANCED_ENSEMBLES),
       'n_advanced_shap_local_rows':len(ADVANCED_SHAP_LOCAL),
       'advanced_explanation_status':STATUS.get('ADVANCED_SHAP_QA'),
       'advanced_temporal_canary':ADVANCED_TEMPORAL_CANARY.to_dict('records'),
       'advanced_shap_interaction_rows':len(ADVANCED_SHAP_INTERACTIONS),
       'advanced_vs_dev_best_single':ADVANCED_SINGLE_DUEL.to_dict('records') if len(ADVANCED_SINGLE_DUEL) else [],
       'github_colab_secret_name':CFG['GITHUB_COLAB_SECRET'],
       'publication_date_audit':PUBLICATION_AVAILABILITY_QA.to_dict('records'),
       'organization_grain_exclusions':ORGANIZATION_GRAIN_EXCLUSIONS,
       'h5_exclusion_audit':H5_CALENDAR_GAPS.to_dict('records'),
       'v5_budget_balance_qc':V5_BUDGET_BALANCE_QA.to_dict('records'),
       'v5_road_qc':V5_ROADS_QA.to_dict('records'),
       'v5_semantic_fixes':V5_SEMANTIC_FIXES.to_dict('records'),
       'v5_hypothesis_decisions':V5_HYPOTHESIS_DECISIONS.to_dict('records'),
       'v5_sample_reconciliation':V5_FROZEN_PREVIOUS_COUNTS.to_dict('records'),
       'v403_exact_key_reconciliation':V5_ROW_REFERENCE_QA.to_dict('records'),
       'v403_key_exclusion_reasons_count':len(V5_ROW_EXCLUSION_REASONS),
       'v5_prediction_volume_audit':V5_PREDICTION_VOLUME_QA.to_dict('records'),
       'v5_seed_family_delta_count':len(V5_SEED_FAMILY_PARITY),
       'v5_execution_history_matched_cells':int(V5_EXECUTION_TRACE.source_matches_history.sum()),
       'v403_reference_archive_qa':V5_REFERENCE_DOWNLOAD_QA.to_dict('records'),
       'progress_widget_count':1 if _PROGRESS_BAR.widget is not None else 0,
       'excel_shap_local_exported_rows':sum(r['rows'] for r in EXCEL_SHEET_INDEX if r['source']=='SHAP_локальные'),
       'supplemental_bdmo_receipts':BDMO_EXTERNAL_RECEIPTS,
       'supplemental_bdmo_load_audit':BDMO_EXTERNAL_QA.to_dict('records'),
       'groups':{g:{'n_features':len(ff),'model_features':ff} for g,ff in GROUP_FEATURES.items()},
       'status':'FULL_SCIENTIFIC_PASS' if scientific_ready else 'DIAGNOSTIC_FULL_RUN_SCIENTIFIC_PARTIAL',
       'technical_status':'PASS' if all(STATUS.get(k,{}).get('status')=='PASS'
             for k in ('PASS_INPUT','PASS_DATA','PASS_TRAINING','PASS_OUTPUTS','PASS_NOTEBOOK')) else 'PARTIAL',
       'scientific_status':'PASS' if scientific_ready else 'PARTIAL',
       'scientific_blockers':[k for k in critical if STATUS.get(k,{}).get('status')!='PASS'] +
           ([] if canonical_confirmatory else ['CANONICAL_M0_M4_COMPLETE_SEQUENCE_NOT_CONFIRMED']) +
           ([] if full_calendar else ['PRIMARY_TWO_YEAR_CALENDAR_INCOMPLETE']) +
           ([] if no_fatal_qa else ['FATAL_RUNTIME_QA_PRESENT']) + SCIENTIFIC_EXPLICIT_BLOCKERS,
       'run_completion_status':RUN_COMPLETION_STATUS,
       'scientific_ready':scientific_ready,'causal_status':'EXCLUDED_FROM_V5_SCOPE',
       'research_scope':'OBSERVATIONAL_PREDICTIVE_H1_H3_ONLY',
       'status_details':STATUS,'task_plan':TASK_STATUSES.to_dict('records'),
       'selected_models':DEV_METRICS.to_dict('records'),
       'factor_cutoffs_by_horizon':FACTOR_CUTOFF_BY_H,
       'target_specific_test_windows':TARGET_TEST_WINDOWS,
       'stage1_physical_qa':STAGE1_PHYSICAL_AUDIT.to_dict('records'),
       'economy_manifest_audit':ECONOMY_SOURCE_MANIFEST_QA.to_dict('records'),
       'regional_context_not_municipal_g1':True,
       'regional_context_dictionary':REGIONAL_CONTEXT_META,
       'regional_context_audit':REGIONAL_CONTEXT_AUDIT.to_dict('records'),
       'regional_unit_qa':G1R_UNIT_QA.to_dict('records'),
       'feature_staleness_qc':STALENESS_QA.to_dict('records'),
       'discovery_exhaustive':DISCOVERY_EXHAUSTIVE,
       'scientific_open_limitations':[
          'G5_HISTORICAL_VINTAGE_UNVERIFIED' if not GROUP_FEATURES.get('G5') else '',
          'H5_EXCLUDED_BY_DESIGN_NO_MODEL_FIT',
          'HISTORIC_MUNICIPAL_BOUNDARY_CROSSWALK_REQUIRED' if int(PANEL.boundary_year_at_risk.sum())>0 else '',
          'G1_INCOME_LABOUR_COVERAGE_INSUFFICIENT' if STATUS.get('G1_DOMAIN_COMPLETENESS',{}).get('status')!='PASS' else '',
          'SOURCE_PUBLICATION_DATES_MISSING' if STATUS.get('HISTORICAL_PUBLICATION_DATES',{}).get('status')!='PASS' else '',
       ],
       'data_smoke':DATA_SMOKE_REPORT, 'sql_duckdb_unit_contract':SQL_DUCKDB_CONTRACT_REPORT, 'real_sql_preflight':REAL_SQL_PREFLIGHT, 'source_parity':SOURCE_PARITY_REPORT,
       'unadapted_municipal_wide_assets':UNADAPTED_MUNICIPAL_WIDE,
       'g1_empty_cause':G1_EMPTY_CAUSE if not GROUP_FEATURES['G1'] else None,
       'target_availability':TARGET_AVAILABILITY.to_dict('records'),
       'distinct_seeds':SEED_QA.to_dict('records'),
       'test_metrics':METRICS.to_dict('records'),
       'ablation_results':ABLA.to_dict('records') if len(ABLA) else [],
       'advanced_comparisons':ADVANCED_COMPARISONS.to_dict('records') if len(ADVANCED_COMPARISONS) else [],
       'advanced_model_selection':ADVANCED_ENSEMBLES.to_dict('records') if len(ADVANCED_ENSEMBLES) else [],
       'advanced_additivity_audit':ADVANCED_SHAP_QA.to_dict('records') if len(ADVANCED_SHAP_QA) else [],
       'outputs':{f.name:{'sha256':digest(f),'bytes':f.stat().st_size} for f in (IPYNB,XLSX,ZIP)},
       'publication':pub,'elapsed_seconds':int(time.monotonic()-START),
       'interpretation':'Municipal retrospective predictive associations only, NEVER causal estimates.'}
    MANIFEST.write_text(json.dumps(json_safe(payload),ensure_ascii=False,allow_nan=False,indent=2),encoding='utf-8')
    ensure(json.loads(MANIFEST.read_text(encoding='utf-8'))['version']==CFG['VERSION'],'MANIFEST_INVALID')
RUN_COMPLETION_STATUS='PREDICTIVE_COMPUTATIONS_COMPLETE_NO_CAUSAL_SCOPE'
write_manifest()

# GitHub writing is intentionally unreachable before FULL scientific acceptance.
def create_or_update_git_file(remote,local,branch,content_sha=None):
    request={'message':f"ML03 {CFG['VERSION']}: {Path(remote).name}",
      'content':base64.b64encode(Path(local).read_bytes()).decode(),'branch':branch}
    if content_sha:request['sha']=content_sha
    obj=api('PUT','/contents/'+urllib.parse.quote(remote,safe='/'),json=request,expected=(200,201)).json()
    return obj['commit']['sha'],obj['content']['sha']
def verify_github_file(remote,branch,path):
    obj=api('GET',f"/contents/{urllib.parse.quote(remote,safe='/')}?ref={urllib.parse.quote(branch,safe='')}").json()
    blob=base64.b64decode(obj['content']) if obj.get('content') else base64.b64decode(
        api('GET','/git/blobs/'+obj['sha']).json()['content'])
    ensure(hashlib.sha256(blob).hexdigest()==digest(path),'GITHUB_READBACK_SHA_MISMATCH: '+Path(remote).name)
def verify_release_asset(asset_id,expected):
    url=API+'/releases/assets/'+str(asset_id)
    r=SESSION.get(url,headers={'Accept':'application/octet-stream'},stream=True,timeout=(30,1200))
    ensure(r.status_code==200,'RELEASE_READBACK_NOT_200')
    if 'json' in r.headers.get('Content-Type','').lower():
        meta=r.json();r.close();u=meta.get('browser_download_url','')
        p=urllib.parse.urlparse(u)
        ensure(p.scheme=='https' and p.hostname in ('github.com','api.github.com'),'RELEASE_ASSET_URL_INVALID')
        r=SESSION.get(u,stream=True,timeout=(30,1200))
    h=hashlib.sha256()
    try:
        ensure(r.status_code==200 and 'json' not in r.headers.get('Content-Type','').lower(),'RELEASE_ASSET_DOWNLOAD_FAILED')
        for b in r.iter_content(4*1024*1024):
            if b:h.update(b)
    finally:r.close()
    ensure(h.hexdigest()==expected,'RELEASE_ASSET_READBACK_SHA_MISMATCH')

def publish_verified():
    ensure(scientific_ready and CFG['MODE']=='FULL' and CFG['PUBLISH_FULL_PASS'] is True,'FULL_QA_REQUIRED')
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    branch='ml03-results/'+CFG['VERSION'].replace('.','-')+'-'+stamp
    tag=CFG['VERSION']+'-'+stamp
    api('POST','/git/refs',json={'ref':'refs/heads/'+branch,'sha':source_sha},expected=(201,))
    npath=CFG['SEGMENT']+'/'+IPYNB.name;xpath=CFG['SEGMENT']+'/'+XLSX.name
    mpath=CFG['SEGMENT']+'/'+MANIFEST.name
    create_or_update_git_file(npath,IPYNB,branch)
    revision,_=create_or_update_git_file(xpath,XLSX,branch)
    # A draft release accepts assets before publishing a tag. Publish the tag only after
    # the manifest and release asset have been re-downloaded and verified.
    release=api('POST','/releases',json={'tag_name':tag,'target_commitish':branch,
      'name':'ML03 '+tag,'draft':True,'prerelease':False,
      'body':'Stage II ML03; scientific PASS; evidence SHA-256 '+digest(ZIP)},expected=(201,)).json()
    with open(ZIP,'rb') as f:
        upload=SESSION.post(f"https://uploads.github.com/repos/{CFG['REPO']}/releases/{release['id']}/assets",
         params={'name':ZIP.name},data=f,headers={'Content-Type':'application/zip','Accept':'application/vnd.github+json'},timeout=(30,1200))
    ensure(upload.status_code==201,'RELEASE_UPLOAD_HTTP_'+str(upload.status_code))
    asset=upload.json()
    verify_github_file(npath,branch,IPYNB);verify_github_file(xpath,branch,XLSX)
    verify_release_asset(asset['id'],digest(ZIP))
    publication={'status':'PASS_GITHUB','branch':branch,'tag':tag,
      'release_url':release['html_url'],'evidence_asset_id':asset['id'],
      'evidence_url':asset.get('browser_download_url',''),
      'notebook_url':f"https://github.com/{CFG['REPO']}/blob/{branch}/{npath}",
      'excel_url':f"https://github.com/{CFG['REPO']}/blob/{branch}/{xpath}",
      'initial_commit_sha':revision,'readback_sha_verified':True,
      'publication_phase':'draft_assets_verified_before_tag_publication'}
    write_manifest(publication)
    manifest_commit,_=create_or_update_git_file(mpath,MANIFEST,branch)
    verify_github_file(mpath,branch,MANIFEST)
    published=api('PATCH','/releases/'+str(release['id']),
       json={'draft':False,'target_commitish':branch},expected=(200,)).json()
    ensure(published.get('draft') is False,'RELEASE_REMAINS_DRAFT')
    ref=api('GET','/git/ref/tags/'+urllib.parse.quote(tag,safe='')).json()
    tagsha=ref['object']['sha']
    if ref['object']['type']=='tag':
        annotated=api('GET','/git/tags/'+tagsha).json();tagsha=annotated['object']['sha']
    ensure(tagsha==manifest_commit,'RELEASE_TAG_DOES_NOT_POINT_TO_VERIFIED_MANIFEST_COMMIT')
    verify_release_asset(asset['id'],digest(ZIP))
    mark('PASS_GITHUB','PASS','Release tag, git artifacts, manifest and evidence passed SHA readback')
    publication['manifest_commit_sha']=manifest_commit
    return publication

PUBLISHED={'status':'NOT_PUBLISHED'}
if CFG['MODE']=='FULL' and CFG['PUBLISH_FULL_PASS']:
    try:PUBLISHED=publish_verified()
    except Exception as ex:
        mark('PASS_GITHUB','PUBLICATION_FAIL',type(ex).__name__+': '+str(ex)[:130])
        PUBLISHED={'status':'PUBLICATION_FAIL','error_type':type(ex).__name__}
        write_manifest(PUBLISHED)
else:mark('PASS_GITHUB','NOT_PUBLISHED','No write without explicit switch and scientific FULL QA')
if STATUS.get('PASS_NOTEBOOK',{}).get('status')=='PASS' and MANIFEST.exists():
    RUN_COMPLETION_STATUS='PREDICTIVE_COMPUTATIONS_COMPLETE_NO_CAUSAL_SCOPE'
    progress('FINAL',1,1,'Все прогнозные модели и проверки завершены; причинный анализ исключён')
else:
    RUN_COMPLETION_STATUS='PARTIAL'
ensure(_PROGRESS_BAR.n==925,'CORE_STAGE_PROGRESS_INCOMPLETE')
# The unified progress remains active until causal diagnostics and BOTH archives are verified.
quiet_print('CORE_STAGE_COMPLETE',CFG['VERSION'], 'comparisons',len(ABLA),'evaluations',len(METRICS))
quiet_print('Local outputs:',*[str(x) for x in (IPYNB,XLSX,ZIP,MANIFEST)],sep='\n · ')
if PUBLISHED['status']=='PASS_GITHUB':quiet_print(PUBLISHED['release_url'])


#### Полная исследовательская упаковка, GitHub Evidence, проверка итогового ZIP.

In [21]:

# 21 · V5.00 · ИТОГОВЫЙ ИССЛЕДОВАТЕЛЬСКИЙ ZIP, ПОЛНЫЙ EVIDENCE И GITHUB QA
import datetime as _dt
import json as _json
import zipfile as _zf
import base64 as _b64
import zlib as _zlib
import nbformat as _nbf
import os as _os
import hashlib as _hashlib
import ast as _ast
import re as _re

def _v5_is_export_cell(cell):
    if cell.cell_type != 'code': return False
    source = cell.source
    return ('RESEARCH_DATA={' in source and "ML03_RESEARCH_SUMMARY" in source and "GITHUB_PUBLICATION_STATUS" in source and "FAST_ZIP" in source)

def _v5_decode_snapshot():
    return _nbf.reads(_zlib.decompress(_b64.b85decode(NB_V5_SOURCE_B85.encode('ascii'))).decode('utf-8'), as_version=4)

def _v5_refresh_cell21_source():
    global NB_V5_SOURCE_B85
    source = _v5_decode_snapshot()
    candidates = [c for c in source.cells if _v5_is_export_cell(c)]
    ensure(len(candidates) == 1, 'V5_EXPORT_SOURCE_NOT_UNIQUE: '+str(len(candidates)))
    target = candidates[0]
    history_code = None
    try:
        history = get_ipython().history_manager.input_hist_raw
        for candidate in reversed(history):
            if not isinstance(candidate,str) or not candidate.lstrip().startswith('# 21'): continue
            if 'RESEARCH_DATA={' not in candidate or 'FAST_ZIP' not in candidate: continue
            try:
                compile(candidate, '<V5_RUNNING_LAST_CELL>', 'exec')
                history_code = candidate
                break
            except SyntaxError: continue
    except (NameError, AttributeError): pass
    if history_code:
        target.source = history_code.rstrip('\n')+'\n'
    else:
        for imp in ('import datetime as _dt','import json as _json','import zipfile as _zf'):
            if not _re.search(r'(?m)^\s*'+_re.escape(imp)+r'\s*$',target.source): target.source = imp+'\n'+target.source
        fragile = ("ensure('import datetime as _dt' in next(c.source for c in "
                   "_nbf.read(UNIFIED_IPYNB,as_version=4).cells\n"
                   "       if c.cell_type=='code' and c.source.startswith('# 21 · V5.00')), "
                   "'CELL21_IMPORT_NOT_IN_EXPORTED_NOTEBOOK')")
        if fragile in target.source: target.source=target.source.replace(fragile,"# V5: проверка структуры без next().")
    compile(target.source, '<V5_LAST_CELL_REPAIRED>', 'exec')
    NB_V5_SOURCE_B85 = _b64.b85encode(_zlib.compress(_nbf.writes(source).encode('utf-8'), 9)).decode('ascii')
    return bool(history_code)

def _v5_export_ok(nb):
    final = [c for c in nb.cells if _v5_is_export_cell(c)]
    if len(final)!=1: return False
    return all(x in final[0].source for x in ('import datetime as _dt','import json as _json','import zipfile as _zf'))

V5_CELL21_SOURCE_CAPTURED = _v5_refresh_cell21_source()
UNIFIED_IPYNB, FINAL_PARITY = materialize_own_notebook()
UNIFIED_PARITY = WORK/'source_parity_report.json'
exported_nb = _nbf.read(UNIFIED_IPYNB, as_version=4)

if not _v5_export_ok(exported_nb):
    recovered_nb = _v5_decode_snapshot()
    source_marker = "NB_V5_SOURCE_B85 = '"+"__ML03_V5_"+"SELF_ARCHIVE__'"
    for cell in recovered_nb.cells:
        if cell.cell_type != 'code': continue
        if source_marker in cell.source: cell.source=cell.source.replace(source_marker,'NB_V5_SOURCE_B85 = '+repr(NB_V5_SOURCE_B85),1)
        compile(cell.source, '<V5_RECOVERED_COMPLETE_NOTEBOOK>', 'exec')
    ensure(len([c for c in recovered_nb.cells if c.cell_type=='code']) == 18,'V5_RECOVERY_EXPECTED_18_CODE_CELLS')
    ensure(_v5_export_ok(recovered_nb), 'V5_RECOVERY_FINAL_CELL_INVALID')
    config_source = next((c.source for c in recovered_nb.cells if c.cell_type=='code' and _re.match(r'^#\s*01\s*·',c.source)),None)
    ensure(config_source is not None,'V5_RECOVERY_CONFIG_CELL_MISSING')
    cfg_nodes = [n for n in _ast.parse(config_source).body if isinstance(n,_ast.Assign) and any(isinstance(t,_ast.Name) and t.id=='CFG' for t in n.targets)]
    ensure(len(cfg_nodes)==1 and _ast.literal_eval(cfg_nodes[0].value)==CFG,'V5_RECOVERY_CONFIG_DIFFERENT_FROM_RUNTIME')
    _nbf.validate(recovered_nb)
    fixed_path = UNIFIED_IPYNB.with_name(UNIFIED_IPYNB.stem+'.__building__.ipynb')
    _nbf.write(recovered_nb,fixed_path)
    ensure(_v5_export_ok(_nbf.read(fixed_path,as_version=4)),'V5_RECOVERY_WRITE_VERIFY_FAILED')
    if TOKEN and len(TOKEN)>12: ensure(TOKEN not in fixed_path.read_text(encoding='utf-8'),'GITHUB_TOKEN_LEAK_IN_V5_EXPORT')
    _os.replace(fixed_path,UNIFIED_IPYNB)
    exported_nb = _nbf.read(UNIFIED_IPYNB,as_version=4)
    FINAL_PARITY.update({
       'source_parity_verified':True,'code_cells':18,'configuration_matches_runtime':True,
       'source_cell_sha256':[_hashlib.sha256(c.source.encode('utf-8')).hexdigest() for c in exported_nb.cells if c.cell_type=='code'],
       'notebook_sha256':digest(UNIFIED_IPYNB),'source_mode':'RECOVERED_FROM_FULL_EMBEDDED_SNAPSHOT','recovered_after_incomplete_export':True})
    UNIFIED_PARITY.write_text(_json.dumps(FINAL_PARITY,ensure_ascii=False,indent=2),encoding='utf-8')
    print('V5: восстановлена полная программа из встроенного снимка; 18 ячеек, без повторного обучения.')

ensure(FINAL_PARITY.get('source_parity_verified') and _v5_export_ok(exported_nb),'V5_FINAL_EXPORTED_NOTEBOOK_INCOMPLETE')
ensure(len([c for c in exported_nb.cells if c.cell_type=='code'])==18,'V5_FINAL_EXPORTED_NOTEBOOK_CELL_COUNT')
write_manifest(PUBLISHED)
ensure(ZIP.is_file() and ZIP.stat().st_size>0,'TECHNICAL_EVIDENCE_NOT_CREATED')

RESEARCH_DATA={
 '01_test_metrics':METRICS,'02_confirmatory_exploratory_comparisons':ABLA,
 '03_model_selection_DEV':DEV_METRICS,'04_ensemble_test':ADVANCED_ENSEMBLES,
 '05_ensemble_vs_best_single':ADVANCED_SINGLE_DUEL,'06_ensemble_intervals':ADVANCED_INTERVALS,
 '07_factor_dictionary':FEATURE_DICT,'08_factor_support':MODEL_DOMAIN_COVERAGE,
 '09_global_SHAP':ADVANCED_SHAP,'10_SHAP_region_CI':ADVANCED_SHAP_CI,
 '11_SHAP_stability':ADVANCED_SHAP_STABILITY,'12_SHAP_interactions':ADVANCED_SHAP_INTERACTIONS,
 '13_SHAP_additivity_QA':ADVANCED_SHAP_QA,'14_grouped_permutations':ADVANCED_GROUP_PERM,
 '15_PDP':ADVANCED_PDP,'16_ICE':ADVANCED_ICE,'17_ALE':ALE,
 '18_publication_dates_QA':PUBLICATION_AVAILABILITY_QA,'19_h5_excluded_audit':H5_CALENDAR_GAPS,
 '20_task_plan':TASK_STATUSES,'21_municipality_boundaries':BOUNDARY_QA,
 '22_source_and_units':SEMANTIC_QA,'23_model_group_coverage':DOMAIN_COVERAGE,
 '24_future_canary':CANARY_QA,'25_ensemble_future_canary':ADVANCED_TEMPORAL_CANARY,
 '26_source_grain_audit':SOURCE_GRAIN_AUDIT,'27_real_SQL_QA':REAL_SQL_PREFLIGHT_QA,
 '28_extended_source_recovery':BDMO_EXTERNAL_RECOVERY,'29_correlation_sensitivity':SENSITIVITY,
 '30_size_heterogeneity':SIZE_SENSITIVITY,'31_hypotheses':HYPOTHESIS_REGISTRY,
 '32_regional_errors':REGIONAL_ERRORS,
 '33_research_status':pd.DataFrame([{'gate':k,**v} for k,v in STATUS.items()]),
 '34_shap_local_sample':ADVANCED_SHAP_LOCAL.head(int(CFG['RESEARCH_LOCAL_SHAP_SAMPLE'])),
 '35_v5_budget_balance_qa':V5_BUDGET_BALANCE_QA,'36_v5_roads_qa':V5_ROADS_QA,
 '37_v5_semantic_fixes':V5_SEMANTIC_FIXES,'38_v5_sample_window':V5_SAMPLE_AUDIT,
 '39_v5_previous_test_counts':V5_FROZEN_PREVIOUS_COUNTS,'40_v5_cross_version_key_qa':V5_ROW_REFERENCE_QA,
 '42_v5_old_key_exclusion_reasons':V5_ROW_EXCLUSION_REASONS,
 '43_v5_reference_download_audit':V5_REFERENCE_DOWNLOAD_QA,
 '44_v5_prediction_volume':V5_PREDICTION_VOLUME_QA,
 '45_v5_seed_family_parity':V5_SEED_FAMILY_PARITY,'46_v5_execution_trace':V5_EXECUTION_TRACE,
 '41_v5_hypothesis_decisions':V5_HYPOTHESIS_DECISIONS,
}

RS=WORK/'research_fast'; RS.mkdir(exist_ok=True)
V5_EXECUTION_TRACE.to_csv(RS/'V5_REAL_EXECUTION_HISTORY.csv',index=False,encoding='utf-8-sig')
SCIENTIFIC_CSV=[]; RESEARCH_INDEX=[]
for name,frame in RESEARCH_DATA.items():
    if frame is None or not isinstance(frame,pd.DataFrame): continue
    path=RS/(name+'.csv'); frame.to_csv(path,index=False,encoding='utf-8-sig'); SCIENTIFIC_CSV.append(path)
    RESEARCH_INDEX.append({'name':name,'rows_exported':len(frame),'rows_total':len(ADVANCED_SHAP_LOCAL) if name=='34_shap_local_sample' else len(frame),'cols':len(frame.columns),'sha256':digest(path)})
progress('ARCHIVE_RESEARCH',3,4,f'{len(SCIENTIFIC_CSV)} научных таблиц')

from openpyxl import Workbook as _Wb
from openpyxl.styles import Font as _Font,PatternFill as _Fill,Alignment as _Align
from openpyxl.cell import WriteOnlyCell as _Cell
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _illegal
from openpyxl import load_workbook as _load
import numpy as _np

lite=RS/'ML03_RESEARCH_SUMMARY.xlsx'
lite_tmp=RS/'ML03_RESEARCH_SUMMARY.__building__.xlsx'
w=_Wb(write_only=True)

def safe(v):
    if v is None or v is pd.NA or v is pd.NaT: return None
    if isinstance(v,_np.integer): return int(v)
    if isinstance(v,_np.bool_): return bool(v)
    if isinstance(v,_np.floating): v=float(v)
    if isinstance(v,float): return v if _np.isfinite(v) else None
    if isinstance(v,(dict,list,tuple,set)): v=_json.dumps(v,ensure_ascii=False,default=str)
    elif isinstance(v,(pd.Timestamp,_dt.datetime,_dt.date)): v=v.isoformat()
    else:
        try:
            if pd.isna(v): return None
        except (TypeError,ValueError): pass
    if isinstance(v,str):
        v=v[:32767]
        if v.startswith(('=','+','-','@')): v="'"+v[:32766]
        return _illegal.sub('',v)
    return str(v)[:32767] if not isinstance(v,(int,bool)) else v

for name,frame in RESEARCH_DATA.items():
    if frame is None or not isinstance(frame,pd.DataFrame): continue
    ws=w.create_sheet(name[:31]); hdr=[]
    for col in frame.columns:
        cell=_Cell(ws,value=str(col)); cell.font=_Font(name='Aptos',size=10,color='FFFFFF',bold=True)
        cell.fill=_Fill('solid',fgColor='17324D'); cell.alignment=_Align(wrap_text=True); hdr.append(cell)
    ws.append(hdr)
    for row in frame.itertuples(index=False,name=None): ws.append([safe(v) for v in row])

w.save(lite_tmp)
verify=_load(lite_tmp,read_only=True,data_only=True)
try:
    ensure(len(verify.sheetnames)==len([f for f in RESEARCH_DATA.values() if isinstance(f,pd.DataFrame)]),'LITE_XLSX_SHEET_COUNT')
    for item in RESEARCH_INDEX:
        r=verify[item['name'][:31]].iter_rows(min_row=1,max_row=1,values_only=True)
        ensure(next(r,None) is not None,'LITE_XLSX_HEADER_MISSING_'+item['name'])
finally: verify.close()
_os.replace(lite_tmp,lite)

REMOTE_PUBLISH={
 'status':'NOT_STARTED','repository':CFG['REPO'],
 'scientific_status':'PASS' if scientific_ready else 'PARTIAL',
 'technical_archive_name':ZIP.name,'technical_archive_bytes':ZIP.stat().st_size,
 'technical_archive_sha256':digest(ZIP),'verification':'not_started'
}

def _publish_heavy_evidence():
    if not CFG.get('GITHUB_TECHNICAL_PUBLISH',True): return {**REMOTE_PUBLISH,'status':'DISABLED'}
    ensure(CFG['MODE']=='FULL','REMOTE_PUBLICATION_REQUIRES_FULL_MODE')
    ensure(bool(TOKEN),'GITHUB_TOKEN_NOT_AVAILABLE')
    head=api('GET','/git/ref/heads/main').json()['object']['sha']
    stamp=_dt.datetime.now(_dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    branch='ml03-results/v500-'+stamp; tag='ml03-v500-results-'+stamp
    segment=CFG['SEGMENT'].strip('/')+'/runs/'+stamp
    progress('ARCHIVE_TECH',1,5,'GitHub: создание изолированной ветки')
    api('POST','/git/refs',json={'ref':'refs/heads/'+branch,'sha':head},expected=(201,))
    create_or_update_git_file(segment+'/'+UNIFIED_IPYNB.name,UNIFIED_IPYNB,branch)
    create_or_update_git_file(segment+'/run_manifest.json',MANIFEST,branch)
    create_or_update_git_file(segment+'/source_parity.json',UNIFIED_PARITY,branch)
    progress('ARCHIVE_TECH',2,5,'GitHub: ноутбук, методика и реестр')
    body=('ML-03 V5.00 — reproducible computed technical evidence. Scientific status: '+
          ('PASS' if scientific_ready else 'PARTIAL')+
          '. Parquet/models are research evidence, NOT approved scientific claims. SHA256: '+digest(ZIP))
    release=api('POST','/releases',json={
      'tag_name':tag,'target_commitish':branch,'name':'ML-03 V5.00 · '+stamp,
      'body':body,'draft':True,'prerelease':not scientific_ready},expected=(201,)).json()
    with open(ZIP,'rb') as stream:
        response=SESSION.post('https://uploads.github.com/repos/'+CFG['REPO']+'/releases/'+str(release['id'])+'/assets',
          params={'name':'ML03_V5_TECHNICAL_EVIDENCE_'+stamp+'.zip'},data=stream,
          headers={'Content-Type':'application/zip','Accept':'application/vnd.github+json'},timeout=(30,1800))
    ensure(response.status_code==201,'GITHUB_TECH_ASSET_UPLOAD_FAILED_'+str(response.status_code))
    asset=response.json()
    ensure(asset.get('state') in ('uploaded',None),'GITHUB_ASSET_NOT_UPLOADED')
    ensure(int(asset.get('size',-1))==ZIP.stat().st_size,'GITHUB_ASSET_BYTES_MISMATCH')
    progress('ARCHIVE_TECH',3,5,'GitHub: технический архив загружен')
    meta=api('GET','/releases/assets/'+str(asset['id'])).json()
    ensure(int(meta.get('size',-1))==ZIP.stat().st_size,'GITHUB_ASSET_READBACK_SIZE_MISMATCH')
    digest_remote=str(meta.get('digest') or asset.get('digest') or '')
    if digest_remote.startswith('sha256:'):
        ensure(digest_remote.partition(':')[2].lower()==digest(ZIP),'GITHUB_ASSET_SHA_READBACK_MISMATCH')
        strength='SHA256_FROM_GITHUB'
    else:
        verify_release_asset(asset['id'],digest(ZIP))
        strength='SHA256_INDEPENDENT_RELEASE_READBACK'
    index={**REMOTE_PUBLISH,'status':'ASSET_VERIFIED','branch':branch,'tag':tag,
           'release_url':release['html_url'],'asset_id':asset['id'],
           'asset_url':asset.get('browser_download_url'),'verification':strength,
           'scientific_ready':bool(scientific_ready),'source_sha':source_sha}
    stampfile=RS/'github_evidence_index.json'
    stampfile.write_text(_json.dumps(index,ensure_ascii=False,indent=2),encoding='utf-8')
    create_or_update_git_file(segment+'/github_evidence_index.json',stampfile,branch)
    verify_github_file(segment+'/github_evidence_index.json',branch,stampfile)
    progress('ARCHIVE_TECH',4,5,'GitHub: реестр и удалённая проверка')
    published=api('PATCH','/releases/'+str(release['id']),json={'draft':False},expected=(200,)).json()
    ensure(published.get('draft') is False,'GITHUB_RELEASE_STILL_DRAFT')
    index['status']='PUBLISHED'; index['release_url']=published['html_url']
    index['git_segment_url']='https://github.com/'+CFG['REPO']+'/tree/'+branch+'/'+segment
    progress('ARCHIVE_TECH',5,5,'GitHub: публикация завершена')
    return index

try:
    REMOTE_PUBLISH=_publish_heavy_evidence()
    mark('GITHUB_TECH_STORAGE','PASS' if REMOTE_PUBLISH['status']=='PUBLISHED' else 'NOT_PUBLISHED',
         REMOTE_PUBLISH.get('release_url','Technical publication not enabled'))
except Exception as ex:
    REMOTE_PUBLISH={**REMOTE_PUBLISH,'status':'UPLOAD_FAILED','error':type(ex).__name__+': '+str(ex)[:350],
                    'recovery':'Technical source files remain in Colab WORK; no scientific PASS is implied'}
    mark('GITHUB_TECH_STORAGE','UPLOAD_FAILED',REMOTE_PUBLISH['error'])
    progress('ARCHIVE_TECH',1,1,'GitHub: ошибка, технические файлы остались в Colab')

remote_record=RS/'GITHUB_PUBLICATION_STATUS.json'
remote_record.write_text(_json.dumps(REMOTE_PUBLISH,ensure_ascii=False,indent=2,default=str),encoding='utf-8')

science_data={
 'version':CFG['VERSION'],'source_repo':CFG['REPO'],'source_commit_sha':source_sha,
 'scientific_ready':bool(scientific_ready),'full_science_gate':STATUS.get('FULL_SCIENCE_GATE',{}),
 'municipalities':int(base.entity_id.nunique()),'municipality_years':len(PANEL),
 'prediction_rows_in_remote_evidence':len(PREDICTIONS),'model_metric_rows':len(METRICS),
 'paired_comparisons':len(ABLA),'ensemble_models':len(ADVANCED_ENSEMBLES),
 'shap_global_rows':len(ADVANCED_SHAP),'shap_local_total':len(ADVANCED_SHAP_LOCAL),
 'shap_local_in_download':min(len(ADVANCED_SHAP_LOCAL),int(CFG['RESEARCH_LOCAL_SHAP_SAMPLE'])),
 'publication':REMOTE_PUBLISH,'table_index':RESEARCH_INDEX,
 'causal_inference_status':'EXCLUDED_FROM_V5_RESEARCH_SCOPE',
 'v403_reference_reconciliation':V5_ROW_REFERENCE_QA.to_dict('records'),
 'notes':'Predictive associations only; no causal effects. Full test predictions and local SHAP in technical Evidence (GitHub only if publication succeeded).'
}

science_manifest=RS/'SCIENCE_MANIFEST.json'
science_manifest.write_text(_json.dumps(science_data,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
readme=RS/'README_НАУЧНЫЕ_РЕЗУЛЬТАТЫ.md'
readme.write_text(
 '# ML-03 V5.00 — материалы для научного анализа\n\n'
 'Содержит компактную книгу Excel, полные итоговые статистические таблицы CSV, объяснения SHAP (агрегаты), '
 'визуализации и независимые QA протоколы.\n\n'
 'Полные наблюдения, прогнозы, локальные SHAP и сохранённые модели **не входят** в скачиваемый архив; '
 'они находятся в техническом Evidence GitHub Release (адрес в `GITHUB_PUBLICATION_STATUS.json`).\n'
 'Если статус `UPLOAD_FAILED`, публикация не прошла: технические данные остались в Colab, их нельзя считать загруженными в GitHub.\n'
 'Научный PASS нельзя выводить из успешной упаковки.\n',encoding='utf-8')

FAST_ZIP=WORK/'ML03_V5_RESEARCH_FAST.zip'
FAST_ZIP_TMP=WORK/'ML03_V5_RESEARCH_FAST.__building__.zip'
figs=sorted(p for p in (WORK/'charts').glob('*.png') if p.is_file())
excluded_dir=WORK/'excluded_design'; excluded_dir.mkdir(parents=True,exist_ok=True)
H5_CALENDAR_GAPS.to_csv(excluded_dir/'H5_excluded_no_training.csv',index=False,encoding='utf-8-sig')
excluded_design=sorted(excluded_dir.glob('*.csv'))

with _zf.ZipFile(FAST_ZIP_TMP,'w',compression=_zf.ZIP_DEFLATED,compresslevel=6,allowZip64=True) as z:
    z.write(lite,'tables/'+lite.name)
    for p in SCIENTIFIC_CSV: z.write(p,'tables/csv/'+p.name)
    for p in figs: z.write(p,'charts/'+p.name)
    for p in excluded_design: z.write(p,'excluded_design/'+p.name)
    z.write(RS/'V5_REAL_EXECUTION_HISTORY.csv','execution/V5_REAL_EXECUTION_HISTORY.csv')
    z.write(UNIFIED_IPYNB,'notebook/'+UNIFIED_IPYNB.name)
    for qc in sorted((WORK/'checkpoints').glob('*.json')) if (WORK/'checkpoints').exists() else []: z.write(qc,'checkpoints/'+qc.name)
    for p in (remote_record,science_manifest,readme,UNIFIED_PARITY): z.write(p,p.name)

ensure(FAST_ZIP_TMP.stat().st_size<CFG['RESEARCH_ARCHIVE_MAX_MB']*1024*1024,
       'SCIENTIFIC_ZIP_EXCEEDS_FAST_LIMIT: '+str(FAST_ZIP_TMP.stat().st_size))

with _zf.ZipFile(FAST_ZIP_TMP) as z:
    required={'tables/'+lite.name,'SCIENCE_MANIFEST.json','GITHUB_PUBLICATION_STATUS.json',readme.name}
    ensure(required.issubset(z.namelist()),'SCIENCE_ZIP_REQUIRED_FILES_MISSING')
    ensure(z.testzip() is None,'SCIENCE_ZIP_CRC_FAIL')
    ensure(len([x for x in z.namelist() if x.startswith('charts/') and x.endswith('.png')])==len(figs),
           'SCIENCE_ZIP_CHARTS_INCOMPLETE')

_os.replace(FAST_ZIP_TMP,FAST_ZIP)
report={'version':CFG['VERSION'],'fast_archive':FAST_ZIP.name,'bytes':FAST_ZIP.stat().st_size,
        'sha256':digest(FAST_ZIP),'tables':len(SCIENTIFIC_CSV),'charts':len(figs),
        'technical_remote':REMOTE_PUBLISH,'scientific_ready':bool(scientific_ready)}
(WORK/'ML03_DELIVERY_REPORT.json').write_text(_json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
progress('DELIVERY',1,1,'Один научный архив проверен, технический сохранён удалённо или статус ошибки зафиксирован')
ensure(_PROGRESS_BAR.n==1000,'UNIFIED_PROGRESS_NOT_COMPLETE')
_PROGRESS_BAR.close()

print('ML-03 V5.00 | Научный архив:',FAST_ZIP.name,
      '| размер, МиБ:',round(FAST_ZIP.stat().st_size/2**20,2),
      '| GitHub технический:',REMOTE_PUBLISH['status'])
if REMOTE_PUBLISH.get('release_url'): print('GitHub:',REMOTE_PUBLISH['release_url'])
print('Полный технический Evidence сохранён локально:',ZIP,'| размер, МиБ:',round(ZIP.stat().st_size/2**20,2))

if CFG.get('AUTO_COLAB_DOWNLOAD') and CFG['MODE']=='FULL':
    try:
        from google.colab import files as _files
        _files.download(str(FAST_ZIP))
        if REMOTE_PUBLISH.get('status') != 'PUBLISHED':
            print('GitHub не подтвердил публикацию. Сохраняю дополнительно полный технический Evidence.')
            _files.download(str(ZIP))
    except ImportError:
        quiet_print('Not a Google Colab runtime; archive remains at',FAST_ZIP)


---
**Приёмка V5.** Манифест отделяет факт вычисления от научной готовности; достоверные отрицательные результаты сохраняются. Для невыполнимых направлений не генерируются синтетические данные и оценки. Программная проверка при создании файла не заменяет полный Colab-прогон на приватных данных.